In [ ]:
# ==== CONFIG ==============================
import warnings
warnings.filterwarnings("ignore")

import os
import math
import numpy as np
import pandas as pd
from pathlib import Path

from scipy import signal
from mne.time_frequency import psd_array_multitaper

# Optional metrics (install if you have these packages)
try:
    from entropy import sample_entropy, permutation_entropy
except Exception:
    sample_entropy = None
    permutation_entropy = None

try:
    from lempel_ziv_complexity import lempel_ziv_complexity as lzc
except Exception:
    lzc = None

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import seaborn as sns

from mne.time_frequency import psd_array_welch

In [ ]:
# Sampling rate and channel definitions
FS = 128  # Hz (change if your CSV has a different rate)
ELECTRODES = ['AF3','AF4','F7','F8','F3','F4','FC5','FC6','P7','P8','T7','T8','O1','O2']
BRAINWAVES = ['Delta','Theta','Alpha','BetaL','BetaH','Gamma']
RANGES  = {'Delta':[1,4],'Theta':[4,8],'Alpha':[8,12],'BetaL':[12,16], 'BetaH':[16,25],'Gamma':[25,45]}

# Helper to make full column names used in your CSV (e.g., "EEG.AF3")
col = lambda e: f"EEG.{e}"

In [ ]:
%load_ext autoreload
%autoreload 2
import os ; import sys
sys.path.insert(0, os.path.abspath(os.path.join('./lib')))

import utilities

FILENAME = "data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv"
FS = 128

RECORDS = utilities.load_eeg_csv(FILENAME, electrodes=ELECTRODES)
RECORDS

In [ ]:
# Cross‑Frequency & Cross‑Region Coupling — Simple Graphs & Validation
# Pairs built ONLY from this 14‑channel set (no extras):
ELECTRODES = ['AF3','AF4','F7','F8','F3','F4','FC5','FC6','P7','P8','T7','T8','O1','O2']

# Region partitions
FRONTAL = ['AF3','AF4','F7','F8','F3','F4','FC5','FC6']
POSTERIOR = ['P7','P8','T7','T8','O1','O2']  # parietal, temporal, occipital

LEFT_FR = ['AF3','F7','F3','FC5']
RIGHT_FR = ['AF4','F8','F4','FC6']
LEFT_POST = ['P7','T7','O1']
RIGHT_POST = ['P8','T8','O2']

# -------------------------
# PAC (directed): phase@FRONTAL  → amplitude@POSTERIOR
# -------------------------
# All front→back (48 pairs)
PAIRS_PAC_ALL = [
    ('AF3','P7'), ('AF3','P8'), ('AF3','T7'), ('AF3','T8'), ('AF3','O1'), ('AF3','O2'),
    ('AF4','P7'), ('AF4','P8'), ('AF4','T7'), ('AF4','T8'), ('AF4','O1'), ('AF4','O2'),
    ('F7','P7'), ('F7','P8'), ('F7','T7'), ('F7','T8'), ('F7','O1'), ('F7','O2'),
    ('F8','P7'), ('F8','P8'), ('F8','T7'), ('F8','T8'), ('F8','O1'), ('F8','O2'),
    ('F3','P7'), ('F3','P8'), ('F3','T7'), ('F3','T8'), ('F3','O1'), ('F3','O2'),
    ('F4','P7'), ('F4','P8'), ('F4','T7'), ('F4','T8'), ('F4','O1'), ('F4','O2'),
    ('FC5','P7'), ('FC5','P8'), ('FC5','T7'), ('FC5','T8'), ('FC5','O1'), ('FC5','O2'),
    ('FC6','P7'), ('FC6','P8'), ('FC6','T7'), ('FC6','T8'), ('FC6','O1'), ('FC6','O2')
]

# Ipsilateral front→back (24 pairs)
PAIRS_PAC_IPSI = [
    # left→left
    ('AF3','P7'), ('AF3','T7'), ('AF3','O1'),
    ('F7','P7'),  ('F7','T7'),  ('F7','O1'),
    ('F3','P7'),  ('F3','T7'),  ('F3','O1'),
    ('FC5','P7'), ('FC5','T7'), ('FC5','O1'),
    # right→right
    ('AF4','P8'), ('AF4','T8'), ('AF4','O2'),
    ('F8','P8'),  ('F8','T8'),  ('F8','O2'),
    ('F4','P8'),  ('F4','T8'),  ('F4','O2'),
    ('FC6','P8'), ('FC6','T8'), ('FC6','O2'),
]

# Contralateral front→back (24 pairs)
PAIRS_PAC_CONTRA = [
    # left frontal → right posterior
    ('AF3','P8'), ('AF3','T8'), ('AF3','O2'),
    ('F7','P8'),  ('F7','T8'),  ('F7','O2'),
    ('F3','P8'),  ('F3','T8'),  ('F3','O2'),
    ('FC5','P8'), ('FC5','T8'), ('FC5','O2'),
    # right frontal → left posterior
    ('AF4','P7'), ('AF4','T7'), ('AF4','O1'),
    ('F8','P7'),  ('F8','T7'),  ('F8','O1'),
    ('F4','P7'),  ('F4','T7'),  ('F4','O1'),
    ('FC6','P7'), ('FC6','T7'), ('FC6','O1'),
]

# EEG.-prefixed versions (if your dataframe uses EEG.<name> columns)
PAIRS_PAC_ALL_EEG    = [('EEG.'+a, 'EEG.'+b) for (a,b) in PAIRS_PAC_ALL]
PAIRS_PAC_IPSI_EEG   = [('EEG.'+a, 'EEG.'+b) for (a,b) in PAIRS_PAC_IPSI]
PAIRS_PAC_CONTRA_EEG = [('EEG.'+a, 'EEG.'+b) for (a,b) in PAIRS_PAC_CONTRA]

# -------------------------
# n:m PLV (phase↔phase): undirected by nature; list each combo once
# -------------------------
# Use the same front↔back pairs but undirected (order doesn’t matter for PLV)
PAIRS_PLV_UNDIRECTED = sorted({tuple(sorted(p)) for p in PAIRS_PAC_ALL})  # 48 unique combos
PAIRS_PLV_UNDIRECTED_EEG = [('EEG.'+a, 'EEG.'+b) for (a,b) in PAIRS_PLV_UNDIRECTED]

# -------------------------
# Suggested minimal sets (quick runs)
# -------------------------
# Strong, interpretable anatomical paths
PAIRS_MIN_IPSI = [
    ('F3','P7'), ('F4','P8'),  # fronto-parietal ipsi
    ('F3','O1'), ('F4','O2'),  # fronto-occipital ipsi
    ('F7','T7'), ('F8','T8'),  # fronto-temporal ipsi
]
PAIRS_MIN_CONTRA = [
    ('F3','P8'), ('F4','P7'),  # fronto-parietal contra
    ('F3','O2'), ('F4','O1'),  # fronto-occipital contra
    ('F7','T8'), ('F8','T7'),  # fronto-temporal contra
]
PAIRS_MIN_ALL = PAIRS_MIN_IPSI + PAIRS_MIN_CONTRA
PAIRS_MIN_ALL_EEG = [('EEG.'+a, 'EEG.'+b) for (a,b) in PAIRS_MIN_ALL]

# Example usage:
# res = run_cfc_cross_region(RECORDS,
#     pairs=PAIRS_PAC_ALL_EEG,  # or PAIRS_MIN_ALL_EEG for quick run
#     ignition_windows=[(290,310),(580,600)],
#     baseline_windows=[(0,290),(325,580)],
#     time_col='Timestamp', out_dir='exports_cfc/S01', show=False,
#     n_perm=200, limit_high_hz=60.0)


In [ ]:
# Define bands or use default
bands = {'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}

result = utilities.run_event_detection_pipeline(
    df=RECORDS,
    electrodes=ELECTRODES,
    fs=FS,
    bands=bands,
    baseline_slice=(100, int(30*FS)),  # first 60s as baseline (or use slice(None) etc.)
    z_thresh=1.0,
    min_bands=1,
    min_duration_s=0.5,
    use_existing_cols=True  # True to use your precomputed EEG.<E>.<Band> if present
)

events = result['events']
events

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional

# This pipeline expects that you have already executed the other canvas modules:
# - Graph Eeg By Band (graph_eeg_timeline, plot_eeg_timeline_grid)
# - Eeg Cfc Records Compatible (run_crossfreq_suite_records, infer_fs_from_records)
# - Criticality_Suite_Records (run_criticality_suite_records) [optional]

# We re-declare a light-weight infer_fs for convenience if not already in scope
_DEF_TIME_COL = 'Timestamp'

def _infer_fs_from_records(RECORDS: pd.DataFrame, time_col: str = _DEF_TIME_COL) -> float:
    t = np.asarray(RECORDS[time_col].values, dtype=float)
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        raise ValueError("Cannot infer fs: Timestamp spacing invalid.")
    return float(1.0 / np.median(dt))


def run_full_eeg_pipeline_records(
    RECORDS: pd.DataFrame,
    electrodes: List[str],
    ranges: Dict[str, Tuple[float, float]],
    # timeline window (seconds)
    timeline_start: Optional[float] = None,
    timeline_end: Optional[float] = None,
    # cross-frequency windows (seconds)
    cfc_windows_ignition: Optional[List[Tuple[float, float]]] = None,
    cfc_windows_rebound: Optional[List[Tuple[float, float]]] = None,
    pac_phase_bands: List[Tuple[float,float]] = [(4,8),(8,13)],
    pac_amp_bands: List[Tuple[float,float]] = [(13,30),(30,80)],
    pac_method: str = 'mi',
    pac_surrogates: int = 0,
    # criticality options
    run_criticality: bool = False,
    criticality_channel: Optional[str] = None,
    metastab_electrodes: Optional[List[str]] = None,
    avalanche_band: Tuple[float,float] = (1,40),
    time_col: str = _DEF_TIME_COL,
    # >>> NEW: autofill options
    auto_autofill_bandpower: bool = True,
    autofill_smooth_sec: float = 0.25
) -> Dict[str, object]:

    """
    Master pipeline: (A) timeline plots, (B) cross-frequency analysis (PAC/bicoherence/shape),
    and optionally (C) criticality structure — all directly from RECORDS.
    
    Returns a dict with keys 'timeline', 'cfc', and optionally 'criticality'.
    """
    report: Dict[str, object] = {}

    # --- Infer fs
    try:
        if 'infer_fs_from_records' in globals():
            fs = infer_fs_from_records(RECORDS, time_col=time_col)  # from Eeg Cfc Records Compatible
        else:
            fs = _infer_fs_from_records(RECORDS, time_col=time_col)
    except Exception as e:
        print(f"[error] fs inference failed: {e}")
        fs = None
    report['fs'] = fs

    # --- (A) Timeline plots (gracefully skip if functions are not in scope)
    # --- (A0) Optional band-power autofill so timeline grids have data
    if auto_autofill_bandpower:
        if 'ensure_band_power_columns' in globals():
            try:
                if fs is not None:
                    ensure_band_power_columns(
                        RECORDS, electrodes=electrodes, ranges=ranges,
                        time_col=time_col, smooth_sec=autofill_smooth_sec, fs=fs
                    )
                else:
                    ensure_band_power_columns(
                        RECORDS, electrodes=electrodes, ranges=ranges,
                        time_col=time_col, smooth_sec=autofill_smooth_sec
                    )
            except Exception as e:
                print(f"[warn] band-power autofill failed: {e}")
        else:
            print('[info] ensure_band_power_columns not found; skipping autofill')

    timeline_out = {}
    try:
        if 'graph_eeg_timeline' in globals():
            graph_eeg_timeline(
                df=RECORDS,
                electrodes=electrodes,
                ranges=ranges,
                time_col=time_col,
                start_time=timeline_start,
                end_time=timeline_end,
            )
            timeline_out['stacked'] = True
        else:
            print('[warn] graph_eeg_timeline not found; skipping stacked timeline plot')
        if 'plot_eeg_timeline_grid' in globals():
            plot_eeg_timeline_grid(
                df=RECORDS,
                electrodes=electrodes,
                ranges=ranges,
                time_col=time_col,
                start_time=timeline_start,
                end_time=timeline_end,
            )
            timeline_out['grid'] = True
        else:
            print('[warn] plot_eeg_timeline_grid not found; skipping grid timeline plot')
    except Exception as e:
        print(f"[warn] timeline plotting error: {e}")
    report['timeline'] = timeline_out

    # --- (B) Cross-frequency analysis (PAC / bicoherence / waveform shape)
    cfc_out = {}
    if cfc_windows_ignition:
        if 'run_crossfreq_suite_records' in globals():
            try:
                cfc_results = run_crossfreq_suite_records(
                    RECORDS,
                    ignition_windows=cfc_windows_ignition,
                    rebound_windows=cfc_windows_rebound,
                    sensor_phase_ch=electrodes[0] if electrodes else 'F4',
                    sensor_amp_chs=tuple(electrodes) if electrodes else ('O1','O2','P7','P8','T7','T8'),
                    phase_bands=pac_phase_bands,
                    amp_bands=pac_amp_bands,
                    method=pac_method,
                    n_sur=pac_surrogates,
                    time_col=time_col
                )
                cfc_out.update(cfc_results)
            except Exception as e:
                print(f"[warn] cross-frequency suite error: {e}")
        else:
            print('[warn] run_crossfreq_suite_records not found; skipping CFC stage')
    else:
        print('[info] No ignition windows provided; skipping CFC')
    report['cfc'] = cfc_out

    # --- (C) Criticality (optional)
    crit_out = {}
    if run_criticality:
        if 'run_criticality_suite_records' in globals():
            try:
                ch = criticality_channel or (electrodes[0] if electrodes else 'F4')
                crit_out = run_criticality_suite_records(
                    RECORDS,
                    windows=cfc_windows_ignition or [],
                    ch=ch,
                    metastab_electrodes=metastab_electrodes or electrodes or [ch],
                    avalanche_band=avalanche_band,
                    dfa_inside_only=True,
                    time_col=time_col
                )
            except Exception as e:
                print(f"[warn] criticality suite error: {e}")
        else:
            print('[warn] run_criticality_suite_records not found; skipping criticality stage')
    report['criticality'] = crit_out if run_criticality else None

    # Quick summary printout
    if cfc_out:
        print(cfc_out.get('verdict_notes', ''))
    if run_criticality and crit_out:
        print('Criticality summary:', crit_out.get('summary', {}))

    return report

# ---------------- Example (commented) ----------------
# electrodes = ['F4','O1','O2']
# ranges = {"theta":(4,8), "alpha":(8,12), "beta":(13,30)}
# windows_ign = [(12,17),(33,38)]
# out = run_full_eeg_pipeline_records(
#     RECORDS,
#     electrodes=electrodes,
#     ranges=ranges,
#     timeline_start=0,
#     timeline_end=60,
#     cfc_windows_ignition=windows_ign,
#     cfc_windows_rebound=None,
#     pac_phase_bands=[(4,8),(8,13)],
#     pac_amp_bands=[(13,30),(30,80)],
#     pac_method='mi',
#     pac_surrogates=0,
#     run_criticality=True,
#     criticality_channel='F4',
#     metastab_electrodes=['F4','O1','O2']
# )
# print(out.keys())


In [ ]:
out = run_full_eeg_pipeline_records(
    RECORDS,
    electrodes=ELECTRODES,
    ranges=RANGES,
    timeline_start=0,
    timeline_end=60,
    cfc_windows_ignition=[180,200],
    cfc_windows_rebound=None,
    pac_phase_bands=[(4,8),(8,13)],
    pac_amp_bands=[(13,30),(30,80)],
    pac_method='mi',
    pac_surrogates=0,
    run_criticality=True,
    criticality_channel='F4',
    metastab_electrodes=['F4','O1','O2']
)

In [ ]:
windows=[(12,17),(33,38)]
out = run_criticality_suite_records(RECORDS, windows, ch='F4', metastab_electrodes=['F4','O1','O2'])
print(out['summary'])


In [ ]:
# Define your windows/electrodes/bands (or keep defaults)
IGNITION_WINDOWS = [(120.0, 150.0)]
REBOUND_WINDOWS  = [(300.0, 330.0)]
ELECTRODES = ['F4','O1','O2']

summary = run_entanglement_geometry_minCut_PLV(
    RECORDS,
    ignition_windows=IGNITION_WINDOWS,
    rebound_windows=REBOUND_WINDOWS,
    electrodes=ELECTRODES,
    bands={'theta':(4,8), 'alpha':(8,13), 'beta':(13,30)},
    do_control=True   # set to False to skip surrogates
)

print(summary['delta_table'])


In [ ]:
plot_entanglement_geometry_deltas(summary['delta_table'])

In [ ]:
plot_entanglement_geometry_levels(summary['delta_table'])

In [ ]:
plot_entanglement_geometry_scatter(summary['delta_table'])

In [ ]:
"""
One‑Click Session Report — Holographic Analyses (fs=128)
-------------------------------------------------------
Runs the core analyses end‑to‑end and exports CSVs + PNGs + a compact PDF
summary for the given session windows.

Configured windows (from your message):
    ignition_windows = [(290,310), (580,600)]
    rebound_windows  = [(310,325)]

What this runs
1) Entanglement–Geometry: Δmin‑cut, Δentropy, ΔPLV
2) Ridge–PAC coupling: lag scan (peak r, lag)
3) Criticality: Δβ, Δα, avalanche CCDFs
4) Harmonics breadth: ΔH, ΔPR, ΔTop10 (+ surrogates)
   (builds functional harmonics from baseline if H not provided)
5) Overlap ETAs: PLV/PAC/min‑cut/β around K≥3 with bootstrap null
6) Multi‑seed “surfaces”: ΔMultiCut via GH tree (+ degree‑preserving controls)
7) Phase embedding: trustworthiness, continuity, geodesic stress (+ surrogates)
8) Temporal holography (optional): phase‑bin AUC + PAC fingerprints if onsets/labels given

Exports
- exports/<session_name>/CSV/*.csv
- exports/<session_name>/FIG/*.png
- exports/<session_name>/<session_name>.pdf (summary via PdfPages)

Usage
-----
report = run_one_click_session_report(
    RECORDS,
    session_name='session_A',
    time_col='Timestamp',
    electrodes=None,                # autodetect EEG.* if None
    ignition_windows=[(290,310),(580,600)],
    rebound_windows=[(310,325)],
    control_windows=None,           # optional
    event_onsets=None,              # optional for temporal holography
    event_labels=None,              # optional
    H=None                          # optional harmonics matrix; builds if None
)
"""
from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from typing import Dict, List, Tuple, Optional
from scipy import signal

# ---------- tiny utils ----------
def _ensure_dir(d):
    os.makedirs(d, exist_ok=True)
    return d

def _autoelectrodes(RECORDS, time_col):
    els = [c.split('.',1)[1] for c in RECORDS.columns if c.startswith('EEG.') and c!=time_col]
    return els or ['F4','O1','O2']

# quick PSD ranker to pick a clean Schumann channel
def pick_best_channel_for_schumann(RECORDS, time_col='Timestamp'):
    fs = infer_fs_from_records(RECORDS, time_col=time_col)
    cands = [c for c in RECORDS.columns if c.startswith('EEG.') and c!=time_col]
    scores = []
    for ch in cands:
        x = np.asarray(RECORDS[ch].values, float)
        f, p = signal.welch(x, fs=fs, nperseg=4*int(fs))
        def band(a,b):
            sel=(f>=a)&(f<=b)
            return np.trapz(p[sel], f[sel]) if np.any(sel) else 0.0
        low = band(4,12); emg=band(40,90); mains=band(55,65)
        score = low/(emg+1e-12) - 0.2*mains
        scores.append((score, ch))
    scores.sort(reverse=True)
    return scores[0][1] if scores else cands[0]

# ---------- main runner ----------
def run_one_click_session_report(
    RECORDS: pd.DataFrame,
    session_name: str = 'session',
    time_col: str = 'Timestamp',
    electrodes: Optional[List[str]] = None,
    ignition_windows: List[Tuple[float,float]] = [(290,310),(580,600)],
    rebound_windows: Optional[List[Tuple[float,float]]] = [(310,325)],
    control_windows: Optional[List[Tuple[float,float]]] = None,
    event_onsets: Optional[List[float]] = None,
    event_labels: Optional[List] = None,
    H: Optional[np.ndarray] = None,
) -> Dict[str, object]:
    fs = infer_fs_from_records(RECORDS, time_col=time_col)
    electrodes = electrodes or _autoelectrodes(RECORDS, time_col)

    # export dirs
    base = _ensure_dir(os.path.join('exports', session_name))
    d_csv = _ensure_dir(os.path.join(base, 'CSV'))
    d_fig = _ensure_dir(os.path.join(base, 'FIG'))
    pdf_path = os.path.join(base, f'{session_name}.pdf')

    # 0) Fused Schumann micro‑grid on best channel
    sigcol = pick_best_channel_for_schumann(RECORDS, time_col=time_col)
    fused = detect_and_plot_schumann_microgrid_with_global_tf(
        RECORDS, signal_col=sigcol, time_col=time_col, show=False
    )
    # Save SAI/overlap
    pd.DataFrame({'t': fused['index'], 'SAI': fused['sai']}).to_csv(os.path.join(d_csv,'sai.csv'), index=False)

    with PdfPages(pdf_path) as pdf:
        # 1) Entanglement–Geometry
        eg = run_entanglement_geometry_minCut_PLV(
            RECORDS,
            ignition_windows=ignition_windows,
            rebound_windows=rebound_windows,
            electrodes=electrodes,
            bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30)},
            do_control=True
        )
        eg['delta_table'].to_csv(os.path.join(d_csv,'entanglement_geometry_deltas.csv'), index=False)
        plt.close('all'); plot_entanglement_geometry_deltas(eg['delta_table']); pdf.savefig(); plt.savefig(os.path.join(d_fig,'entanglement_deltas.png'))

        # 2) Ridge–PAC coupling
        rpc = run_ridge_pac_coupling(
            RECORDS, fused=fused, electrodes=electrodes, time_col=time_col,
            pac_pairs={'theta→gamma':((7,9),(30,80)), 'alpha→gamma':((8,12),(30,80))},
            max_lag_sec=2.0, pac_win_sec=2.0, step_sec=0.25, smooth_sec=0.20,
            off_resonant_bands=[(16,18)], show=True
        )
        rpc['ridge_pac_corr'].to_csv(os.path.join(d_csv,'ridge_pac_corr.csv'), index=False)
        pdf.savefig(); plt.savefig(os.path.join(d_fig,'ridge_pac_lagcurves.png')); plt.close('all')

        # 3) Criticality
        crit = run_criticality_analysis(
            RECORDS,
            ignition_windows=ignition_windows,
            rebound_windows=rebound_windows,
            control_windows=control_windows,
            electrodes=electrodes,
        )
        crit['delta_table'].to_csv(os.path.join(d_csv,'criticality_deltas.csv'), index=False)
        plt.close('all'); plot_criticality_deltas(crit['delta_table']); pdf.savefig(); plt.savefig(os.path.join(d_fig,'criticality_deltas.png'))
        plt.close('all'); plot_avalanche_ccdf(crit['avalanches']); pdf.savefig(); plt.savefig(os.path.join(d_fig,'avalanche_ccdf.png'))

        # 4) Harmonics breadth (build H if needed)
        if H is None:
            H = build_functional_harmonics_from_baseline(
                RECORDS,
                electrodes=electrodes,
                ignition_windows=ignition_windows,
                time_col=time_col,
                fband=(4,40), n_modes=64
            )
        hb = run_connectome_harmonics_breadth(
            RECORDS, H=H, electrodes=electrodes,
            ignition_windows=ignition_windows, rebound_windows=rebound_windows,
            time_col=time_col, orthonormal=True, do_surrogate=True, n_surr=200
        )
        hb['delta_table'].to_csv(os.path.join(d_csv,'harmonics_breadth_deltas.csv'), index=False)
        plt.close('all'); plot_harmonics_power_spectra(hb['spectra']); pdf.savefig(); plt.savefig(os.path.join(d_fig,'harmonics_power.png'))
        plt.close('all'); plot_harmonics_breadth_deltas(hb['delta_table']); pdf.savefig(); plt.savefig(os.path.join(d_fig,'harmonics_deltas.png'))

        # 5) Overlap ETAs (K≥3)
        etas = run_overlap_coherence_etas(
            RECORDS, fused=fused, electrodes=electrodes, time_col=time_col,
            K=3, win_sec=2.0, step_sec=0.25, span_sec=5.0,
            plv_band=(8,13), pac_pairs={'theta→gamma':((4,8),(30,80))},
            mincut_band=(8,13), beta_band=(1,40), n_boot=200, show=True
        )
        pdf.savefig(); plt.savefig(os.path.join(d_fig,'overlap_etas.png')); plt.close('all')

        # 6) Multi‑seed surfaces (F,P,O,T)
        ms = run_multi_seed_surface_cuts(
            RECORDS,
            ignition_windows=ignition_windows,
            rebound_windows=rebound_windows,
            time_col=time_col,
            bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30)},
            electrodes=electrodes, clusters=None,
            control_mode='degree_rewire', n_shuffle=200, graph_density=0.3, show=True
        )
        ms['delta_table'].to_csv(os.path.join(d_csv,'multiseed_deltas.csv'), index=False)
        pdf.savefig(); plt.savefig(os.path.join(d_fig,'multiseed_deltas.png')); plt.close('all')

        # 7) Phase embedding (alpha)
        pe = run_phase_embedding_emergent_geometry(
            RECORDS, ignition_windows=ignition_windows, rebound_windows=rebound_windows,
            control_windows=control_windows, time_col=time_col, electrodes=electrodes,
            band=(8,13), n_neighbors=6, n_components=2, method='isomap', k_quality=5,
            n_surr=100, show=True
        )
        pe['metrics_table'].to_csv(os.path.join(d_csv,'phase_embedding_metrics.csv'), index=False)
        pdf.savefig(); plt.savefig(os.path.join(d_fig,'phase_embedding_metrics.png')); plt.close('all')

        # 8) Temporal holography (optional if onsets provided)
        if event_onsets is not None:
            th = run_temporal_holography_multiplexed(
                RECORDS, event_onsets=event_onsets, labels=event_labels,
                time_col=time_col, electrodes=electrodes,
                ref_electrodes=[e for e in ['O1','O2','Oz','Pz'] if ('EEG.'+e) in RECORDS.columns] or electrodes[:1],
                ref_band='theta', n_bins=6, feat_window=(-0.5, 1.0), n_shuffle=200, show=True
            )
            th['auc_table'].to_csv(os.path.join(d_csv,'temporal_holography_auc.csv'), index=False)
            pdf.savefig(); plt.savefig(os.path.join(d_fig,'temporal_holography_auc.png')); plt.close('all')

    return {
        'export_dir': base,
        'pdf': pdf_path,
        'csv_dir': d_csv,
        'fig_dir': d_fig,
        'fused': fused
    }


In [ ]:
report = run_one_click_session_report(
    RECORDS,
    session_name='session_A',
    time_col='Timestamp',
    electrodes=None,   # autodetect EEG.*; pass a list to force an order
    ignition_windows=[(290,310),(580,600)],
    rebound_windows=[(310,325)],
    control_windows=None,     # optional deep-rest windows
    event_onsets=None,        # optional: for Temporal Holography
    event_labels=None,        # optional
    H=None                    # optional: harmonics matrix; builds if None
)

In [ ]:
import numpy as np
from scipy import signal

def auto_pick_posterior_channels(RECORDS, n_pick=8, time_col='Timestamp'):
    fs = infer_fs(RECORDS, time_col=time_col)
    # candidate posterior labels to prefer, in priority order
    prefer = ['O1','O2','Oz','POz','PO3','PO4','Pz','P3','P4','PO7','PO8','TP7','TP8','T7','T8']
    # keep those that exist
    avail = [lab for lab in prefer if ('EEG.'+lab) in RECORDS.columns]
    # score by low (4–12 Hz) / high (40–90 Hz) – mains penalty
    scores = []
    for lab in avail:
        x = np.asarray(RECORDS['EEG.'+lab].values, float)
        f, p = signal.welch(x, fs=fs, nperseg=4*int(fs))
        def band(a,b): 
            sel=(f>=a)&(f<=b)
            return np.trapz(p[sel], f[sel]) if np.any(sel) else 0.0
        low = band(4,12); emg = band(40,90); mains = band(55,65)
        score = low/(emg+1e-12) - 0.2*mains
        scores.append((score, 'EEG.'+lab))
    scores.sort(reverse=True)
    return [ch for _, ch in scores[:n_pick]]

In [ ]:
"""
Cross-frequency interactions involving Schumann (stand-alone)

3a) Schumann-locked PAC (event-related PAC; ERPAC)
    - Detect Schumann bursts (threshold on ELF envelope around ~7.8 Hz)
    - Time-lock EEG θ→γ PAC to burst onsets (±10–20 s)
    - Optional cluster-based permutation across trials (sign-flip null)

3b) Cross-bicoherence / cross-bispectrum
    - Cross-bicoherence between Schumann band (f1≈7.8 Hz) and EEG γ (f2≈30–80 Hz)
      predicting energy at f1+f2 in EEG
    - Returns bicoherence matrix B(f1,f2) and a simple circular-shift surrogate

Assumes a pandas.DataFrame RECORDS with a time column (default 'Timestamp')
and signal columns like 'EEG.O1', 'EEG.O2' (EEG), and a Schumann reference channel
(e.g., 'EEG.O1' or a magnetometer if available).
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from scipy import signal
import matplotlib.pyplot as plt

# -------------------- basic helpers --------------------

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0:
        raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    """Return a numeric signal array. Accepts 'EEG.O1' or bare 'O1' (will try 'EEG.O1')."""
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.' + name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found in RECORDS.")

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order: int = 4) -> np.ndarray:
    ny = 0.5 * fs
    f1 = max(1e-6, min(f1, ny * 0.99))
    f2 = max(f1 + 1e-6, min(f2, ny * 0.999))
    b, a = signal.butter(order, [f1 / ny, f2 / ny], btype='band')
    return signal.filtfilt(b, a, x)

def slice_epoch(x: np.ndarray, idx0: int, idx1: int) -> Optional[np.ndarray]:
    idx0 = max(0, idx0); idx1 = min(len(x), idx1)
    if idx1 <= idx0:
        return None
    return x[idx0:idx1]

# -------------------- 3a) Schumann-locked, event-related PAC --------------------

def detect_schumann_bursts(RECORDS: pd.DataFrame,
                           sr_channel: str,
                           time_col: str = 'Timestamp',
                           center_hz: float = 7.83,
                           half_bw_hz: float = 0.6,
                           smooth_sec: float = 0.25,
                           thresh_mode: str = 'z',
                           z_thresh: float = 2.5,
                           perc_thresh: float = 95.0,
                           min_isi_sec: float = 2.0
                           ) -> Dict[str, object]:
    """
    Detect Schumann bursts on a reference signal by thresholding the narrowband envelope.
    Returns {'onsets_sec': [...], 'env': env, 't': t}.
    """
    fs = infer_fs(RECORDS, time_col)
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    y = get_series(RECORDS, sr_channel)
    yb = bandpass(y, fs, center_hz - half_bw_hz, center_hz + half_bw_hz)
    env = np.abs(signal.hilbert(yb))
    # smooth envelope
    n = max(1, int(round(fs * smooth_sec)))
    if n > 1:
        w = np.hanning(n) / np.sum(np.hanning(n))
        env = np.convolve(env, w, mode='same')
    # threshold
    if thresh_mode == 'z':
        z = (env - env.mean()) / (env.std() + 1e-12)
        mask = z >= z_thresh
    else:
        thr = np.percentile(env, perc_thresh)
        mask = env >= thr
    # rising edges
    on_idx = np.where(np.diff(mask.astype(int)) == 1)[0] + 1
    # enforce minimum ISI
    on = []
    last_t = -np.inf
    for i in on_idx:
        if t[i] - last_t >= min_isi_sec:
            on.append(t[i]); last_t = t[i]
    return {'onsets_sec': on, 'env': env, 't': t}

def pac_mi_phase_amp(x_phase: np.ndarray,
                     x_amp: np.ndarray,
                     nbins: int = 18) -> float:
    """Tort MI: KL divergence of phase-binned amplitude from uniform."""
    ph = np.angle(signal.hilbert(x_phase))
    am = np.abs(signal.hilbert(x_amp))
    edges = np.linspace(-np.pi, np.pi, nbins + 1)
    digit = np.digitize(ph, edges) - 1
    digit = np.clip(digit, 0, nbins - 1)
    m = np.zeros(nbins)
    for k in range(nbins):
        sel = (digit == k)
        m[k] = np.mean(am[sel]) if np.any(sel) else 0.0
    if m.sum() <= 0:
        return 0.0
    p = m / m.sum()
    eps = 1e-12
    mi = np.sum(p * np.log((p + eps) / (1.0 / nbins))) / np.log(nbins)
    return float(mi)

def epochwise_pac_timecourse(RECORDS: pd.DataFrame,
                             eeg_channels: List[str],
                             time_col: str,
                             onsets_sec: List[float],
                             win_sec: Tuple[float, float] = (-10.0, 10.0),
                             pac_phase_band: Tuple[float, float] = (4, 8),
                             pac_amp_band: Tuple[float, float] = (30, 80),
                             step_sec: float = 0.25,
                             win_pac_sec: float = 1.0,
                             nbins: int = 18) -> Dict[str, object]:
    """
    Build trial x time PAC(t) around onsets, averaged over channels.
    Sliding window (win_pac_sec) with step (step_sec).
    """
    fs = infer_fs(RECORDS, time_col)
    n_step = max(1, int(round(step_sec * fs)))
    L = int(round((win_sec[1] - win_sec[0]) * fs))
    # assemble channel matrix
    X = []
    for ch in eeg_channels:
        X.append(get_series(RECORDS, ch))
    X = np.vstack(X)  # (n_ch, N)
    N = X.shape[1]
    # time vector relative to onset for centers of windows
    centers = np.arange(int(round(win_sec[0] * fs + win_pac_sec * fs / 2)),
                        int(round(win_sec[1] * fs - win_pac_sec * fs / 2)) + 1,
                        n_step)
    t_rel = centers / fs
    # compute PAC per trial
    PAC = []  # trials x time
    keep_onsets = []
    for on in onsets_sec:
        i_on = int(round(on * fs))
        i0 = i_on + int(round(win_sec[0] * fs))
        i1 = i_on + int(round(win_sec[1] * fs))
        if i0 < 0 or i1 > N or (i1 - i0) < int(round(win_pac_sec * fs)):
            continue
        trial = []
        for c in centers:
            s = i_on + int(round(c - win_pac_sec * fs / 2))
            e = s + int(round(win_pac_sec * fs))
            if s < 0 or e > N:
                trial.append(np.nan)
                continue
            # band-limit per channel and average MI across channels
            mis = []
            for x in X:
                xp = bandpass(x[s:e], fs, pac_phase_band[0], pac_phase_band[1])
                xa = bandpass(x[s:e], fs, pac_amp_band[0], pac_amp_band[1])
                mis.append(pac_mi_phase_amp(xp, xa, nbins=nbins))
            trial.append(np.nanmean(mis))
        PAC.append(trial)
        keep_onsets.append(on)
    PAC = np.array(PAC, float)  # (n_trials, n_time)
    return {'t_rel': t_rel, 'PAC_trials': PAC, 'onsets_used': keep_onsets}

def cluster_permutation_1d(mean_tc: np.ndarray,
                           trials_tc: np.ndarray,
                           alpha: float = 0.05,
                           n_perm: int = 200,
                           rng_seed: int = 7) -> Dict[str, object]:
    """
    Simple 1D cluster-based permutation along time for ERPAC curve.
    - Observed: mean_tc (T,) from trials_tc (N,T) relative to baseline 0
    - Null: sign-flip trials randomly to build max-cluster distribution
    Returns significant mask and cluster boundaries.
    """
    rng = np.random.default_rng(rng_seed)
    T = mean_tc.size
    # Threshold = percentile of permuted means at each time (one-sided)
    null_means = []
    for _ in range(n_perm):
        signs = rng.choice([-1, 1], size=trials_tc.shape[0])
        perm = np.nanmean(signs[:, None] * trials_tc, axis=0)
        null_means.append(perm)
    null_means = np.array(null_means)
    thr = np.nanpercentile(null_means, 100 * (1 - alpha), axis=0)  # timepoint-wise threshold

    # observed clusters
    sig = mean_tc > thr
    # cluster mass = sum over contiguous sig points
    clusters = []
    start = None
    for i in range(T):
        if sig[i] and start is None:
            start = i
        elif (not sig[i]) and start is not None:
            clusters.append((start, i - 1))
            start = None
    if start is not None:
        clusters.append((start, T - 1))

    # Null cluster masses
    null_max = []
    for p in null_means:
        s = p > thr  # reuse same threshold
        maxmass = 0.0; run = 0.0
        for i in range(T):
            if s[i]:
                run += p[i]
                maxmass = max(maxmass, run)
            else:
                run = 0.0
        null_max.append(maxmass)
    thresh_mass = np.nanpercentile(null_max, 95)

    # Which observed clusters exceed mass threshold?
    sig_clusters = []
    for (a, b) in clusters:
        mass = np.nansum(mean_tc[a:b+1])
        if mass >= thresh_mass:
            sig_clusters.append((a, b))
    mask = np.zeros(T, dtype=bool)
    for (a, b) in sig_clusters:
        mask[a:b+1] = True
    return {'sig_mask': mask, 'sig_clusters': sig_clusters, 'thr_point': thr, 'thr_mass': thresh_mass}

def run_schumann_locked_erpac(RECORDS: pd.DataFrame,
                              sr_channel: str,
                              eeg_channels: List[str],
                              time_col: str = 'Timestamp',
                              detect_params: Dict = None,
                              erpac_params: Dict = None,
                              baseline_window: Tuple[float, float] = (-10.0, -2.0),
                              do_permutation: bool = True) -> Dict[str, object]:
    """
    Full ERPAC:
      1) detect Schumann bursts on sr_channel
      2) build trial x time PAC around onsets
      3) baseline-correct per trial by subtracting mean PAC in baseline_window
      4) cluster-based permutation along time (optional)
    """
    detect_params = detect_params or {}
    erpac_params = erpac_params or {}
    fs = infer_fs(RECORDS, time_col)
    # 1) detect bursts
    det = detect_schumann_bursts(RECORDS, sr_channel, time_col=time_col, **detect_params)
    onsets = det['onsets_sec']
    if len(onsets) == 0:
        raise ValueError("No Schumann bursts detected with current threshold.")

    # 2) epochwise PAC
    ep = epochwise_pac_timecourse(RECORDS, eeg_channels, time_col, onsets, **erpac_params)
    PAC = ep['PAC_trials']  # (n_trials, T)
    t_rel = ep['t_rel']

    # 3) baseline-correct per trial
    bsel = (t_rel >= baseline_window[0]) & (t_rel <= baseline_window[1])
    PAC_bc = PAC - np.nanmean(PAC[:, bsel], axis=1, keepdims=True)
    mean_tc = np.nanmean(PAC_bc, axis=0)

    out = {'t_rel': t_rel, 'PAC_trials': PAC, 'PAC_bc': PAC_bc, 'mean_tc': mean_tc, 'onsets': ep['onsets_used']}
    if do_permutation:
        perm = cluster_permutation_1d(mean_tc, PAC_bc)
        out.update({'perm': perm})
    return out

# -------------------- 3b) Cross-bicoherence / cross-bispectrum --------------------

def segment_fft(sig: np.ndarray, fs: float, nperseg: int, noverlap: int) -> np.ndarray:
    """Return STFT-like complex spectra array (n_seg, n_freq) using Hann windows."""
    step = nperseg - noverlap
    win = signal.hann(nperseg, sym=False)
    n_fft = int(2 ** np.ceil(np.log2(nperseg)))
    segs = []
    for start in range(0, len(sig) - nperseg + 1, step):
        seg = sig[start:start+nperseg] * win
        S = np.fft.rfft(seg, n=n_fft)   # (n_freq,)
        segs.append(S)
    return np.array(segs), np.fft.rfftfreq(n_fft, d=1/fs)

def cross_bicoherence(RECORDS: pd.DataFrame,
                      x_sr: str,              # Schumann reference channel (for f1 ≈ 7.8 Hz)
                      y_eeg: str,             # EEG channel for gamma (f2)
                      z_eeg: Optional[str] = None,  # EEG channel for f1+f2 (default = y_eeg)
                      time_col: str = 'Timestamp',
                      f1_list: List[float] = (7.83,),    # cyclic base(s)
                      f2_min: float = 30.0, f2_max: float = 80.0, n_f2: int = 40,
                      nperseg: int = 2048, noverlap: int = 1024,
                      do_surrogate: bool = True, n_surr: int = 200, rng_seed: int = 11
                      ) -> Dict[str, object]:
    """
    Compute cross-bicoherence b_xy(f1,f2) predicting Z at f1+f2:
      b_xy(f1,f2) = E[X(f1)Y(f2)Z*(f1+f2)] / sqrt( E|X(f1)Y(f2)|^2 * E|Z(f1+f2)|^2 )
    Returns matrix over (f1_list,f2_grid) and a simple circular-shift surrogate null for y/z.
    """
    fs = infer_fs(RECORDS, time_col)
    x = get_series(RECORDS, x_sr)
    y = get_series(RECORDS, y_eeg)
    if z_eeg is None:
        z = y
    else:
        z = get_series(RECORDS, z_eeg)

    X, f = segment_fft(x, fs, nperseg, noverlap)   # (n_seg, n_freq)
    Y, _ = segment_fft(y, fs, nperseg, noverlap)
    Z, _ = segment_fft(z, fs, nperseg, noverlap)
    if X.size == 0 or Y.size == 0 or Z.size == 0:
        raise ValueError("Not enough data for the chosen nperseg/noverlap.")

    # f2 grid and indexing helpers
    f2_grid = np.linspace(f2_min, f2_max, n_f2)
    def idx_of(freq):
        return int(np.argmin(np.abs(f - freq)))

    B = np.zeros((len(f1_list), n_f2), float)
    for i, f1 in enumerate(f1_list):
        i1 = idx_of(f1)
        for j, f2 in enumerate(f2_grid):
            i2 = idx_of(f2)
            i12 = idx_of(f1 + f2)
            num = np.mean(X[:, i1] * Y[:, i2] * np.conj(Z[:, i12]))
            den = np.sqrt(np.mean(np.abs(X[:, i1] * Y[:, i2])**2) * np.mean(np.abs(Z[:, i12])**2) + 1e-24)
            B[i, j] = np.abs(num) / (den + 1e-24)

    out = {'f1_list': np.array(f1_list), 'f2_grid': f2_grid, 'bicoherence': B, 'freqs': f}
    # Simple surrogate: circularly shift Y (gamma) segments per realization
    if do_surrogate:
        rng = np.random.default_rng(rng_seed)
        null_max = []
        for _ in range(n_surr):
            # circular shift each epoch spectrum by a random small amount in time domain
            # (approximate by reordering epochs; simpler robust null)
            Y_perm = Y.copy()
#             rng.shuffle(Y_perm, axis=0)
            Y_perm = Y_perm[rng.permutation(Y_perm.shape[0]), :]
            B0 = np.zeros_like(B)
            for i, f1 in enumerate(f1_list):
                i1 = idx_of(f1)
                for j, f2 in enumerate(f2_grid):
                    i2 = idx_of(f2); i12 = idx_of(f1 + f2)
                    num = np.mean(X[:, i1] * Y_perm[:, i2] * np.conj(Z[:, i12]))
                    den = np.sqrt(np.mean(np.abs(X[:, i1] * Y_perm[:, i2])**2) * np.mean(np.abs(Z[:, i12])**2) + 1e-24)
                    B0[i, j] = np.abs(num) / (den + 1e-24)
            null_max.append(np.nanmax(B0))
        out['null_thresh95'] = float(np.nanpercentile(null_max, 95))
    return out

def plot_bicoherence(out: Dict[str, object], i_f1: int = 0, title: Optional[str] = None) -> None:
    """Heatmap of cross-bicoherence at a fixed f1 index across f2_grid."""
    f2 = out['f2_grid']; B = out['bicoherence']; f1_list = out['f1_list']
    plt.figure(figsize=(7, 3))
    plt.plot(f2, B[i_f1], lw=1.8)
    if 'null_thresh95' in out:
        plt.axhline(out['null_thresh95'], color='k', ls='--', lw=1, label='null 95%')
    plt.xlabel('f2 (Hz, EEG γ)'); plt.ylabel('cross-bicoherence |b_xy|')
    plt.title(title or f'Cross-bicoherence at f1={f1_list[i_f1]:.2f} Hz')
    plt.grid(alpha=0.2)
    if 'null_thresh95' in out:
        plt.legend()
    plt.tight_layout(); plt.show()

# -------------------- examples --------------------
if __name__ == "__main__":
    # 3a) Schumann-locked ERPAC:
    # det_params = {'center_hz':7.83, 'half_bw_hz':0.6, 'thresh_mode':'z', 'z_thresh':2.5}
    # er_params  = {'win_sec':(-10,10), 'pac_phase_band':(4,8), 'pac_amp_band':(30,80), 'step_sec':0.25, 'win_pac_sec':1.0}
    # er = run_schumann_locked_erpac(RECORDS, sr_channel='EEG.O1',
    #         eeg_channels=['EEG.O1','EEG.O2','EEG.Pz'],
    #         time_col='Timestamp',
    #         detect_params=det_params,
    #         erpac_params=er_params,
    #         baseline_window=(-10,-2),
    #         do_permutation=True)
    # print('n_onsets:', len(er['onsets']))
    # # Plot ERPAC mean with significant clusters if present
    # plt.figure(figsize=(8,3))
    # plt.plot(er['t_rel'], er['mean_tc'], lw=1.8, label='ERPAC (θ→γ)')
    # if 'perm' in er:
    #     m = er['perm']['sig_mask']
    #     plt.fill_between(er['t_rel'], 0, er['mean_tc'], where=m, color='tab:red', alpha=0.25, step='pre', label='sig cluster')
    # plt.axvline(0, color='k', lw=1); plt.xlabel('Time (s)'); plt.ylabel('PAC (MI)')
    # plt.title('Event-related PAC around Schumann bursts'); plt.legend(); plt.tight_layout(); plt.show()

    # 3b) Cross-bicoherence:
    # bi = cross_bicoherence(RECORDS, x_sr='EEG.O1', y_eeg='EEG.O2', z_eeg=None,
    #                        time_col='Timestamp', f1_list=[7.83], f2_min=30, f2_max=80, n_f2=40,
    #                        nperseg=2048, noverlap=1024, do_surrogate=True, n_surr=200)
    # plot_bicoherence(bi, i_f1=0)
    pass


In [ ]:
# 3a) Schumann-locked ERPAC:
det_params = {'center_hz':7.83, 'half_bw_hz':0.6, 'thresh_mode':'z', 'z_thresh':2.5}
er_params  = {'win_sec':(-10,10), 'pac_phase_band':(4,8), 'pac_amp_band':(30,80), 'step_sec':0.25, 'win_pac_sec':1.0}
er = run_schumann_locked_erpac(RECORDS, sr_channel='EEG.F4',
        eeg_channels=['EEG.AF4','EEG.FC6', 'EEG.O2','EEG.T8','EEG.P8'],
        time_col='Timestamp',
        detect_params=det_params,
        erpac_params=er_params,
        baseline_window=(-10,-2),
        do_permutation=True)
print('n_onsets:', len(er['onsets']))
# Plot ERPAC mean with significant clusters if present
plt.figure(figsize=(8,3))
plt.plot(er['t_rel'], er['mean_tc'], lw=1.8, label='ERPAC (θ→γ)')
if 'perm' in er:
    m = er['perm']['sig_mask']
    plt.fill_between(er['t_rel'], 0, er['mean_tc'], where=m, color='tab:red', alpha=0.25, step='pre', label='sig cluster')
plt.axvline(0, color='k', lw=1); plt.xlabel('Time (s)'); plt.ylabel('PAC (MI)')
plt.title('Event-related PAC around Schumann bursts'); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 3b) Cross-bicoherence:
bi = cross_bicoherence(RECORDS, x_sr='EEG.F4', y_eeg='EEG.O2', z_eeg=None,
                       time_col='Timestamp', f1_list=[7.83], f2_min=35, f2_max=60, n_f2=40,
                       nperseg=2048, noverlap=1024, do_surrogate=True, n_surr=200)
plot_bicoherence(bi, i_f1=0)

In [ ]:
"""
Directionality & Information Flow (stand-alone)

4a) Frequency-domain Granger / Partial Directed Coherence (PDC) / DTF
    • Fit VAR (bivariate or multivariate) on EEG + Schumann reference
    • Diagnostics: order selection (AIC/BIC), stability (roots<1), residual whiteness (Ljung-Box)
    • Spectral DTF/PDC and (optional) time-domain Granger tests
    • Report values at Schumann harmonics (≈7.83, 14.3, 20.8, 27.3, 33.8 Hz)

4b) Transfer Entropy (TE) / Conditional TE (lag-resolved)
    • kNN (Kraskov-style) estimator for TE X→Y at specified lags
    • Surrogate significance via circular time-shift

4c) Time-varying (state-space flavored) AR “Kalman-RLS”
    • Recursive least-squares with forgetting to track AR coefficients over time
    • Extract directed coupling gains X→Y(t), Y→X(t) and a time-varying DTF at 7.83 Hz

Assumes a pandas.DataFrame RECORDS with a time column (default 'Timestamp') and
columns like 'EEG.O1', 'EEG.O2', or a Schumann reference (EEG or magnetometer).
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from scipy import signal, stats
import matplotlib.pyplot as plt

# optional: VAR & Ljung-Box from statsmodels
try:
    from statsmodels.tsa.api import VAR
    from statsmodels.stats.diagnostic import acorr_ljungbox
    _HAS_SM = True
except Exception:
    _HAS_SM = False

# ------------------ generic helpers ------------------

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0:
        raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    """Return numeric signal. Accepts 'EEG.O1' or bare 'O1' (tries 'EEG.O1')."""
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.' + name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found.")

def slice_concat(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> np.ndarray:
    if not windows: return x.copy()
    segs=[]; n=len(x)
    for (t0,t1) in windows:
        i0,i1 = int(round(t0*fs)), int(round(t1*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def stack_channels(RECORDS: pd.DataFrame, channels: List[str],
                   fs: float, windows: Optional[List[Tuple[float,float]]],
                   demean: bool=True) -> np.ndarray:
    X = []
    for ch in channels:
        x = get_series(RECORDS, ch)
        x = slice_concat(x, fs, windows)
        if demean: x = x - np.mean(x)
        X.append(x)
    # truncate to min length
    L = min(map(len, X))
    X = np.vstack([x[:L] for x in X])  # (n_ch, L)
    return X

# ------------------ 4a) VAR / DTF / PDC ------------------

def fit_var_model(X: np.ndarray,
                  order_max: int = 20,
                  crit: str = 'bic') -> Dict[str, object]:
    """
    Fit VAR to (n_ch, L) array X.T using statsmodels.
    Returns chosen order p, A matrices (p, n_ch, n_ch), noise cov Sigma_u,
    stability flag, Ljung-Box p-values (per channel), and model object (if available).
    """
    if not _HAS_SM:
        raise RuntimeError("statsmodels is required for VAR fitting.")

    Y = X.T  # (L, n_ch)
    model = VAR(Y)

    # --- order selection: support both attribute and dict-like outputs
    sel = model.select_order(maxlags=order_max)
    if hasattr(sel, crit):
        p = int(getattr(sel, crit))
    else:
        # older statsmodels returns a dict-like
        p = int(sel[crit])

    # fit VAR(p)
    res = model.fit(p)
    A = np.array(res.coefs)            # shape (p, n_ch, n_ch)
    Sigma_u = np.array(res.sigma_u)    # (n_ch, n_ch)

    # stability: all roots inside unit circle
    stable = bool(np.all(np.abs(res.roots) < 1.0))

    # --- residual whiteness: Ljung–Box, compatible with old/new statsmodels
    def _lb_last_pvalue(x: np.ndarray, max_lag: int) -> float:
        lag = max(1, min(20, max_lag))
        try:
            # newer API (return_df=True)
            lb_df = acorr_ljungbox(x, lags=[lag], return_df=True)
            return float(lb_df['lb_pvalue'].iloc[-1])
        except TypeError:
            # older API: returns (stat, pvalue)
            stat, p = acorr_ljungbox(x, lags=lag)
            # ensure we return the last lag's p-value
            p = np.atleast_1d(p)
            return float(p[-1])

    resid = res.resid  # (L-p, n_ch)
    lb_pvals = []
    for j in range(resid.shape[1]):
        lb_pvals.append(_lb_last_pvalue(resid[:, j], max_lag=len(resid)//5))

    return {
        'order': p,
        'A': A,
        'Sigma_u': Sigma_u,
        'stable': stable,
        'lb_pvals': lb_pvals,
        'res': res
    }

def _A_of_f(A: np.ndarray, f: np.ndarray, fs: float) -> np.ndarray:
    """
    A(f) = I - sum_{k=1..p} A_k e^{-i 2π f k / fs}, shape: (n_freq, n_ch, n_ch).
    """
    p, n, _ = A.shape
    I = np.eye(n)
    Af = []
    for ff in f:
        z = np.zeros((n, n), dtype=complex)
        for k in range(1, p+1):
            z += A[k-1] * np.exp(-1j * 2*np.pi*ff * k / fs)
        Af.append(I - z)
    return np.array(Af)  # (n_freq, n, n)

def _H_of_f(Af: np.ndarray) -> np.ndarray:
    """
    Transfer matrix H(f) = A(f)^{-1}, for each frequency. Af shape (n_freq, n, n).
    """
    Hf = np.zeros_like(Af, dtype=complex)
    for i in range(Af.shape[0]):
        Hf[i] = np.linalg.inv(Af[i])
    return Hf

def spectral_dtf_pdc(A: np.ndarray, Sigma_u: np.ndarray, fs: float,
                     fmin: float = 0.0, fmax: float = 50.0, n_freq: int = 256) -> Dict[str, np.ndarray]:
    """
    Compute DTF and PDC spectra from VAR(A, Sigma_u).
    DTF_{i<-j}(f) = |H_{ij}(f)| / sqrt(sum_k |H_{ik}(f)|^2)  (outflow from j to i)
    PDC_{i<-j}(f) = |A_{ij}(f)| / sqrt(sum_k |A_{kj}(f)|^2)  (column-normalized A(f))
    """
    n = A.shape[1]
    f = np.linspace(fmin, fmax, n_freq)
    Af = _A_of_f(A, f, fs)                # (n_f, n, n)
    Hf = _H_of_f(Af)                      # (n_f, n, n)

    # DTF
    DTF = np.zeros((n, n, n_freq))
    for k in range(n_freq):
        H = Hf[k]
        num = np.abs(H)**2
        den = np.sum(num, axis=1, keepdims=True) + 1e-24  # row sum: to targets i
        DTF[:, :, k] = np.sqrt(num / den)                 # sqrt often used; use num/den if you prefer

    # PDC
    PDC = np.zeros((n, n, n_freq))
    for k in range(n_freq):
        A_k = Af[k]
        num = np.abs(A_k)**2
        den = np.sum(num, axis=0, keepdims=True) + 1e-24  # column sum: out of source j
        PDC[:, :, k] = np.sqrt(num / den)

    return {'f': f, 'DTF': DTF, 'PDC': PDC}

def summarize_dtf_pdc_at_harmonics(spec: Dict[str, np.ndarray],
                                   harm: List[float]) -> pd.DataFrame:
    f = spec['f']; DTF = spec['DTF']; PDC = spec['PDC']
    n = DTF.shape[0]
    rows=[]
    for hf in harm:
        idx = int(np.argmin(np.abs(f - hf)))
        for i in range(n):
            for j in range(n):
                rows.append({'freq': float(f[idx]),
                             'target_i': i, 'source_j': j,
                             'DTF': float(DTF[i, j, idx]),
                             'PDC': float(PDC[i, j, idx])})
    return pd.DataFrame(rows)

def run_freq_granger_pdc_dtf(RECORDS: pd.DataFrame,channels: List[str], windows: Optional[List[Tuple[float,float]]] = None,time_col: str = 'Timestamp',order_max: int = 20,crit: str = 'bic',fmin: float = 0.0, fmax: float = 45.0, n_freq: int = 256,harmonics: List[float] = (7.83,14.3,20.8,27.3,33.8),run_granger_tests: bool = True) -> Dict[str, object]:
    """
    Fit VAR, compute DTF/PDC spectra, and report harmonic values.
    Optionally run time-domain Granger tests (F-tests) if statsmodels available.
    """
    fs = infer_fs(RECORDS, time_col)
    X = stack_channels(RECORDS, channels, fs, windows)   # (n_ch, L)
    var = fit_var_model(X, order_max=order_max, crit=crit)

    spec = spectral_dtf_pdc(var['A'], var['Sigma_u'], fs, fmin=fmin, fmax=fmax, n_freq=n_freq)
    table_hp = summarize_dtf_pdc_at_harmonics(spec, list(harmonics))

    gtests = None
    if run_granger_tests and _HAS_SM:
        # test_causality on fitted VAR
        res = var['res']
        pairs=[]
        for i in range(len(channels)):
            for j in range(len(channels)):
                if i==j: continue
                try:
                    # does j cause i?
                    out = res.test_causality(caused=i, causing=[j], kind='f')
                    pairs.append({'target_i':i,'source_j':j,'F':float(out.statistic),'p':float(out.pvalue)})
                except Exception:
                    pairs.append({'target_i':i,'source_j':j,'F':np.nan,'p':np.nan})
        gtests = pd.DataFrame(pairs)

    return {'order': var['order'], 'stable': var['stable'], 'lb_pvals': var['lb_pvals'],'A': var['A'], 'Sigma_u': var['Sigma_u'],'spec': spec, 'harmonics_table': table_hp, 'granger_tests': gtests}

# ------------------ 4b) Transfer Entropy (kNN) ------------------

def _knn_entropy(points: np.ndarray, k: int = 4) -> float:
    """
    Shannon differential entropy via Kozachenko–Leonenko (Euclidean metric).
    H ≈ ψ(n) − ψ(k) + log(c_d) + d * mean(log r_k)
    where r_k is the distance to the k-th NN, c_d = π^{d/2} / Γ(d/2 + 1).
    """
    from sklearn.neighbors import NearestNeighbors
    from scipy.special import gamma as Gamma, digamma

    points = np.asarray(points, float)
    n, dim = points.shape
    if n <= k:
        return np.nan

    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(points)
    dists, _ = nbrs.kneighbors(points)          # (n, k+1) includes self at col 0
    rk = dists[:, -1]                           # distance to k-th neighbor

    c_d = (np.pi ** (dim / 2.0)) / Gamma(dim / 2.0 + 1.0)   # unit-ball volume
    H = digamma(n) - digamma(k) + np.log(c_d) + dim * np.mean(np.log(rk + 1e-24))
    return float(H) + (dim*np.log(2)) - stats.digamma(k) + stats.digamma(n)
    
def transfer_entropy_knn(x: np.ndarray, y: np.ndarray,lag: int,k_embed_x: int = 1, k_embed_y: int = 1,k: int = 4) -> float:
    """
    TE X→Y at a given *positive* lag (samples): predicts y_{t+lag} from [y_t^(k_embed_y), x_t^(k_embed_x)].
    Uses kNN differential entropy approximation (simple, small-sample friendly).
    """
    assert lag > 0
    N = min(len(x), len(y)) - lag
    if N <= max(k_embed_x, k_embed_y):
        return np.nan
    # build delay vectors
    def embed(sig, kdim):
        if kdim <= 0: return None
        X = []
        for d in range(kdim):
            X.append(sig[d:N+d])
        return np.column_stack(X)
    x0 = x[:N]
    y0 = y[:N]
    y_future = y[lag:lag+N]

    Xy = embed(y0, k_embed_y)          # past of Y
    Xx = embed(x0, k_embed_x)          # past of X
    if Xy is None and Xx is None:
        return np.nan
    # assemble joint vectors
    if Xy is None:
        YF_X = np.column_stack([y_future, Xx])
        YF = y_future[:,None]
        H_yf_x = _knn_entropy(YF_X, k=k)
        H_yf   = _knn_entropy(YF, k=k)
        return float(H_yf - H_yf_x)
    if Xx is None:
        YF_Y = np.column_stack([y_future, Xy])
        H_yf_y = _knn_entropy(YF_Y, k=k)
        H_yf   = _knn_entropy(y_future[:,None], k=k)
        return float(H_yf - H_yf_y)
    YF_YX = np.column_stack([y_future, Xy, Xx])
    YF_Y  = np.column_stack([y_future, Xy])
    H_yf_yx = _knn_entropy(YF_YX, k=k)
    H_yf_y  = _knn_entropy(YF_Y,  k=k)
    return float(H_yf_y - H_yf_yx)

def run_transfer_entropy(RECORDS: pd.DataFrame,x_channel: str, y_channel: str,windows: Optional[List[Tuple[float,float]]] = None,time_col: str = 'Timestamp',lags_ms: List[float] = (10, 20, 40, 80, 160, 320),k_embed_x: int = 1, k_embed_y: int = 1,k: int = 4,n_surr: int = 200,rng_seed: int = 13) -> Dict[str, object]:
    """
    Compute TE(X→Y) and TE(Y→X) across lags (ms). Returns arrays and surrogate 95% thresholds.
    """
    fs = infer_fs(RECORDS, time_col)
    x = get_series(RECORDS, x_channel)
    y = get_series(RECORDS, y_channel)
    x = slice_concat(x, fs, windows)
    y = slice_concat(y, fs, windows)

    lags = [max(1, int(round(fs * lm / 1000.0))) for lm in lags_ms]
    te_xy, te_yx = [], []
    for L in lags:
        te_xy.append(transfer_entropy_knn(x, y, lag=L, k_embed_x=k_embed_x, k_embed_y=k_embed_y, k=k))
        te_yx.append(transfer_entropy_knn(y, x, lag=L, k_embed_x=k_embed_y, k_embed_y=k_embed_x, k=k))
    te_xy = np.array(te_xy, float)
    te_yx = np.array(te_yx, float)

    # surrogate null via circular shift of X and Y independently
    rng = np.random.default_rng(rng_seed)
    null_xy = []
    null_yx = []
    n = len(x)
    for _ in range(n_surr):
        sx = int(rng.integers(1, n-1))
        sy = int(rng.integers(1, n-1))
        xs = np.r_[x[-sx:], x[:-sx]]
        ys = np.r_[y[-sy:], y[:-sy]]
        row_xy=[]; row_yx=[]
        for L in lags:
            row_xy.append(transfer_entropy_knn(xs, y, lag=L, k_embed_x=k_embed_x, k_embed_y=k_embed_y, k=k))
            row_yx.append(transfer_entropy_knn(ys, x, lag=L, k_embed_x=k_embed_y, k_embed_y=k_embed_x, k=k))
        null_xy.append(row_xy); null_yx.append(row_yx)
    null_xy = np.array(null_xy, float)
    null_yx = np.array(null_yx, float)
    thr_xy = np.nanpercentile(null_xy, 95, axis=0)
    thr_yx = np.nanpercentile(null_yx, 95, axis=0)

    return {'lags_ms': np.array(lags_ms, float), 'TE_xy': te_xy, 'TE_yx': te_yx,'thr_xy_95': thr_xy, 'thr_yx_95': thr_yx,'null_xy': null_xy, 'null_yx': null_yx}

# ------------------ 4c) Time-varying AR via Kalman-RLS ------------------

def kalman_rls_tvar_ar(X: np.ndarray, order: int = 4, lam: float = 0.995) -> Dict[str, object]:
    """
    Track time-varying AR coefficients for a multivariate series X (n_ch, L)
    using recursive least squares with forgetting factor lam.
    State vector stacks AR mats row-wise: for n_ch and order p, size = n_ch*n_ch*p.
    Returns coeffs[t] shaped (p, n_ch, n_ch).
    """
    n, L = X.shape
    p = order
    d = n*n*p
    theta = np.zeros(d)                     # initial coeff vector
    P = np.eye(d) * 1e3                     # large initial covariance
    coeffs = []

    def phi_t(t_idx):
        # design vector for y_t = sum_k A_k y_{t-k}
        rows=[]
        for k in range(1, p+1):
            rows.append(X[:, t_idx-k])      # shape (n,)
        Ypast = np.concatenate(rows, axis=0)  # (n*p,)
        # Build block-diag kron for all rows (for each output dim)
        Phi = np.zeros((n, d))
        # For output i, its row params live at offsets
        for i in range(n):
            # coefficients for output i are contiguous blocks of length n across lags
            # index mapping: offset = i + n*j + n*n*(k-1) over (k,j)
            col = 0
            for kk in range(p):
                for j in range(n):
                    idx = i + j*n + kk*n*n   # row-major per-lag
                    Phi[i, idx] = Ypast[kk*n + j]
                    col += 1
        return Phi

    for t in range(p, L):
        Phi = phi_t(t)               # (n, d)
        y  = X[:, t]                 # (n,)
        # RLS update for each output eq combined (matrix form)
        # flatten to a big observation by stacking rows
        H = Phi                      # (n, d)
        R = np.eye(n) * 1e-3
        # predict
        P = P / lam
        # Kalman gain
        S = H @ P @ H.T + R
        K = (P @ H.T) @ np.linalg.pinv(S)
        # residual
        y_hat = H @ theta
        err = y - y_hat
        # update
        theta = theta + K @ err
        P = (np.eye(d) - K @ H) @ P
        # store reshaped coeffs at time t
        A_t = np.zeros((p, n, n))
        for kk in range(p):
            for j in range(n):
                for i in range(n):
                    idx = i + j*n + kk*n*n
                    A_t[kk, i, j] = theta[idx]
        coeffs.append(A_t)
    coeffs = np.array(coeffs)  # (L-p, p, n, n)
    return {'A_t': coeffs, 'order': p}

def dtf_at_freq_from_A(A: np.ndarray, fs: float, f0: float) -> np.ndarray:
    """
    DTF at a single frequency f0 from AR matrices A (p,n,n).
    """
    f = np.array([f0])
    Af = _A_of_f(A, f, fs)            # (1, n, n)
    Hf = _H_of_f(Af)                  # (1, n, n)
    H = Hf[0]
    num = np.abs(H)**2
    den = np.sum(num, axis=1, keepdims=True) + 1e-24
    return np.sqrt(num/den)           # (n, n)

def run_tvar_dtf(RECORDS: pd.DataFrame,channels: List[str],windows: Optional[List[Tuple[float,float]]] = None,time_col: str = 'Timestamp',order: int = 4, lam: float = 0.995,f0: float = 7.83) -> Dict[str, object]:

    """
    Time-varying AR (Kalman-RLS) and DTF(t, i<-j) at f0.
    Returns dict with A_t (T,p,n,n) and DTF_t (T,n,n).
    """
    fs = infer_fs(RECORDS, time_col)
    X = stack_channels(RECORDS, channels, fs, windows)  # (n, L)
    tv = kalman_rls_tvar_ar(X, order=order, lam=lam)
    A_t = tv['A_t']                      # (T, p, n, n)
    T, p, n, _ = A_t.shape
    D = np.zeros((T, n, n))
    for t in range(T):
        D[t] = dtf_at_freq_from_A(A_t[t], fs, f0)
    return {'A_t': A_t, 'DTF_t': D, 'fs': fs, 'f0': f0, 'channels': channels}

# ------------------ usage examples ------------------
if __name__ == "__main__":
    # 4a) VAR / PDC / DTF
    # res = run_freq_granger_pdc_dtf(RECORDS,
    #         channels=['EEG.Oz','EEG.O1','EEG.O2'],  # include Schumann ref if available
    #         windows=[(290,310),(580,600)],
    #         time_col='Timestamp',
    #         order_max=20, crit='bic',
    #         fmin=0.0, fmax=45.0, n_freq=256,
    #         harmonics=[7.83,14.3,20.8,27.3,33.8],
    #         run_granger_tests=True)
    # print('VAR order:', res['order'], 'stable:', res['stable'], 'LB pvals:', res['lb_pvals'])
    # print(res['harmonics_table'].head())
    # if res['granger_tests'] is not None: print(res['granger_tests'])

    # 4b) Transfer Entropy
    # te = run_transfer_entropy(RECORDS, x_channel='EEG.Oz', y_channel='EEG.O1',
    #         windows=[(290,310),(580,600)], time_col='Timestamp',
    #         lags_ms=[10,20,40,80,160,320], k_embed_x=1, k_embed_y=1, k=4, n_surr=200)
    # print('TE X->Y:', te['TE_xy'], 'thr95:', te['thr_xy_95'])
    # print('TE Y->X:', te['TE_yx'], 'thr95:', te['thr_yx_95'])

    # 4c) Time-varying DTF at 7.83 Hz
    # tv = run_tvar_dtf(RECORDS, channels=['EEG.Oz','EEG.O1'],
    #         windows=[(290,310),(580,600)], time_col='Timestamp',
    #         order=4, lam=0.995, f0=7.83)
    # print('DTF_t shape:', tv['DTF_t'].shape)
    pass


In [ ]:
# 4a) VAR / PDC / DTF
res = run_freq_granger_pdc_dtf(RECORDS,
        channels=['EEG.F4','EEG.O1','EEG.O2', 'EEG.T8'],  # include Schumann ref if available
        windows=[(290,310),(580,600)],
        time_col='Timestamp',
        order_max=2, crit='aic',
        fmin=0.0, fmax=45.0, n_freq=256,
        harmonics=[7.83,14.3,20.8,27.3,33.8],
        run_granger_tests=True)
print('VAR order:', res['order'], 'stable:', res['stable'], 'LB pvals:', res['lb_pvals'])
print(res['harmonics_table'].head())
if res['granger_tests'] is not None: print(res['granger_tests'])

In [ ]:
# ===== Transfer Entropy helpers (drop-in replacement) =====
import numpy as np
from typing import Optional
from scipy.special import gamma as Gamma, digamma
from sklearn.neighbors import NearestNeighbors

def _knn_entropy(points: np.ndarray, k: int = 4) -> float:
    """
    Differential entropy via Kozachenko–Leonenko (Euclidean).
    H ≈ ψ(n) − ψ(k) + log(c_d) + d * mean(log r_k),
    where r_k is distance to the k-th NN; c_d = π^{d/2} / Γ(d/2 + 1).
    """
    P = np.asarray(points, float)
    if P.ndim == 1:
        P = P[:, None]
    n, d = P.shape
    if n <= k or d < 1:
        return np.nan

    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(P)
    dists, _ = nbrs.kneighbors(P)           # includes self at [:,0]
    rk = dists[:, -1]                       # k-th neighbor distance

    c_d = (np.pi ** (d / 2.0)) / Gamma(d / 2.0 + 1.0)  # unit-ball volume
    H = digamma(n) - digamma(k) + np.log(c_d) + d * np.mean(np.log(rk + 1e-24))
    return float(H)

def _embed(sig: np.ndarray, kdim: int) -> Optional[np.ndarray]:
    """Simple delay embedding with unit delay: [x_t, x_{t+1}, ..., x_{t+kdim-1}] aligned to t."""
    if kdim <= 0:
        return None
    N = sig.size - (kdim - 1)
    if N <= 0:
        return None
    cols = [sig[i:i+N] for i in range(kdim)]
    return np.column_stack(cols)

def transfer_entropy_knn(x: np.ndarray, y: np.ndarray,
                         lag: int,
                         k_embed_x: int = 1, k_embed_y: int = 1,
                         k: int = 4) -> float:
    """
    TE X→Y at positive sample lag.
    TE = H(Y_{t+lag}, Y_t^(k)) - H(Y_t^(k)) - H(Y_{t+lag}, Y_t^(k), X_t^(l)) + H(Y_t^(k), X_t^(l))
    """
    assert lag > 0
    L = min(x.size, y.size)
    # Align so that future y is available
    y_future = y[lag:L]
    y_past = y[:L-lag]
    x_past = x[:L-lag]

    Yp = _embed(y_past, k_embed_y)    # (N, k_y) or None
    Xp = _embed(x_past, k_embed_x)    # (N, k_x) or None
    if Yp is None and Xp is None:
        return np.nan

    # Trim y_future to match embedded rows
    N = None
    if Yp is not None:
        N = Yp.shape[0]
    if Xp is not None:
        N = Xp.shape[0] if N is None else min(N, Xp.shape[0])
    if N is None or N <= k+1:
        return np.nan
    yF = y_future[:N, None]
    if Yp is not None: Yp = Yp[:N, :]
    if Xp is not None: Xp = Xp[:N, :]

    # Build joint vectors
    if Yp is None:    # TE reduces to H(yF) - H(yF, Xp)
        H1 = _knn_entropy(yF, k=k)
        H2 = _knn_entropy(np.column_stack([yF, Xp]), k=k)
        return float(H1 - H2)
    if Xp is None:    # TE reduces to H(yF, Yp) - H(Yp) - [H(yF, Yp) - H(Yp)] = 0
        H_yF_Yp = _knn_entropy(np.column_stack([yF, Yp]), k=k)
        H_Yp    = _knn_entropy(Yp, k=k)
        return float((H_yF_Yp - H_Yp) - (H_yF_Yp - H_Yp))

    H_yF_Yp     = _knn_entropy(np.column_stack([yF, Yp]), k=k)
    H_Yp        = _knn_entropy(Yp, k=k)
    H_yF_Yp_Xp  = _knn_entropy(np.column_stack([yF, Yp, Xp]), k=k)
    H_Yp_Xp     = _knn_entropy(np.column_stack([Yp, Xp]), k=k)

    TE = (H_yF_Yp - H_Yp) - (H_yF_Yp_Xp - H_Yp_Xp)
    return float(TE)


In [ ]:
# 4b) Transfer Entropy
te = run_transfer_entropy(RECORDS, x_channel='EEG.F4', y_channel='EEG.O1',
        windows=[(290,310),(580,600)], time_col='Timestamp',
        lags_ms=[10,20,40,80,160,320], k_embed_x=1, k_embed_y=1, k=4, n_surr=200)
print('TE X->Y:', te['TE_xy'], 'thr95:', te['thr_xy_95'])
print('TE Y->X:', te['TE_yx'], 'thr95:', te['thr_yx_95'])


In [ ]:
# 4c) Time-varying DTF at 7.83 Hz
tv = run_tvar_dtf(RECORDS, channels=['EEG.F4','EEG.O2'],
        windows=[(290,310),(580,600)], time_col='Timestamp',
        order=4, lam=0.995, f0=7.83)
print('DTF_t shape:', tv['DTF_t'].shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example: j=0 → i=1 (channels[0] -> channels[1])
i, j = 1, 0
dtf_ij = tv['DTF_t'][:, i, j]
dtf_ji = tv['DTF_t'][:, j, i]
di = dtf_ij - dtf_ji

# light smoothing (e.g., 0.5 s)
win = max(1, int(round(0.5 * tv['fs'])))
w = np.ones(win)/win
dtf_ij_s = np.convolve(dtf_ij, w, mode='same')
dtf_ji_s = np.convolve(dtf_ji, w, mode='same')
di_s     = np.convolve(di,     w, mode='same')

plt.figure(figsize=(9,3))
plt.plot(dtf_ij_s, label=f"{tv['channels'][j]}→{tv['channels'][i]} @ {tv['f0']} Hz")
plt.plot(dtf_ji_s, label=f"{tv['channels'][i]}→{tv['channels'][j]}")
plt.plot(di_s,     label='Directionality index (Δ)')
plt.axhline(0, color='k', lw=0.8); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
def mean_in_windows(x, fs, windows):
    if not windows: return np.nan
    out=[]
    for (t0,t1) in windows:
        s,e = int(t0*fs), int(t1*fs)
        s = max(0, min(len(x)-1, s)); e = max(s+1, min(len(x), e))
        out.append(np.nanmean(x[s:e]))
    return float(np.nanmean(out)) if out else np.nan

ign_wins = [(290,310),(580,600)]
# baseline = complement of ignition (quick and dirty): everything else in the analyzed segment
# or use your own control windows
base_mean = np.nanmean(dtf_ij_s)  # replace with explicit control windows if you have them
ign_mean  = mean_in_windows(dtf_ij_s, tv['fs'], ign_wins)

print("DTF mean ignition:", ign_mean, "baseline:", base_mean, "Δ:", ign_mean - base_mean)


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
n_perm = 500
null = []
for _ in range(n_perm):
    # random circular shift of one channel's time course before TV-AR (quick proxy):
    # For a more correct null, rerun run_tvar_dtf on shifted data; this quick version
    # shifts the already-computed DTF, which is conservative for testing a mean diff.
    s = int(rng.integers(1, len(dtf_ij_s)-1))
    dtf_ij_perm = np.r_[dtf_ij_s[-s:], dtf_ij_s[:-s]]
    ign = mean_in_windows(dtf_ij_perm, tv['fs'], ign_wins)
    null.append(ign - base_mean)
thr95 = np.nanpercentile(null, 95)
print("Δ threshold (95% null):", thr95)


In [ ]:
"""
DTF grid + text report
----------------------
Create a grid of DTF(t, target <- source) plots for ALL electrodes (targets),
and print a text table with:
  • mean DTF in ignition windows
  • Δ = mean_ign − mean_base
  • Δ threshold (95% null) from a simple permutation (circular shift) null

USAGE
=====
# 1) Compute time-varying DTF separately for ignition and baseline:
tv_ign = run_tvar_dtf(
    RECORDS,
    channels=['EEG.Oz','EEG.O1','EEG.O2'],  # include your chosen source channel too
    windows=[(290,310),(580,600)],
    time_col='Timestamp',
    order=4, lam=0.995, f0=7.83
)
tv_base = run_tvar_dtf(
    RECORDS,
    channels=['EEG.Oz','EEG.O1','EEG.O2'],
    windows=[(0, 280), (325, 575)],         # supply your baseline windows explicitly
    time_col='Timestamp',
    order=4, lam=0.995, f0=7.83
)

# 2) Make the grid and printed report for source='EEG.Oz' (or whichever)
plot_dtf_grid_and_report(
    tv_ign, tv_base,
    src_channel='EEG.Oz',
    smooth_sec=0.5,
    n_cols=3,
    session_name='session_A'
)
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Optional

def _moving_average(x: np.ndarray, win: int) -> np.ndarray:
    if win <= 1: return x
    w = np.ones(win)/win
    return np.convolve(x, w, mode='same')

def _mean_in_seconds(x: np.ndarray, fs: float, windows: Optional[list]) -> float:
    """
    Mean over a set of windows defined in SECONDS relative to x's indexing (0..T/fs).
    If windows is None, simply return the mean over the whole series.
    """
    if not windows:
        return float(np.nanmean(x))
    vals = []
    T = len(x)
    for (t0, t1) in windows:
        s = int(round(t0 * fs)); e = int(round(t1 * fs))
        s = max(0, min(T-1, s)); e = max(s+1, min(T, e))
        vals.append(np.nanmean(x[s:e]))
    return float(np.nanmean(vals)) if vals else np.nan

def plot_dtf_grid_and_report(tv_ign: Dict[str, object],
                             tv_base: Dict[str, object],
                             src_channel: str,
                             smooth_sec: float = 0.5,
                             n_cols: int = 4,
                             ign_windows_sec: Optional[list] = None,
                             base_windows_sec: Optional[list] = None,
                             n_perm: int = 500,
                             rng_seed: int = 7,
                             session_name: Optional[str] = None) -> None:
    """
    Grid of DTF(t, target <- source) for all targets (all electrodes except source).
    For each target:
      - plot smoothed DTF time series
      - compute mean ignition (over ign_windows_sec or whole ign series)
      - compute baseline mean (over base_windows_sec or whole base series)
      - permutation null on Δ = mean_ign − mean_base  -> threshold (95%)
    Print a text summary of mean_ign and Δ threshold.

    tv_ign / tv_base: dicts from run_tvar_dtf; must be computed with the SAME 'channels' ordering.
    src_channel: e.g., 'EEG.Oz' — must be present in tv_ign['channels'].
    """
    # --- extract metadata & align ---
    chs = tv_ign['channels']
    assert chs == tv_base['channels'], "Channel lists differ between tv_ign and tv_base."
    if src_channel not in chs:
        # allow bare names like 'Oz'
        if src_channel.startswith('EEG.'):
            alt = src_channel.split('.',1)[1]
        else:
            alt = 'EEG.' + src_channel
        if alt in chs:
            src_channel = alt
        else:
            raise ValueError(f"Source channel '{src_channel}' not found in {chs}")

    src_idx = chs.index(src_channel)
    fs = float(tv_ign['fs'])
    f0 = float(tv_ign['f0'])
    D_ign = np.asarray(tv_ign['DTF_t'])    # (T_ign, n, n)
    D_base = np.asarray(tv_base['DTF_t'])  # (T_base, n, n)

    # smoothing
    win = max(1, int(round(smooth_sec * fs)))
    # generate subplot grid
    targets = [i for i in range(len(chs)) if i != src_idx]
    n_t = len(targets)
    n_cols = max(1, n_cols)
    n_rows = int(np.ceil(n_t / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 2.8*n_rows), squeeze=False)
    axes = axes.ravel()

    rng = np.random.default_rng(rng_seed)
    text_lines = []
    for idx, tgt in enumerate(targets):
        ax = axes[idx]
        # DTF series for source->target in each state
        dtf_ign = D_ign[:, tgt, src_idx]
        dtf_base = D_base[:, tgt, src_idx]
        # smooth
        dtf_ign_s = _moving_average(dtf_ign, win)
        dtf_base_s = _moving_average(dtf_base, win)

        # mean in windows (if provided) else whole
        mean_ign = _mean_in_seconds(dtf_ign_s, fs, ign_windows_sec)
        mean_base = _mean_in_seconds(dtf_base_s, fs, base_windows_sec)
        delta_obs = mean_ign - mean_base

        # permutation null for Δ: random circular shift of ign time series (conservative)
        null = []
        T = len(dtf_ign_s)
        for _ in range(n_perm):
            s = int(rng.integers(1, max(2, T-1)))  # 1..T-1
            perm_ign = np.r_[dtf_ign_s[-s:], dtf_ign_s[:-s]]
            perm_mean_ign = _mean_in_seconds(perm_ign, fs, ign_windows_sec)
            null.append(perm_mean_ign - mean_base)
        thr95 = float(np.nanpercentile(null, 95))

        # plot
        ax.plot(dtf_ign_s, label=f'Ignition', color='tab:blue', lw=1.2)
        ax.plot(np.linspace(0, len(dtf_base_s)-1, len(dtf_base_s)), dtf_base_s,
                label='Baseline', color='tab:orange', lw=1.0, alpha=0.9)
        ax.axhline(mean_base, color='tab:orange', ls='--', lw=1.0, alpha=0.8)

        # annotate text on plot
        ch_name = chs[tgt]
        ax.set_title(f"{ch_name}  ({src_channel}→{ch_name}) @ {f0:.2f} Hz")
        ax.set_xlabel('Samples (after AR warm-up)')
        ax.set_ylabel('DTF')
        ax.set_ylim(0, 1.0)
        ax.grid(alpha=0.25)
        ax.legend(loc='upper right', fontsize=8)
        ax.text(0.01, 0.02,
                f"mean_ign={mean_ign:.3f}\nΔ={delta_obs:.3f}\nthr95={thr95:.3f}\nSig={delta_obs>thr95}",
                transform=ax.transAxes, fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.6, lw=0.5))

        # collect text line
        text_lines.append(f"{ch_name}: mean_ign={mean_ign:.4f}, Δ={delta_obs:.4f}, thr95={thr95:.4f}, Sig={delta_obs>thr95}")

    # hide extra axes (if any)
    for k in range(n_t, n_rows*n_cols):
        fig.delaxes(axes[k])

    supt = f"DTF grid: {src_channel}→targets @ {f0:.2f} Hz"
    if session_name:
        supt = f"{session_name} — " + supt
    fig.suptitle(supt, y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

    # print text summary
    print("\n=== DTF mean ignition and Δ thresholds ===")
    for line in text_lines:
        print(line)

    # optional: return the summary as a list/dict if you want to save it
    # return text_lines


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Optional, List, Tuple

def _moving_average(x: np.ndarray, win: int) -> np.ndarray:
    if win <= 1: return x
    w = np.ones(win) / win
    return np.convolve(x, w, mode='same')

def _mean_in_seconds(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> float:
    if not windows:
        return float(np.nanmean(x))
    vals = []
    T = len(x)
    for (t0, t1) in windows:
        s = int(round(t0 * fs)); e = int(round(t1 * fs))
        s = max(0, min(T-1, s)); e = max(s+1, min(T, e))
        vals.append(np.nanmean(x[s:e]))
    return float(np.nanmean(vals)) if vals else np.nan

def plot_dtf_grid_bidir_like_single(tv_ign: Dict[str, object],
                                    tv_base: Optional[Dict[str, object]],
                                    src_channel: str,
                                    smooth_sec: float = 0.5,
                                    n_cols: int = 3,
                                    ign_windows_sec: Optional[List[Tuple[float,float]]] = None,
                                    base_windows_sec: Optional[List[Tuple[float,float]]] = None,
                                    show_baseline: bool = False,
                                    n_perm: int = 500,
                                    rng_seed: int = 7,
                                    session_name: Optional[str] = None) -> None:
    """
    Make a grid of panels that match the earlier single-link style:
      • blue:   DTF(source → target) in ignition
      • orange: DTF(target → source) in ignition
      • green:  Directionality Index Δ = (src→tgt) − (tgt→src)
    Optionally overlay faint baseline series for context.
    Text box per panel: mean_ign (src→tgt), Δ, thr95 (permutation), Sig flag.

    tv_ign, tv_base: outputs from run_tvar_dtf (must share identical 'channels').
    src_channel: e.g., 'EEG.F4' (must be in tv_ign['channels']).
    """
    chs = tv_ign['channels']
    if src_channel not in chs:
        # allow bare names
        src_channel = ('EEG.' + src_channel) if ('EEG.' + src_channel) in chs else src_channel
    assert src_channel in chs, f"{src_channel} not found in {chs}"
    if tv_base is not None:
        assert chs == tv_base['channels'], "Channel lists differ between tv_ign and tv_base."

    src_idx = chs.index(src_channel)
    fs  = float(tv_ign['fs'])
    f0  = float(tv_ign['f0'])
    D_I = np.asarray(tv_ign['DTF_t'])    # (T_ign, n, n)
    D_B = np.asarray(tv_base['DTF_t']) if tv_base is not None else None

    # smoothing window
    win = max(1, int(round(smooth_sec * fs)))

    # targets = all except source
    targets = [i for i in range(len(chs)) if i != src_idx]
    n_t = len(targets)
    n_cols = max(1, n_cols)
    n_rows = int(np.ceil(n_t / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8*n_cols, 2.4*n_rows), squeeze=False)
    axes = axes.ravel()

    rng = np.random.default_rng(rng_seed)
    text_lines = []

    for m, tgt in enumerate(targets):
        ax = axes[m]

        # ignition: src→tgt and tgt→src
        ij_I = _moving_average(D_I[:, tgt, src_idx], win)
        ji_I = _moving_average(D_I[:, src_idx, tgt], win)
        di_I = ij_I - ji_I

        # baseline (optional overlay)
        if show_baseline and D_B is not None:
            ij_B = _moving_average(D_B[:, tgt, src_idx], win)
            ji_B = _moving_average(D_B[:, src_idx, tgt], win)
        else:
            ij_B = ji_B = None

        # means for stats (per windows if provided)
        mean_ign = _mean_in_seconds(ij_I, fs, ign_windows_sec)
        if D_B is not None:
            mean_base = _mean_in_seconds(ij_B if ij_B is not None else D_B[:, tgt, src_idx], fs, base_windows_sec)
        else:
            mean_base = np.nan
        delta_obs = mean_ign - mean_base if np.isfinite(mean_base) else np.nan

        # permutation null for Δ: circularly shift ignition ij series
        thr95 = np.nan
        if np.isfinite(delta_obs):
            null = []
            T = len(ij_I)
            for _ in range(n_perm):
                s = int(rng.integers(1, max(2, T-1)))
                ij_perm = np.r_[ij_I[-s:], ij_I[:-s]]
                perm_mean_ign = _mean_in_seconds(ij_perm, fs, ign_windows_sec)
                null.append(perm_mean_ign - mean_base)
            thr95 = float(np.nanpercentile(null, 95))

        # ---- plotting (match single-link style) ----
        ax.plot(ij_I, color='tab:blue',  lw=1.6, label=f'{src_channel}→{chs[tgt]} @ {f0:.2f} Hz')
        ax.plot(ji_I, color='tab:orange', lw=1.2, label=f'{chs[tgt]}→{src_channel}')
        ax.plot(di_I, color='tab:green', lw=1.2, label='Directionality Index (Δ)')

        if show_baseline and D_B is not None:
            ax.plot(np.linspace(0, len(D_B)-1, len(D_B)), ij_B, color='tab:blue',  alpha=0.25, lw=0.8)
            ax.plot(np.linspace(0, len(D_B)-1, len(D_B)), ji_B, color='tab:orange', alpha=0.25, lw=0.8)

#         ax.set_ylim(-1.0, 1.0)
        # dynamic limits that include src→tgt, tgt→src, and Δ, with a small pad
        ymin = float(np.nanmin([ij_I.min(), ji_I.min(), di_I.min()]))
        ymax = float(np.nanmax([ij_I.max(), ji_I.max(), di_I.max()]))

        pad  = 0.05 * (ymax - ymin + 1e-9)
        lo   = max(ymin - pad, -0.2)          # don’t let it go crazy low
        hi   = min(max(ymax + pad, 0.2), 1.05)  # cap near 1

        ax.set_ylim(lo, hi)
        ax.grid(alpha=0.25)
        ax.set_title(f"{chs[tgt]}  ({src_channel}↔{chs[tgt]}) @ {f0:.2f} Hz", fontsize=10)
        ax.set_xlabel('Samples (after AR warm-up)')
        ax.set_ylabel('DTF')
        ax.legend(loc='upper right', fontsize=8, ncol=1, framealpha=0.8)

        # text box
        txt = [f"mean_ign={mean_ign:.3f}"]
        if np.isfinite(delta_obs):
            txt.append(f"Δ={delta_obs:.3f}")
            txt.append(f"thr95={thr95:.3f}")
            txt.append(f"Sig={delta_obs>thr95}")
        ax.text(0.01, 0.02, "\n".join(txt), transform=ax.transAxes, fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, lw=0.5))

        text_lines.append(f"{chs[tgt]}: mean_ign={mean_ign:.4f}" +
                          (f", Δ={delta_obs:.4f}, thr95={thr95:.4f}, Sig={delta_obs>thr95}"
                           if np.isfinite(delta_obs) else ""))

    # drop unused axes
    for k in range(n_t, n_rows*n_cols):
        fig.delaxes(axes[k])

    supt = f"DTF grid: {src_channel}↔targets @ {f0:.2f} Hz"
    if session_name: supt = f"{session_name} — " + supt
    fig.suptitle(supt, y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

    print("\n=== DTF mean ignition and Δ thresholds ===")
    for line in text_lines:
        print(line)


In [ ]:
# 1) Compute TV-DTF for ignition & baseline (same channels & params)
tv_ign = run_tvar_dtf(RECORDS, channels=['EEG.F4','EEG.F3','EEG.AF3','EEG.AF4','EEG.FC5','EEG.FC6','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2'],
                      windows=[(290,310),(580,600)], time_col='Timestamp', order=4, lam=0.995, f0=7.83)
tv_base = run_tvar_dtf(RECORDS, channels=tv_ign['channels'],
                       windows=[(0,290),(325,580)], time_col='Timestamp', order=4, lam=0.995, f0=7.83)

# 2) Plot the bidirectional grid styled like the single-link figure
plot_dtf_grid_bidir_like_single(tv_ign, tv_base,
                                src_channel='EEG.F4',
                                smooth_sec=0.6,
                                n_cols=3,
                                show_baseline=False,     # set True to overlay faint baseline
                                n_perm=500,
                                session_name='session_A')


In [ ]:
"""
Network-level coupling — simple validity tests & graphs
======================================================

Implements two quick, self-contained analyses:

5a) Cross-domain graph alignment
    • Build EEG PLV graph in a chosen band (e.g., alpha).
    • Compute per-electrode Schumann coherence at harmonics (MSC by Welch).
    • Make a Schumann-weighted PLV graph:
          A_weighted[i,j] = PLV[i,j] * sqrt( MSC_i * MSC_j )
      (MSC_i is the mean MSC of channel i across the harmonics.)
    • Compare graph Laplacian entropy and global min-cut with/without Schumann weighting.
    • Plots: adjacency heatmaps, Δ edge histogram, bar chart of entropy/min-cut.

5b) Source-space (ROI) mapping (sensor ROI proxy)
    • Define conservative ROIs (occipital/parietal/frontal/temporal groups).
    • Build ROI time series (mean across available sensors) and optionally
      apply symmetric orthogonalization (leakage reduction).
    • For each ROI, compute:
        - PLV with Schumann narrowband (~7.83 Hz ± half_bw)
        - MSC with Schumann across harmonics (Welch)
      with circular-shift surrogates → p-values.
    • Plots: bar charts with 95% surrogate bands; table of stats.

Assumptions:
- RECORDS: pandas.DataFrame with a time column (default 'Timestamp')
  and EEG/sensor columns like 'EEG.O1', 'EEG.O2', ...
- sr_channel: a Schumann/ELF reference; if none, use a clean posterior EEG.

Copy-paste this module, then see the usage examples at the bottom.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from typing import Dict, List, Tuple, Optional
from scipy import signal

# ----------------------------- generic helpers -----------------------------

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0:
        raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    """Return a numeric signal array. Accepts 'EEG.O1' or bare 'O1'."""
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.' + name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found in RECORDS.")

def slice_concat(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> np.ndarray:
    if not windows: return x.copy()
    segs=[]; n=len(x)
    for (t0,t1) in windows:
        i0,i1 = int(round(t0*fs)), int(round(t1*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order: int = 4) -> np.ndarray:
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny)); f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

# ----------------------------- PLV & MSC -----------------------------------

def plv_matrix(RECORDS: pd.DataFrame,
               channels: List[str],
               band: Tuple[float,float],
               windows: Optional[List[Tuple[float,float]]] = None,
               time_col: str = 'Timestamp') -> np.ndarray:
    """
    Pairwise PLV matrix (N×N) within 'band'. Uses analytic phases from Hilbert
    after band-pass. Windows are concatenated before PLV.
    """
    fs = infer_fs(RECORDS, time_col)
    X = []
    for ch in channels:
        x = get_series(RECORDS, ch)
        x = slice_concat(x, fs, windows)
        xb = bandpass(x, fs, band[0], band[1])
        X.append(np.angle(signal.hilbert(xb)))
    X = np.vstack(X)  # (N, T)
    N = len(channels)
    PLV = np.zeros((N,N))
    for i in range(N):
        for j in range(i, N):
            dphi = X[i]-X[j]
            v = np.abs(np.mean(np.exp(1j*dphi)))
            PLV[i,j]=PLV[j,i]=float(v)
    np.fill_diagonal(PLV, 0.0)
    return PLV

def msc_vs_sr(RECORDS: pd.DataFrame,
              channels: List[str], sr_channel: str,
              windows: Optional[List[Tuple[float,float]]] = None,
              time_col: str = 'Timestamp',
              nperseg: Optional[int] = None, noverlap: Optional[int] = None,
              harmonics: List[float] = (7.83,14.3,20.8,27.3,33.8)) -> pd.DataFrame:
    """
    Per-channel magnitude-squared coherence with SR at harmonics
    (Welch MSC; simple & fast). Returns DataFrame:
      ['channel','MSC_mean','MSC_at_<freq>']
    """
    fs = infer_fs(RECORDS, time_col)
    if nperseg is None:
        nperseg = int(4*fs)
    if noverlap is None:
        noverlap = int(0.5*nperseg)
    y = get_series(RECORDS, sr_channel)
    y = slice_concat(y, fs, windows)

    rows=[]
    for ch in channels:
        x = get_series(RECORDS, ch)
        x = slice_concat(x, fs, windows)
        f, Cxy = signal.coherence(x, y, fs=fs, nperseg=nperseg, noverlap=noverlap)
        row = {'channel': ch}
        vals=[]
        for hf in harmonics:
            idx = int(np.argmin(np.abs(f - hf)))
            row[f"MSC_{hf:.2f}"] = float(Cxy[idx])
            vals.append(float(Cxy[idx]))
        row['MSC_mean'] = float(np.mean(vals))
        rows.append(row)
    return pd.DataFrame(rows)

# --------------------- Graph metrics: entropy & min-cut ---------------------

def laplacian_entropy(adj: np.ndarray) -> float:
    """Shannon entropy of positive Laplacian eigenvalues (normalized)."""
    deg = np.sum(adj, axis=1)
    L = np.diag(deg) - adj
    L = 0.5*(L+L.T)
    vals = np.linalg.eigvalsh(L)
    vals = vals[vals > 1e-12]
    if vals.size == 0: return np.nan
    p = vals / np.sum(vals)
    return float(-np.sum(p*np.log(p)))

def global_mincut(adj: np.ndarray) -> float:
    """Stoer–Wagner global min-cut on weighted undirected graph."""
    G = nx.from_numpy_array(adj)
    if G.number_of_edges()==0: return 0.0
    try:
        cut_val, _ = nx.stoer_wagner(G)
        return float(cut_val)
    except Exception:
        return float('nan')

# ---------------------- 5a) Cross-domain graph alignment --------------------

def cross_domain_graph_alignment(RECORDS: pd.DataFrame,
                                 eeg_channels: List[str],
                                 sr_channel: str,
                                 band: Tuple[float,float] = (8,13),
                                 harmonics: List[float] = (7.83,14.3,20.8,27.3,33.8),
                                 windows: Optional[List[Tuple[float,float]]] = None,
                                 time_col: str = 'Timestamp') -> Dict[str, object]:
    """
    Build EEG PLV graph in 'band'. Weight edges by geometric mean of the nodes'
    MSC with SR at the given harmonics. Compare entropy & min-cut.
    Plots heatmaps and summary bars.
    """
    # PLV matrix
    PLV = plv_matrix(RECORDS, eeg_channels, band, windows, time_col)
    # per-node MSC mean across harmonics
    msc_tbl = msc_vs_sr(RECORDS, eeg_channels, sr_channel, windows, time_col)
    node_msc = {row['channel']: row['MSC_mean'] for _,row in msc_tbl.iterrows()}
    # Weighted edge factor: sqrt(msc_i * msc_j)
    N = len(eeg_channels)
    W = np.zeros((N,N))
    for i,ch_i in enumerate(eeg_channels):
        for j,ch_j in enumerate(eeg_channels):
            if i==j: continue
            fct = np.sqrt(max(0.0, node_msc[ch_i]) * max(0.0, node_msc[ch_j]))
            W[i,j] = fct
    A_plain = PLV.copy()
    A_weight = PLV * W

    # Graph metrics
    ent_plain  = laplacian_entropy(A_plain)
    ent_weight = laplacian_entropy(A_weight)
    cut_plain  = global_mincut(A_plain)
    cut_weight = global_mincut(A_weight)

    # ----- Plots -----
    fig, axs = plt.subplots(1,2, figsize=(10,4))
    im0 = axs[0].imshow(A_plain, vmin=0, vmax=1, cmap='viridis')
    axs[0].set_title(f'PLV adjacency ({band[0]}–{band[1]} Hz)')
    plt.colorbar(im0, ax=axs[0], fraction=0.046)
    im1 = axs[1].imshow(A_weight, vmin=0, vmax=np.nanmax(A_weight)+1e-9, cmap='viridis')
    axs[1].set_title('Schumann-weighted PLV')
    plt.colorbar(im1, ax=axs[1], fraction=0.046)
    for ax in axs:
        ax.set_xticks(range(N)); ax.set_yticks(range(N))
        ax.set_xticklabels([c.split('.',1)[-1] for c in eeg_channels], rotation=90, fontsize=8)
        ax.set_yticklabels([c.split('.',1)[-1] for c in eeg_channels], fontsize=8)
    plt.tight_layout(); plt.show()

    # Δ edge histogram
    dA = A_weight - A_plain
    plt.figure(figsize=(6,3))
    plt.hist(dA[np.triu_indices(N,1)].ravel(), bins=30, color='tab:blue', alpha=0.8)
    plt.xlabel('Δ edge weight (weighted − plain)'); plt.ylabel('count')
    plt.title('Edge weight changes due to Schumann weighting'); plt.tight_layout(); plt.show()

    # Summary bars
    labels = ['Entropy','Min-cut']
    vals_plain  = [ent_plain, cut_plain]
    vals_weight = [ent_weight, cut_weight]
    x = np.arange(2); w = 0.38
    plt.figure(figsize=(6,3))
    plt.bar(x-w/2, vals_plain,  width=w, label='Plain')
    plt.bar(x+w/2, vals_weight, width=w, label='Weighted')
    plt.xticks(x, labels); plt.title('Graph metrics'); plt.legend(); plt.tight_layout(); plt.show()

    summary = pd.DataFrame([{
        'entropy_plain': ent_plain, 'entropy_weighted': ent_weight,
        'mincut_plain': cut_plain, 'mincut_weighted': cut_weight
    }])
    return {'A_plain':A_plain, 'A_weight':A_weight, 'node_msc':msc_tbl, 'summary':summary}

# ---------------------- 5b) Source-space ROI mapping (sensor proxy) ---------

def symmetric_orthogonalize(ts: np.ndarray) -> np.ndarray:
    """
    Symmetric orthogonalization (Colclough et al., 2015) for leakage reduction.
    Input ts: (n_roi, T); output has orthonormal columns in least-squares sense.
    """
    X = ts.T  # (T, n)
    C = X.T @ X
    vals, vecs = np.linalg.eigh(C)
    W = vecs @ np.diag(1.0/np.sqrt(np.maximum(vals, 1e-12))) @ vecs.T
    Y = X @ W
    return Y.T

def roi_time_series(RECORDS: pd.DataFrame,
                    roi_map: Dict[str, List[str]],
                    windows: Optional[List[Tuple[float,float]]],
                    time_col: str = 'Timestamp',
                    orthogonalize: bool = True) -> Tuple[np.ndarray, List[str], float]:
    """
    Build (n_roi, T) ROI matrix by averaging available channels per ROI,
    with optional symmetric orthogonalization across ROIs.
    """
    fs = infer_fs(RECORDS, time_col)
    ts = []
    names = []
    for roi, chans in roi_map.items():
        present = [ch for ch in chans if (ch in RECORDS.columns) or ('EEG.'+ch in RECORDS.columns)]
        if not present: continue
        X = []
        for ch in present:
            x = get_series(RECORDS, ch)
            x = slice_concat(x, fs, windows)
            X.append(x)
        L = min(map(len, X))
        X = np.vstack([x[:L] for x in X])
        ts.append(np.mean(X, axis=0))
        names.append(roi)
    if not ts:
        raise ValueError("No ROI could be formed from the provided mapping.")
    TS = np.vstack(ts)
    if orthogonalize and TS.shape[0] > 1:
        TS = symmetric_orthogonalize(TS)
    return TS, names, fs

def roi_plv_msc_vs_sr(RECORDS: pd.DataFrame,
                      roi_map: Dict[str,List[str]],
                      sr_channel: str,
                      windows: Optional[List[Tuple[float,float]]] = None,
                      time_col: str = 'Timestamp',
                      phase_band: Tuple[float,float] = (7.3, 8.3),
                      harmonics: List[float] = (7.83,14.3,20.8,27.3,33.8),
                      nperseg: Optional[int] = None, noverlap: Optional[int] = None,
                      n_surr: int = 200, rng_seed: int = 17) -> Dict[str, object]:
    """
    ROI-level PLV (phase_band) and MSC (harmonics) vs SR with circular-shift surrogates.
    Returns DataFrame with PLV, MSC_mean, p-values; and plots bars + null bands.
    """
    TS, roi_names, fs = roi_time_series(RECORDS, roi_map, windows, time_col, orthogonalize=True)
    y = get_series(RECORDS, sr_channel)
    y = slice_concat(y, fs, windows)

    # PLV per ROI
    xb = bandpass(y, fs, phase_band[0], phase_band[1])
    ph_y = np.angle(signal.hilbert(xb))
    rows = []
    # coherence via Welch
    if nperseg is None: nperseg = int(4*fs)
    if noverlap is None: noverlap = int(0.5*nperseg)
    fY, _ = signal.welch(y, fs=fs, nperseg=nperseg, noverlap=noverlap)

    # surrogates
    rng = np.random.default_rng(rng_seed)
    for r_idx, roi in enumerate(roi_names):
        x = TS[r_idx]
        # PLV
        xr = bandpass(x, fs, phase_band[0], phase_band[1])
        ph_x = np.angle(signal.hilbert(xr))
        plv = float(np.abs(np.mean(np.exp(1j*(ph_x - ph_y)))))
        # MSC across harmonics (Welch)
        fX, Px = signal.welch(x, fs=fs, nperseg=nperseg, noverlap=noverlap)
        fC, Cxy = signal.coherence(x, y, fs=fs, nperseg=nperseg, noverlap=noverlap)
        msc_vals=[]
        for hf in harmonics:
            idx = int(np.argmin(np.abs(fC - hf)))
            msc_vals.append(float(Cxy[idx]))
        msc_mean = float(np.mean(msc_vals))

        # surrogate nulls by circular shift of SR
        null_plv=[]; null_msc=[]
        n = len(y)
        for _ in range(n_surr):
            s = int(rng.integers(1, n-1))
            ys = np.r_[y[-s:], y[:-s]]
            # PLV null
            ysb = bandpass(ys, fs, phase_band[0], phase_band[1])
            ph_ys = np.angle(signal.hilbert(ysb))
            null_plv.append(np.abs(np.mean(np.exp(1j*(ph_x - ph_ys)))))
            # MSC null at harmonics
            _, C0 = signal.coherence(x, ys, fs=fs, nperseg=nperseg, noverlap=noverlap)
            vals=[]
            for hf in harmonics:
                idx = int(np.argmin(np.abs(fC - hf)))
                vals.append(float(C0[idx]))
            null_msc.append(np.mean(vals))
        plv_thr = float(np.nanpercentile(null_plv, 95))
        msc_thr = float(np.nanpercentile(null_msc, 95))
        p_plv = float((np.sum(np.array(null_plv) >= plv)+1)/(n_surr+1))
        p_msc = float((np.sum(np.array(null_msc)>= msc_mean)+1)/(n_surr+1))

        rows.append({'ROI':roi, 'PLV':plv, 'PLV_thr95':plv_thr, 'p_PLV':p_plv,
                     'MSC_mean':msc_mean, 'MSC_thr95':msc_thr, 'p_MSC':p_msc})
    df = pd.DataFrame(rows)

    # --- plots ---
    # PLV bars
    plt.figure(figsize=(8,3))
    plt.bar(df['ROI'], df['PLV'], color='tab:blue', alpha=0.9)
    for i,(thr) in enumerate(df['PLV_thr95']):
        plt.plot([i-0.4, i+0.4],[thr,thr], 'k--', lw=1)
    plt.ylabel(f'PLV ({phase_band[0]}–{phase_band[1]} Hz)'); plt.title('ROI PLV vs Schumann (95% null dashed)')
    plt.tight_layout(); plt.show()

    # MSC bars
    plt.figure(figsize=(8,3))
    plt.bar(df['ROI'], df['MSC_mean'], color='tab:orange', alpha=0.9)
    for i,(thr) in enumerate(df['MSC_thr95']):
        plt.plot([i-0.4, i+0.4],[thr,thr], 'k--', lw=1)
    plt.ylabel('MSC (mean across harmonics)'); plt.title('ROI coherence vs Schumann (95% null dashed)')
    plt.tight_layout(); plt.show()

    return {'roi_table': df, 'roi_names': roi_names, 'fs': fs}

# ----------------------------- usage examples ------------------------------
if __name__ == "__main__":
    # Example channel lists / ROIs (edit to your montage)
    eeg_channels = ['EEG.O1','EEG.O2','EEG.Oz','EEG.P3','EEG.P4','EEG.Pz','EEG.F3','EEG.F4']
    sr_channel  = 'EEG.Oz'   # or a magnetometer
    # windows = [(290,310),(580,600)]   # optional

    # 5a) Cross-domain graph alignment
    # res = cross_domain_graph_alignment(RECORDS,
    #         eeg_channels=eeg_channels, sr_channel=sr_channel,
    #         band=(8,13), harmonics=[7.83,14.3,20.8,27.3,33.8],
    #         windows=[(290,310),(580,600)])
    # print(res['summary'])

    # 5b) ROI mapping (sensor proxy)
    # conservative ROI map (use whatever exists)
    # roi_map = {
    #   'OCC': ['O1','O2','Oz'],
    #   'PAR': ['P3','P4','Pz','POz'],
    #   'FR':  ['F3','F4','Fz'],
    #   'TEMP':['T7','T8','TP7','TP8']
    # }
    # roi_res = roi_plv_msc_vs_sr(RECORDS, roi_map, sr_channel=sr_channel,
    #         windows=[(290,310),(580,600)], phase_band=(7.3,8.3),
    #         harmonics=[7.83,14.3,20.8,27.3,33.8], n_surr=200)
    # print(roi_res['roi_table'])
    pass


In [ ]:
eeg_channels = ['EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.F3','EEG.F4','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2']
sr_channel  = 'EEG.F4'   # or a magnetometer
windows = [(290,310),(580,600)]   # optional

# 5a) Cross-domain graph alignment
res = cross_domain_graph_alignment(RECORDS,
        eeg_channels=eeg_channels, sr_channel=sr_channel,
        band=(8,13), harmonics=[7.83,14.3,20.8,27.3,33.8],
        windows=[(290,310),(580,600)])
print(res['summary'])


In [ ]:
# 5b) ROI mapping (sensor proxy)
# conservative ROI map (use whatever exists)
roi_map = {
  'OCC': ['O1','O2'],
  'PAR': ['P7','P8'],
  'FR':  ['AF3','AF4','F3','F4','F7','F8','FC5','FC6'],
  'TEMP':['T7','T8']
}
roi_res = roi_plv_msc_vs_sr(RECORDS, roi_map, sr_channel=sr_channel,
        windows=[(290,310),(580,600)], phase_band=(7.3,8.3),
        harmonics=[7.83,14.3,20.8,27.3,33.8], n_surr=200)
print(roi_res['roi_table'])

In [ ]:
"""
Event-related & HMM approaches — simple validity tests & graphs
===============================================================

6a) Schumann-burst ERP/ERSP/ITC (with simple cluster-perm tests)
    • Detect Schumann bursts on a reference channel (7.83 ± half_bw Hz envelope).
    • Time-lock EEG trials to burst onsets; build:
        – ERP (trial-average time-domain)
        – ERSP (time–frequency power via Morlet CWT)
        – ITC (inter-trial coherence)
    • Simple permutation tests:
        – ERP: sign-flip trials → time-wise threshold + max-cluster mass along time.
        – ERSP/ITC: sign-flip trials → TF threshold + TF cluster mass (4-connectivity).
    • Plots: ERP with significant time mask; ERSP & ITC maps with TF clusters.

6b) HMM-like state analysis on EEG spectrograms (GaussianMixture fallback)
    • Build sliding-window band-power features (θ/α/β/γ) from EEG (mean over channels).
    • Fit GaussianMixture (K states) to features; decode state(t).
    • Validate vs Schumann amplitude:
        – Event-triggered state occupancy around Schumann peaks (ETA + null by circular shift)
        – Logistic regression of state transitions vs Schumann amplitude (ROC-AUC + null)

Only depends on NumPy/SciPy/matplotlib/scikit-learn (+NetworkX for clustering).
Copy/paste into your notebook. See usage at bottom.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from typing import Dict, List, Tuple, Optional
from scipy import signal
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ------------------------------- generic helpers -------------------------------

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.' + name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found in RECORDS.")

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order: int = 4) -> np.ndarray:
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny)); f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

def slice_epoch(x: np.ndarray, i0: int, i1: int) -> Optional[np.ndarray]:
    i0 = max(0, i0); i1 = min(len(x), i1)
    if i1 <= i0: return None
    return x[i0:i1]

# --------------------- Schumann burst detection (re-used) ----------------------

def detect_schumann_bursts(RECORDS: pd.DataFrame, sr_channel: str,
                           time_col: str = 'Timestamp',
                           center_hz: float = 7.83, half_bw_hz: float = 0.6,
                           smooth_sec: float = 0.25,
                           thresh_mode: str = 'z', z_thresh: float = 2.5,
                           perc_thresh: float = 95.0,
                           min_isi_sec: float = 2.0) -> Dict[str, object]:
    fs = infer_fs(RECORDS, time_col)
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    y = get_series(RECORDS, sr_channel)
    yb = bandpass(y, fs, center_hz-half_bw_hz, center_hz+half_bw_hz)
    env = np.abs(signal.hilbert(yb))
    # smooth
    n = max(1, int(round(fs*smooth_sec)))
    if n>1:
        w = np.hanning(n); w /= w.sum()
        env = np.convolve(env, w, mode='same')
    # threshold
    if thresh_mode == 'z':
        z = (env - env.mean())/(env.std()+1e-12)
        mask = z >= z_thresh
    else:
        thr = np.percentile(env, perc_thresh)
        mask = env >= thr
    on_idx = np.where(np.diff(mask.astype(int))==1)[0] + 1
    on = []
    last = -np.inf
    for i in on_idx:
        if t[i] - last >= min_isi_sec:
            on.append(t[i]); last = t[i]
    return {'onsets_sec': on, 'env': env, 't': t}

# --------------------------- 6a) ERP / ERSP / ITC ------------------------------

def morlet_cwt(sig: np.ndarray, fs: float, freqs: np.ndarray, w0: float = 6.0) -> np.ndarray:
    """
    Complex Morlet CWT via FFT-convolution.
    Returns array of shape (n_freq, N) with complex coefficients.
    """
    sig = np.asarray(sig, float)
    N = sig.size
    Wx = []

    # Precompute FFT of the (padded) signal once per maximum kernel length
    # We build each kernel with length L depending on f0; use linear convolution length N+L-1
    for f0 in freqs:
        # Build complex Morlet kernel in time
        # duration: a few cycles at f0 (wider at low f to keep frequency resolution)
        dur = max(2.0, 8.0 / f0)                 # seconds
        L = int(np.ceil(dur * fs))
        if L % 2 == 0:
            L += 1
        tt = (np.arange(-(L // 2), L // 2 + 1)) / fs
        sigma_t = w0 / (2 * np.pi * f0)
        mw = np.exp(-0.5 * (tt / sigma_t) ** 2) * np.exp(1j * 2 * np.pi * f0 * tt)
        # zero-mean correction + L2 normalize
        mw = mw - np.mean(mw)
        mw = mw / (np.sqrt(np.sum(np.abs(mw) ** 2)) + 1e-24)

        # Linear convolution via FFT (complex)
        n_lin = N + L - 1
        n_fft = int(2 ** np.ceil(np.log2(n_lin)))   # next power of two
        S = np.fft.fft(sig, n=n_fft)
        H = np.fft.fft(mw,  n=n_fft)
        conv = np.fft.ifft(S * H)[:n_lin]          # complex result

        # center-trim to length N (align kernel center at each t)
        # shift so kernel center aligns with signal sample
        start = (L - 1) // 2
        end = start + N
        Wx.append(conv[start:end])

    return np.array(Wx)  # (n_freq, N) complex


def erp_ersp_itc(RECORDS: pd.DataFrame, eeg_channels: List[str], sr_channel: str,
                 time_col: str = 'Timestamp',
                 win_sec: Tuple[float,float] = (-5.0, 5.0),
                 baseline_sec: Tuple[float,float] = (-4.0,-1.0),
                 center_hz: float = 7.83, half_bw_hz: float = 0.6,
                 detect_kwargs: Dict = None,
                 fmin: float = 4.0, fmax: float = 40.0, n_freq: int = 48,
                 w0: float = 6.0,
                 n_perm: int = 200, alpha: float = 0.05,
                 show: bool = True) -> Dict[str, object]:
    """
    Build ERP/ERSP/ITC time-locked to Schumann bursts and run simple cluster-perm tests.
    Returns dict with curves/maps and significance masks.
    """
    detect_kwargs = detect_kwargs or {}
    fs = infer_fs(RECORDS, time_col)
    det = detect_schumann_bursts(RECORDS, sr_channel, time_col=time_col,
                                 center_hz=center_hz, half_bw_hz=half_bw_hz, **detect_kwargs)
    onsets = det['onsets_sec']
    if len(onsets)==0:
        raise ValueError("No Schumann bursts detected. Loosen threshold or check channel.")

    # build trials for each EEG channel -> ERP first (avg all channels at the end)
    t_axis = np.arange(int(win_sec[0]*fs), int(win_sec[1]*fs)) / fs
    trials = []        # (n_trials, n_time)
    tf_trials = []     # list of (n_trials, n_freq, n_time) per channel (we'll average across channels)
    freqs = np.exp(np.linspace(np.log(fmin), np.log(fmax), n_freq))
    for ch in eeg_channels:
        x = get_series(RECORDS, ch)
        # time-lock segments
        segs=[]
        tf_segs=[]
        for on in onsets:
            i_on = int(round(on*fs))
            i0 = i_on + int(round(win_sec[0]*fs))
            i1 = i_on + int(round(win_sec[1]*fs))
            seg = slice_epoch(x, i0, i1)
            if seg is None or len(seg) != len(t_axis):
                continue
            segs.append(seg)
            # TF (power)
            W = morlet_cwt(seg, fs, freqs, w0=w0)
            tf_segs.append(np.abs(W)**2)
        if segs:
            arr = np.vstack(segs)                  # (n_trials, n_time)
            tf_arr = np.stack(tf_segs, axis=0)     # (n_trials, n_freq, n_time)
            trials.append(arr)
            tf_trials.append(tf_arr)

    if not trials:
        raise ValueError("No valid trials formed (edge effects or too few bursts).")
    # average across channels → trial x time (ERP) and trial x freq x time (ERSP base)
    ERP_trials = np.nanmean(np.stack(trials, axis=0), axis=0)            # (n_trials, n_time)
    ERSP_trials = np.nanmean(np.stack(tf_trials, axis=0), axis=0)        # (n_trials, n_freq, n_time)

    # baseline correction for ERSP (dB)
    bsel = (t_axis>=baseline_sec[0]) & (t_axis<=baseline_sec[1])
    ERSP_db = 10*np.log10(ERSP_trials / (np.nanmean(ERSP_trials[:,:,bsel], axis=2, keepdims=True)+1e-24))
    # ITC = |mean(W/|W|)| across trials (use channel-avg phase: we already averaged power across channels;
    # compute ITC from first channel's complex CWT to keep it simple)
    # Use the first channel's TF trials to get phases:
    Wphase = np.stack(tf_trials, axis=0)[0]  # (n_trials, n_freq, n_time) power, not phase — recompute phases from one channel
    # recompute from first EEG channel to keep code consistent:
    x0 = get_series(RECORDS, eeg_channels[0])
    W_trials=[]
    for on in onsets[:ERP_trials.shape[0]]:
        i_on = int(round(on*fs))
        seg = slice_epoch(x0, i_on+int(round(win_sec[0]*fs)), i_on+int(round(win_sec[1]*fs)))
        if seg is None or len(seg)!=len(t_axis): continue
        W_trials.append(morlet_cwt(seg, fs, freqs, w0=w0))
    W_trials = np.stack(W_trials, axis=0)           # (n_trials, n_freq, n_time)
    ITC = np.abs(np.nanmean(W_trials/np.maximum(np.abs(W_trials),1e-24), axis=0))  # (n_freq, n_time)

    # ---------------- permutation tests ----------------
    rng = np.random.default_rng(11)
    # ERP sign-flip → time threshold + cluster mass (1D)
    mean_erp = np.nanmean(ERP_trials, axis=0)
    null_erp=[]
    for _ in range(n_perm):
        signs = rng.choice([-1,1], size=ERP_trials.shape[0])
        null_erp.append(np.nanmean(signs[:,None]*ERP_trials, axis=0))
    null_erp = np.stack(null_erp, axis=0)
    thr_erp = np.nanpercentile(null_erp, 100*(1-alpha), axis=0)
    sig_time = mean_erp > thr_erp
    # cluster mass (1D)
    max_mass=0.0; cur=0.0
    for i,flag in enumerate(sig_time):
        if flag: cur += mean_erp[i]
        else: max_mass=max(max_mass,cur); cur=0.0
    erp_mass = max(max_mass, cur)
    null_mass=[]
    for p in null_erp:
        cur=0.0; mm=0.0
        for i in range(len(p)):
            if p[i]>thr_erp[i]:
                cur+=p[i]; mm=max(mm,cur)
            else:
                cur=0.0
        null_mass.append(mm)
    erp_sig = erp_mass >= np.nanpercentile(null_mass, 95)

    # ERSP/ITC TF clustering (4-connectivity, one-sided)
    # ERSP: positive deviations (power increases)
    ERSP_mean = np.nanmean(ERSP_db, axis=0)         # (n_freq, n_time)
    # null by sign-flip trials
    null_tf=[]
    for _ in range(n_perm):
        signs = rng.choice([-1,1], size=ERSP_db.shape[0])[:,None,None]
        null_tf.append(np.nanmean(signs*ERSP_db, axis=0))
    null_tf = np.stack(null_tf, axis=0)
    thr_tf = np.nanpercentile(null_tf, 100*(1-alpha), axis=0)  # per-TF threshold
    tf_sig = ERSP_mean > thr_tf

    # cluster mass on TF grid
    G = nx.grid_2d_graph(tf_sig.shape[0], tf_sig.shape[1])
    def max_cluster_mass(mask, value_map):
        mask_idx = set(zip(*np.where(mask)))
        visited=set(); best=0.0
        for node in list(mask_idx):
            if node in visited: continue
            stack=[node]; mass=0.0
            while stack:
                u=stack.pop()
                if u in visited or u not in mask_idx: continue
                visited.add(u); mass += float(value_map[u[0], u[1]])
                for v in G.neighbors(u):
                    if v in mask_idx and v not in visited:
                        stack.append(v)
            best=max(best,mass)
        return best
    ersp_mass = max_cluster_mass(tf_sig, ERSP_mean)
    # null TF cluster mass
    null_mass_tf=[]
    for p in null_tf:
        null_mass_tf.append(max_cluster_mass(p>thr_tf, p))
    ersp_sig = ersp_mass >= np.nanpercentile(null_mass_tf, 95)

    # ITC: same procedure
    # Build null by random sign of trials' unit-phase (approximate)
    # (We already averaged trials → approximate null by time-circular shift of phase map)
    def circ_shift_2d(A, sh0, sh1):
        return np.roll(np.roll(A, sh0, axis=0), sh1, axis=1)
    null_itc=[]
    for _ in range(n_perm):
        sh0 = int(rng.integers(1, ITC.shape[0]-1))
        sh1 = int(rng.integers(1, ITC.shape[1]-1))
        null_itc.append(circ_shift_2d(ITC, sh0, sh1))
    null_itc = np.stack(null_itc, axis=0)
    thr_itc = np.nanpercentile(null_itc, 100*(1-alpha), axis=0)
    itc_sig = ITC > thr_itc
    itc_mass = max_cluster_mass(itc_sig, ITC)
    itc_sig_global = itc_mass >= np.nanpercentile([max_cluster_mass(n>thr_itc, n) for n in null_itc], 95)

    # ---------------- plots ----------------
    if show:
        # ERP
        plt.figure(figsize=(9,3))
        plt.plot(t_axis, mean_erp, lw=1.8, label='ERP (avg EEG)')
        plt.fill_between(t_axis, 0, mean_erp, where=sig_time, color='tab:red', alpha=0.25, step='pre', label='sig (time-wise)')
        plt.axvline(0, color='k', lw=1); plt.axhline(0, color='k', lw=0.5, alpha=0.4)
        plt.title(f"Schumann-locked ERP  (cluster-sig={erp_sig})"); plt.xlabel('Time (s)'); plt.ylabel('uV (a.u.)')
        plt.legend(); plt.tight_layout(); plt.show()

        # ERSP
        plt.figure(figsize=(9,3.2))
        extent = [t_axis[0], t_axis[-1], freqs[0], freqs[-1]]
        plt.imshow(ERSP_mean, aspect='auto', origin='lower', extent=extent, cmap='magma')
        plt.colorbar(label='ERSP (dB vs baseline)')
        # overlay significant TF mask
        yy, xx = np.where(tf_sig)
        plt.scatter(t_axis[xx], freqs[yy], s=2, c='cyan', alpha=0.6, label='sig TF')
        plt.title(f"Schumann-locked ERSP (TF cluster-sig={ersp_sig})")
        plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)'); plt.legend(loc='upper right', fontsize=8)
        plt.tight_layout(); plt.show()

        # ITC
        plt.figure(figsize=(9,3.2))
        plt.imshow(ITC, aspect='auto', origin='lower', extent=extent, cmap='viridis', vmin=0, vmax=1)
        plt.colorbar(label='ITC')
        yi, xi = np.where(itc_sig)
        plt.scatter(t_axis[xi], freqs[yi], s=2, c='white', alpha=0.7, label='sig TF')
        plt.title(f"Schumann-locked ITC (TF cluster-sig={itc_sig_global})")
        plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)'); plt.legend(loc='upper right', fontsize=8)
        plt.tight_layout(); plt.show()

    return {'t': t_axis, 'freqs': freqs,
            'ERP_mean': mean_erp, 'ERP_sig_time': sig_time, 'ERP_cluster_sig': erp_sig,
            'ERSP_mean_db': ERSP_mean, 'ERSP_sig_tf': tf_sig, 'ERSP_cluster_sig': ersp_sig,
            'ITC': ITC, 'ITC_sig_tf': itc_sig, 'ITC_cluster_sig': itc_sig_global}

# --------------------------- 6b) HMM-like state analysis -----------------------

def bandpower_features(RECORDS: pd.DataFrame, eeg_channels: List[str],
                       time_col: str = 'Timestamp',
                       bands: Dict[str, Tuple[float,float]] = None,
                       win_sec: float = 2.0, step_sec: float = 0.25) -> Dict[str, object]:
    """
    Sliding-window mean band power per band, averaged over EEG channels.
    Returns {'t': t_centers, 'X': feature_matrix (T, nbands)}.
    """
    bands = bands or {'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,80)}
    fs = infer_fs(RECORDS, time_col)
    Xsig = [get_series(RECORDS, ch) for ch in eeg_channels]
    Xsig = np.vstack(Xsig)  # (n_ch, N)
    N = Xsig.shape[1]
    win = int(round(win_sec*fs)); step = int(round(step_sec*fs))
    centers = np.arange(win//2, N - win//2, step)
    feats=[]
    for c in centers:
        s = c - win//2; e = c + win//2
        seg = Xsig[:, s:e]                      # (n_ch, win)
        row=[]
        for (f1,f2) in bands.values():
            bp = []
            for ch in range(seg.shape[0]):
                xb = bandpass(seg[ch], fs, f1, f2)
                bp.append(np.mean(xb**2))
            row.append(np.mean(bp))
        feats.append(row)
    feats = np.asarray(feats)     # (T, nbands)
    return {'t': centers/fs, 'X': feats, 'bands': list(bands.keys())}

def schumann_amplitude(RECORDS: pd.DataFrame, sr_channel: str,
                       time_col: str = 'Timestamp',
                       center_hz: float = 7.83, half_bw_hz: float = 0.6,
                       win_sec: float = 2.0, step_sec: float = 0.25) -> Dict[str, object]:
    """Sliding-window Schumann envelope mean in the same grid as bandpower_features."""
    fs = infer_fs(RECORDS, time_col)
    y = get_series(RECORDS, sr_channel)
    env = np.abs(signal.hilbert(bandpass(y, fs, center_hz-half_bw_hz, center_hz+half_bw_hz)))
    N = len(env); win = int(round(win_sec*fs)); step = int(round(step_sec*fs))
    centers = np.arange(win//2, N-win//2, step)
    amp=[]
    for c in centers:
        s=c-win//2; e=c+win//2
        amp.append(np.mean(env[s:e]))
    return {'t': centers/fs, 'amp': np.asarray(amp)}

def hmm_states_gmm(features: np.ndarray, K: int = 3, random_state: int = 0) -> Dict[str, object]:
    """Fit GaussianMixture as a simple HMM surrogate; decode state(t)."""
    gmm = GaussianMixture(n_components=K, covariance_type='full', random_state=random_state)
    gmm.fit(features)
    gamma = gmm.predict_proba(features)        # (T, K)
    z = np.argmax(gamma, axis=1)               # hard states
    return {'model': gmm, 'states': z, 'post': gamma}

def eta_state_occupancy(states: np.ndarray, T: np.ndarray,
                        event_times: np.ndarray, span_sec: float = 10.0, n_states: int = None) -> Dict[str, object]:
    """Event-triggered state occupancy around event_times (ETA)."""
    if n_states is None: n_states = int(states.max()+1)
    # time index mapping
    tau = np.arange(-span_sec, span_sec, np.median(np.diff(T)))
    occ = np.zeros((n_states, tau.size))
    counts = np.zeros(tau.size)
    for et in event_times:
        i0 = np.argmin(np.abs(T - et))
        # build relative samples
        for k,dt in enumerate(tau):
            idx = i0 + int(round(dt / np.median(np.diff(T))))
            if 0 <= idx < len(states):
                s = states[idx]
                occ[s, k] += 1
                counts[k] += 1
    occ = occ / np.maximum(counts, 1e-12)
    return {'tau': tau, 'occ': occ, 'counts': counts}

def logistic_state_transition_vs_amp(states: np.ndarray, amp: np.ndarray) -> Dict[str, float]:
    """Binary transition Y: 1 if state changes at t+1; regress on Schumann amp(t)."""
    y = (np.diff(states) != 0).astype(int)
    X = amp[:-1][:,None]
    if np.all(y==y[0]):
        return {'auc': np.nan, 'coef': np.nan}
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(X, y)
    p = clf.predict_proba(X)[:,1]
    auc = roc_auc_score(y, p)
    return {'auc': float(auc), 'coef': float(clf.coef_[0,0])}

def run_hmm_state_tests(RECORDS: pd.DataFrame, eeg_channels: List[str], sr_channel: str,
                        time_col: str = 'Timestamp',
                        K: int = 3, bands: Dict[str, Tuple[float,float]] = None,
                        win_sec: float = 2.0, step_sec: float = 0.25,
                        span_sec: float = 10.0,
                        peak_perc: float = 95.0,
                        n_perm: int = 200, rng_seed: int = 23,
                        show: bool = True) -> Dict[str, object]:
    """
    1) Build band-power features; fit GMM(K) → state(t).
    2) Build Schumann amplitude; find peaks (> percentile).
    3) ETA of state occupancies around peaks + circular-shift null.
    4) Logistic regression: state transitions vs Schumann amplitude (AUC + null).
    """
    feat = bandpower_features(RECORDS, eeg_channels, time_col=time_col, bands=bands,
                              win_sec=win_sec, step_sec=step_sec)
    amp  = schumann_amplitude(RECORDS, sr_channel, time_col=time_col,
                              center_hz=7.83, half_bw_hz=0.6,
                              win_sec=win_sec, step_sec=step_sec)

    # align to common time grid
    t  = feat['t']; X = feat['X']
    ta = amp['t'];  A = amp['amp']
    if len(ta) != len(t):
        # interpolate Schumann amp to feature grid
        A = np.interp(t, ta, A)

    hmm = hmm_states_gmm(X, K=K, random_state=0)
    z = hmm['states']

    # peak events
    thr = np.percentile(A, peak_perc)
    peaks = np.where((A[1:-1]>A[:-2]) & (A[1:-1]>A[2:]) & (A[1:-1]>=thr))[0] + 1
    events = t[peaks]

    eta = eta_state_occupancy(z, t, events, span_sec=span_sec, n_states=K)
    tau, occ = eta['tau'], eta['occ']

    # ETA null by circular shift of A
    rng = np.random.default_rng(rng_seed)
    null_occ = []
    for _ in range(n_perm):
        s = int(rng.integers(1, len(A)-1))
        Ap = np.r_[A[-s:], A[:-s]]
        pk = np.where((Ap[1:-1]>Ap[:-2]) & (Ap[1:-1]>Ap[2:]) & (Ap[1:-1]>=thr))[0] + 1
        ev = t[pk]
        et = eta_state_occupancy(z, t, ev, span_sec=span_sec, n_states=K)
        null_occ.append(et['occ'])
    null_occ = np.stack(null_occ, axis=0)   # (n_perm, K, Ttau)

    # summarize: max occupancy boost per state vs null 95%
    obs_boost = np.max(occ - np.nanmean(null_occ, axis=0), axis=1)  # (K,)
    thr95 = np.nanpercentile(np.max(null_occ - np.nanmean(null_occ, axis=0), axis=2), 95, axis=0)  # (K,)

    # logistic regression: transitions vs A
    lr = logistic_state_transition_vs_amp(z, A)
    # null AUC by circular shift
    auc_null=[]
    for _ in range(n_perm):
        s = int(rng.integers(1, len(A)-1))
        Ap = np.r_[A[-s:], A[:-s]]
        lr0 = logistic_state_transition_vs_amp(z, Ap)
        if not np.isnan(lr0['auc']):
            auc_null.append(lr0['auc'])
    auc_thr95 = np.nanpercentile(auc_null, 95) if auc_null else np.nan

    if show:
        # plot occupancy ETAs
        plt.figure(figsize=(9, 3.2))
        for k in range(K):
            plt.plot(tau, occ[k], lw=1.6, label=f'State {k}')
        plt.axvline(0, color='k', lw=1)
        plt.title('ETA: state occupancy around Schumann peaks')
        plt.xlabel('Time (s)'); plt.ylabel('Occupancy')
        plt.legend(); plt.tight_layout(); plt.show()

        # bar: max boost vs null
        plt.figure(figsize=(6,3))
        x = np.arange(K)
        plt.bar(x, obs_boost, color='tab:blue', alpha=0.9)
        for i,thr in enumerate(thr95):
            plt.plot([i-0.35,i+0.35],[thr,thr],'k--',lw=1)
        plt.xticks(x, [f'S{k}' for k in range(K)])
        plt.ylabel('Max occupancy boost'); plt.title('ETA boost vs null (95% dashed)')
        plt.tight_layout(); plt.show()

        # logistic regression summary
        print(f"LogReg transitions ~ Schumann amp: AUC={lr['auc']:.3f}, coef={lr['coef']:.3f}, null95={auc_thr95:.3f}")

    return {'t': t, 'states': z, 'post': hmm['post'], 'amp': A,
            'events': events,
            'eta_tau': tau, 'eta_occ': occ, 'eta_null': null_occ,
            'eta_boost': obs_boost, 'eta_boost_thr95': thr95,
            'lr_auc': lr['auc'], 'lr_coef': lr['coef'], 'lr_auc_thr95': auc_thr95}

# ------------------------------- usage examples --------------------------------
if __name__ == "__main__":
    # 6a) ERP/ERSP/ITC around Schumann bursts
    # er = erp_ersp_itc(RECORDS,
    #                   eeg_channels=['EEG.O1','EEG.O2','EEG.Pz'],
    #                   sr_channel='EEG.Oz',
    #                   time_col='Timestamp',
    #                   win_sec=(-5, 5),
    #                   baseline_sec=(-4,-1),
    #                   center_hz=7.83, half_bw_hz=0.6,
    #                   detect_kwargs={'thresh_mode':'z','z_thresh':2.5},
    #                   fmin=4, fmax=40, n_freq=48, w0=6.0,
    #                   n_perm=200, alpha=0.05, show=True)

    # 6b) HMM-like state tests on EEG band-power vs Schumann amplitude
    # st = run_hmm_state_tests(RECORDS,
    #                          eeg_channels=['EEG.O1','EEG.O2','EEG.Pz','EEG.F4'],
    #                          sr_channel='EEG.Oz',
    #                          time_col='Timestamp',
    #                          K=3,
    #                          bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,80)},
    #                          win_sec=2.0, step_sec=0.25,
    #                          span_sec=10.0, peak_perc=95.0,
    #                          n_perm=200, show=True)
    pass


In [ ]:
def erp_ersp_itc_safe(RECORDS, eeg_channels, sr_channel,
                      time_col='Timestamp',
                      win_sec=(-3.0, 3.0),           # shorter default
                      baseline_sec=(-2.0, -0.5),     # inside window
                      center_hz=7.83, half_bw_hz=0.6,
                      detect_kwargs=None,
                      fmin=4.0, fmax=40.0, n_freq=48, w0=6.0,
                      n_perm=200, alpha=0.05,
                      edge_policy='pad',              # 'pad' or 'drop'
                      pad_mode='reflect',             # or 'constant'
                      show=True):
    """Schumann-locked ERP/ERSP/ITC with edge padding and TF cluster-perm."""
    detect_kwargs = detect_kwargs or {}
    fs = infer_fs(RECORDS, time_col)
    det = detect_schumann_bursts(RECORDS, sr_channel, time_col=time_col,
                                 center_hz=center_hz, half_bw_hz=half_bw_hz, **detect_kwargs)
    onsets = np.array(det['onsets_sec'], float)
    if onsets.size == 0:
        raise ValueError("No Schumann bursts detected.")

    t_axis = np.arange(int(win_sec[0]*fs), int(win_sec[1]*fs)) / fs
    L = len(t_axis)
    freqs = np.exp(np.linspace(np.log(fmin), np.log(fmax), n_freq))

    # helper: pad segment to length L
    def take_segment(x, i_on):
        i0 = i_on + int(round(win_sec[0]*fs))
        i1 = i_on + int(round(win_sec[1]*fs))
        if 0 <= i0 and i1 <= len(x):
            seg = x[i0:i1]
            if len(seg) != L: return None
            return seg
        if edge_policy == 'drop':
            return None
        # pad
        left = max(0, -i0)
        right = max(0, i1 - len(x))
        s = max(i0, 0); e = min(i1, len(x))
        seg = x[s:e]
        if left or right:
            seg = np.pad(seg, (left, right), mode=pad_mode)
        if len(seg) != L:  # last resort
            return None
        return seg

    # collect trials per channel
    trials_by_ch = []
    cwt_by_ch   = []
    for ch in eeg_channels:
        x = get_series(RECORDS, ch)
        segs=[]; cplx=[]
        for on in onsets:
            i_on = int(round(on*fs))
            seg = take_segment(x, i_on)
            if seg is None: continue
            seg_hilb = morlet_cwt(seg, fs, freqs, w0=w0)   # (n_freq, L) complex
            segs.append(seg)
            cplx.append(seg_hilb)
        if segs:
            trials_by_ch.append(np.vstack(segs))                   # (n_trials, L)
            cwt_by_ch.append(np.stack(cplx, axis=0))               # (n_trials, n_freq, L)

    if not trials_by_ch:
        raise ValueError("No valid trials formed after padding.")
    # average across channels
    ERP_trials  = np.nanmean(np.stack(trials_by_ch, axis=0), axis=0)      # (n_trials, L)
    CWT_trials  = np.nanmean(np.stack(cwt_by_ch,   axis=0), axis=0)       # (n_trials, n_freq, L)
    ERSP_trials = np.abs(CWT_trials)**2                                   # power

    # ERSP baseline (dB)
    bsel = (t_axis >= baseline_sec[0]) & (t_axis <= baseline_sec[1])
    ERSP_db = 10*np.log10(ERSP_trials / (np.nanmean(ERSP_trials[:,:,bsel], axis=2, keepdims=True)+1e-24))

    # ITC = |mean(exp(i*phase))| across trials
    phases = CWT_trials / np.maximum(np.abs(CWT_trials), 1e-24)
    ITC = np.abs(np.nanmean(phases, axis=0))                              # (n_freq, L)

    # ERP permutation (sign-flip)
    mean_erp = np.nanmean(ERP_trials, axis=0)
    rng = np.random.default_rng(11)
    null_erp=[]
    for _ in range(n_perm):
        signs = rng.choice([-1,1], size=ERP_trials.shape[0])[:,None]
        null_erp.append(np.nanmean(signs*ERP_trials, axis=0))
    null_erp = np.stack(null_erp, axis=0)
    thr_erp  = np.nanpercentile(null_erp, 100*(1-alpha), axis=0)
    sig_time = mean_erp > thr_erp

    # ERSP TF cluster
    ERSP_mean = np.nanmean(ERSP_db, axis=0)                               # (n_freq, L)
    null_tf=[]
    for _ in range(n_perm):
        signs = rng.choice([-1,1], size=ERSP_db.shape[0])[:,None,None]
        null_tf.append(np.nanmean(signs*ERSP_db, axis=0))
    null_tf = np.stack(null_tf, axis=0)
    thr_tf  = np.nanpercentile(null_tf, 100*(1-alpha), axis=0)
    tf_sig  = ERSP_mean > thr_tf

    # cluster mass (4-connectivity)
    G = nx.grid_2d_graph(*ERSP_mean.shape)
    def max_cluster_mass(mask, val):
        idx = set(zip(*np.where(mask))); seen=set(); best=0.0
        for u in list(idx):
            if u in seen: continue
            stack=[u]; mass=0.0
            while stack:
                v = stack.pop()
                if v in seen or v not in idx: continue
                seen.add(v); mass += float(val[v[0], v[1]])
                for w in G.neighbors(v):
                    if w in idx and w not in seen:
                        stack.append(w)
            best = max(best, mass)
        return best
    ersp_mass = max_cluster_mass(tf_sig, ERSP_mean)
    null_mass = [max_cluster_mass(p>thr_tf, p) for p in null_tf]
    ersp_sig  = ersp_mass >= np.nanpercentile(null_mass, 95)

    # ITC TF cluster
    null_itc=[]
    for _ in range(n_perm):
        sh0 = int(rng.integers(1, ITC.shape[0]-1)); sh1 = int(rng.integers(1, ITC.shape[1]-1))
        null_itc.append(np.roll(np.roll(ITC, sh0, axis=0), sh1, axis=1))
    null_itc = np.stack(null_itc, axis=0)
    thr_itc  = np.nanpercentile(null_itc, 100*(1-alpha), axis=0)
    itc_sig  = ITC > thr_itc
    itc_mass = max_cluster_mass(itc_sig, ITC)
    itc_sig_global = itc_mass >= np.nanpercentile([max_cluster_mass(n>thr_itc, n) for n in null_itc], 95)

    # plots (same as before)
    if show:
        plt.figure(figsize=(9,3))
        plt.plot(t_axis, mean_erp, lw=1.8, label='ERP (avg EEG)')
        plt.fill_between(t_axis, 0, mean_erp, where=sig_time, color='tab:red', alpha=0.25, step='pre', label='sig (time-wise)')
        plt.axvline(0, color='k', lw=1); plt.axhline(0, color='k', lw=0.5, alpha=0.4)
        plt.title(f"Schumann-locked ERP  (cluster-sig={ersp_sig})"); plt.xlabel('Time (s)'); plt.ylabel('uV (a.u.)')
        plt.legend(); plt.tight_layout(); plt.show()

        extent = [t_axis[0], t_axis[-1], freqs[0], freqs[-1]]
        plt.figure(figsize=(9,3.2))
        plt.imshow(ERSP_mean, aspect='auto', origin='lower', extent=extent, cmap='magma')
        plt.colorbar(label='ERSP (dB vs baseline)')
        yx = np.where(tf_sig); plt.scatter(t_axis[yx[1]], freqs[yx[0]], s=2, c='cyan', alpha=0.6, label='sig TF')
        plt.title(f"Schumann-locked ERSP (TF cluster-sig={ersp_sig})")
        plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)'); plt.legend(loc='upper right', fontsize=8); plt.tight_layout(); plt.show()

        plt.figure(figsize=(9,3.2))
        plt.imshow(ITC, aspect='auto', origin='lower', extent=extent, cmap='viridis', vmin=0, vmax=1)
        plt.colorbar(label='ITC')
        yi, xi = np.where(itc_sig)
        plt.scatter(t_axis[xi], freqs[yi], s=2, c='white', alpha=0.7, label='sig TF')
        plt.title(f"Schumann-locked ITC (TF cluster-sig={itc_sig_global})")
        plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)'); plt.legend(loc='upper right', fontsize=8)
        plt.tight_layout(); plt.show()

    return {'t': t_axis, 'freqs': freqs,
            'ERP_mean': mean_erp, 'ERP_sig_time': sig_time,
            'ERSP_mean_db': ERSP_mean, 'ERSP_sig_tf': tf_sig, 'ERSP_cluster_sig': ersp_sig,
            'ITC': ITC, 'ITC_sig_tf': itc_sig, 'ITC_cluster_sig': itc_sig_global}


In [ ]:
eeg_channels = ['EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.F3','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.O1','EEG.O2']


# 6a) ERP/ERSP/ITC around Schumann bursts
er = erp_ersp_itc(RECORDS,
                  eeg_channels=eeg_channels,
                  sr_channel='EEG.F4',
                  time_col='Timestamp',
                  win_sec=(-3, 3),
                  baseline_sec=(-4,-1),
                  center_hz=7.83, half_bw_hz=0.6,
                  detect_kwargs={'thresh_mode':'z','z_thresh':1.8},
                  fmin=4, fmax=40, n_freq=48, w0=6.0,
                  n_perm=200, alpha=0.05, show=True)

In [ ]:
er = erp_ersp_itc_safe(
    RECORDS,
    eeg_channels=eeg_channels, #['EEG.O1','EEG.O2'],
    sr_channel='EEG.F4',
    time_col='Timestamp',
    win_sec=(-3, 3),                 # shorter window to avoid edges
    baseline_sec=(-2, -0.5),
    center_hz=7.83, half_bw_hz=0.6,
    detect_kwargs={'thresh_mode':'z','z_thresh':1.8, 'min_isi_sec':1.0},  # looser detection
    fmin=4, fmax=40, n_freq=48, w0=6.0,
    n_perm=200, alpha=0.05,
    edge_policy='pad', pad_mode='reflect',
    show=True
)


In [ ]:
eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.O1','EEG.O2']

# 6b) HMM-like state tests on EEG band-power vs Schumann amplitude
st = run_hmm_state_tests(RECORDS,
                         eeg_channels=eeg_channels,
                         sr_channel='EEG.F4',
                         time_col='Timestamp',
                         K=3,
                         bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,80)},
                         win_sec=2.0, step_sec=0.25,
                         span_sec=10.0, peak_perc=95.0,
                         n_perm=200, show=True)

In [ ]:
"""
ONE-CLICK SESSION REPORT — v2
=============================

What’s new in v2 (vs your v1 runner):
• Uses the PATCHED complex Morlet CWT (no rfft bug) for TF pages.
• Adds the Frequency-Domain Coupling page (MSC at harmonics + WTC).
• Adds the DTF bidirectional GRID (like your single-link figure) with dynamic y-limits.
• Adds Event-related ERP/ERSP/ITC (safe edge-padding).
• Adds an Off-harmonic negative-control page (MSC @ 16–18 Hz).
• Builds a compact “at-a-glance” summary page.
• Skips gracefully (with print) if a helper isn’t defined in your env.

Requires that your helper blocks from earlier steps are in scope (most are),
e.g.:
  - detect_and_plot_schumann_microgrid_with_global_tf
  - run_entanglement_geometry_minCut_PLV + plot_* helpers
  - run_ridge_pac_coupling
  - run_criticality_analysis + plot_* helpers
  - build_functional_harmonics_from_baseline + run_connectome_harmonics_breadth + plot_* helpers
  - run_overlap_coherence_etas
  - run_multi_seed_surface_cuts + plot_multicut_deltas
  - run_phase_embedding_emergent_geometry + plot_phase_embedding_quality
  - run_temporal_holography_multiplexed (optional if you supply onsets/labels)
  - run_tvar_dtf + plot_dtf_grid_bidir_like_single
  - run_multitaper_msc_harmonics + plot_msc_harmonics_compare
  - run_wavelet_coherence

If a block is missing, the report runner will skip it and print a short note.
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from scipy import signal
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ---------------- PATCHED Morlet CWT (complex FFT) ---------------- #
def morlet_cwt(sig: np.ndarray, fs: float, freqs: np.ndarray, w0: float = 6.0) -> np.ndarray:
    """
    Complex Morlet CWT via FFT-convolution (linear convolution, centered).
    Returns array (n_freq, N) complex.
    """
    sig = np.asarray(sig, float)
    N = sig.size
    Wx = []
    for f0 in freqs:
        dur = max(2.0, 8.0 / f0)
        L = int(np.ceil(dur * fs))
        if L % 2 == 0: L += 1
        tt = (np.arange(-(L // 2), L // 2 + 1)) / fs
        sigma_t = w0 / (2 * np.pi * f0)
        mw = np.exp(-0.5 * (tt / sigma_t) ** 2) * np.exp(1j * 2 * np.pi * f0 * tt)
        mw = mw - np.mean(mw)
        mw = mw / (np.sqrt(np.sum(np.abs(mw) ** 2)) + 1e-24)

        n_lin = N + L - 1
        n_fft = int(2 ** np.ceil(np.log2(n_lin)))
        S = np.fft.fft(sig, n=n_fft)
        H = np.fft.fft(mw,  n=n_fft)
        conv = np.fft.ifft(S * H)[:n_lin]    # complex
        start = (L - 1) // 2; end = start + N
        Wx.append(conv[start:end])
    return np.array(Wx)

# -------------- small helpers for this runner -------------- #
def _ensure_dir(d):
    os.makedirs(d, exist_ok=True)
    return d

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def pick_best_channel_for_schumann(RECORDS, time_col='Timestamp'):
    """PSD ranker favoring 4–12 Hz, penalizing 55–65 & 40–90."""
    fs = infer_fs(RECORDS, time_col=time_col)
    cands = [c for c in RECORDS.columns if c.startswith('EEG.') and c != time_col]
    scores = []
    for ch in cands:
        x = np.asarray(RECORDS[ch].values, float)
        f, p = signal.welch(x, fs=fs, nperseg=4*int(fs))
        def band(a,b): 
            sel=(f>=a)&(f<=b)
            return np.trapz(p[sel], f[sel]) if np.any(sel) else 0.0
        low = band(4,12); emg=band(40,90); mains=band(55,65)
        score = low/(emg+1e-12) - 0.2*mains
        scores.append((score, ch))
    scores.sort(reverse=True)
    return scores[0][1] if scores else None

# ---------------- main: one-click report v2 ---------------- #
def run_one_click_session_report_v2(
    RECORDS: pd.DataFrame,
    session_name: str = 'session_v2',
    time_col: str = 'Timestamp',
    electrodes: Optional[List[str]] = None,     # autodetect if None (EEG.*)
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    rebound_windows: Optional[List[Tuple[float,float]]] = None,
    control_windows: Optional[List[Tuple[float,float]]] = None,
    event_onsets: Optional[List[float]] = None, # optional (Temporal Holography)
    event_labels: Optional[List] = None,        # optional labels
    H: Optional[np.ndarray] = None,             # optional harmonics matrix for breadth
    include_offharmonic_control: bool = True,   # MSC off-band control
    f0: float = 7.83
) -> Dict[str, object]:
    """
    Ships the v2 report as a PDF + CSV + PNGs under exports/<session_name>/
    Skips gracefully if a helper is missing. Uses patched Morlet CWT.
    """
    fs = infer_fs(RECORDS, time_col)
    # autodetect electrodes
    if electrodes is None:
        electrodes = [c for c in RECORDS.columns if c.startswith('EEG.')]
    if not electrodes:
        raise ValueError("No EEG.* channels found.")

    base_dir = _ensure_dir(os.path.join('exports', session_name))
    fig_dir  = _ensure_dir(os.path.join(base_dir, 'FIG'))
    csv_dir  = _ensure_dir(os.path.join(base_dir, 'CSV'))
    pdf_path = os.path.join(base_dir, f'{session_name}.pdf')

    # Choose Schumann ref
    sigcol = pick_best_channel_for_schumann(RECORDS, time_col=time_col) or electrodes[0]

    # Build fused Schumann reference (if available)
    fused = None
    if 'detect_and_plot_schumann_microgrid_with_global_tf' in globals():
        try:
            fused = detect_and_plot_schumann_microgrid_with_global_tf(
                RECORDS, signal_col=sigcol, time_col=time_col, show=False
            )
            pd.DataFrame({'t': fused['index'], 'SAI': fused['sai']}).to_csv(os.path.join(csv_dir,'sai.csv'), index=False)
        except Exception as e:
            print("[WARN] Fused micro-grid failed:", e)
    else:
        print("[SKIP] fused micro-grid (helper not found)")

    with PdfPages(pdf_path) as pdf:
        # 0) Summary page header stub
        plt.figure(figsize=(8.5, 3.5))
        plt.axis('off')
        text = f"Session Report v2 — {session_name}\nfs={fs:.2f} Hz\nsource={sigcol}\n" \
               f"Ignition={ignition_windows}\nRebound={rebound_windows}\nControl={control_windows}"
        plt.text(0.01, 0.8, text, fontsize=12, va='top')
        plt.text(0.01, 0.5, "Blocks:\n• Frequency-Domain Coupling (MSC/WTC)"
                            "\n• Entanglement–Geometry"
                            "\n• Ridge–PAC coupling"
                            "\n• Criticality"
                            "\n• Harmonics breadth"
                            "\n• Overlap ETAs (K≥3)"
                            "\n• Multi-seed surfaces"
                            "\n• Phase Embedding"
                            "\n• Temporal Holography (if onsets)"
                            "\n• ERP/ERSP/ITC (safe)"
                            "\n• DTF grid"
                            "\n• Off-harmonic control", fontsize=10, va='top')
        pdf.savefig(); plt.savefig(os.path.join(fig_dir, '00_cover.png')); plt.close('all')

        # 0.5) Frequency-domain coupling (MSC + WTC)
        if 'run_multitaper_msc_harmonics' in globals():
            try:
                x_chs = electrodes
                msc_ign = run_multitaper_msc_harmonics(RECORDS, x_channels=x_chs, y_channel=sigcol,
                                                       windows=ignition_windows, time_col=time_col,
                                                       half_bw_hz=3.0,
                                                       harmonics=[7.83,14.3,20.8,27.3,33.8])
                msc_base = run_multitaper_msc_harmonics(RECORDS, x_channels=x_chs, y_channel=sigcol,
                                                        windows=control_windows, time_col=time_col,
                                                        half_bw_hz=3.0,
                                                        harmonics=[7.83,14.3,20.8,27.3,33.8]) if control_windows else None
                pd.DataFrame(msc_ign['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_harmonics_ign.csv'), index=False)
                if msc_base is not None:
                    pd.DataFrame(msc_base['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_harmonics_base.csv'), index=False)
                if 'plot_msc_harmonics_compare' in globals() and (msc_base is not None):
                    plot_msc_harmonics_compare(msc_ign, msc_base, title='MSC @ harmonics (Ign vs Base)')
                    pdf.savefig(); plt.savefig(os.path.join(fig_dir,'01_msc_harmonics.png')); plt.close('all')
                else:
                    # single-state plot
                    if 'plot_msc_harmonics_table' in globals():
                        plot_msc_harmonics_table(msc_ign['harmonics_table'], title='MSC @ harmonics (Ignition)')
                        pdf.savefig(); plt.savefig(os.path.join(fig_dir,'01_msc_harmonics_ign.png')); plt.close('all')
            except Exception as e:
                print("[WARN] MSC page:", e)
        else:
            print("[SKIP] MSC (helper not found)")

        if 'run_wavelet_coherence' in globals():
            try:
                # pick a posterior channel if present
                pref = [e for e in ['EEG.O1','EEG.O2','EEG.Oz'] if e in electrodes]
                x_wtc = pref[0] if pref else electrodes[0]
                wtc = run_wavelet_coherence(RECORDS, x_channel=x_wtc, y_channel=sigcol,
                                            time_col=time_col, fmin=4, fmax=40, n_freq=64,
                                            w0=6.0, n_perm=200, p_cluster=0.05, show=True)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'02_wtc_map.png')); plt.close('all')
            except Exception as e:
                print("[WARN] WTC page:", e)
        else:
            print("[SKIP] WTC (helper not found)")

        # 1) Entanglement–Geometry (Δmin-cut/Δentropy/ΔPLV)
        if 'run_entanglement_geometry_minCut_PLV' in globals() and 'plot_entanglement_geometry_deltas' in globals():
            try:
                eg = run_entanglement_geometry_minCut_PLV(RECORDS, ignition_windows, rebound_windows,
                                                          electrodes=electrodes, bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30)},
                                                          time_col=time_col, do_control=True)
                eg['delta_table'].to_csv(os.path.join(csv_dir,'entanglement_deltas.csv'), index=False)
                plot_entanglement_geometry_deltas(eg['delta_table']); pdf.savefig(); plt.savefig(os.path.join(fig_dir,'03_entanglement_deltas.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Entanglement–Geometry:", e)

        # 2) Ridge–PAC coupling
        if fused and ('run_ridge_pac_coupling' in globals()):
            try:
                rpc = run_ridge_pac_coupling(RECORDS, fused=fused, electrodes=electrodes, time_col=time_col,
                                             pac_pairs={'theta→gamma':((7,9),(30,80)), 'alpha→gamma':((8,12),(30,80))},
                                             max_lag_sec=2.0, pac_win_sec=2.0, step_sec=0.25, smooth_sec=0.20,
                                             off_resonant_bands=[(16,18)], show=True)
                pd.DataFrame(rpc['ridge_pac_corr']).to_csv(os.path.join(csv_dir,'ridge_pac_corr.csv'), index=False)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'04_ridge_pac.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Ridge–PAC:", e)

        # 3) Criticality
        if 'run_criticality_analysis' in globals() and 'plot_criticality_deltas' in globals():
            try:
                crit = run_criticality_analysis(RECORDS, ignition_windows, rebound_windows, control_windows,
                                                electrodes=electrodes)
                crit['delta_table'].to_csv(os.path.join(csv_dir,'criticality_deltas.csv'), index=False)
                plot_criticality_deltas(crit['delta_table']); pdf.savefig(); plt.savefig(os.path.join(fig_dir,'05_criticality_deltas.png')); plt.close('all')
                if 'plot_avalanche_ccdf' in globals():
                    plot_avalanche_ccdf(crit['avalanches']); pdf.savefig(); plt.savefig(os.path.join(fig_dir,'05_avalanche_ccdf.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Criticality:", e)

        # 4) Harmonics breadth (build H if needed)
        if 'run_connectome_harmonics_breadth' in globals():
            try:
                if H is None and 'build_functional_harmonics_from_baseline' in globals():
                    H = build_functional_harmonics_from_baseline(RECORDS, electrodes, ignition_windows, time_col=time_col, fband=(4,40), n_modes=64)
                if H is not None:
                    hb = run_connectome_harmonics_breadth(RECORDS, H=H, electrodes=electrodes,
                                                          ignition_windows=ignition_windows, rebound_windows=rebound_windows,
                                                          time_col=time_col, orthonormal=True, do_surrogate=True, n_surr=200)
                    hb['delta_table'].to_csv(os.path.join(csv_dir,'harmonics_breadth_deltas.csv'), index=False)
                    if 'plot_harmonics_power_spectra' in globals():
                        plot_harmonics_power_spectra(hb['spectra']); pdf.savefig(); plt.savefig(os.path.join(fig_dir,'06_harmonics_power.png')); plt.close('all')
                    if 'plot_harmonics_breadth_deltas' in globals():
                        plot_harmonics_breadth_deltas(hb['delta_table']); pdf.savefig(); plt.savefig(os.path.join(fig_dir,'06_harmonics_deltas.png')); plt.close('all')
                else:
                    print("[SKIP] Harmonics breadth (no H and builder not found)")
            except Exception as e:
                print("[WARN] Harmonics breadth:", e)

        # 5) Overlap ETAs (K≥3)
        if fused and ('run_overlap_coherence_etas' in globals()):
            try:
                etas = run_overlap_coherence_etas(RECORDS, fused=fused, electrodes=electrodes, time_col=time_col,
                                                  K=3, win_sec=2.0, step_sec=0.25, span_sec=5.0,
                                                  plv_band=(8,13), pac_pairs={'theta→gamma':((4,8),(30,80))},
                                                  mincut_band=(8,13), beta_band=(1,40), n_boot=200, show=True)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'07_overlap_etas.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Overlap ETAs:", e)

        # 6) Multi-seed surfaces
        if 'run_multi_seed_surface_cuts' in globals():
            try:
                ms = run_multi_seed_surface_cuts(RECORDS, ignition_windows=ignition_windows,
                                                 rebound_windows=rebound_windows, time_col=time_col,
                                                 bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30)},
                                                 electrodes=electrodes, clusters=None,
                                                 control_mode='degree_rewire', n_shuffle=200, graph_density=0.3, show=True)
                pd.DataFrame(ms['delta_table']).to_csv(os.path.join(csv_dir,'multiseed_deltas.csv'), index=False)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'08_multiseed_deltas.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Multi-seed surfaces:", e)

        # 7) Phase metric embedding
        if 'run_phase_embedding_emergent_geometry' in globals():
            try:
                pe = run_phase_embedding_emergent_geometry(RECORDS, ignition_windows=ignition_windows,
                                                           rebound_windows=rebound_windows, control_windows=control_windows,
                                                           time_col=time_col, electrodes=electrodes,
                                                           band=(8,13), n_neighbors=6, n_components=2,
                                                           method='isomap', k_quality=5, n_surr=100, show=True)
                pd.DataFrame(pe['metrics_table']).to_csv(os.path.join(csv_dir,'phase_embedding_metrics.csv'), index=False)
                if 'plot_phase_embedding_quality' in globals():
                    plot_phase_embedding_quality(pe);  # prints surrogate bands
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'09_phase_embedding_metrics.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Phase embedding:", e)

        # 8) Temporal holography (optional)
        if (event_onsets is not None) and ('run_temporal_holography_multiplexed' in globals()):
            try:
                th = run_temporal_holography_multiplexed(RECORDS, event_onsets=event_onsets, labels=event_labels,
                                                         time_col=time_col, electrodes=electrodes,
                                                         ref_electrodes=[e for e in ['O1','O2','Oz','Pz'] if ('EEG.'+e) in RECORDS.columns] or [electrodes[0]],
                                                         ref_band='theta', n_bins=6, feat_window=(-0.5,1.0), n_shuffle=200, show=True)
                pd.DataFrame(th['auc_table']).to_csv(os.path.join(csv_dir,'temporal_holography_auc.csv'), index=False)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'10_temporal_holography_auc.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Temporal holography:", e)

        # 9) ERP/ERSP/ITC (safe) — if you want event-related page regardless of onsets
        if 'erp_ersp_itc_safe' in globals():
            try:
                er = erp_ersp_itc_safe(RECORDS, eeg_channels=[e for e in electrodes if e.endswith(('O1','O2','Oz','Pz'))] or electrodes[:3],
                                       sr_channel=sigcol, time_col=time_col, win_sec=(-3,3), baseline_sec=(-2,-0.5),
                                       center_hz=f0, half_bw_hz=0.6, detect_kwargs={'thresh_mode':'z','z_thresh':2.0},
                                       fmin=4, fmax=40, n_freq=48, w0=6.0, n_perm=200, alpha=0.05, show=True)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'11_erp_ersp_itc.png')); plt.close('all')
            except Exception as e:
                print("[WARN] ERP/ERSP/ITC:", e)
        else:
            print("[SKIP] ERP/ERSP/ITC safe (helper not found)")

        # 10) DTF grid (bidirectional, single-link style)
        if 'run_tvar_dtf' in globals() and 'plot_dtf_grid_bidir_like_single' in globals():
            try:
                # compute DTF for ignition and baseline; supply explicit baseline if missing
                base_windows = control_windows
                if base_windows is None:
                    # coarse complement: everything except ignition (+ buffers)
                    t_all = np.asarray(pd.to_numeric(RECORDS[time_col]).values, float)
                    t0, t1 = float(t_all[0]), float(t_all[-1])
                    buf = 2.0
                    spans = []
                    last = t0
                    for (a, b) in sorted(ignition_windows or []):
                        if a-buf > last: spans.append((last, a-buf))
                        last = b+buf
                    if last < t1: spans.append((last, t1))
                    base_windows = [(a,b) for (a,b) in spans if (b-a)>2.0]

                tv_ign = run_tvar_dtf(RECORDS, channels=electrodes, windows=ignition_windows, time_col=time_col,
                                      order=4, lam=0.995, f0=f0)
                tv_base = run_tvar_dtf(RECORDS, channels=electrodes, windows=base_windows,    time_col=time_col,
                                      order=4, lam=0.995, f0=f0)
                src = [e for e in electrodes if e.endswith(('Oz','F4','O1'))]
                src_channel = src[0] if src else electrodes[0]
                plot_dtf_grid_bidir_like_single(tv_ign, tv_base, src_channel=src_channel,
                                                smooth_sec=0.6, n_cols=3, show_baseline=False, n_perm=500,
                                                session_name=session_name)
                pdf.savefig(); plt.savefig(os.path.join(fig_dir,'12_dtf_grid.png')); plt.close('all')
            except Exception as e:
                print("[WARN] DTF grid:", e)
        else:
            print("[SKIP] DTF grid (helpers not found)")

        # 11) Off-harmonic negative control (MSC @ 16–18 Hz)
        if include_offharmonic_control and ('run_multitaper_msc_harmonics' in globals()):
            try:
                off = run_multitaper_msc_harmonics(RECORDS, x_channels=electrodes, y_channel=sigcol,
                                                   windows=ignition_windows, time_col=time_col,
                                                   half_bw_hz=3.0,
                                                   harmonics=[16.0, 18.0])
                pd.DataFrame(off['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_offharmonic_ign.csv'), index=False)
                if 'plot_msc_harmonics_table' in globals():
                    plot_msc_harmonics_table(off['harmonics_table'], title='MSC @ off-harmonic (Ignition)',
                                             label='Ign', color='tab:gray')
                    pdf.savefig(); plt.savefig(os.path.join(fig_dir,'13_msc_offharmonic.png')); plt.close('all')
            except Exception as e:
                print("[WARN] Off-harmonic MSC:", e)

        # FINAL: At-a-glance summary (lightweight)
        plt.figure(figsize=(8.5, 3.5)); plt.axis('off')
        summary_lines = []
        for name, csv in [('entanglement_deltas','Δmin-cut/Δentropy'),
                          ('criticality_deltas','Δβ/Δα'),
                          ('harmonics_breadth_deltas','ΔH/ΔPR/ΔTop10'),
                          ('multiseed_deltas','ΔMultiCut'),
                          ('phase_embedding_metrics','T/C/Stress')]:
            f = os.path.join(csv_dir, f"{name}.csv")
            if os.path.exists(f):
                df = pd.read_csv(f)
                summary_lines.append(f"{name}: OK (rows={len(df)})")
            else:
                summary_lines.append(f"{name}: —")
        plt.text(0.02, 0.9, "Summary:", fontsize=12, weight='bold')
        plt.text(0.02, 0.7, "\n".join(summary_lines), fontsize=10, va='top')
        pdf.savefig(); plt.savefig(os.path.join(fig_dir,'99_summary.png')); plt.close('all')

    return {'export_dir': base_dir, 'pdf': pdf_path, 'fig_dir': fig_dir, 'csv_dir': csv_dir, 'schumann_ref': sigcol}


In [ ]:
report = run_one_click_session_report_v2(
    RECORDS,
    session_name='session_v2',
    time_col='Timestamp',
    electrodes=None,                            # autodetect EEG.*
    ignition_windows=[(290,310),(580,600)],
    rebound_windows=[(310,325)],
    control_windows=None,                       # optional
    event_onsets=None,                          # optional (Temporal Holography)
    event_labels=None,
    H=None                                      # optional harmonics matrix
)
print("PDF:", report['pdf'])


In [ ]:
# ==============================
# HoloPipeline v1 — Orchestrator
# ==============================
from __future__ import annotations
import os, json
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
from scipy import signal

# ---- tiny utils ----
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d
def infer_fs(RECORDS: pd.DataFrame, time_col='Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs")
    return float(1.0/np.median(dt))
def pick_best_channel_for_schumann(RECORDS, time_col='Timestamp'):
    fs = infer_fs(RECORDS, time_col)
    cands = [c for c in RECORDS.columns if c.startswith('EEG.')]
    if not cands: return None
    scores=[]
    for ch in cands:
        x = np.asarray(RECORDS[ch].values, float)
        f, p = signal.welch(x, fs=fs, nperseg=4*int(fs))
        def band(a,b): sel=(f>=a)&(f<=b); 
        return np.trapz(p[sel], f[sel]) if np.any(sel) else 0.0
        low = band(4,12); emg = band(40,90); mains = band(55,65)
        scores.append((low/(emg+1e-12) - 0.2*mains, ch))
    scores.sort(reverse=True); return scores[0][1]

# ---------------- config ----------------
@dataclass
class PipelineConfig:
    session_name: str = 'session_pipeline'
    time_col: str = 'Timestamp'
    electrodes: Optional[List[str]] = None              # autodetect EEG.* if None
    sr_channel: Optional[str] = None                    # Schumann ref; if None, auto-pick
    ignition_windows: Optional[List[Tuple[float,float]]] = None
    baseline_windows: Optional[List[Tuple[float,float]]] = None
    rebound_windows: Optional[List[Tuple[float,float]]] = None
    # bands & harmonics
    bands: Dict[str, Tuple[float,float]] = field(default_factory=lambda: {'theta':(4,8),'alpha':(8,13),'beta':(13,30)})
    harmonics: List[float] = field(default_factory=lambda: [7.83,14.3,20.8,27.3,33.8])
    # scoring weights (0..1)
    weights: Dict[str, float] = field(default_factory=lambda: {
        'msc':0.12, 'dtf':0.12, 'entanglement':0.10, 'criticality':0.10,
        'breadth':0.10, 'overlap_etas':0.08, 'multiseed':0.08, 'embedding':0.08,
        'erp_tf':0.12, 'scf':0.05, 'plv_topo':0.05
    })
    out_dir: str = 'exports_pipeline'
    show_figs: bool = True   # set False to suppress interactive plots

# ---------------- scoring helpers ----------------
def _norm_pos(x, ref):  # positive is good; scale by ref to 0..1
    if ref<=0: return float(x>0)
    return float(np.clip(x/ref, 0, 1))
def _sigmoid(x, k=4.0): return float(1/(1+np.exp(-k*x)))      # soft mapping -> 0..1
def _clip01(x): return float(np.clip(x,0,1))

# --- individual step scorers (consume dicts returned by helpers) ---
def score_msc(msc_ign: Dict, msc_base: Optional[Dict]) -> float:
    try:
        ign = pd.DataFrame(msc_ign['harmonics_table'])['MSC'].mean()
        if msc_base is not None:
            bas = pd.DataFrame(msc_base['harmonics_table'])['MSC'].mean()
            delta = ign - bas
            return _clip01(_sigmoid(delta*3))      # higher ign coherence than base
        return _clip01(ign)                         # fallback: absolute
    except Exception: return 0.0

def score_dtf(tv_ign: Dict, tv_base: Dict, electrodes: List[str], smooth_sec=0.5) -> float:
    try:
        fs = tv_ign['fs']; win = max(1,int(round(smooth_sec*fs)))
        D_I = np.asarray(tv_ign['DTF_t']); D_B = np.asarray(tv_base['DTF_t'])
        # choose a source (favor Oz/F4 if present)
        src = [e for e in electrodes if e.endswith(('Oz','F4','O1'))]
        src_idx = electrodes.index(src[0] if src else electrodes[0])
        targets = [i for i in range(len(electrodes)) if i!=src_idx]
        # fraction of targets where mean(ign) − mean(base) > 95% null (circular-shift ign)
        rng = np.random.default_rng(7)
        sig_count=0
        for tgt in targets:
            ijI = np.convolve(D_I[:,tgt,src_idx], np.ones(win)/win, mode='same')
            ijB = np.convolve(D_B[:,tgt,src_idx], np.ones(win)/win, mode='same')
            mean_I = np.nanmean(ijI); mean_B = np.nanmean(ijB)
            # null
            null=[]
            T=len(ijI)
            for _ in range(200):
                s=int(rng.integers(1,T-1)); null.append(np.nanmean(np.r_[ijI[-s:],ijI[:-s]]) - mean_B)
            thr95 = np.nanpercentile(null,95)
            if (mean_I - mean_B) > thr95: sig_count+=1
        return _clip01(sig_count/max(1,len(targets)))
    except Exception: return 0.0

def score_entanglement(eg: Dict) -> float:
    try:
        df = eg['delta_table']
        # average across bands: want Δmin-cut>0 and ΔPLV>0
        if 'd_mincut' in df and 'd_plv' in df:
            v = float(np.nanmean(df['d_mincut'])) + 0.7*float(np.nanmean(df['d_plv']))
            return _clip01(_sigmoid(v))
    except Exception: ...
    return 0.0

def score_criticality(crit: Dict) -> float:
    try:
        d = crit['delta_table'].iloc[0]
        # β should go down; α should approach 1
        s_beta = _sigmoid(-float(d['d_beta'])*3)               # flatter 1/f → better
        s_alpha= _sigmoid((float(d['d_alpha']))*3)              # increase toward ~1
        return _clip01(0.6*s_beta + 0.4*s_alpha)
    except Exception: return 0.0

def score_breadth(hb: Dict) -> float:
    try:
        d = hb['delta_table'].iloc[0]
        v = 0.6*float(d.get('d_H',0)) + 0.4*float(d.get('d_PR',0))
        return _clip01(_sigmoid(v*3))
    except Exception: return 0.0

def score_overlap_etas(etas: Dict) -> float:
    try:
        # use PLV ETA amplitude at 0s relative to median |ETA|
        tau = np.asarray(etas['eta_time']); idx = int(np.argmin(np.abs(tau)))
        v = float(etas['eta_plv'][idx])
        base = np.nanmedian(np.abs(etas['eta_plv']))
        return _clip01(_sigmoid((v-base)*4))
    except Exception: return 0.0

def score_multiseed(ms: Dict) -> float:
    try:
        df = ms['delta_table']; v = float(np.nanmean(df['d_cap']))
        # normalize by 95% of null if provided
        if ms.get('shuffle_null') is not None and not ms['shuffle_null'].empty:
            null95 = np.nanpercentile(ms['shuffle_null']['cap_perm'],95)
            return _clip01(v/(null95+1e-12))
        return _clip01(_sigmoid(v))
    except Exception: return 0.0

def score_embedding(pe: Dict) -> float:
    try:
        mt = pe['metrics_table'].set_index('state')
        if 'ignition' in mt.index and 'baseline' in mt.index:
            dv = (mt.loc['ignition','trust'] - mt.loc['baseline','trust']) \
               + (mt.loc['ignition','cont']  - mt.loc['baseline','cont']) \
               - (mt.loc['ignition','stress']- mt.loc['baseline','stress'])
            return _clip01(_sigmoid(float(dv)))
    except Exception: return 0.0

def score_erp_tf(er: Dict) -> float:
    try:
        # 1 if ERSP TF cluster significant; boost if ITC also significant
        s = 1.0 if er.get('ERSP_cluster_sig',False) else 0.0
        if er.get('ITC_cluster_sig',False): s = min(1.0, s+0.25)
        return s
    except Exception: return 0.0

def score_scf(scf: Dict) -> float:
    try:
        # use integrated |SCF| across α harmonics, z-score vs 16–18 Hz (if available)
        tbl = scf['table']
        return _clip01(_sigmoid(float(tbl['SCF_int'].mean())*0.5))
    except Exception: return 0.0

def score_plv_topo(plv_res: Dict) -> float:
    try:
        # mean PLV across channels @ fundamental
        df = plv_res['table']; v = float(np.nanmean(df[df['freq']==7.83]['PLV']))
        return _clip01(v)
    except Exception: return 0.0

# ---------------- main pipeline ----------------
def run_holo_pipeline(RECORDS: pd.DataFrame, cfg: PipelineConfig) -> Dict[str, object]:
    fs = infer_fs(RECORDS, cfg.time_col)
    # electrodes & SR
    electrodes = cfg.electrodes or [c for c in RECORDS.columns if c.startswith('EEG.')]
    sr = cfg.sr_channel or pick_best_channel_for_schumann(RECORDS, cfg.time_col) or (electrodes[0] if electrodes else None)
    if sr is None: raise ValueError("No sr_channel and no EEG.* found.")

    # outputs
    out_dir = _ensure_dir(os.path.join(cfg.out_dir, cfg.session_name))
    fig_dir = _ensure_dir(os.path.join(out_dir, 'FIG'))
    csv_dir = _ensure_dir(os.path.join(out_dir, 'CSV'))

    scores = {}
    blocks = {}

    # 0) MSC & WTC
    if 'run_multitaper_msc_harmonics' in globals():
        try:
            msc_ign = run_multitaper_msc_harmonics(RECORDS, x_channels=electrodes, y_channel=sr,
                                                   windows=cfg.ignition_windows, time_col=cfg.time_col,
                                                   half_bw_hz=3.0, harmonics=cfg.harmonics)
            msc_base = run_multitaper_msc_harmonics(RECORDS, x_channels=electrodes, y_channel=sr,
                                                    windows=cfg.baseline_windows, time_col=cfg.time_col,
                                                    half_bw_hz=3.0, harmonics=cfg.harmonics) if cfg.baseline_windows else None
            pd.DataFrame(msc_ign['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_ign.csv'), index=False)
            if msc_base is not None:
                pd.DataFrame(msc_base['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_base.csv'), index=False)
            scores['msc'] = score_msc(msc_ign, msc_base)
            blocks['msc'] = msc_ign
        except Exception as e:
            print("[MSC] skipped:", e); scores['msc']=0.0

    # 1) Entanglement–Geometry
    if 'run_entanglement_geometry_minCut_PLV' in globals():
        try:
            eg = run_entanglement_geometry_minCut_PLV(RECORDS, cfg.ignition_windows, cfg.rebound_windows,
                                                      electrodes=electrodes, bands=cfg.bands, time_col=cfg.time_col, do_control=True)
            eg['delta_table'].to_csv(os.path.join(csv_dir,'entanglement.csv'), index=False)
            scores['entanglement'] = score_entanglement(eg)
            blocks['entanglement'] = eg
        except Exception as e:
            print("[Entanglement] skipped:", e); scores['entanglement']=0.0

    # 2) DTF grid (TV-AR)
    if 'run_tvar_dtf' in globals():
        try:
            base_wins = cfg.baseline_windows
            if base_wins is None and cfg.ignition_windows:
                # quick complement baseline
                t_all = np.asarray(pd.to_numeric(RECORDS[cfg.time_col]).values, float)
                t0, t1 = float(t_all[0]), float(t_all[-1]); buf=2.0
                spans=[]; last=t0
                for (a,b) in sorted(cfg.ignition_windows):
                    if a-buf > last: spans.append((last, a-buf))
                    last=b+buf
                if last<t1: spans.append((last,t1))
                base_wins=[(a,b) for (a,b) in spans if (b-a)>2.0]
            tvI = run_tvar_dtf(RECORDS, channels=electrodes, windows=cfg.ignition_windows, time_col=cfg.time_col,
                               order=4, lam=0.995, f0=cfg.harmonics[0])
            tvB = run_tvar_dtf(RECORDS, channels=electrodes, windows=base_wins, time_col=cfg.time_col,
                               order=4, lam=0.995, f0=cfg.harmonics[0])
            scores['dtf'] = score_dtf(tvI, tvB, electrodes)
            blocks['dtf'] = {'ign':tvI, 'base':tvB}
            # simple grid figure
            if 'plot_dtf_grid_bidir_like_single' in globals() and cfg.show_figs:
                src = [e for e in electrodes if e.endswith(('Oz','F4','O1'))]
                src_channel = src[0] if src else electrodes[0]
                plot_dtf_grid_bidir_like_single(tvI, tvB, src_channel=src_channel, smooth_sec=0.6, n_cols=3,
                                                show_baseline=False, n_perm=300, session_name=cfg.session_name)
                plt.savefig(os.path.join(fig_dir,'dtf_grid.png')); plt.close('all')
        except Exception as e:
            print("[DTF] skipped:", e); scores['dtf']=0.0

    # 3) Criticality
    if 'run_criticality_analysis' in globals():
        try:
            crit = run_criticality_analysis(RECORDS, cfg.ignition_windows, cfg.rebound_windows, cfg.baseline_windows,
                                            electrodes=electrodes)
            crit['delta_table'].to_csv(os.path.join(csv_dir,'criticality.csv'), index=False)
            scores['criticality']=score_criticality(crit); blocks['criticality']=crit
            if 'plot_criticality_deltas' in globals() and cfg.show_figs:
                plot_criticality_deltas(crit['delta_table']); plt.savefig(os.path.join(fig_dir,'criticality.png')); plt.close('all')
        except Exception as e:
            print("[Criticality] skipped:", e); scores['criticality']=0.0

    # 4) Harmonics breadth (functional harmonics if H not supplied)
    if 'run_connectome_harmonics_breadth' in globals():
        try:
            H = None
            if 'build_functional_harmonics_from_baseline' in globals():
                H = build_functional_harmonics_from_baseline(RECORDS, electrodes, cfg.ignition_windows,
                                                             time_col=cfg.time_col, fband=(4,40), n_modes=64)
            if H is not None:
                hb = run_connectome_harmonics_breadth(RECORDS, H=H, electrodes=electrodes,
                                                      ignition_windows=cfg.ignition_windows, rebound_windows=cfg.rebound_windows,
                                                      time_col=cfg.time_col, orthonormal=True, do_surrogate=True, n_surr=200)
                hb['delta_table'].to_csv(os.path.join(csv_dir,'breadth.csv'), index=False)
                scores['breadth']=score_breadth(hb); blocks['breadth']=hb
        except Exception as e:
            print("[Breadth] skipped:", e); scores['breadth']=0.0

    # 5) Overlap ETAs (needs fused)
    if 'run_overlap_coherence_etas' in globals() and 'detect_and_plot_schumann_microgrid_with_global_tf' in globals():
        try:
            fused = detect_and_plot_schumann_microgrid_with_global_tf(RECORDS, signal_col=sr, time_col=cfg.time_col, show=False)
            etas = run_overlap_coherence_etas(RECORDS, fused=fused, electrodes=electrodes, time_col=cfg.time_col,
                                              K=3, win_sec=2.0, step_sec=0.25, span_sec=5.0,
                                              plv_band=(8,13), pac_pairs={'theta→gamma':((4,8),(30,80))},
                                              mincut_band=(8,13), beta_band=(1,40), n_boot=200, show=cfg.show_figs)
            scores['overlap_etas']=score_overlap_etas(etas); blocks['overlap_etas']=etas
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'overlap_etas.png')); plt.close('all')
        except Exception as e:
            print("[Overlap-ETAs] skipped:", e); scores['overlap_etas']=0.0

    # 6) Multi-seed surfaces
    if 'run_multi_seed_surface_cuts' in globals():
        try:
            ms = run_multi_seed_surface_cuts(RECORDS, ignition_windows=cfg.ignition_windows,
                                             rebound_windows=cfg.rebound_windows, time_col=cfg.time_col,
                                             bands=cfg.bands, electrodes=electrodes,
                                             control_mode='degree_rewire', n_shuffle=200, graph_density=0.3, show=cfg.show_figs)
            pd.DataFrame(ms['delta_table']).to_csv(os.path.join(csv_dir,'multiseed.csv'), index=False)
            scores['multiseed']=score_multiseed(ms); blocks['multiseed']=ms
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'multiseed.png')); plt.close('all')
        except Exception as e:
            print("[Multi-seed] skipped:", e); scores['multiseed']=0.0

    # 7) Phase embedding
    if 'run_phase_embedding_emergent_geometry' in globals():
        try:
            pe = run_phase_embedding_emergent_geometry(RECORDS, ignition_windows=cfg.ignition_windows, rebound_windows=cfg.rebound_windows,
                                                       control_windows=cfg.baseline_windows, time_col=cfg.time_col,
                                                       electrodes=electrodes, band=(8,13),
                                                       n_neighbors=6, n_components=2, method='isomap', k_quality=5,
                                                       n_surr=100, show=cfg.show_figs)
            pd.DataFrame(pe['metrics_table']).to_csv(os.path.join(csv_dir,'embedding.csv'), index=False)
            scores['embedding']=score_embedding(pe); blocks['embedding']=pe
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'embedding.png')); plt.close('all')
        except Exception as e:
            print("[Embedding] skipped:", e); scores['embedding']=0.0

    # 8) ERP/ERSP/ITC (safe)
    if 'erp_ersp_itc_safe' in globals():
        try:
            er = erp_ersp_itc_safe(RECORDS, eeg_channels=[e for e in electrodes if e.endswith(('O1','O2','Oz','Pz'))] or electrodes[:3],
                                   sr_channel=sr, time_col=cfg.time_col, win_sec=(-3,3), baseline_sec=(-2,-0.5),
                                   center_hz=cfg.harmonics[0], half_bw_hz=0.6,
                                   detect_kwargs={'thresh_mode':'z','z_thresh':2.0},
                                   fmin=4, fmax=40, n_freq=48, w0=6.0, n_perm=200, alpha=0.05, show=cfg.show_figs)
            scores['erp_tf']=score_erp_tf(er); blocks['erp_tf']=er
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'erp_tf.png')); plt.close('all')
        except Exception as e:
            print("[ERP/TF] skipped:", e); scores['erp_tf']=0.0

    # 9) SCF (optional; simple)
    if 'scf_at_harmonics' in globals():
        try:
            scf = scf_at_harmonics(RECORDS, sr, harmonics=cfg.harmonics, windows=cfg.ignition_windows, time_col=cfg.time_col)
            pd.DataFrame(scf['table']).to_csv(os.path.join(csv_dir,'scf.csv'), index=False)
            scores['scf']=score_scf(scf); blocks['scf']=scf
        except Exception as e:
            print("[SCF] skipped:", e); scores['scf']=0.0

    # 10) PLV topography (optional quick score)
    if 'run_plv_harmonics_topography' in globals():
        try:
            plv_res = run_plv_harmonics_topography(RECORDS, eeg_channels=electrodes, sr_channel=sr,
                                                   harmonics=cfg.harmonics, half_bw_hz=0.6,
                                                   windows=cfg.ignition_windows, time_col=cfg.time_col)
            pd.DataFrame(plv_res['table']).to_csv(os.path.join(csv_dir,'plv_topo.csv'), index=False)
            scores['plv_topo']=score_plv_topo(plv_res); blocks['plv_topo']=plv_res
        except Exception as e:
            print("[PLV topo] skipped:", e); scores['plv_topo']=0.0

    # ---------- composite overall score ----------
    # normalize weights over steps actually present
    present = {k:v for k,v in scores.items() if not np.isnan(v)}
    Wsum = sum(cfg.weights.get(k,0.0) for k in present.keys())
    if Wsum<=0: overall=0.0
    else:
        overall = float(sum(cfg.weights.get(k,0.0)*present[k] for k in present.keys())/Wsum)

    # save scores + JSON
#     pd.DataFrame([scores | {'overall':overall}]).to_csv(os.path.join(csv_dir,'scores.csv'), index=False)
#     with open(os.path.join(out_dir,'scores.json'),'w') as f: json.dump(scores | {'overall':overall}, f, indent=2)

    final_scores = scores.copy()
    final_scores['overall'] = overall

    pd.DataFrame([final_scores]).to_csv(os.path.join(csv_dir,'scores.csv'), index=False)
    with open(os.path.join(out_dir,'scores.json'),'w') as f:
    json.dump(final_scores, f, indent=2)

    
    
    print(f"\nComposite overall score: {overall:.3f}")
    return {'scores':scores, 'overall':overall, 'blocks':blocks,
            'export_dir':out_dir, 'fig_dir':fig_dir, 'csv_dir':csv_dir,
            'electrodes':electrodes, 'sr_channel':sr, 'fs':fs}


In [ ]:
cfg = PipelineConfig(
    session_name='S01',
    time_col='Timestamp',
    electrodes=None,                         # autodetect EEG.*
    sr_channel="EEG.F4",                         # auto-pick posterior channel
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],    # explicit baseline windows (recommended)
    rebound_windows=[(310,325)],
    # optional: adjust weights per your priorities
    weights={'msc':0.12,'dtf':0.12,'entanglement':0.10,'criticality':0.10,
             'breadth':0.10,'overlap_etas':0.08,'multiseed':0.08,'embedding':0.08,
             'erp_tf':0.12,'scf':0.05,'plv_topo':0.05},
    out_dir='exports_pipeline',
    show_figs=True
)

out = run_holo_pipeline(RECORDS, cfg)
print("Scores:", out['scores'])
print("Overall:", out['overall'])
print("Results in:", out['export_dir'])


In [ ]:
# ==============================
# HoloPipeline v1 — Orchestrator
# ==============================
from __future__ import annotations
import os, json
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
from scipy import signal

# ---- tiny utils ----
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d
def infer_fs(RECORDS: pd.DataFrame, time_col='Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs")
    return float(1.0/np.median(dt))
def pick_best_channel_for_schumann(RECORDS, time_col='Timestamp'):
    fs = infer_fs(RECORDS, time_col)
    cands = [c for c in RECORDS.columns if c.startswith('EEG.')]
    if not cands: return None
    scores=[]
    for ch in cands:
        x = np.asarray(RECORDS[ch].values, float)
        f, p = signal.welch(x, fs=fs, nperseg=4*int(fs))
        def band(a,b): sel=(f>=a)&(f<=b); 
        return np.trapz(p[sel], f[sel]) if np.any(sel) else 0.0
        low = band(4,12); emg = band(40,90); mains = band(55,65)
        scores.append((low/(emg+1e-12) - 0.2*mains, ch))
    scores.sort(reverse=True); return scores[0][1]

# ---------------- config ----------------
@dataclass
class PipelineConfig:
    session_name: str = 'session_pipeline'
    time_col: str = 'Timestamp'
    electrodes: Optional[List[str]] = None              # autodetect EEG.* if None
    sr_channel: Optional[str] = None                    # Schumann ref; if None, auto-pick
    ignition_windows: Optional[List[Tuple[float,float]]] = None
    baseline_windows: Optional[List[Tuple[float,float]]] = None
    rebound_windows: Optional[List[Tuple[float,float]]] = None
    # bands & harmonics
    bands: Dict[str, Tuple[float,float]] = field(default_factory=lambda: {'theta':(4,8),'alpha':(8,13),'beta':(13,30)})
    harmonics: List[float] = field(default_factory=lambda: [7.83,14.3,20.8,27.3,33.8])
    # scoring weights (0..1)
    weights: Dict[str, float] = field(default_factory=lambda: {
        'msc':0.12, 'dtf':0.12, 'entanglement':0.10, 'criticality':0.10,
        'breadth':0.10, 'overlap_etas':0.08, 'multiseed':0.08, 'embedding':0.08,
        'erp_tf':0.12, 'scf':0.05, 'plv_topo':0.05
    })
    out_dir: str = 'exports_pipeline'
    show_figs: bool = True   # set False to suppress interactive plots

# ---------------- scoring helpers ----------------
def _norm_pos(x, ref):  # positive is good; scale by ref to 0..1
    if ref<=0: return float(x>0)
    return float(np.clip(x/ref, 0, 1))
def _sigmoid(x, k=4.0): return float(1/(1+np.exp(-k*x)))      # soft mapping -> 0..1
def _clip01(x): return float(np.clip(x,0,1))

# --- individual step scorers (consume dicts returned by helpers) ---
def score_msc(msc_ign: Dict, msc_base: Optional[Dict]) -> float:
    try:
        ign = pd.DataFrame(msc_ign['harmonics_table'])['MSC'].mean()
        if msc_base is not None:
            bas = pd.DataFrame(msc_base['harmonics_table'])['MSC'].mean()
            delta = ign - bas
            return _clip01(_sigmoid(delta*3))      # higher ign coherence than base
        return _clip01(ign)                         # fallback: absolute
    except Exception: return 0.0

def score_dtf(tv_ign: Dict, tv_base: Dict, electrodes: List[str], smooth_sec=0.5) -> float:
    try:
        fs = tv_ign['fs']; win = max(1,int(round(smooth_sec*fs)))
        D_I = np.asarray(tv_ign['DTF_t']); D_B = np.asarray(tv_base['DTF_t'])
        # choose a source (favor Oz/F4 if present)
        src = [e for e in electrodes if e.endswith(('Oz','F4','O1'))]
        src_idx = electrodes.index(src[0] if src else electrodes[0])
        targets = [i for i in range(len(electrodes)) if i!=src_idx]
        # fraction of targets where mean(ign) − mean(base) > 95% null (circular-shift ign)
        rng = np.random.default_rng(7)
        sig_count=0
        for tgt in targets:
            ijI = np.convolve(D_I[:,tgt,src_idx], np.ones(win)/win, mode='same')
            ijB = np.convolve(D_B[:,tgt,src_idx], np.ones(win)/win, mode='same')
            mean_I = np.nanmean(ijI); mean_B = np.nanmean(ijB)
            # null
            null=[]
            T=len(ijI)
            for _ in range(200):
                s=int(rng.integers(1,T-1)); null.append(np.nanmean(np.r_[ijI[-s:],ijI[:-s]]) - mean_B)
            thr95 = np.nanpercentile(null,95)
            if (mean_I - mean_B) > thr95: sig_count+=1
        return _clip01(sig_count/max(1,len(targets)))
    except Exception: return 0.0

def score_entanglement(eg: Dict) -> float:
    try:
        df = eg['delta_table']
        # average across bands: want Δmin-cut>0 and ΔPLV>0
        if 'd_mincut' in df and 'd_plv' in df:
            v = float(np.nanmean(df['d_mincut'])) + 0.7*float(np.nanmean(df['d_plv']))
            return _clip01(_sigmoid(v))
    except Exception: ...
    return 0.0

def score_criticality(crit: Dict) -> float:
    try:
        d = crit['delta_table'].iloc[0]
        # β should go down; α should approach 1
        s_beta = _sigmoid(-float(d['d_beta'])*3)               # flatter 1/f → better
        s_alpha= _sigmoid((float(d['d_alpha']))*3)              # increase toward ~1
        return _clip01(0.6*s_beta + 0.4*s_alpha)
    except Exception: return 0.0

def score_breadth(hb: Dict) -> float:
    try:
        d = hb['delta_table'].iloc[0]
        v = 0.6*float(d.get('d_H',0)) + 0.4*float(d.get('d_PR',0))
        return _clip01(_sigmoid(v*3))
    except Exception: return 0.0

def score_overlap_etas(etas: Dict) -> float:
    try:
        # use PLV ETA amplitude at 0s relative to median |ETA|
        tau = np.asarray(etas['eta_time']); idx = int(np.argmin(np.abs(tau)))
        v = float(etas['eta_plv'][idx])
        base = np.nanmedian(np.abs(etas['eta_plv']))
        return _clip01(_sigmoid((v-base)*4))
    except Exception: return 0.0

def score_multiseed(ms: Dict) -> float:
    try:
        df = ms['delta_table']; v = float(np.nanmean(df['d_cap']))
        # normalize by 95% of null if provided
        if ms.get('shuffle_null') is not None and not ms['shuffle_null'].empty:
            null95 = np.nanpercentile(ms['shuffle_null']['cap_perm'],95)
            return _clip01(v/(null95+1e-12))
        return _clip01(_sigmoid(v))
    except Exception: return 0.0

def score_embedding(pe: Dict) -> float:
    try:
        mt = pe['metrics_table'].set_index('state')
        if 'ignition' in mt.index and 'baseline' in mt.index:
            dv = (mt.loc['ignition','trust'] - mt.loc['baseline','trust']) \
               + (mt.loc['ignition','cont']  - mt.loc['baseline','cont']) \
               - (mt.loc['ignition','stress']- mt.loc['baseline','stress'])
            return _clip01(_sigmoid(float(dv)))
    except Exception: return 0.0

def score_erp_tf(er: Dict) -> float:
    try:
        # 1 if ERSP TF cluster significant; boost if ITC also significant
        s = 1.0 if er.get('ERSP_cluster_sig',False) else 0.0
        if er.get('ITC_cluster_sig',False): s = min(1.0, s+0.25)
        return s
    except Exception: return 0.0

def score_scf(scf: Dict) -> float:
    try:
        # use integrated |SCF| across α harmonics, z-score vs 16–18 Hz (if available)
        tbl = scf['table']
        return _clip01(_sigmoid(float(tbl['SCF_int'].mean())*0.5))
    except Exception: return 0.0

def score_plv_topo(plv_res: Dict) -> float:
    try:
        # mean PLV across channels @ fundamental
        df = plv_res['table']; v = float(np.nanmean(df[df['freq']==7.83]['PLV']))
        return _clip01(v)
    except Exception: return 0.0

# ---------------- main pipeline ----------------
def run_holo_pipeline(RECORDS: pd.DataFrame, cfg: PipelineConfig) -> Dict[str, object]:
    fs = infer_fs(RECORDS, cfg.time_col)
    # electrodes & SR
    electrodes = cfg.electrodes or [c for c in RECORDS.columns if c.startswith('EEG.')]
    sr = cfg.sr_channel or pick_best_channel_for_schumann(RECORDS, cfg.time_col) or (electrodes[0] if electrodes else None)
    if sr is None: raise ValueError("No sr_channel and no EEG.* found.")

    # outputs
    out_dir = _ensure_dir(os.path.join(cfg.out_dir, cfg.session_name))
    fig_dir = _ensure_dir(os.path.join(out_dir, 'FIG'))
    csv_dir = _ensure_dir(os.path.join(out_dir, 'CSV'))

    scores = {}
    blocks = {}

    # 0) MSC & WTC
    if 'run_multitaper_msc_harmonics' in globals():
        try:
            msc_ign = run_multitaper_msc_harmonics(RECORDS, x_channels=electrodes, y_channel=sr,
                                                   windows=cfg.ignition_windows, time_col=cfg.time_col,
                                                   half_bw_hz=3.0, harmonics=cfg.harmonics)
            msc_base = run_multitaper_msc_harmonics(RECORDS, x_channels=electrodes, y_channel=sr,
                                                    windows=cfg.baseline_windows, time_col=cfg.time_col,
                                                    half_bw_hz=3.0, harmonics=cfg.harmonics) if cfg.baseline_windows else None
            pd.DataFrame(msc_ign['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_ign.csv'), index=False)
            if msc_base is not None:
                pd.DataFrame(msc_base['harmonics_table']).to_csv(os.path.join(csv_dir,'msc_base.csv'), index=False)
            scores['msc'] = score_msc(msc_ign, msc_base)
            blocks['msc'] = msc_ign
        except Exception as e:
            print("[MSC] skipped:", e); scores['msc']=0.0

    # 1) Entanglement–Geometry
    if 'run_entanglement_geometry_minCut_PLV' in globals():
        try:
            eg = run_entanglement_geometry_minCut_PLV(RECORDS, cfg.ignition_windows, cfg.rebound_windows,
                                                      electrodes=electrodes, bands=cfg.bands, time_col=cfg.time_col, do_control=True)
            eg['delta_table'].to_csv(os.path.join(csv_dir,'entanglement.csv'), index=False)
            scores['entanglement'] = score_entanglement(eg)
            blocks['entanglement'] = eg
        except Exception as e:
            print("[Entanglement] skipped:", e); scores['entanglement']=0.0

    # 2) DTF grid (TV-AR)
    if 'run_tvar_dtf' in globals():
        try:
            base_wins = cfg.baseline_windows
            if base_wins is None and cfg.ignition_windows:
                # quick complement baseline
                t_all = np.asarray(pd.to_numeric(RECORDS[cfg.time_col]).values, float)
                t0, t1 = float(t_all[0]), float(t_all[-1]); buf=2.0
                spans=[]; last=t0
                for (a,b) in sorted(cfg.ignition_windows):
                    if a-buf > last: spans.append((last, a-buf))
                    last=b+buf
                if last<t1: spans.append((last,t1))
                base_wins=[(a,b) for (a,b) in spans if (b-a)>2.0]
            tvI = run_tvar_dtf(RECORDS, channels=electrodes, windows=cfg.ignition_windows, time_col=cfg.time_col,
                               order=4, lam=0.995, f0=cfg.harmonics[0])
            tvB = run_tvar_dtf(RECORDS, channels=electrodes, windows=base_wins, time_col=cfg.time_col,
                               order=4, lam=0.995, f0=cfg.harmonics[0])
            scores['dtf'] = score_dtf(tvI, tvB, electrodes)
            blocks['dtf'] = {'ign':tvI, 'base':tvB}
            # simple grid figure
            if 'plot_dtf_grid_bidir_like_single' in globals() and cfg.show_figs:
                src = [e for e in electrodes if e.endswith(('Oz','F4','O1'))]
                src_channel = src[0] if src else electrodes[0]
                plot_dtf_grid_bidir_like_single(tvI, tvB, src_channel=src_channel, smooth_sec=0.6, n_cols=3,
                                                show_baseline=False, n_perm=300, session_name=cfg.session_name)
                plt.savefig(os.path.join(fig_dir,'dtf_grid.png')); plt.close('all')
        except Exception as e:
            print("[DTF] skipped:", e); scores['dtf']=0.0

    # 3) Criticality
    if 'run_criticality_analysis' in globals():
        try:
            crit = run_criticality_analysis(RECORDS, cfg.ignition_windows, cfg.rebound_windows, cfg.baseline_windows,
                                            electrodes=electrodes)
            crit['delta_table'].to_csv(os.path.join(csv_dir,'criticality.csv'), index=False)
            scores['criticality']=score_criticality(crit); blocks['criticality']=crit
            if 'plot_criticality_deltas' in globals() and cfg.show_figs:
                plot_criticality_deltas(crit['delta_table']); plt.savefig(os.path.join(fig_dir,'criticality.png')); plt.close('all')
        except Exception as e:
            print("[Criticality] skipped:", e); scores['criticality']=0.0

    # 4) Harmonics breadth (functional harmonics if H not supplied)
    if 'run_connectome_harmonics_breadth' in globals():
        try:
            H = None
            if 'build_functional_harmonics_from_baseline' in globals():
                H = build_functional_harmonics_from_baseline(RECORDS, electrodes, cfg.ignition_windows,
                                                             time_col=cfg.time_col, fband=(4,40), n_modes=64)
            if H is not None:
                hb = run_connectome_harmonics_breadth(RECORDS, H=H, electrodes=electrodes,
                                                      ignition_windows=cfg.ignition_windows, rebound_windows=cfg.rebound_windows,
                                                      time_col=cfg.time_col, orthonormal=True, do_surrogate=True, n_surr=200)
                hb['delta_table'].to_csv(os.path.join(csv_dir,'breadth.csv'), index=False)
                scores['breadth']=score_breadth(hb); blocks['breadth']=hb
        except Exception as e:
            print("[Breadth] skipped:", e); scores['breadth']=0.0

    # 5) Overlap ETAs (needs fused)
    if 'run_overlap_coherence_etas' in globals() and 'detect_and_plot_schumann_microgrid_with_global_tf' in globals():
        try:
            fused = detect_and_plot_schumann_microgrid_with_global_tf(RECORDS, signal_col=sr, time_col=cfg.time_col, show=False)
            etas = run_overlap_coherence_etas(RECORDS, fused=fused, electrodes=electrodes, time_col=cfg.time_col,
                                              K=3, win_sec=2.0, step_sec=0.25, span_sec=5.0,
                                              plv_band=(8,13), pac_pairs={'theta→gamma':((4,8),(30,80))},
                                              mincut_band=(8,13), beta_band=(1,40), n_boot=200, show=cfg.show_figs)
            scores['overlap_etas']=score_overlap_etas(etas); blocks['overlap_etas']=etas
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'overlap_etas.png')); plt.close('all')
        except Exception as e:
            print("[Overlap-ETAs] skipped:", e); scores['overlap_etas']=0.0

    # 6) Multi-seed surfaces
    if 'run_multi_seed_surface_cuts' in globals():
        try:
            ms = run_multi_seed_surface_cuts(RECORDS, ignition_windows=cfg.ignition_windows,
                                             rebound_windows=cfg.rebound_windows, time_col=cfg.time_col,
                                             bands=cfg.bands, electrodes=electrodes,
                                             control_mode='degree_rewire', n_shuffle=200, graph_density=0.3, show=cfg.show_figs)
            pd.DataFrame(ms['delta_table']).to_csv(os.path.join(csv_dir,'multiseed.csv'), index=False)
            scores['multiseed']=score_multiseed(ms); blocks['multiseed']=ms
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'multiseed.png')); plt.close('all')
        except Exception as e:
            print("[Multi-seed] skipped:", e); scores['multiseed']=0.0

    # 7) Phase embedding
    if 'run_phase_embedding_emergent_geometry' in globals():
        try:
            pe = run_phase_embedding_emergent_geometry(RECORDS, ignition_windows=cfg.ignition_windows, rebound_windows=cfg.rebound_windows,
                                                       control_windows=cfg.baseline_windows, time_col=cfg.time_col,
                                                       electrodes=electrodes, band=(8,13),
                                                       n_neighbors=6, n_components=2, method='isomap', k_quality=5,
                                                       n_surr=100, show=cfg.show_figs)
            pd.DataFrame(pe['metrics_table']).to_csv(os.path.join(csv_dir,'embedding.csv'), index=False)
            scores['embedding']=score_embedding(pe); blocks['embedding']=pe
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'embedding.png')); plt.close('all')
        except Exception as e:
            print("[Embedding] skipped:", e); scores['embedding']=0.0

    # 8) ERP/ERSP/ITC (safe)
    if 'erp_ersp_itc_safe' in globals():
        try:
            er = erp_ersp_itc_safe(RECORDS, eeg_channels=[e for e in electrodes if e.endswith(('O1','O2','Oz','Pz'))] or electrodes[:3],
                                   sr_channel=sr, time_col=cfg.time_col, win_sec=(-3,3), baseline_sec=(-2,-0.5),
                                   center_hz=cfg.harmonics[0], half_bw_hz=0.6,
                                   detect_kwargs={'thresh_mode':'z','z_thresh':2.0},
                                   fmin=4, fmax=40, n_freq=48, w0=6.0, n_perm=200, alpha=0.05, show=cfg.show_figs)
            scores['erp_tf']=score_erp_tf(er); blocks['erp_tf']=er
            if cfg.show_figs: plt.savefig(os.path.join(fig_dir,'erp_tf.png')); plt.close('all')
        except Exception as e:
            print("[ERP/TF] skipped:", e); scores['erp_tf']=0.0

    # 9) SCF (optional; simple)
    if 'scf_at_harmonics' in globals():
        try:
            scf = scf_at_harmonics(RECORDS, sr, harmonics=cfg.harmonics, windows=cfg.ignition_windows, time_col=cfg.time_col)
            pd.DataFrame(scf['table']).to_csv(os.path.join(csv_dir,'scf.csv'), index=False)
            scores['scf']=score_scf(scf); blocks['scf']=scf
        except Exception as e:
            print("[SCF] skipped:", e); scores['scf']=0.0

    # 10) PLV topography (optional quick score)
    if 'run_plv_harmonics_topography' in globals():
        try:
            plv_res = run_plv_harmonics_topography(RECORDS, eeg_channels=electrodes, sr_channel=sr,
                                                   harmonics=cfg.harmonics, half_bw_hz=0.6,
                                                   windows=cfg.ignition_windows, time_col=cfg.time_col)
            pd.DataFrame(plv_res['table']).to_csv(os.path.join(csv_dir,'plv_topo.csv'), index=False)
            scores['plv_topo']=score_plv_topo(plv_res); blocks['plv_topo']=plv_res
        except Exception as e:
            print("[PLV topo] skipped:", e); scores['plv_topo']=0.0

    # ---------- composite overall score ----------
    # normalize weights over steps actually present
    present = {k:v for k,v in scores.items() if not np.isnan(v)}
    Wsum = sum(cfg.weights.get(k,0.0) for k in present.keys())
    if Wsum<=0: overall=0.0
    else:
        overall = float(sum(cfg.weights.get(k,0.0)*present[k] for k in present.keys())/Wsum)

    # save scores + JSON
    pd.DataFrame([scores | {'overall':overall}]).to_csv(os.path.join(csv_dir,'scores.csv'), index=False)
    with open(os.path.join(out_dir,'scores.json'),'w') as f: json.dump(scores | {'overall':overall}, f, indent=2)
    print(f"\nComposite overall score: {overall:.3f}")
    return {'scores':scores, 'overall':overall, 'blocks':blocks,
            'export_dir':out_dir, 'fig_dir':fig_dir, 'csv_dir':csv_dir,
            'electrodes':electrodes, 'sr_channel':sr, 'fs':fs}


In [ ]:
cfg = PipelineConfig(
    session_name='S01',
    time_col='Timestamp',
    electrodes=None,                         # autodetect EEG.*
    sr_channel="EEG.F4",                         # auto-pick posterior channel
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],    # explicit baseline windows (recommended)
    rebound_windows=[(310,325)],
    # optional: adjust weights per your priorities
    weights={'msc':0.12,'dtf':0.12,'entanglement':0.10,'criticality':0.10,
             'breadth':0.10,'overlap_etas':0.08,'multiseed':0.08,'embedding':0.08,
             'erp_tf':0.12,'scf':0.05,'plv_topo':0.05},
    out_dir='exports_pipeline',
    show_figs=True
)

out = run_holo_pipeline(RECORDS, cfg)
print("Scores:", out['scores'])
print("Overall:", out['overall'])
print("Results in:", out['export_dir'])


In [ ]:
# ================================
# HoloBatch — “Change filename” run
# ================================
from __future__ import annotations
import os, json
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from scipy import signal
import matplotlib.pyplot as plt

# ---------- tiny utils ----------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(RECORDS: pd.DataFrame, time_col='Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer fs")
    return float(1.0/np.median(dt))

def pick_best_channel_for_schumann(RECORDS, time_col='Timestamp'):
    fs = infer_fs(RECORDS, time_col=time_col)
    cands = [c for c in RECORDS.columns if c.startswith('EEG.')]
    if not cands: return None
    scores=[]
    for ch in cands:
        x = np.asarray(pd.to_numeric(RECORDS[ch], errors='coerce').fillna(0.0).values, float)
        f, p = signal.welch(x, fs=fs, nperseg=4*int(fs))
        def band(a,b):
            sel=(f>=a)&(f<=b); 
            return np.trapz(p[sel], f[sel]) if np.any(sel) else 0.0
        low = band(4,12); emg = band(40,90); mains = band(55,65)
        scores.append((low/(emg+1e-12) - 0.2*mains, ch))
    scores.sort(reverse=True); return scores[0][1]

# ---------- auto-detect ignition/baseline windows ----------
def auto_detect_windows(RECORDS: pd.DataFrame,
                        time_col: str = 'Timestamp',
                        sr_channel: Optional[str] = None,
                        mode: str = 'fused',           # 'fused' | 'envelope'
                        k_overlap: int = 3,            # K≥k for ignition seeds (fused)
                        win_len_sec: float = 20.0,     # ignition window length
                        merge_gap_sec: float = 5.0,    # merge close seeds
                        z_thresh: float = 2.0,         # envelope mode
                        min_isi_sec: float = 2.0,
                        buffer_sec: float = 2.0
                       ) -> Dict[str, List[Tuple[float,float]]]:
    """
    Returns {'ignition_windows': [...], 'rebound_windows': [...], 'baseline_windows': [...]}
    """
    fs = infer_fs(RECORDS, time_col)
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    t0, t1 = float(t[0]), float(t[-1])

    if sr_channel is None:
        sr_channel = pick_best_channel_for_schumann(RECORDS, time_col=time_col) or [c for c in RECORDS.columns if c.startswith('EEG.')][0]

    seeds = []
    if mode == 'fused' and 'detect_and_plot_schumann_microgrid_with_global_tf' in globals():
        # Use fused micro-grid overlap K≥k
        fused = detect_and_plot_schumann_microgrid_with_global_tf(RECORDS, signal_col=sr_channel, time_col=time_col, show=False)
        z_thr = float(fused.get('params',{}).get('z_thresh', 3.5))
        overlap = np.sum((np.asarray(fused['z_ridge']) >= z_thr).astype(int), axis=0)
        on_idx = np.where(np.diff((overlap >= k_overlap).astype(int)) == 1)[0] + 1
        seeds = fused['index'][on_idx].tolist()
    else:
        # Envelope threshold around 7.83±0.6 Hz
        y = np.asarray(pd.to_numeric(RECORDS[sr_channel], errors='coerce').fillna(0.0).values, float)
        # bandpass
        ny = 0.5*fs
        b,a = signal.butter(4, [7.23/ny, 8.43/ny], btype='band')
        env = np.abs(signal.hilbert(signal.filtfilt(b,a,y)))
        z = (env - env.mean())/(env.std()+1e-12)
        mask = z >= z_thresh
        on_idx = np.where(np.diff(mask.astype(int))==1)[0] + 1
        # min ISI
        last = -np.inf
        for i in on_idx:
            if t[i] - last >= min_isi_sec:
                seeds.append(float(t[i])); last = float(t[i])

    # merge seeds into fixed-length windows
    seeds = sorted(seeds)
    ign = []
    last_end = -np.inf
    for s in seeds:
        start = s - win_len_sec/2
        end   = s + win_len_sec/2
        if start <= last_end + merge_gap_sec:
            # extend last window
            ign[-1] = (ign[-1][0], end)
            last_end = end
        else:
            ign.append((start, end))
            last_end = end

    # clip to session bounds
    ign = [(max(t0, a), min(t1, b)) for (a,b) in ign if (b-a) > 1.0]
    # rebound: small window immediately after each ignition
    reb = [(min(t1, b), min(t1, b+0.5*win_len_sec)) for (_,b) in ign]
    # baseline: complement with buffer
    spans=[]; last=t0
    for (a,b) in ign:
        if a-buffer_sec > last: spans.append((last, a-buffer_sec))
        last = b+buffer_sec
    if last < t1: spans.append((last, t1))
    base = [(a,b) for (a,b) in spans if (b-a) > 2.0]

    return {'ignition_windows': ign, 'rebound_windows': reb, 'baseline_windows': base, 'sr_channel': sr_channel}

# ---------- one-file analyze ----------
def analyze_session_file(file_path: str,
                         session_name: Optional[str] = None,
                         time_col: str = 'Timestamp',
                         electrodes: Optional[List[str]] = None,
                         sr_channel: Optional[str] = None,
                         auto_mode: str = 'fused',          # 'fused' or 'envelope'
                         win_len_sec: float = 20.0,
                         k_overlap: int = 3,
                         z_thresh: float = 2.0,
                         out_root: str = 'exports_batch'
                        ) -> Dict[str, object]:
    """
    Load a file → auto-detect ignition/baseline → run full report v2 + scoring pipeline.
    Returns dict with important paths and scores.
    """
    # 1) Load RECORDS (assumes CSV; tweak if TSV/parquet)
    RECORDS = pd.read_csv(file_path)
    if session_name is None:
        session_name = os.path.splitext(os.path.basename(file_path))[0]

    # 2) Auto-detect windows & SR
    auto = auto_detect_windows(RECORDS, time_col=time_col, sr_channel=sr_channel,
                               mode=auto_mode, k_overlap=k_overlap, win_len_sec=win_len_sec, z_thresh=z_thresh)
    sr = auto['sr_channel']

    # 3) Electrode set (once for all steps)
    if electrodes is None:
        electrodes = [c for c in RECORDS.columns if c.startswith('EEG.')]

    # 4) Report v2 (PDF+FIG+CSV)
    report = None
    if 'run_one_click_session_report_v2' in globals():
        try:
            report = run_one_click_session_report_v2(
                RECORDS,
                session_name=session_name,
                time_col=time_col,
                electrodes=electrodes,
                ignition_windows=auto['ignition_windows'],
                rebound_windows=auto['rebound_windows'],
                control_windows=auto['baseline_windows'],
                event_onsets=None,
                event_labels=None,
                H=None,
                include_offharmonic_control=True,
                f0=7.83
            )
        except Exception as e:
            print("[Report v2] skipped:", e)

    # 5) Scoring pipeline (CSV + overall score)
    scores = None
    if 'PipelineConfig' in globals() and 'run_holo_pipeline' in globals():
        try:
            cfg = PipelineConfig(
                session_name=session_name,
                time_col=time_col,
                electrodes=electrodes,
                sr_channel=sr,
                ignition_windows=auto['ignition_windows'],
                baseline_windows=auto['baseline_windows'],
                rebound_windows=auto['rebound_windows'],
                out_dir=os.path.join(out_root)  # export path
            )
            scores = run_holo_pipeline(RECORDS, cfg)
        except Exception as e:
            print("[Pipeline scoring] skipped:", e)

    return {
        'session': session_name,
        'sr_channel': sr,
        'windows': auto,
        'report': report,
        'scores': scores
    }

# ---------- batch multiple files ----------
def batch_analyze(file_list: List[str],
                  time_col: str = 'Timestamp',
                  auto_mode: str = 'fused',
                  out_root: str = 'exports_batch'
                 ) -> pd.DataFrame:
    """
    Run analyze_session_file on each file and collect the overall scores into a table.
    """
    rows=[]
    for f in file_list:
        print(f"\n=== Analyzing: {f} ===")
        out = analyze_session_file(f, time_col=time_col, auto_mode=auto_mode, out_root=out_root)
        sess = out['session']
        overall = out['scores']['overall'] if out.get('scores') else np.nan
        row = {'session': sess, 'overall': overall}
        if out.get('scores') and out['scores'].get('scores'):
            row.update(out['scores']['scores'])
        rows.append(row)
    df = pd.DataFrame(rows)
    # save group summary
    _ensure_dir(out_root)
    df.to_csv(os.path.join(out_root, 'batch_scores.csv'), index=False)
    print("\nBatch summary saved to:", os.path.join(out_root, 'batch_scores.csv'))
    return df


In [ ]:
def detect_time_col(RECORDS: pd.DataFrame,
                    candidates: Tuple[str,...] = ('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')) -> Optional[str]:
    cols = list(RECORDS.columns)
    # exact match
    for c in candidates:
        if c in RECORDS.columns:
            return c
    # heuristic: first numeric, monotonic-increasing column
    for c in cols:
        s = pd.to_numeric(RECORDS[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(RECORDS)):
            arr = s.values.astype(float)
            dt = np.diff(arr[np.isfinite(arr)])
            if dt.size and np.nanmedian(dt) > 0:
                return c
    # heuristic: first datetime-like
    for c in cols:
        try:
            sd = pd.to_datetime(RECORDS[c], errors='raise')
            return c
        except Exception:
            continue
    return None

def ensure_timestamp_column(RECORDS: pd.DataFrame,
                            time_col: Optional[str] = None,
                            default_fs: float = 128.0,
                            out_name: str = 'Timestamp') -> str:
    """
    Ensure RECORDS has a numeric seconds column named out_name.
    Returns the column name used (out_name).
    """
    col = time_col or detect_time_col(RECORDS)
    if col is None:
        # synthesize at default_fs
        N = len(RECORDS)
        RECORDS[out_name] = np.arange(N, dtype=float) / float(default_fs)
        return out_name

    s = RECORDS[col]
    # datetime → seconds since first sample
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        RECORDS[out_name] = tsec.values
        return out_name

    # numeric-ish → coerce and use directly (shift to t=0)
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(RECORDS)):
        # last resort: synthesize
        N = len(RECORDS)
        RECORDS[out_name] = np.arange(N, dtype=float) / float(default_fs)
        return out_name

    # normalize to start at 0
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    RECORDS[out_name] = sn.values
    return out_name


In [ ]:
file_path='data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv'
RECORDS = pd.read_csv(file_path)


# # Single session
# out = analyze_session_file(
#     file_path='data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',               # ← change only this
#     session_name='S01',
#     time_col='time', 
#     auto_mode='fused',                          # uses the fused micro-grid if available
#     win_len_sec=20.0, k_overlap=3,              # ignition window length & K-threshold
#     z_thresh=2.0                                # used only if auto_mode='envelope'
# )


out = analyze_session_file(
    file_path='data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',   # ← only change filename
    session_name='S01',
    auto_mode='fused',
    win_len_sec=20.0, k_overlap=3, z_thresh=2.0
)
print("Overall:", out['scores']['overall'] if out['scores'] else '—')
print("Windows:", out['windows'])


print("Overall score:", out['scores']['overall'] if out['scores'] else '—')
print("Windows:", out['windows'])

# # Batch sessions
# files = [
#     'sessions/S01.csv',
#     'sessions/S02.csv',
#     'sessions/S03.csv'
# ]
# batch_df = batch_analyze(files, auto_mode='fused')
# display(batch_df)


In [ ]:
"""
Attractor Topology via Nonlinear Dimensional Embedding — Simple Graphs & Validity Tests
======================================================================================

This module reconstructs EEG phase-space attractors (Takens delay embedding) and computes:
  • Delay τ (autocorrelation-based)
  • Embedding dimension m (False Nearest Neighbors; FNN)
  • Correlation (fractal) dimension D2 (Grassberger-Procaccia)
  • Largest Lyapunov exponent λ_max (Rosenstein)
  • Recurrence plot (RP) + simple RQA (recurrence rate RR, determinism DET)
  • (Optional) Persistent homology summaries if `ripser` is available

It includes:
  • Ignition vs Baseline comparison (windows you pass in)
  • Surrogate tests (phase-randomized and time-shuffled) → p-values
  • Clean, readable plots saved alongside a concise text/CSV summary

Assumptions:
  • RECORDS is a pandas.DataFrame with a numeric time column (default 'Timestamp')
    and EEG channels named 'EEG.*'. You pass one or more EEG channels; we average them
    as a robust single drive signal (or you can pass a single channel).
  • Python 3.7+ with NumPy, SciPy, scikit-learn (neighbors), matplotlib, pandas.
    (If scikit-learn is missing, we fall back to naïve kNN with NumPy.)

Usage (minimal):
----------------
res = run_attractor_topology(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.Oz'],           # averaged
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_attractor/S01',
    show=True
)
print(res['summary_df'])

Files saved to out_dir:
  • attractor_3D_[state].png
  • corr_dimension_[state].png
  • lyapunov_[state].png
  • recurrence_[state].png
  • (if available) persistence_[state].png
  • summary.csv + summary.txt
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
from numpy.linalg import norm

# register the 3D projection with Matplotlib
try:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    _HAS_MPL_3D = True
except Exception:
    _HAS_MPL_3D = False


# Optional: scikit-learn neighbors
try:
    from sklearn.neighbors import KDTree
    _HAS_SK = True
except Exception:
    _HAS_SK = False

# Optional: ripser (persistent homology)
try:
    from ripser import ripser
    from persim import plot_diagrams
    _HAS_RIPSER = True
except Exception:
    _HAS_RIPSER = False


# ------------------------- I/O + preproc helpers -------------------------

def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0:
        raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.' + name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found in RECORDS.")

def slice_concat(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> np.ndarray:
    if not windows: return x.copy()
    segs=[]; n=len(x)
    for (t0,t1) in windows:
        i0,i1 = int(round(t0*fs)), int(round(t1*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x = np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)


# ------------------------- Delay & embedding tools -------------------------

def estimate_delay_tau(x: np.ndarray, fs: float, max_lag_sec: float = 2.0, method: str = 'acf-1e') -> int:
    """
    Pick τ from the first time where autocorrelation falls below 1/e (default) or crosses 0.
    """
    nlag = int(max(1, round(max_lag_sec*fs)))
    x = zscore(x)
    acf = signal.correlate(x, x, mode='full')
    acf = acf[acf.size//2:acf.size//2+nlag+1]
    acf = acf / (acf[0] + 1e-12)
    if method == 'zero':
        idx = np.where(np.sign(acf[1:]) != np.sign(acf[:-1]))[0]
        tau = int(idx[0]+1) if idx.size else max(1, int(0.05*fs))
    else:  # 'acf-1e'
        idx = np.where(acf <= 1/np.e)[0]
        tau = int(idx[0]) if idx.size else max(1, int(0.05*fs))
    return max(1, tau)

def takens_embedding(x: np.ndarray, m: int, tau: int) -> np.ndarray:
    """
    Return (N_eff, m) embedded matrix: [x_t, x_{t+τ}, ..., x_{t+(m-1)τ}]
    """
    N = len(x) - (m-1)*tau
    if N <= 10:
        raise ValueError("Time series too short for requested embedding.")
    Y = np.column_stack([x[i:i+N] for i in range(0, m*tau, tau)]).astype(float)
    return Y

def false_nearest_neighbors(x: np.ndarray, tau: int, m_list: List[int],
                            theiler: int = 10) -> pd.DataFrame:
    """
    Simple FNN percentage vs m. If sklearn is present, uses KDTree; else brute force sample.
    """
    rows=[]
    for m in m_list:
        try:
            X_m = takens_embedding(x, m, tau)       # (N, m)
            X_m1= takens_embedding(x, m+1, tau)     # (N', m+1) with N' slightly smaller
            N = min(len(X_m), len(X_m1))
            X_m  = X_m[:N]; X_m1 = X_m1[:N]
        except Exception:
            rows.append({'m': m, 'FNN%': np.nan}); continue

        if _HAS_SK:
            tree = KDTree(X_m)
            # 1-NN with Theiler window
            dist, idx = tree.query(X_m, k=2)
            nn = idx[:,1]
            # apply Theiler: replace neighbors within theiler samples by next best
            for i in range(N):
                if abs(nn[i]-i) <= theiler:
                    # find next NN not within Theiler window
                    dists, idxs = tree.query(X_m[i:i+1], k=10)
                    for cand in idxs[0,1:]:
                        if abs(cand - i) > theiler:
                            nn[i] = cand; break
        else:
            # fallback: naïve nearest neighbor (sampled)
            nn = np.zeros(N, dtype=int)
            for i in range(N):
                j = np.argmin(np.where(np.arange(N)==i, np.inf, norm(X_m - X_m[i], axis=1)))
                if abs(j-i) <= theiler:
                    # pick next best
                    d = norm(X_m - X_m[i], axis=1)
                    d[i] = np.inf
                    order = np.argsort(d)
                    for cand in order:
                        if abs(cand - i) > theiler:
                            j = cand; break
                nn[i] = j

        # FNN criterion (Kennel et al.): ratio of neighbor distance in m+1 vs m exceeding threshold
        Rtol = 15.0  # typical 10–15
        dist_m  = norm(X_m  - X_m[nn],  axis=1)
        dist_m1 = norm(X_m1 - X_m1[nn], axis=1)
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = dist_m1 / (dist_m + 1e-12)
        fnn = np.mean(ratio > Rtol) * 100.0
        rows.append({'m': m, 'FNN%': float(fnn)})
    return pd.DataFrame(rows)

# ------------------------- Fractal dimension (GP) -------------------------

def correlation_dimension_gp(X: np.ndarray,
                             r_min_quant: float = 0.05,
                             r_max_quant: float = 0.30,
                             n_r: int = 20,
                             max_pairs: int = 30000) -> Dict[str, object]:
    """
    Grassberger-Procaccia correlation sum C(r) and slope (D2) over a mid-range.
    Subsamples pairs to limit O(N^2).
    """
    N = len(X)
    # pairwise distances on subsample
    idx = np.random.choice(N, size=min(N, 1000), replace=False)
    D = np.sqrt(((X[idx,None,:] - X[None,idx,:])**2).sum(axis=2)).ravel()
    D = D[D>0]
    Dsorted = np.sort(D)
    rmin = Dsorted[int(r_min_quant*len(Dsorted))]
    rmax = Dsorted[int(r_max_quant*len(Dsorted))]
    r_vals = np.exp(np.linspace(np.log(rmin+1e-12), np.log(rmax+1e-12), n_r))
    # correlation sum C(r) ~ fraction of pairs with distance < r
    if len(D) > max_pairs:
        D = np.random.choice(D, size=max_pairs, replace=False)
    C = np.array([np.mean(D < r) for r in r_vals])
    # slope via linear fit on log-log
    with np.errstate(divide='ignore', invalid='ignore'):
        x = np.log(r_vals + 1e-24); y = np.log(C + 1e-24)
    A = np.vstack([x, np.ones_like(x)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    return {'r': r_vals, 'C': C, 'D2': float(slope), 'fit': (float(slope), float(intercept))}

# ------------------------- Largest Lyapunov (Rosenstein) -------------------------

def lyapunov_rosenstein(x: np.ndarray, m: int, tau: int, fs: float,
                        theiler: int = 10,
                        t_fit: Tuple[int,int] = (1, 30)) -> Dict[str, object]:
    """
    Rosenstein et al. method. Returns λ_max (1/s) and the divergence curve.
    t_fit in samples (embedded-time steps).
    """
    X = takens_embedding(x, m, tau)  # (N, m)
    N = len(X)
    # nearest neighbor with Theiler window
    if _HAS_SK:
        tree = KDTree(X)
        dist, idx = tree.query(X, k=2)
        nn = idx[:,1]
        # Theiler correction
        for i in range(N):
            if abs(nn[i]-i) <= theiler:
                dists, idxs = tree.query(X[i:i+1], k=10)
                for cand in idxs[0,1:]:
                    if abs(cand - i) > theiler:
                        nn[i] = cand; break
    else:
        nn = np.zeros(N, dtype=int)
        for i in range(N):
            d = norm(X - X[i], axis=1)
            d[i] = np.inf
            j = np.argmin(d)
            if abs(j-i) <= theiler:
                order = np.argsort(d)
                for cand in order:
                    if abs(cand - i) > theiler:
                        j = cand; break
            nn[i] = j

    # Average log divergence over lead times k
    max_k = min(100, N-1)
    L = []
    for k in range(1, max_k):
        valid = (np.arange(N) + k < N) & (nn + k < N)
        if not np.any(valid): break
        d_k = norm(X[valid + k] - X[nn[valid] + k], axis=1)
        d_0 = norm(X[valid]     - X[nn[valid]],     axis=1) + 1e-24
        L.append(np.mean(np.log(d_k / d_0)))
    L = np.array(L)
    k0, k1 = t_fit
    k1 = min(k1, len(L)-1)
    if k1 <= k0:
        return {'lambda': np.nan, 'L': L, 'k': np.arange(1, len(L)+1)}
    # linear fit on L(k) ~ λ * k * Δt
    ks = np.arange(1, len(L)+1)
    A = np.vstack([ks[k0:k1], np.ones(k1-k0)]).T
    slope, intercept = np.linalg.lstsq(A, L[k0:k1], rcond=None)[0]
    lam = slope * fs / (tau)       # convert per embedded step to per-second approx
    return {'lambda': float(lam), 'L': L, 'k': ks}

# ------------------------- Recurrence plot + simple RQA -------------------------

def recurrence_plot(X: np.ndarray, eps_quant: float = 0.1) -> Dict[str, object]:
    """
    Binary RP thresholded at eps = quantile(eps_quant) of distances.
    Simple RQA: Recurrence Rate (RR), Determinism (DET) via diagonal line counts ≥2.
    """
    N = len(X)
    # distance matrix on subsample for efficiency
    idx = np.random.choice(N, size=min(N, 1200), replace=False)
    Y = X[idx]
    D = np.sqrt(((Y[:,None,:]-Y[None,:,:])**2).sum(axis=2))
    eps = np.quantile(D, eps_quant)
    R = (D <= eps).astype(int)
    np.fill_diagonal(R, 0)
    RR = np.mean(R)
    # DET: crude diagonal line detector
    det_lines=0; total_lines=0
    for i in range(R.shape[0]-1):
        run=0
        for j in range(R.shape[1]-1):
            if R[i,j]==1 and R[i+1,j+1]==1:
                run+=1
            else:
                if run>=1:
                    total_lines+=1
                    if run+1>=2: det_lines+=1
                run=0
        if run>=1:
            total_lines+=1
            if run+1>=2: det_lines+=1
    DET = det_lines / (total_lines + 1e-12)
    return {'R': R, 'RR': float(RR), 'DET': float(DET), 'eps': float(eps)}

# ------------------------- Persistent homology (optional) -------------------------

def persistent_homology_summary(X: np.ndarray, maxdim: int = 2) -> Dict[str, object]:
    """
    If ripser is installed, compute persistence and return diagrams + simple counts.
    """
    if not _HAS_RIPSER:
        return {'available': False}
    # subsample to keep compute reasonable
    N = len(X)
    idx = np.random.choice(N, size=min(N, 800), replace=False)
    Y = X[idx]
    dgms = ripser(Y, maxdim=maxdim)['dgms']
    # simple summaries: count H0/H1/H2 bars above small persistence
    def count_persistent(dgm, thr=0.02):
        return int(np.sum((dgm[:,1]-dgm[:,0]) > thr))
    summ = {
        'H0_count': count_persistent(dgms[0], thr=0.0),
        'H1_count': count_persistent(dgms[1]) if len(dgms)>1 else 0,
        'H2_count': count_persistent(dgms[2]) if len(dgms)>2 else 0
    }
    return {'available': True, 'dgms': dgms, 'summary': summ}

# ------------------------- Surrogates -------------------------

def phase_randomize(x: np.ndarray) -> np.ndarray:
    X = np.fft.rfft(x)
    mag = np.abs(X)
    ph  = np.angle(X)
    k = len(ph)
    rand = np.random.uniform(-np.pi, np.pi, size=k)
    rand[0] = ph[0]
    if k % 2 == 0:
        rand[-1] = ph[-1]
    Xs = mag * np.exp(1j*rand)
    return np.fft.irfft(Xs, n=len(x)).astype(float)

def time_shuffle(x: np.ndarray) -> np.ndarray:
    return np.random.permutation(x)

def metric_vs_surrogates(metric_func, x: np.ndarray, n_surr: int = 100, kind: str = 'phase') -> Tuple[float, float]:
    """
    Compute metric on x, build null from surrogates (phase or shuffle). Return (value, p-value).
    """
    val = metric_func(x)
    null=[]
    for _ in range(n_surr):
        xs = phase_randomize(x) if kind=='phase' else time_shuffle(x)
        null.append(metric_func(xs))
    null = np.asarray(null, float)
    p = (np.sum(null >= val) + 1) / (n_surr + 1)
    return float(val), float(p)

# ------------------------- Top-level runner -------------------------

def run_attractor_topology(RECORDS: pd.DataFrame,
                           eeg_channels: List[str],
                           ignition_windows: Optional[List[Tuple[float,float]]],
                           baseline_windows: Optional[List[Tuple[float,float]]],
                           time_col: str = 'Timestamp',
                           out_dir: str = 'exports_attractor/session',
                           show: bool = True,
                           max_lag_sec: float = 2.0,
                           m_list: List[int] = [2,3,4,5,6,7,8],
                           n_surrogates: int = 100) -> Dict[str, object]:
    """
    Build attractor embeddings and tests for Ignition and Baseline.
    Returns dict with summary DataFrame and paths.
    """
    _ensure_dir(out_dir)

    # >>> NEW: make sure a valid numeric-seconds column exists <<<
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    # -------------------------------------------------------------

    fs = infer_fs(RECORDS, time_col)
    
    # robust drive signal (mean of channels)
    Xsig=[]
    for ch in eeg_channels:
        Xsig.append(get_series(RECORDS, ch))
    Xsig = np.vstack(Xsig)
    x_full = zscore(np.mean(Xsig, axis=0))
    # per state
    states = {'ignition': ignition_windows, 'baseline': baseline_windows}
    summaries=[]
    outputs={}
    for state, wins in states.items():
        if not wins: continue
        x = slice_concat(x_full, fs, wins)
        x = zscore(x)
        # τ and m
        tau = estimate_delay_tau(x, fs, max_lag_sec=max_lag_sec, method='acf-1e')
        fnn_df = false_nearest_neighbors(x, tau, m_list)
        # choose m* at elbow (min m where FNN% < 5% or lowest)
        fnn_df = fnn_df.dropna()
        m_star = int(fnn_df.loc[fnn_df['FNN%'].le(5.0).idxmax(),'m']) if np.any(fnn_df['FNN%']<=5.0) else int(fnn_df['m'].iloc[np.argmin(fnn_df['FNN%'])])
        m_star = max(3, m_star)

        # embed
        X = takens_embedding(x, m_star, tau)    # (N,m)
        # (A) D2
        D2 = correlation_dimension_gp(X)['D2']
        # (B) λ_max
        lyap = lyapunov_rosenstein(x, m_star, tau, fs, theiler=int(0.5*fs/tau), t_fit=(1, 30))
        lam = lyap['lambda']
        # (C) RP + RQA
        rqa = recurrence_plot(X, eps_quant=0.1)
        # (D) persistent homology (optional)
        ph = persistent_homology_summary(X)

        # surrogate tests on D2 and λ_max
        d2_val, d2_p = metric_vs_surrogates(lambda xs: correlation_dimension_gp(takens_embedding(zscore(xs), m_star, tau))['D2'],
                                            x, n_surr=n_surrogates, kind='phase')
        lam_val, lam_p = metric_vs_surrogates(lambda xs: lyapunov_rosenstein(zscore(xs), m_star, tau, fs)['lambda'],
                                              x, n_surr=n_surrogates, kind='phase')

        summaries.append({'state':state, 'tau':tau, 'm':m_star,
                          'D2':float(D2), 'D2_p':float(d2_p),
                          'lambda':float(lam), 'lambda_p':float(lam_p),
                          'RR':rqa['RR'], 'DET':rqa['DET'],
                          'PH_available': ph['available']})

        # ===== Plots =====
        # 3D attractor
        # 3D (or 2D fallback) attractor plot
        stride = max(1, len(X)//8000)  # keep plots light; adjust as you like

        if _HAS_MPL_3D and X.shape[1] >= 3:
            fig = plt.figure(figsize=(5,4))
            ax = fig.add_subplot(111, projection='3d')
            ax.plot(X[::stride,0], X[::stride,1], X[::stride,2], lw=0.6, alpha=0.8)
            ax.set_title(f'Attractor (m={m_star}, τ={tau}) — {state}')
            plt.tight_layout()
            fig.savefig(os.path.join(out_dir, f'attractor_3D_{state}.png'), dpi=140)
            if show: plt.show()
            plt.close(fig)
        else:
            # 2D fallback (pairwise)
            fig, axs = plt.subplots(1, 2, figsize=(8,3.2))
            axs[0].plot(X[::stride,0], X[::stride,1], lw=0.6, alpha=0.8)
            axs[0].set_title(f'X1 vs X2 — {state}')
            if X.shape[1] >= 3:
                axs[1].plot(X[::stride,1], X[::stride,2], lw=0.6, alpha=0.8)
                axs[1].set_title(f'X2 vs X3 — {state}')
            else:
                axs[1].plot(X[::stride,0], X[::stride,0], lw=0.6, alpha=0.3)  # dummy if m<3
                axs[1].set_title('2D fallback')
            for ax in axs: ax.set_xlabel(''); ax.set_ylabel('')
            plt.tight_layout()
            fig.savefig(os.path.join(out_dir, f'attractor_2D_{state}.png'), dpi=140)
            if show: plt.show()
            plt.close(fig)


        # Correlation dimension log-log
        gp = correlation_dimension_gp(X)
        fig = plt.figure(figsize=(5,3))
        plt.plot(np.log(gp['r']+1e-24), np.log(gp['C']+1e-24), 'o-', lw=1)
        s, b = gp['fit']
        plt.plot(np.log(gp['r']+1e-24), s*np.log(gp['r']+1e-24)+b, 'r--', lw=1)
        plt.title(f'Correlation sum (D2≈{gp["D2"]:.2f}) — {state}')
        plt.xlabel('log r'); plt.ylabel('log C(r)'); plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f'corr_dimension_{state}.png'), dpi=140)
        if show: plt.show()
        plt.close(fig)

        # Lyapunov curve
        fig = plt.figure(figsize=(5,3))
        plt.plot(lyap['k']/fs*tau, lyap['L'], lw=1.2)
        plt.title(f'Lyapunov divergence (λ≈{lam:.3f} 1/s) — {state}')
        plt.xlabel('Time (s)'); plt.ylabel('⟨log(d_k/d_0)⟩'); plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f'lyapunov_{state}.png'), dpi=140)
        if show: plt.show()
        plt.close(fig)

        # Recurrence plot
        R = rqa['R']
        fig = plt.figure(figsize=(4,4))
        plt.imshow(R, origin='lower', cmap='binary')
        plt.title(f'Recurrence plot — {state}\nRR={rqa["RR"]:.3f}, DET={rqa["DET"]:.3f}')
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f'recurrence_{state}.png'), dpi=140)
        if show: plt.show()
        plt.close(fig)

        # Persistent homology diagram (optional)
        if ph['available']:
            fig = plt.figure(figsize=(4,3))
            plot_diagrams(ph['dgms'])
            plt.title(f'Persistence diagrams — {state}')
            plt.tight_layout()
            fig.savefig(os.path.join(out_dir, f'persistence_{state}.png'), dpi=140)
            if show: plt.show()
            plt.close(fig)

        outputs[state] = {'fnn':fnn_df, 'D2':D2, 'lambda':lam, 'rqa':rqa, 'ph':ph}

    # ===== Summary table & save =====
    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
    with open(os.path.join(out_dir, 'summary.txt'),'w') as f:
        f.write(summary_df.to_string(index=False))

    return {'summary_df': summary_df, 'outputs': outputs, 'out_dir': out_dir}


In [ ]:
import numpy as np
import pandas as pd

def detect_time_col(RECORDS: pd.DataFrame,
                    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')) -> str | None:
    # 1) direct match
    for c in candidates:
        if c in RECORDS.columns:
            return c
    # 2) first numeric, roughly monotonic column
    for c in RECORDS.columns:
        s = pd.to_numeric(RECORDS[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(RECORDS)):
            arr = s.values.astype(float)
            dt = np.diff(arr[np.isfinite(arr)])
            if dt.size and np.nanmedian(dt) > 0:
                return c
    # 3) first datetime-like column
    for c in RECORDS.columns:
        try:
            _ = pd.to_datetime(RECORDS[c], errors='raise')
            return c
        except Exception:
            continue
    return None

def ensure_timestamp_column(RECORDS: pd.DataFrame,
                            time_col: str | None = None,
                            default_fs: float = 128.0,
                            out_name: str = 'Timestamp') -> str:
    """
    Ensure RECORDS[out_name] exists as numeric seconds (t=0 at first sample).
    Returns the column name used ('Timestamp' by default).
    """
    col = time_col or detect_time_col(RECORDS)
    if col is None:
        # synthesize uniform time if none exists
        N = len(RECORDS)
        RECORDS[out_name] = np.arange(N, dtype=float) / float(default_fs)
        return out_name

    s = RECORDS[col]
    # datetime -> seconds since first
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        RECORDS[out_name] = tsec.values
        return out_name

    # numeric-ish -> coerce & shift to start at 0
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(RECORDS)):
        N = len(RECORDS)
        RECORDS[out_name] = np.arange(N, dtype=float) / float(default_fs)
        return out_name

    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    RECORDS[out_name] = sn.values
    return out_name


In [ ]:
def lyapunov_rosenstein(x: np.ndarray,
                        m: int,
                        tau: int,
                        fs: float,
                        theiler: int = 10,
                        t_fit: Tuple[int,int] = (1, 30)) -> Dict[str, object]:
    """
    Largest Lyapunov exponent (Rosenstein et al.).
    Returns {'lambda': λ_max (1/s), 'L': divergence curve, 'k': lead steps}.
    """
    # Build embedding
    X = takens_embedding(zscore(x), m, tau)  # (N, m)
    N = len(X)
    if N <= theiler + 2:
        return {'lambda': np.nan, 'L': np.array([]), 'k': np.array([])}

    # Nearest neighbor indices with Theiler exclusion
    if _HAS_SK:
        tree = KDTree(X)
        # pull more than 2 neighbors to have replacements if Theiler excludes
        dists, idxs = tree.query(X, k=min(20, N-1))
        nn = np.zeros(N, dtype=int)
        for i in range(N):
            # skip self at idxs[i,0]; find first neighbor beyond Theiler window
            chosen = None
            for cand in idxs[i, 1:]:
                if abs(int(cand) - i) > theiler:
                    chosen = int(cand); break
            nn[i] = chosen if chosen is not None else int(idxs[i,1])
    else:
        nn = np.zeros(N, dtype=int)
        for i in range(N):
            d = np.linalg.norm(X - X[i], axis=1)
            d[i] = np.inf
            order = np.argsort(d)
            chosen = None
            for cand in order:
                if abs(int(cand) - i) > theiler:
                    chosen = int(cand); break
            nn[i] = chosen if chosen is not None else int(order[0])

    # Mean log divergence over lead steps k
    max_k = max(2, min(100, N-1))
    Lvals = []
    ks = []
    for k in range(1, max_k):
        # only indices where both i+k and nn[i]+k are valid
        valid = (np.arange(N) + k < N) & (nn + k < N)
        if not np.any(valid):
            break
        idx = np.where(valid)[0]
        if idx.size < 5:  # too few pairs for a stable average
            break
        d0 = np.linalg.norm(X[idx]       - X[nn[idx]],       axis=1) + 1e-24
        dk = np.linalg.norm(X[idx + k]   - X[nn[idx] + k],   axis=1)
        Lvals.append(np.mean(np.log(dk / d0)))
        ks.append(k)

    Lvals = np.asarray(Lvals, float)
    ks = np.asarray(ks, int)
    if Lvals.size < 5:
        return {'lambda': np.nan, 'L': Lvals, 'k': ks}

    # Linear fit region (auto-cap to available ks)
    k0, k1 = t_fit
    k1 = min(k1, int(0.6*len(Lvals)))  # avoid late saturation, keep early linear regime
    if k1 <= k0+1:
        return {'lambda': np.nan, 'L': Lvals, 'k': ks}

    A = np.vstack([ks[k0:k1], np.ones(k1-k0)]).T
    slope, intercept = np.linalg.lstsq(A, Lvals[k0:k1], rcond=None)[0]
    # Convert per-embedded-step slope to per-second λ: one embedded step = tau samples
    lam = float(slope) * fs / float(tau)
    return {'lambda': lam, 'L': Lvals, 'k': ks}


In [ ]:
res = run_attractor_topology(
    RECORDS,
#     eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.F3','EEG.F4', 'EEG.FC5','EEG.FC6'],           # averaged
    eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F4','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_attractor/S01',
    show=True
)
summary = res['summary_df']
print(summary.head(10).to_string(index=False))      # preview only
# Full results are in: exports_attractor/S01/summary.csv and summary.txt

In [ ]:
"""
Entanglement Entropy Analogs & Integrative Information — Simple Graphs & Tests
=============================================================================

This module computes classical analogs of “entanglement/integration” for EEG:

  (A) Multichannel information & complexity (Gaussian & algorithmic):
      • Total Correlation (TC) (a.k.a. multi-information; Gaussian)
      • Dual Total Correlation (DTC) (Gaussian)
      • O-information (O = TC − DTC), redundancy/synergy indicator (Gaussian)
      • Entropy h(X) (Gaussian, z-scored)
      • Lempel–Ziv complexity (LZc) of a global binary sequence (PCA1→binarize)
      • Permutation entropy (PE) (order=3) averaged across channels

  (B) Network integration via connectivity:
      • PLV adjacency inside a band (e.g., alpha)
      • Laplacian spectral entropy (graph) as an integration/complexity index

  (C) Ignition vs Baseline comparisons + Surrogate tests:
      • Circular-shift surrogates → 95% null bands and p-values
      • Simple bar/heatmap plots, saved to out_dir

  (D) Time-resolved coupling to Schumann amplitude:
      • Sliding-window integration score vs SR envelope (7.83±0.6 Hz)
      • Correlation coefficient r and a quick null via circular shift

Inputs: 
  - RECORDS: pandas.DataFrame with a time column (default 'Timestamp') and EEG.* columns
  - eeg_channels: list of EEG.* channel names (or bare labels like 'O1')

Usage:
------
res = run_integration_analogs(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.Oz','EEG.Pz'],
    band=(8,13),
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    sr_channel='EEG.Oz',                 # if None, picks a posterior channel automatically
    time_col='Timestamp',
    out_dir='exports_integration/S01',
    show=True
)
print(res['summary'])   # per-state metrics and p-values
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
from numpy.linalg import det, inv

# ---------------------- small utilities ----------------------

def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found.")

def slice_concat(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> np.ndarray:
    if not windows: return x.copy()
    segs=[]; n=len(x)
    for (t0,t1) in windows:
        i0,i1 = int(round(t0*fs)), int(round(t1*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order=4) -> np.ndarray:
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny)); f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

def zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x,float)
    return (x - np.mean(x)) / (np.std(x)+1e-12)

# ---------------------- PLV & graph metrics ----------------------

def plv_matrix(RECORDS, channels, band, windows, time_col='Timestamp') -> np.ndarray:
    fs = infer_fs(RECORDS, time_col)
    phases=[]
    for ch in channels:
        x = slice_concat(get_series(RECORDS, ch), fs, windows)
        xb = bandpass(x, fs, band[0], band[1])
        phases.append(np.angle(signal.hilbert(xb)))
    P = np.vstack(phases)  # (N, T)
    N = P.shape[0]
    A = np.zeros((N,N))
    for i in range(N):
        for j in range(i, N):
            dphi = P[i]-P[j]
            A[i,j]=A[j,i]=float(np.abs(np.mean(np.exp(1j*dphi))))
    np.fill_diagonal(A, 0.0)
    return A

def laplacian_spectral_entropy(A: np.ndarray) -> float:
    D = np.diag(A.sum(axis=1))
    L = D - A
    L = 0.5*(L+L.T)
    vals = np.linalg.eigvalsh(L)
    vals = vals[vals>1e-12]
    if vals.size == 0: return np.nan
    p = vals/np.sum(vals)
    return float(-np.sum(p*np.log(p)))

# ---------------------- Gaussian information measures ----------------------

def gaussian_entropies(X: np.ndarray) -> Dict[str, float]:
    """
    X: (n_ch, T) z-scored
    Gaussian differential entropies:
      h(X) = 0.5 * [ n ln(2πe) + ln det Σ ]
      TC   = Σ h(X_i) − h(X)
      DTC  = h(X) − Σ h(X_i | X_{-i})  with  h(X_i | X_{-i}) = 0.5 ln(2πe σ^2_{i|-i})
      O    = TC − DTC
    """
    n, T = X.shape
    # covariance
    Sigma = np.cov(X)
    # add tiny ridge for stability
    Sigma = 0.5*(Sigma+Sigma.T) + 1e-9*np.eye(n)
    # entropies
    hXi = 0.5*(np.log(2*np.pi*np.e)*np.ones(n) + np.log(np.diag(Sigma)+1e-24))
    hX  = 0.5*(n*np.log(2*np.pi*np.e) + np.log(det(Sigma)+1e-24))
    TC  = float(np.sum(hXi) - hX)
    # conditional variances via precision
    Prec = inv(Sigma)
    cond_vars = 1.0 / np.diag(Prec)
    hXi_cond = 0.5*(np.log(2*np.pi*np.e) + np.log(cond_vars + 1e-24))
    DTC = float(hX - np.sum(hXi_cond))
    Oinfo = float(TC - DTC)
    return {'hX': float(hX), 'TC': TC, 'DTC': DTC, 'O': Oinfo}

# ---------------------- Algorithmic & ordinal complexities ----------------------

def pca_first_component(X: np.ndarray) -> np.ndarray:
    # X: (n_ch, T)
    Xc = X - X.mean(axis=1, keepdims=True)
    C = Xc @ Xc.T / Xc.shape[1]
    vals, vecs = np.linalg.eigh(C)
    v = vecs[:, -1]            # first PC (eigenvector)
    y = v @ Xc                 # PC1 time series
    return np.asarray(y).ravel()

def lz_complexity_binary(seq: np.ndarray) -> float:
    """
    LZ76 complexity of a binary sequence (0/1), normalized by n/log2(n).
    """
    s = ''.join('1' if v else '0' for v in (seq>0))
    n = len(s)
    i = 0; k = 1; l = 1; c = 1
    while True:
        if s[i+k-1] == s[l+k-1]:
            k += 1
            if l+k > n:
                c += 1; break
        else:
            if k > 1:
                i += 1
                if i == l:
                    c += 1
                    l += k
                    if l+1 > n:
                        break
                    i = 0; k = 1
            else:
                c += 1
                l += 1
                if l+1 > n:
                    break
                i = 0; k = 1
    norm = n/np.log2(max(2,n))
    return float(c / norm)

def permutation_entropy(x: np.ndarray, m: int = 3, tau: int = 1) -> float:
    """
    Band-limited x; simple permutation entropy of order m (permutation count m! bins).
    """
    x = np.asarray(x, float)
    T = len(x) - (m-1)*tau
    if T <= m: return np.nan
    patterns = {}
    for i in range(T):
        w = x[i:i+m*tau:tau]
        perm = tuple(np.argsort(w))
        patterns[perm] = patterns.get(perm, 0) + 1
    p = np.array(list(patterns.values()), float)
    p = p / p.sum()
    H = -np.sum(p*np.log(p+1e-24)) / np.log(np.math.factorial(m))
    return float(H)

# ---------------------- Surrogates ----------------------

def circular_shift_null(xmat: np.ndarray, n_surr: int = 200) -> List[np.ndarray]:
    """
    Circularly shift each channel independently; returns list of surrogates (n_ch, T).
    """
    n, T = xmat.shape
    rng = np.random.default_rng(11)
    sur=[]
    for _ in range(n_surr):
        Xs = []
        for i in range(n):
            s = int(rng.integers(1, T-1))
            Xs.append(np.r_[xmat[i,-s:], xmat[i,:-s]])
        sur.append(np.vstack(Xs))
    return sur

# ---------------------- Main runner ----------------------

def run_integration_analogs(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    band: Tuple[float,float] = (8,13),
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    sr_channel: Optional[str] = None,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_integration/session',
    show: bool = True,
    n_surr: int = 200
) -> Dict[str, object]:
    """
    Compute integration/complexity measures and simple tests; produce figures + CSV summary.
    """
    _ensure_dir(out_dir)
    fs = infer_fs(RECORDS, time_col)
    # choose SR if not provided
    if sr_channel is None:
        # simple PSD preference: pick Oz if exists, else first EEG.*
        sr_channel = 'EEG.Oz' if 'EEG.Oz' in RECORDS.columns else next((c for c in RECORDS.columns if c.startswith('EEG.')), None)
    # build X (n_ch, T) for both states
    Xall=[]
    for ch in eeg_channels:
        Xall.append(get_series(RECORDS, ch))
    Xall = np.vstack(Xall)      # (n_ch, T)
    states = {'ignition': ignition_windows, 'baseline': baseline_windows}

    def compute_state(wins, state_name):
        if not wins: return None
        X = np.vstack([slice_concat(x, fs, wins) for x in Xall])    # (n_ch, Tstate)
        # z-score each channel
        Xz = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True)+1e-12)
        # Gaussian integration measures
        g = gaussian_entropies(Xz.copy())
        # graph via PLV
        A = plv_matrix(RECORDS, eeg_channels, band, wins, time_col=time_col)
        H_L = laplacian_spectral_entropy(A)
        # LZc on PCA1
        pc1 = pca_first_component(Xz)
        # binarize by median
        lzc = lz_complexity_binary(pc1 - np.median(pc1))
        # perm entropy averaged across channels (in-band)
        pe_ch=[]
        for i in range(X.shape[0]):
            xb = bandpass(X[i], fs, band[0], band[1])
            pe_ch.append(permutation_entropy(xb, m=3, tau=1))
        PE = float(np.nanmean(pe_ch))
        return {'X':Xz, 'gauss':g, 'A':A, 'H_L':H_L, 'LZc':float(lzc), 'PE':PE}

    results={}
    for name, wins in states.items():
        res = compute_state(wins, name)
        if res: results[name]=res

    # ------------- Surrogates (ignition) -------------
    surr_pvals = {}
    if 'ignition' in results:
        X = results['ignition']['X']
        sur = circular_shift_null(X, n_surr=n_surr)
        # metrics on surrogates
        g_TC=[]; g_DTC=[]; g_O=[]; H_L_s=[]; LZ_s=[]; PE_s=[]
        for Xs in sur:
            g = gaussian_entropies(Xs)
            g_TC.append(g['TC']); g_DTC.append(g['DTC']); g_O.append(g['O'])
            # PLV graph on surrogate: scramble all channels jointly (consistent shifts)
            # Build PLV on surrogate in same band
            # (We simulate PLV null by randomizing phase via Hilbert-stage circular shift)
            # for speed, use original A as reference; here re-compute with Xs projected back to signals:
            # approximate by recomputing PLV on Xs via direct phase extraction:
            # (convert Xs to signal-like by inverse z-score to amplitude 1; this is a coarse null)
            phases = np.angle(signal.hilbert(np.array([bandpass(x, fs, band[0], band[1]) for x in Xs])))
            N = phases.shape[0]
            A = np.zeros((N,N))
            for i in range(N):
                for j in range(i,N):
                    dphi = phases[i]-phases[j]
                    A[i,j]=A[j,i]=float(np.abs(np.mean(np.exp(1j*dphi))))
            H_L_s.append(laplacian_spectral_entropy(A))
            pc1 = pca_first_component(Xs)
            LZ_s.append(lz_complexity_binary(pc1 - np.median(pc1)))
            PE_s.append(np.nanmean([permutation_entropy(bandpass(x, fs, band[0], band[1])) for x in Xs]))
        def pval(obs, null):
            null = np.asarray(null, float)
            return float((np.sum(null >= obs)+1)/(len(null)+1))
        surr_pvals = {
            'TC_p':  pval(results['ignition']['gauss']['TC'],  g_TC),
            'DTC_p': pval(results['ignition']['gauss']['DTC'], g_DTC),
            'O_p':   pval(results['ignition']['gauss']['O'],   g_O),
            'H_L_p': pval(results['ignition']['H_L'],         H_L_s),
            'LZc_p': pval(results['ignition']['LZc'],         LZ_s),
            'PE_p':  pval(results['ignition']['PE'],          PE_s)
        }

    # ------------- Simple plots -------------
    # adjacency heatmaps
    for st in results:
        fig,ax = plt.subplots(1,2, figsize=(8,3))
        im = ax[0].imshow(results[st]['A'], vmin=0, vmax=1, cmap='viridis')
        ax[0].set_title(f'PLV adjacency ({st})')
        plt.colorbar(im, ax=ax[0], fraction=0.046)
        # Gaussian info bar
        g = results[st]['gauss']
        names = ['hX','TC','DTC','O','H_L','LZc','PE']
        vals = [g['hX'], g['TC'], g['DTC'], g['O'], results[st]['H_L'], results[st]['LZc'], results[st]['PE']]
        ax[1].bar(range(len(names)), vals, color='tab:blue', alpha=0.85)
        ax[1].set_xticks(range(len(names))); ax[1].set_xticklabels(names, rotation=30)
        ax[1].set_title('Integration / Complexity')
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'integration_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # ignition vs baseline bars with null bands (if both present)
    if 'ignition' in results and 'baseline' in results:
        names = ['TC','DTC','O','H_L','LZc','PE']
        ign_vals = [results['ignition']['gauss']['TC'],
                    results['ignition']['gauss']['DTC'],
                    results['ignition']['gauss']['O'],
                    results['ignition']['H_L'],
                    results['ignition']['LZc'],
                    results['ignition']['PE']]
        base_vals= [results['baseline']['gauss']['TC'],
                    results['baseline']['gauss']['DTC'],
                    results['baseline']['gauss']['O'],
                    results['baseline']['H_L'],
                    results['baseline']['LZc'],
                    results['baseline']['PE']]
        x = np.arange(len(names)); w=0.38
        plt.figure(figsize=(8,3.2))
        plt.bar(x-w/2, base_vals, width=w, label='Baseline', color='tab:orange', alpha=0.9)
        plt.bar(x+w/2, ign_vals,  width=w, label='Ignition', color='tab:blue',  alpha=0.9)
        # add surrogate 95% lines for ignition (if available)
        if surr_pvals:
            # create simple line at 95th percentile of each surrogate null (we didn’t store whole null arrays; show p-values text instead)
            for i,name in enumerate(names):
                pkey = f'{name}_p' if f'{name}_p' in surr_pvals else None
                if pkey:
                    plt.text(i+w/2, ign_vals[i], f" p={surr_pvals[pkey]:.3f}", ha='center', va='bottom', fontsize=8)
        plt.xticks(x, names, rotation=0)
        plt.ylabel('Value'); plt.title('Ignition vs Baseline — Integration Indices'); plt.legend()
        plt.tight_layout(); plt.savefig(os.path.join(out_dir,'integration_ign_vs_base.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # ------------- Time-resolved coupling to SR amplitude -------------
    sr = get_series(RECORDS, sr_channel)
    env = np.abs(signal.hilbert(bandpass(sr, fs, 7.83-0.6, 7.83+0.6)))
    # sliding window score: normalize and combine TC+H_L (zwise) as a simple integrative index
    # build sliding windows (2.0 s, 0.25 s step)
    win = int(round(2.0*fs)); step = int(round(0.25*fs))
    idxs = list(range(0, len(Xall[0])-win, step))
    score_ts = []
    for s in idxs:
        seg = Xall[:, s:s+win]
        seg = (seg - seg.mean(axis=1, keepdims=True)) / (seg.std(axis=1, keepdims=True)+1e-12)
        g = gaussian_entropies(seg)
        A = plv_matrix(RECORDS, eeg_channels, band, [(s/fs,(s+win)/fs)], time_col=time_col)
        H_L = laplacian_spectral_entropy(A)
        score_ts.append( 0.7*g['TC'] + 0.3*H_L )
    score_ts = np.asarray(score_ts)
    t_centers = (np.array(idxs)+win//2)/fs
    # resample env to centers
    env_c = np.interp(t_centers, np.arange(len(env))/fs, env)
    # correlation + circular-shift null
    r = np.corrcoef(score_ts, env_c)[0,1]
    rng = np.random.default_rng(5)
    null_r=[]
    for _ in range(200):
        s = int(rng.integers(1,len(env_c)-1))
        null_r.append(np.corrcoef(score_ts, np.r_[env_c[-s:], env_c[:-s]])[0,1])
    thr95 = np.nanpercentile(null_r, 95)

    plt.figure(figsize=(9,3))
    zsc = (score_ts - np.nanmean(score_ts))/ (np.nanstd(score_ts)+1e-12)
    ze  = (env_c - np.nanmean(env_c))/ (np.nanstd(env_c)+1e-12)
    plt.plot(t_centers, zsc, label='Integration score (z)')
    plt.plot(t_centers, ze,  label='SR envelope (z)')
    plt.title(f'Time-resolved integration vs SR envelope  (r={r:.2f}, null95={thr95:.2f})')
    plt.xlabel('Time (s)'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir,'integration_vs_sr_timeseries.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # ------------- Summary & save -------------
    rows=[]
    for st, res in results.items():
        rows.append({
            'state': st,
            'hX': res['gauss']['hX'],
            'TC': res['gauss']['TC'],
            'DTC': res['gauss']['DTC'],
            'O': res['gauss']['O'],
            'H_L': res['H_L'],
            'LZc': res['LZc'],
            'PE': res['PE']
        })
    summary = pd.DataFrame(rows)
    if surr_pvals:
        p_row = {'state':'ignition_pvals'} | surr_pvals if hasattr(dict, '__or__') else dict(**{'state':'ignition_pvals'}, **surr_pvals)
        summary = pd.concat([summary, pd.DataFrame([p_row])], ignore_index=True)
    summary.to_csv(os.path.join(out_dir,'summary.csv'), index=False)

    return {'summary': summary,
            'results': results,
            'corr_env_r': float(r),
            'corr_env_null95': float(thr95),
            'out_dir': out_dir}


In [ ]:
eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2']

res = run_integration_analogs(
    RECORDS,
    eeg_channels=eeg_channels,
    band=(8,13),
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    sr_channel='EEG.F4',                 # if None, picks a posterior channel automatically
    time_col='Timestamp',
    out_dir='exports_integration/S02',
    show=True
)
print(res['summary'])   # per-state metrics and p-values

In [ ]:
"""
Connectome Harmonics & Resonant Mode Analysis — Simple Graphs & Validity Tests
=============================================================================

Goal
----
Project EEG into a set of spatial “harmonic” modes (connectome or sensor functional
harmonics), then test whether mode activations:
  • concentrate in Schumann bands (~7.8, 14.3, 20.8, 27.3, 33.8 Hz),
  • increase in ignition vs baseline,
  • covary with Schumann amplitude/envelope (time-resolved),
  • show increased MSC coherence to Schumann at harmonics.

What this module provides
-------------------------
1) Harmonic basis:
   (A) If you have a **matrix W** (N×N) whose nodes map one-to-one to your EEG channels,
       we compute Laplacian eigenvectors (connectome/sensor harmonics).
   (B) Otherwise, we **build functional harmonics** from PLV adjacency in a band (e.g., alpha).

2) Mode projection:
   X (n_ch × T) → A = H^T X (n_modes × T).  (H columns are Laplacian eigenvectors.)

3) Tests & graphs:
   • Mode power spectrum by state (Ignition vs Baseline) with simple Δ + null (circular-shift).
   • Schumann-band mode power (per mode, per harmonic band).
   • MSC coherence of each mode to SR at harmonics (bars + null).
   • Time series: chosen mode amplitude vs SR envelope (r + null).
   • Heatmap of the first K eigenvectors (“spatial harmonics”) across channels.

Inputs/assumptions
------------------
RECORDS: pandas.DataFrame with a numeric time column (default 'Timestamp')
and EEG signals named 'EEG.*'. You provide `eeg_channels` (or we detect them).
If you have a true connectome mapped to your EEG channels, pass `W_conn` (n_ch×n_ch).

Usage (minimal)
---------------
res = run_connectome_harmonics_resonance(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.Oz','EEG.Pz'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    sr_channel='EEG.Oz',                # or None to auto-pick posterior
    band_for_functional=(8,13),         # used when W_conn=None
    W_conn=None,                        # (optional) provide Laplacian source for harmonics
    n_modes=16,
    out_dir='exports_harmonics/S01',
    show=True
)
print(res['summary'])
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
import networkx as nx

# ------------------------- utils -------------------------

def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.' + name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found in RECORDS.")

def slice_concat(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> np.ndarray:
    if not windows: return x.copy()
    segs=[]; n=len(x)
    for (t0,t1) in windows:
        i0,i1 = int(round(t0*fs)), int(round(t1*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order=4) -> np.ndarray:
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny)); f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

def zscore(x): x = np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

# ------------------------- PLV adjacency & functional harmonics -------------------------

def plv_adj(RECORDS, channels, band, windows, time_col='Timestamp') -> np.ndarray:
    fs = infer_fs(RECORDS, time_col)
    phases=[]
    for ch in channels:
        x = slice_concat(get_series(RECORDS, ch), fs, windows)
        xb = bandpass(x, fs, band[0], band[1])
        phases.append(np.angle(signal.hilbert(xb)))
    P = np.vstack(phases)  # (N, T)
    N = P.shape[0]
    A = np.zeros((N,N))
    for i in range(N):
        for j in range(i,N):
            dphi = P[i]-P[j]
            A[i,j]=A[j,i]=float(np.abs(np.mean(np.exp(1j*dphi))))
    np.fill_diagonal(A, 0.0)
    return A

def laplacian_eigendecomp(W: np.ndarray, n_modes: int) -> Tuple[np.ndarray, np.ndarray]:
    """Return first n_modes Laplacian eigenvalues & eigenvectors (columns)."""
    W = 0.5*(W+W.T)
    D = np.diag(W.sum(axis=1))
    L = D - W
    vals, vecs = np.linalg.eigh(L)
    idx = np.argsort(vals)   # ascending (low spatial freq first)
    vals = vals[idx]; vecs = vecs[:, idx]
    K = min(n_modes, vecs.shape[1])
    # normalize columns to unit norm
    H = vecs[:, :K]
    for k in range(K):
        H[:,k] /= (np.linalg.norm(H[:,k]) + 1e-12)
    return vals[:K], H

# ------------------------- Mode projection & spectra -------------------------

def project_to_harmonics(X: np.ndarray, H: np.ndarray) -> np.ndarray:
    """X: (n_ch, T), H: (n_ch, K) columns orthonormal → A: (K, T)."""
    return H.T @ X

def mode_band_power(A: np.ndarray, fs: float, fband: Tuple[float,float]) -> np.ndarray:
    """A: (K, T) → band power per mode via band-pass + RMS."""
    K = A.shape[0]
    out=[]
    for k in range(K):
        ak = bandpass(A[k], fs, fband[0], fband[1])
        out.append(float(np.mean(ak**2)))
    return np.array(out)

def mode_welch_power(A: np.ndarray, fs: float, nperseg: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Return (f, Pk(f)) where Pk is (K, n_f)."""
    if nperseg is None:
        nperseg = int(2*fs)
    P=[]; freqs=None
    for k in range(A.shape[0]):
        f, p = signal.welch(A[k], fs=fs, nperseg=nperseg, noverlap=nperseg//2)
        freqs = f if freqs is None else freqs
        P.append(p)
    return freqs, np.vstack(P)

# ------------------------- MSC coherence mode↔SR at harmonics -------------------------

def msc_mode_to_sr(A: np.ndarray, sr: np.ndarray, fs: float,
                   harmonics: List[float], nperseg: Optional[int]=None) -> pd.DataFrame:
    if nperseg is None: nperseg = int(4*fs)
    rows=[]
    for k in range(A.shape[0]):
        f, C = signal.coherence(A[k], sr, fs=fs, nperseg=nperseg, noverlap=nperseg//2)
        for hf in harmonics:
            idx = int(np.argmin(np.abs(f - hf)))
            rows.append({'mode': k+1, 'freq': float(f[idx]), 'MSC': float(C[idx])})
    return pd.DataFrame(rows)

# ------------------------- Schumann envelope -------------------------

def schumann_envelope(sr: np.ndarray, fs: float, center_hz: float = 7.83, half_bw: float = 0.6) -> np.ndarray:
    yb = bandpass(sr, fs, center_hz-half_bw, center_hz+half_bw)
    return np.abs(signal.hilbert(yb))

# ------------------------- Runner -------------------------

def run_connectome_harmonics_resonance(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]],
    baseline_windows: Optional[List[Tuple[float,float]]],
    sr_channel: Optional[str] = None,
    time_col: str = 'Timestamp',
    band_for_functional: Tuple[float,float] = (8,13),
    W_conn: Optional[np.ndarray] = None,   # (n_ch x n_ch) adjacency in EEG order
    n_modes: int = 16,
    out_dir: str = 'exports_harmonics/session',
    show: bool = True,
    harmonics: List[float] = (7.83, 14.3, 20.8, 27.3, 33.8),
) -> Dict[str, object]:
    """
    Build harmonic basis (connectome W_conn if provided; else functional PLV) and test resonance:
      • Mode band power spectra in ignition vs baseline (alpha by default)
      • Mode MSC coherence to SR at Schumann harmonics (bars)
      • Time-resolved mode amplitude vs SR envelope (r + null)
      • Heatmap of first K eigenvectors (spatial harmonics)

    Returns summary dict and writes figures/CSV to out_dir.
    """
    _ensure_dir(out_dir)
    fs = infer_fs(RECORDS, time_col)

    # SR channel (auto-pick posterior if None)
    if sr_channel is None:
        sr_channel = 'EEG.Oz' if 'EEG.Oz' in RECORDS.columns else next((c for c in RECORDS.columns if c.startswith('EEG.')), None)
    sr = get_series(RECORDS, sr_channel)

    # Data matrix X
    X = np.vstack([get_series(RECORDS, ch) for ch in eeg_channels])  # (n_ch, T)
    # Build harmonic basis
    if W_conn is not None and W_conn.shape[0] == len(eeg_channels):
        vals, H = laplacian_eigendecomp(W_conn, n_modes)
        basis_name = 'connectome'
    else:
        # functional harmonics from PLV in alpha (or band_for_functional)
        A_plv = plv_adj(RECORDS, eeg_channels, band_for_functional, windows=baseline_windows or ignition_windows, time_col=time_col)
        vals, H = laplacian_eigendecomp(A_plv, n_modes)
        basis_name = 'functional'

    # Project signals to harmonic coefficients for each state
    states = {'ignition': ignition_windows, 'baseline': baseline_windows}
    results = {}
    for st, wins in states.items():
        if not wins: continue
        Xs = np.vstack([slice_concat(get_series(RECORDS, ch), fs, wins) for ch in eeg_channels])
        Xs = (Xs - Xs.mean(axis=1, keepdims=True)) / (Xs.std(axis=1, keepdims=True)+1e-12)
        A = project_to_harmonics(Xs, H)                       # (K, Tst)
        # spectra
        f, Pk = mode_welch_power(A, fs, nperseg=int(2*fs))    # Pk: (K, n_f)
        # band powers (per Schumann harmonic: narrow bands)
        bandpowers={}
        for hf in harmonics:
            bandpowers[f"{hf:.2f}"] = mode_band_power(A, fs, (hf-0.6, hf+0.6))
        # MSC to SR at harmonics
        Y = slice_concat(sr, fs, wins)
        msc_tab = msc_mode_to_sr(A, Y, fs, harmonics, nperseg=int(4*fs))
        results[st] = {'A':A, 'freqs':f, 'Pk':Pk, 'bandpowers':bandpowers, 'msc':msc_tab}

    # -------------- Plots --------------
    # Eigenvectors heatmap (spatial harmonics)
    plt.figure(figsize=(min(10, 0.5*n_modes+2), 3.5))
    im = plt.imshow(H, aspect='auto', cmap='coolwarm', vmin=-np.max(np.abs(H)), vmax=np.max(np.abs(H)))
    plt.colorbar(label='eigenvector weight')
    plt.yticks(range(H.shape[0]), [ch.split('.',1)[-1] for ch in eeg_channels], fontsize=8)
    plt.xticks(range(H.shape[1]), [f'H{k+1}' for k in range(H.shape[1])], fontsize=8)
    plt.title(f'{basis_name.capitalize()} harmonics — first {H.shape[1]} modes')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'harmonics_eigenvectors.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # Mode power spectrum by state
    if 'ignition' in results:
        f = results['ignition']['freqs']
        K = H.shape[1]
        for st in results:
            plt.figure(figsize=(8,3))
            plt.imshow(results[st]['Pk'], aspect='auto', origin='lower',
                       extent=[f[0], f[-1], 1, K], cmap='magma')
            plt.colorbar(label='Power')
            plt.xlabel('Frequency (Hz)'); plt.ylabel('Mode index')
            plt.title(f'Mode power spectrum — {st}')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'mode_spectrum_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

        # Δ band power around fundamental (7.83±0.6) across modes
        bp_ign = results['ignition']['bandpowers']['7.83']
        bp_base= results['baseline']['bandpowers']['7.83'] if 'baseline' in results else np.zeros_like(bp_ign)
        delta = bp_ign - bp_base
        plt.figure(figsize=(8,3))
        x = np.arange(1, len(delta)+1)
        plt.bar(x, delta, color='tab:blue', alpha=0.9)
        plt.xlabel('Mode index'); plt.ylabel('Δ power (Ign−Base)')
        plt.title('Schumann-band (7.83 Hz) mode Δ power')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'mode_delta_7p83.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # MSC bars by mode at harmonics (ignition)
    if 'ignition' in results:
        msc = results['ignition']['msc']
        pivot = msc.pivot(index='mode', columns='freq', values='MSC')
        plt.figure(figsize=(9,3))
        plt.imshow(pivot.values, aspect='auto', origin='lower',
                   extent=[pivot.columns.min(), pivot.columns.max(), 1, pivot.index.max()],
                   vmin=0, vmax=1, cmap='viridis')
        plt.colorbar(label='MSC')
        plt.xlabel('Frequency (Hz)'); plt.ylabel('Mode index')
        plt.title('Mode↔SR MSC at Schumann harmonics (ignition)')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'mode_msc_ignition.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # Time-resolved: strongest Schumann-band mode vs SR envelope (ignition)
    if 'ignition' in results:
        # pick mode k* with largest Δ power at 7.83 Hz (if baseline available)
        if 'baseline' in results:
            k_star = int(np.argmax(results['ignition']['bandpowers']['7.83'] -
                                   results['baseline']['bandpowers']['7.83']))
        else:
            k_star = int(np.argmax(results['ignition']['bandpowers']['7.83']))
        Ak = results['ignition']['A'][k_star]
        # envelope of Ak at 7.83
        envA = schumann_envelope(Ak, fs, center_hz=7.83, half_bw=0.6)
        envSR= schumann_envelope(slice_concat(sr, fs, ignition_windows), fs, center_hz=7.83, half_bw=0.6)
        # correlate with circular-shift null
        r = np.corrcoef(zscore(envA), zscore(envSR))[0,1]
        rng = np.random.default_rng(7)
        null=[]
        for _ in range(200):
            s = int(rng.integers(1, len(envSR)-1))
            null.append(np.corrcoef(zscore(envA), zscore(np.r_[envSR[-s:], envSR[:-s]]))[0,1])
        thr95 = np.nanpercentile(null, 95)
        t = np.arange(len(envA))/fs
        plt.figure(figsize=(9,3))
        plt.plot(t, zscore(envA), label=f'Mode {k_star+1} env (z)')
        plt.plot(t, zscore(envSR), label='SR env (z)')
        plt.title(f'Mode {k_star+1} vs SR envelope @7.83 Hz  (r={r:.2f}, null95={thr95:.2f})')
        plt.xlabel('Time (s)'); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, 'mode_env_vs_sr.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # ---------- Simple summary (safe; no 'names' needed) ----------
    summary_rows = []
    for st, r in results.items():
        row = {'state': st}
        # average MSC across modes at the fundamental
        msc_fund = r['msc']
        # guard for floating frequency labels
        msctab = msc_fund.copy()
        msctab['freq'] = msctab['freq'].round(2)
        row['MSC7p83_mean'] = float(msctab[msctab['freq'] == 7.83]['MSC'].mean())

        # top-1 mode Schumann-band power around 7.83 Hz
        key = f"{7.83:.2f}"
        if key in r['bandpowers']:
            row['MaxModePow7p83'] = float(np.max(r['bandpowers'][key]))
        else:
            row['MaxModePow7p83'] = np.nan

        row['Basis'] = 'connectome' if (W_conn is not None and W_conn.shape[0] == len(eeg_channels)) else 'functional'
        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)

    return {
        'basis': ('connectome' if (W_conn is not None and W_conn.shape[0] == len(eeg_channels)) else 'functional'),
        'eigenvals': vals,
        'H': H,
        'results': results,
        'summary': summary,
        'out_dir': out_dir
    }


In [ ]:
def schumann_envelope(sig: np.ndarray, fs: float,
                      center: float = 7.83, half: float = 0.6,
                      **kwargs) -> np.ndarray:
    """
    Compatibility wrapper: accepts center/half or center_hz/half_bw.
    """
    # allow alternate kw names
    if 'center_hz' in kwargs:
        center = kwargs['center_hz']
    if 'half_bw' in kwargs:
        half = kwargs['half_bw']
    b, a = signal.butter(4, [max(1e-6, (center-half))/(0.5*fs),
                             min(0.999, (center+half)/(0.5*fs))], btype='band')
    yb = signal.filtfilt(b, a, np.asarray(sig, float))
    return np.abs(signal.hilbert(yb))


In [ ]:
res = run_connectome_harmonics_resonance(
    RECORDS,
    
    eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    sr_channel='EEG.F4',                # or None to auto-pick posterior
    band_for_functional=(8,13),         # used when W_conn=None
    W_conn=None,                        # (optional) provide Laplacian source for harmonics
    n_modes=16,
    out_dir='exports_harmonics/S01',
    show=True
)
print(res['summary'])

In [ ]:
"""
Informational Geometry of EEG State Manifolds — Simple Graphs & Validity Tests
=============================================================================

Goal (validations you can run per session)
-----------------------------------------
• Build a *state vector* per short time window from EEG features
  (band powers, a simple integration index from PLV graph entropy, etc.).
• Embed the high-D state space (PCA + Isomap/UMAP) → 2D for visualization.
• Quantify the manifold:
    – Trustworthiness / Continuity (k-NN preservation)
    – Geodesic stress (Isomap geodesics vs 2D Euclidean)
    – Curvature proxy (local geodesic stretch)
    – Entropy of the embedded distribution (2D histogram entropy)
    – SPD (covariance) manifold spread (Log-Euclidean)
    – Silhouette (Ignition vs Baseline separation)
• Time-lock geometry to the Schumann envelope (corr + null).
• Simple graphs + surrogate tests (label permutation, circular shift).

Assumptions
-----------
RECORDS: pandas.DataFrame with a numeric time column (default 'Timestamp')
and EEG signals named 'EEG.*'. You provide `eeg_channels` (or we detect them).
Provide ignition and baseline windows; they will label the state-points.

Usage
-----
res = run_info_geometry_state_manifolds(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.Oz','EEG.Pz'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    sr_channel='EEG.Oz',               # None → auto-pick posterior channel
    time_col='Timestamp',
    out_dir='exports_infogeo/S01',
    show=True
)
print(res['summary'])
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal, sparse
from scipy.sparse.csgraph import shortest_path

# Optional: UMAP
try:
    import umap
    _HAS_UMAP = True
except Exception:
    _HAS_UMAP = False

# Optional: sklearn distances & silhouette
try:
    from sklearn.metrics import pairwise_distances, silhouette_score
    from sklearn.manifold import Isomap
    _HAS_SK = True
except Exception:
    _HAS_SK = False

# ---------------- small utilities ----------------

def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found.")

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order=4) -> np.ndarray:
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny)); f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

def zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

def schumann_envelope(sr: np.ndarray, fs: float, center=7.83, half_bw=0.6) -> np.ndarray:
    yb = bandpass(sr, fs, center-half_bw, center+half_bw)
    return np.abs(signal.hilbert(yb))

def slice_windows(RECORDS: pd.DataFrame, time_col: str, fs: float,
                  win_sec: float, step_sec: float) -> List[Tuple[int,int,float]]:
    """Return list of index windows (s,e, t_center)."""
    N = len(RECORDS)
    win = int(round(win_sec*fs)); step = int(round(step_sec*fs))
    idxs=[]
    for c in range(win//2, N-win//2, step):
        s = c - win//2; e = c + win//2
        idxs.append((s, e, c/fs))
    return idxs

def in_any_window(t: float, windows: List[Tuple[float,float]]) -> bool:
    for a,b in windows:
        if a <= t <= b: return True
    return False

# ---------------- features per window ----------------

def plv_graph_entropy(RECORDS, eeg_channels, fs, s, e, band):
    """Build PLV adjacency on [s:e] and return Laplacian spectral entropy."""
    phases=[]
    for ch in eeg_channels:
        x = get_series(RECORDS, ch)[s:e]
        xb = bandpass(x, fs, band[0], band[1])
        phases.append(np.angle(signal.hilbert(xb)))
    P = np.vstack(phases)
    N = len(eeg_channels)
    A = np.zeros((N,N))
    for i in range(N):
        for j in range(i,N):
            dphi = P[i]-P[j]
            A[i,j]=A[j,i]=float(np.abs(np.mean(np.exp(1j*dphi))))
    np.fill_diagonal(A, 0.0)
    # Laplacian entropy
    D = np.diag(A.sum(axis=1)); L = D - A
    L = 0.5*(L+L.T)
    vals = np.linalg.eigvalsh(L)
    vals = vals[vals>1e-12]
    if vals.size==0: return np.nan
    p = vals/np.sum(vals)
    return float(-np.sum(p*np.log(p)))

def window_features(RECORDS: pd.DataFrame,
                    eeg_channels: List[str],
                    sr_channel: str,
                    time_col: str,
                    win_sec: float = 2.0,
                    step_sec: float = 0.25,
                    bands: Dict[str, Tuple[float,float]] = None,
                    add_plv_entropy_band: Tuple[float,float] = (8,13)) -> pd.DataFrame:
    """
    Build a feature vector per sliding window:
      • band powers (theta/alpha/beta/gamma) averaged across channels
      • PLV graph Laplacian entropy (alpha by default)
      • SR envelope mean (7.83±0.6)
    Returns DataFrame with columns ['t','state','feat_*'] (state unlabeled here).
    """
    bands = bands or {'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,80)}
    fs = infer_fs(RECORDS, time_col)
    idxs = slice_windows(RECORDS, time_col, fs, win_sec, step_sec)
    # prefetch arrays
    X = np.vstack([get_series(RECORDS, ch) for ch in eeg_channels])  # (n_ch, N)
    sr = get_series(RECORDS, sr_channel)
    env_sr = schumann_envelope(sr, fs)
    rows=[]
    for s,e,t in idxs:
        row={'t':t}
        seg = X[:, s:e]
        for name,(f1,f2) in bands.items():
            bp = []
            for ch in range(seg.shape[0]):
                xb = bandpass(seg[ch], fs, f1,f2)
                bp.append(np.mean(xb**2))
            row[f'BP_{name}'] = float(np.mean(bp))
        # PLV graph entropy (alpha by default)
        try:
            row['PLV_H'] = plv_graph_entropy(RECORDS, eeg_channels, fs, s, e, add_plv_entropy_band)
        except Exception:
            row['PLV_H'] = np.nan
        row['SR_env'] = float(np.mean(env_sr[s:e]))
        rows.append(row)
    return pd.DataFrame(rows)

# ---------------- embeddings ----------------

def embed_states(F: pd.DataFrame, method: str = 'pca', n_neighbors: int = 8,
                 n_components: int = 2, random_state: int = 0) -> Dict[str, np.ndarray]:
    """
    Embed feature matrix (T×D) to 2D/3D. Returns {'X':coords, 'method':..., 'components':...}
    """
    X = F.values
    # z-score features
    X = (X - np.nanmean(X, axis=0)) / (np.nanstd(X, axis=0)+1e-12)
    # fill NaNs with col mean 0
    X[np.isnan(X)] = 0.0
    if method == 'umap' and _HAS_UMAP:
        reducer = umap.UMAP(n_neighbors=n_neighbors, n_components=n_components, metric='euclidean',
                            random_state=random_state)
        Z = reducer.fit_transform(X)
        return {'X': Z, 'method': 'umap'}
    elif method == 'isomap' and _HAS_SK:
        iso = Isomap(n_neighbors=n_neighbors, n_components=n_components)
        Z = iso.fit_transform(X)
        return {'X': Z, 'method': 'isomap'}
    else:
        # PCA fallback
        # covariance on columns
        C = np.cov(X, rowvar=False)
        vals, vecs = np.linalg.eigh(C)
        idx = np.argsort(vals)[::-1][:n_components]
        Z = X @ vecs[:, idx]
        return {'X': Z, 'method': 'pca'}

# ---------------- information-geometric metrics ----------------

def trust_continuity(D_high: np.ndarray, D_low: np.ndarray, k: int = 8) -> Tuple[float,float]:
    """Trustworthiness & Continuity (Tenenbaum / van der Maaten)."""
    n = D_high.shape[0]
    def ranks(D):
        R = np.zeros_like(D, dtype=int)
        for i in range(n):
            order = np.argsort(D[i])
            rank = np.empty(n, dtype=int); rank[order] = np.arange(n)
            R[i] = rank
        return R
    R_h = ranks(D_high); R_l = ranks(D_low)
    # neighborhoods
    N_h = [set(np.argsort(D_high[i])[1:k+1]) for i in range(n)]
    N_l = [set(np.argsort(D_low[i])[1:k+1])  for i in range(n)]
    # trust
    t_sum=0.0
    for i in range(n):
        U = N_l[i] - N_h[i]
        if U:
            t_sum += np.sum(R_h[i][list(U)] - k)
    T = 1.0 - (2.0 / (n*k*(2*n - 3*k - 1))) * t_sum if n>(3*k+1) else np.nan
    # continuity
    c_sum=0.0
    for i in range(n):
        V = N_h[i] - N_l[i]
        if V:
            c_sum += np.sum(R_l[i][list(V)] - k)
    C = 1.0 - (2.0 / (n*k*(2*n - 3*k - 1))) * c_sum if n>(3*k+1) else np.nan
    return float(T), float(C)

def geodesic_stress(X_high: np.ndarray, X_low: np.ndarray, k: int = 8) -> float:
    """
    Geodesic stress between k-NN geodesics (from high-D feature space) and Euclidean in embedding.
    """
    # high-D distances
    D_h = pairwise_distances(X_high) if _HAS_SK else np.linalg.norm(X_high[:,None,:]-X_high[None,:,:], axis=-1)
    # build kNN graph
    W = np.full_like(D_h, np.inf, dtype=float)
    for i in range(D_h.shape[0]):
        idx = np.argsort(D_h[i])[1:k+1]
        W[i, idx] = D_h[i, idx]
    W = np.minimum(W, W.T); np.fill_diagonal(W, 0.0)
    G = shortest_path(sparse.csr_matrix(W), directed=False)
    # low-D distances
    D_l = pairwise_distances(X_low) if _HAS_SK else np.linalg.norm(X_low[:,None,:]-X_low[None,:,:], axis=-1)
    num = np.nansum((G - D_l)**2); den = np.nansum(G**2)+1e-12
    return float(np.sqrt(num/den))

def curvature_proxy(X_high: np.ndarray, X_low: np.ndarray, k: int = 8) -> float:
    """
    Mean relative geodesic stretch over k-NN: mean_i mean_{j in N_i} (d_geo - d_euc)/d_euc.
    """
    D_h = pairwise_distances(X_high) if _HAS_SK else np.linalg.norm(X_high[:,None,:]-X_high[None,:,:], axis=-1)
    # kNN geodesic via D_h (graph on k neighbors)
    W = np.full_like(D_h, np.inf, dtype=float)
    for i in range(D_h.shape[0]):
        idx = np.argsort(D_h[i])[1:k+1]
        W[i, idx] = D_h[i, idx]
    W = np.minimum(W, W.T); np.fill_diagonal(W, 0.0)
    G = shortest_path(sparse.csr_matrix(W), directed=False)
    D_l = pairwise_distances(X_low) if _HAS_SK else np.linalg.norm(X_low[:,None,:]-X_low[None,:,:], axis=-1)
    e = (G - D_l) / (D_l + 1e-12)
    # only kNN pairs
    mask = np.isfinite(W) & (W>0)
    return float(np.nanmean(e[mask]))

def entropy_2d_embed(Z: np.ndarray, bins: int = 40) -> float:
    """
    Entropy of the embedded distribution via 2D histogram (Shannon, base e).
    """
    H, xe, ye = np.histogram2d(Z[:,0], Z[:,1], bins=bins, density=False)
    p = H.ravel().astype(float); p = p / (np.sum(p)+1e-12)
    p = p[p>0]
    return float(-np.sum(p*np.log(p)))

def logeuclidean_spread(cov_list: List[np.ndarray]) -> float:
    """
    Spread on SPD manifold (Log-Euclidean): mean pairwise ||log(S_i) − log(S_j)||_F.
    """
    logs=[]
    for S in cov_list:
        S = 0.5*(S+S.T) + 1e-9*np.eye(S.shape[0])
        vals, vecs = np.linalg.eigh(S)
        logs.append(vecs @ np.diag(np.log(np.maximum(vals,1e-12))) @ vecs.T)
    logs = np.stack(logs, axis=0)
    n = logs.shape[0]
    dists=[]
    for i in range(n):
        for j in range(i+1,n):
            dists.append(np.linalg.norm(logs[i]-logs[j],'fro'))
    return float(np.nanmean(dists)) if dists else np.nan

# ---------------- main runner ----------------

def run_info_geometry_state_manifolds(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: List[Tuple[float,float]],
    baseline_windows: List[Tuple[float,float]],
    sr_channel: Optional[str] = None,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_infogeo/session',
    show: bool = True,
    win_sec: float = 2.0,
    step_sec: float = 0.25,
    bands: Dict[str, Tuple[float,float]] = None,
    embed_method: str = 'isomap',   # 'umap'|'isomap'|'pca'
    n_neighbors: int = 8
) -> Dict[str, object]:
    """
    Build state vectors → embeddings → info-geom metrics, with simple graphs + tests.
    Outputs figures + CSV summary in out_dir.
    """
    _ensure_dir(out_dir)
    fs = infer_fs(RECORDS, time_col)
    if sr_channel is None:
        sr_channel = 'EEG.Oz' if 'EEG.Oz' in RECORDS.columns else next((c for c in RECORDS.columns if c.startswith('EEG.')), None)

    # 1) Feature time series
    F = window_features(RECORDS, eeg_channels, sr_channel, time_col, win_sec, step_sec, bands)
    # Label state per window center
    F['state'] = ['ignition' if in_any_window(t, ignition_windows) else
                  ('baseline' if in_any_window(t, baseline_windows) else 'other')
                  for t in F['t'].values]
    F0 = F[F['state']!='other'].reset_index(drop=True)
    feat_cols = [c for c in F0.columns if c.startswith('BP_')] + ['PLV_H','SR_env']
    X_high = F0[feat_cols].values
    # keep covariances per state for SPD spread
    cov_ign = []; cov_bas = []
    if np.any(F0['state']=='ignition'):
        cov_ign.append(np.cov(F0[F0['state']=='ignition'][feat_cols].values, rowvar=False))
    if np.any(F0['state']=='baseline'):
        cov_bas.append(np.cov(F0[F0['state']=='baseline'][feat_cols].values, rowvar=False))

    # 2) Embedding
    emb = embed_states(F0[feat_cols], method=embed_method, n_neighbors=n_neighbors, n_components=2, random_state=0)
    Z = emb['X']  # (T’, 2)

    # BEFORE
    # F0[['Z1','Z2']] = Z

    # AFTER
    F0 = F0.reset_index(drop=True).copy()
    if Z.ndim != 2 or Z.shape[1] < 2:
        raise ValueError(f"Embedding returned shape {Z.shape}; need at least 2D. "
                         "Set n_components=2 or use method='pca'/'isomap'/'umap' accordingly.")
    F0['Z1'] = Z[:, 0]
    F0['Z2'] = Z[:, 1]


    # 3) Metrics
    # high-D distances on features
    D_high = pairwise_distances(X_high) if _HAS_SK else np.linalg.norm(X_high[:,None,:]-X_high[None,:,:], axis=-1)
    D_low  = pairwise_distances(Z) if _HAS_SK else np.linalg.norm(Z[:,None,:]-Z[None,:,:], axis=-1)
    T, C = trust_continuity(D_high, D_low, k=n_neighbors)
    stress = geodesic_stress(X_high, Z, k=n_neighbors)
    curv   = curvature_proxy(X_high, Z, k=n_neighbors)
    H_emb  = entropy_2d_embed(Z, bins=40)
    SPD_ign = logeuclidean_spread(cov_ign) if cov_ign else np.nan
    SPD_bas = logeuclidean_spread(cov_bas) if cov_bas else np.nan

    # silhouette (Ign vs Base)
    labels = (F0['state']=='ignition').astype(int).values
    sil = silhouette_score(Z, labels) if _HAS_SK and len(np.unique(labels))>1 else np.nan
    # permutation null for silhouette
    rng = np.random.default_rng(13)
    null_sil=[]
    for _ in range(200):
        perm = rng.permutation(labels)
        if _HAS_SK and len(np.unique(perm))>1:
            null_sil.append(silhouette_score(Z, perm))
    sil_thr95 = float(np.nanpercentile(null_sil, 95)) if null_sil else np.nan

    # correlation with SR envelope (on aligned centers)
    # re-sample SR envelope on centers corresponding to F0 rows
    sr = get_series(RECORDS, sr_channel)
    env = schumann_envelope(sr, fs)
    t_all = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    env_centers = np.interp(F0['t'].values, t_all, env)
    # pick principal coordinate (Z1) as manifold coordinate, corr with env
    r = np.corrcoef(Z[:,0], env_centers)[0,1]
    null_r=[]
    for _ in range(200):
        s = int(rng.integers(1, len(env_centers)-1))
        null_r.append(np.corrcoef(Z[:,0], np.r_[env_centers[-s:],env_centers[:-s]])[0,1])
    r_thr95 = float(np.nanpercentile(null_r, 95))

    # 4) Plots
    # scatter colored by state
    plt.figure(figsize=(6,5))
    cmap = {'ignition':'tab:red','baseline':'tab:blue'}
    for st in ['baseline','ignition']:
        idx = F0['state']==st
        plt.scatter(Z[idx,0], Z[idx,1], s=12, alpha=0.7, c=cmap[st], label=st)
    plt.title(f"{emb['method'].upper()} manifold — states"); plt.xlabel('Z1'); plt.ylabel('Z2')
    plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir,'embed_states.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # scatter colored by SR envelope
    plt.figure(figsize=(6,5))
    sc = plt.scatter(Z[:,0], Z[:,1], c=env_centers, s=12, cmap='viridis')
    plt.colorbar(sc, label='SR env (a.u.)')
    plt.title('Manifold colored by Schumann envelope'); plt.xlabel('Z1'); plt.ylabel('Z2')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir,'embed_sr_colormap.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # metric bars + null lines
    plt.figure(figsize=(8,3))
    names = ['Trust','Cont','Stress','Curv','H(emb)','Sil','Sil_null95','SPD_ign','SPD_bas','r(Z1,SR)','r_null95']
    vals  = [T, C, stress, curv, H_emb, sil, sil_thr95, SPD_ign, SPD_bas, r, r_thr95]
    plt.bar(range(len(names)), vals, color='tab:purple', alpha=0.85)
    plt.xticks(range(len(names)), names, rotation=30)
    plt.title('Information-geometric metrics'); plt.tight_layout()
    plt.savefig(os.path.join(out_dir,'metrics_bars.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # 5) Save tables
    F0.to_csv(os.path.join(out_dir,'state_features_embedding.csv'), index=False)
    summary = pd.DataFrame([{
        'method': emb['method'], 'trust':T, 'continuity':C, 'stress':stress, 'curvature_proxy':curv,
        'entropy_2d':H_emb, 'silhouette':sil, 'sil_null95':sil_thr95,
        'SPD_spread_ign':SPD_ign, 'SPD_spread_bas':SPD_bas,
        'r_Z1_SR':r, 'r_null95':r_thr95
    }])
    summary.to_csv(os.path.join(out_dir,'summary.csv'), index=False)

    return {'summary': summary,
            'features': F0,
            'embedding': Z,
            'metrics': {'trust':T,'continuity':C,'stress':stress,'curvature':curv,
                        'entropy_2d':H_emb,'silhouette':sil,'sil_null95':sil_thr95,
                        'SPD_ign':SPD_ign,'SPD_bas':SPD_bas,'r':r,'r_null95':r_thr95},
            'out_dir': out_dir}


In [ ]:
res = run_info_geometry_state_manifolds(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6','EEG.F3','EEG.AF3', 'EEG.AF4', 'EEG.T7', 'EEG.T8'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    sr_channel='EEG.F4',               # None → auto-pick posterior channel
    time_col='Timestamp',
    out_dir='exports_infogeo/S01',
    show=True
)
print(res['summary'])

In [ ]:
"""
Directed Connectivity & Causal Routing — Simple Graphs and Validation Tests
============================================================================

What this module does (turn-key):
1) Builds ROI time series from your EEG channels (or uses your small channel set directly).
2) Fits a VAR model per state (Ignition vs Baseline) on **band-limited** multivariate EEG(+SR).
3) Computes **frequency-domain directed measures** from the VAR:
   • DTF (Directed Transfer Function)  i <- j  (per frequency)
   • PDC (Partial Directed Coherence) i <- j  (per frequency)
4) Summarizes:
   • ROI↔ROI directed matrices (mean within bands).
   • **Brain↔Field** direction at ~7.83 Hz (SR→ROI and ROI→SR), with a **circular-shift null** for SR.
5) Plots:
   • Directed heatmaps (Ignition vs Baseline).
   • Bars for SR→ROI and ROI→SR at 7.83 Hz with **95% null** lines.
   • A simple “net flow” index per ROI:  Σ_j DTF(i<-j) − Σ_j DTF(j<-i)
6) CSV outputs with all summaries and p-values.

Requirements
------------
• pandas, numpy, scipy, matplotlib, networkx
• statsmodels (for VAR). If not present, the module will **skip** VAR‐based parts and print a note.

Inputs
------
RECORDS: pandas.DataFrame with a numeric time column (default 'Timestamp')
and EEG channels named 'EEG.*' (e.g., 'EEG.O1', 'EEG.F4', ...). You can also
include a Schumann reference channel; if not provided, we auto-pick a posterior EEG.

Usage
-----
res = run_directed_connectivity_routing(
    RECORDS,
    eeg_channels=['EEG.F4','EEG.Pz','EEG.O1','EEG.O2'],   # or let roi_map group many sensors
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    roi_map=None,                                   # or dict like {'F':['F3','F4','Fz'], 'P':['P3','P4','Pz'], ...}
    sr_channel=None,                                 # None → auto-pick posterior (e.g., Oz)
    bands={'theta':(4,8), 'alpha':(8,13), 'beta':(13,30)},
    f0=7.83,
    time_col='Timestamp',
    out_dir='exports_directed/S01',
    show=True
)
print(res['summary'])
"""

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from typing import Dict, List, Tuple, Optional
from scipy import signal

# statsmodels for VAR
try:
    from statsmodels.tsa.api import VAR
    _HAS_SM = True
except Exception:
    _HAS_SM = False

# ---------------------------- utilities ----------------------------

def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(RECORDS: pd.DataFrame, time_col: str = 'Timestamp') -> float:
    t = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0) & np.isfinite(dt)]
    if dt.size == 0: raise ValueError("Cannot infer sampling rate from time column.")
    return float(1.0 / np.median(dt))

def get_series(RECORDS: pd.DataFrame, name: str) -> np.ndarray:
    if name in RECORDS.columns:
        x = pd.to_numeric(RECORDS[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in RECORDS.columns:
        x = pd.to_numeric(RECORDS[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Signal '{name}' not found.")

def zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

def slice_concat(x: np.ndarray, fs: float, windows: Optional[List[Tuple[float,float]]]) -> np.ndarray:
    if not windows: return x.copy()
    segs=[]; n=len(x)
    for (t0,t1) in windows:
        i0,i1 = int(round(t0*fs)), int(round(t1*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def bandpass(x: np.ndarray, fs: float, f1: float, f2: float, order=4) -> np.ndarray:
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny)); f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

def pick_posterior_sr(RECORDS: pd.DataFrame) -> str:
    # prefer Oz, O1, O2, then any EEG.*
    for c in ['EEG.Oz','EEG.O1','EEG.O2']:
        if c in RECORDS.columns: return c
    for c in RECORDS.columns:
        if c.startswith('EEG.'): return c
    raise ValueError("No EEG.* channel found for SR")

# ---------------------------- ROI builder ----------------------------

def make_roi_series(RECORDS: pd.DataFrame,
                    eeg_channels: List[str],
                    roi_map: Optional[Dict[str, List[str]]] = None,
                    windows: Optional[List[Tuple[float,float]]] = None,
                    time_col: str = 'Timestamp') -> Tuple[np.ndarray, List[str], float]:
    """
    Return (X, names, fs) where X is (n_nodes, T) z-scored per node.
    If roi_map is None, treat eeg_channels as nodes.
    """
    fs = infer_fs(RECORDS, time_col)
    if roi_map:
        X=[]; names=[]
        for roi, chans in roi_map.items():
            present=[]
            for ch in chans:
                nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
                if nm in RECORDS.columns:
                    present.append(slice_concat(get_series(RECORDS, nm), fs, windows))
            if present:
                arr = np.vstack(present)
                L = np.min([len(a) for a in arr])
                arr = arr[:,:L]
                X.append(np.mean(arr, axis=0))
                names.append(roi)
        if not X: raise ValueError("No ROI channels found in RECORDS.")
        X = np.vstack([zscore(x) for x in X])
        return X, names, fs
    else:
        # channels directly
        X=[]
        for ch in eeg_channels:
            x = slice_concat(get_series(RECORDS, ch), fs, windows)
            X.append(zscore(x))
        names = [c.split('.',1)[-1] for c in eeg_channels]
        X = np.vstack(X)
        return X, names, fs

# ---------------------------- VAR → DTF/PDC ----------------------------

def fit_var_robust(X: np.ndarray,
                   fs: float,
                   order_max: int = 16,
                   trend: str = 'n',          # no intercept (we z-score / prefilter)
                   ridge: float = 1e-8) -> Tuple[object, Optional[int], Optional[np.ndarray]]:
    """
    Robust VAR(p) fit with manual BIC selection and PD enforcement on Σ_u.
    X: (n_nodes, T)
    Returns (res, p_star, A) where A has shape (p, n, n). If fit fails, returns (None,None,None).
    """
    if not _HAS_SM:
        return None, None, None

    Y = X.T  # (T, n)
    n = Y.shape[1]
    N = Y.shape[0]

    # keep p small relative to samples
    p_cap = max(1, min(order_max, N // max(8, 2*n)))  # conservative cap
    best = None
    best_bic = np.inf
    best_p = None

    model = VAR(Y)
    for p in range(1, p_cap+1):
        try:
            res = model.fit(p, trend=trend)
            # Residual covariance; ridge to ensure PD
            Sigma = 0.5*(res.sigma_u_mle + res.sigma_u_mle.T) + ridge*np.eye(n)
            # Check PD
            if np.any(np.linalg.eigvalsh(Sigma) <= 0):
                continue
            # BIC (manual): −2ℓ + k ln(T)
            T_eff = res.nobs
            # loglike of Gaussian VAR residuals
            ll = -0.5*T_eff * (n*np.log(2*np.pi) + np.log(np.linalg.det(Sigma)) + n)
            k = n*n*p  # parameters (trend excluded)
            bic = -2*ll + k*np.log(max(1,T_eff))
            if bic < best_bic:
                best_bic = bic
                best = res
                best_p = p
        except Exception:
            continue

    if best is None:
        return None, None, None
    A = np.array(best.coefs)  # (p, n, n)
    return best, best_p, A


def A_of_f(A: np.ndarray, f: np.ndarray, fs: float) -> np.ndarray:
    """A(f) = I − Σ_k A_k e^{−i2πfk/fs};  returns (n_f, n, n)."""
    p, n, _ = A.shape
    I = np.eye(n)
    Af = []
    for ff in f:
        Z = I.copy()
        for k in range(1, p+1):
            Z = Z - A[k-1] * np.exp(-1j*2*np.pi*ff * k / fs)
        Af.append(Z)
    return np.array(Af)

def H_of_f(Af: np.ndarray) -> np.ndarray:
    """Transfer matrix H(f) = A(f)^{-1}, per frequency."""
    Hf = np.zeros_like(Af, dtype=complex)
    for i in range(Af.shape[0]):
        Hf[i] = np.linalg.inv(Af[i])
    return Hf

def spectral_dtf_pdc(A: np.ndarray, fs: float,
                     fmin: float, fmax: float, n_freq: int = 128) -> Dict[str, np.ndarray]:
    """
    DTF_{i<-j}(f) = |H_ij| / sqrt(Σ_k |H_ik|^2)   (row-normalized)
    PDC_{i<-j}(f) = |A_ij| / sqrt(Σ_k |A_kj|^2)   (column-normalized A(f))
    Returns dict with 'f','DTF','PDC' arrays of shape (n, n, n_freq).
    """
    n = A.shape[1]
    f = np.linspace(fmin, fmax, n_freq)
    Af = A_of_f(A, f, fs)         # (n_f, n, n)
    Hf = H_of_f(Af)               # (n_f, n, n)
    DTF = np.zeros((n, n, n_freq))
    PDC = np.zeros((n, n, n_freq))
    for k in range(n_freq):
        H = Hf[k]
        num = np.abs(H)**2
        den = np.sum(num, axis=1, keepdims=True) + 1e-24
        DTF[:, :, k] = np.sqrt(num/den)
        A_k = Af[k]
        numA = np.abs(A_k)**2
        denA = np.sum(numA, axis=0, keepdims=True) + 1e-24
        PDC[:, :, k] = np.sqrt(numA/denA)
    return {'f': f, 'DTF': DTF, 'PDC': PDC}

def band_average(M: np.ndarray, f: np.ndarray, band: Tuple[float,float]) -> np.ndarray:
    """Mean over frequency indices in band; M shape (n,n,n_f)."""
    sel = (f>=band[0]) & (f<=band[1])
    if not np.any(sel): sel = [np.argmin(np.abs(f - np.mean(band)))]
    return np.nanmean(M[:, :, sel], axis=2)

# ---------------------------- Nulls for SR causality ----------------------------

def circular_shift_sr_null(RECORDS: pd.DataFrame, sr_channel: str, fs: float,
                           windows: List[Tuple[float,float]],
                           nodes: List[str], roi_map: Optional[Dict[str,List[str]]],
                           band: Tuple[float,float],
                           order_max: int, n_surr: int = 100) -> float:
    """
    Build null distribution for SR→ROI DTF at ~f0 by circularly shifting SR and refitting VAR.
    Returns the 95th percentile (threshold).
    """
    rng = np.random.default_rng(11)
    vals=[]
    for _ in range(n_surr):
        # shift SR only
        sr = slice_concat(get_series(RECORDS, sr_channel), fs, windows)
        s = int(rng.integers(1, len(sr)-1))
        sr_sh = np.r_[sr[-s:], sr[:-s]]
        # build node matrix with shifted SR (last node)
#         if roi_map:
#             Xroi, names, _ = make_roi_series(RECORDS, nodes, roi_map, windows, time_col='Timestamp')
#         else:
#             Xroi, names, _ = make_roi_series(RECORDS, nodes, None, windows, time_col='Timestamp')
        
        # build ROI
        Xroi, names, _ = make_roi_series(RECORDS, nodes, roi_map, windows, time_col='Timestamp')
        sr  = slice_concat(get_series(RECORDS, sr_channel), fs, windows)

        # align lengths
        L = min(Xroi.shape[1], sr.shape[0])
        Xroi = Xroi[:, :L]
        sr   = sr[:L]

        # shift only SR
        s = int(rng.integers(1, L-1))
        sr_sh = np.r_[sr[-s:], sr[:-s]]

        # prefilter & z-score (same preband as main path)
        preband = (2.0, 45.0)
        for i in range(Xroi.shape[0]):
            Xroi[i] = bandpass(Xroi[i], fs, preband[0], preband[1])
        sr_sh = bandpass(sr_sh, fs, preband[0], preband[1])

        Xroi = (Xroi - Xroi.mean(axis=1, keepdims=True)) / (Xroi.std(axis=1, keepdims=True)+1e-12)
        sr_z = (sr_sh - sr_sh.mean())/(sr_sh.std()+1e-12)

        X = np.vstack([Xroi, sr_z[None, :]])
        X += 1e-6 * rng.standard_normal(X.shape)

        
        # append SR
        X = np.vstack([Xroi, zscore(sr_sh)])
        if not _HAS_SM: return np.nan
        res, p, A = fit_var(X, order_max=order_max)
        if A is None: continue
        spec = spectral_dtf_pdc(A, fs, fmin=band[0], fmax=band[1], n_freq=64)
        D = band_average(spec['DTF'], spec['f'], band=(band[0],band[1]))
        # SR is last node
        sr_idx = X.shape[0]-1
        inbound = np.nanmean(D[:-1, sr_idx])   # ROI <- SR
        vals.append(inbound)
    return float(np.nanpercentile(vals, 95)) if vals else np.nan

# ---------------------------- Runner ----------------------------

def run_directed_connectivity_routing(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]],
    baseline_windows: Optional[List[Tuple[float,float]]],
    roi_map: Optional[Dict[str, List[str]]] = None,
    sr_channel: Optional[str] = None,
    bands: Dict[str, Tuple[float,float]] = None,
    f0: float = 7.83,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_directed/session',
    show: bool = True,
    order_max: int = 16,
    n_surr: int = 100
) -> Dict[str, object]:
    """
    Map directed connectivity inside the brain and between brain and field (SR).
    Produces heatmaps and SR→ROI bars with 95% null thresholds, plus CSV summaries.
    """
    _ensure_dir(out_dir)
    fs = infer_fs(RECORDS, time_col)
    bands = bands or {'theta':(4,8),'alpha':(8,13),'beta':(13,30)}
    sr_channel = sr_channel or pick_posterior_sr(RECORDS)
    # node labels
    node_names = list(roi_map.keys()) if roi_map else [c.split('.',1)[-1] for c in eeg_channels]
    # states
    states = {'ignition': ignition_windows, 'baseline': baseline_windows}

    results = {}
    for st, wins in states.items():
        # --- ROI+SR matrix for this state (wins) ---
        Xroi, names, _ = make_roi_series(RECORDS, eeg_channels, roi_map, wins, time_col=time_col)
        sr  = slice_concat(get_series(RECORDS, sr_channel), fs, wins)

        # length-align
        L = min(Xroi.shape[1], sr.shape[0])
        Xroi = Xroi[:, :L]; sr = sr[:L]

        # prefilter (stabilize)
        preband = (2.0, 45.0)
        for i in range(Xroi.shape[0]):
            Xroi[i] = bandpass(Xroi[i], fs, preband[0], preband[1])
        sr = bandpass(sr, fs, preband[0], preband[1])

        # z-score & tiny jitter
        Xroi = (Xroi - Xroi.mean(axis=1, keepdims=True)) / (Xroi.std(axis=1, keepdims=True)+1e-12)
        sr_z = (sr - sr.mean())/(sr.std()+1e-12)
        rng  = np.random.default_rng(5)
        X    = np.vstack([Xroi, sr_z[None,:]])
        X   += 1e-6 * rng.standard_normal(X.shape)

        # ---------- try safeguarded VAR ----------
        res, p, A = fit_var_safeguarded(X, fs, order_max=order_max)
        if A is None:
            # ---------- PCA fallback ----------
            K = max(2, min(4, Xroi.shape[0]//2, (Xroi.shape[1]//20)))  # 2..4 comps; keep sane w.r.t. T
            if K >= 2:
                Z, U = pca_reduce_nodes(Xroi, k=K)             # (K, T)
                Xp = np.vstack([Z, sr_z[None,:]])              # PCs + SR
                Xp += 1e-6 * rng.standard_normal(Xp.shape)
                res, p, A = fit_var_safeguarded(Xp, fs, order_max=min(order_max, 8))
                if A is not None:
                    # compute DTF/PDC on PCs
                    spec = spectral_dtf_pdc(A, fs, fmin=2.0, fmax=45.0, n_freq=256)
                    band_mats = {bn: band_average(spec['DTF'], spec['f'], bnd) for bn,bnd in bands.items()}
                    band0 = (max(2.0, f0-0.6), min(45.0, f0+0.6))
                    D0 = band_average(spec['DTF'], spec['f'], band0)
                    sr_idx = Xp.shape[0]-1
                    inbound_SR = D0[:-1, sr_idx]; outbound_SR = D0[sr_idx, :-1]
                    # NOTE: labels are PCs (PC1..PCk) not ROIs when in PCA fallback
                    pc_names = [f'PC{k+1}' for k in range(K)] + ['SR']
                    results[st] = {'names': pc_names, 'DTF_spec': spec, 'DTF_bands': band_mats,
                                   'DTF_SR_in': inbound_SR, 'DTF_SR_out': outbound_SR,
                                   'thr95_SR_in': np.nan, 'order': p, 'mode': 'PCA'}
                else:
                    # ---------- bivariate Granger fallback ----------
                    F = granger_bivariate_matrix(X, maxlag=6)   # on nodes+SR
                    names_sr = names + ['SR']
                    results[st] = {'names': names_sr, 'BIV_F': F, 'mode': 'BIV'}
            else:
                # ---------- bivariate fallback directly ----------
                F = granger_bivariate_matrix(X, maxlag=6)
                names_sr = names + ['SR']
                results[st] = {'names': names_sr, 'BIV_F': F, 'mode': 'BIV'}
            # plotting for fallback handled below
            continue

        # ---------- (normal) DTF/PDC path ----------
        spec = spectral_dtf_pdc(A, fs, fmin=2.0, fmax=45.0, n_freq=256)
        band_mats = {bn: band_average(spec['DTF'], spec['f'], bnd) for bn,bnd in bands.items()}
        band0 = (max(2.0, f0-0.6), min(45.0, f0+0.6))
        D0 = band_average(spec['DTF'], spec['f'], band0)
        sr_idx = X.shape[0]-1
        inbound_SR  = D0[:-1, sr_idx]
        outbound_SR = D0[sr_idx, :-1]
        thr95 = circular_shift_sr_null(RECORDS, sr_channel, fs, wins, eeg_channels, roi_map, band0, order_max=order_max, n_surr=n_surr)
        results[st] = {'names': names + ['SR'], 'DTF_spec': spec, 'DTF_bands': band_mats,
                       'DTF_SR_in': inbound_SR, 'DTF_SR_out': outbound_SR, 'thr95_SR_in': thr95,
                       'order': p, 'mode': 'VAR'}

    # ---- Delta matrices (Ign − Base), handle VAR/PCA vs BIV gracefully ----
    if 'ignition' in results and 'baseline' in results:
        mode_i = results['ignition'].get('mode', 'VAR')
        mode_b = results['baseline'].get('mode', 'VAR')

        # helper to align names/matrices if lengths differ
        def _align(A, B):
            n = min(A.shape[0], B.shape[0])
            return A[:n,:n], B[:n,:n]

        names_i = results['ignition']['names']
        names_b = results['baseline']['names']
        names = names_i if names_i == names_b else [f'N{k+1}' for k in range(min(len(names_i), len(names_b)))]

        if ('DTF_bands' in results['ignition']) and ('DTF_bands' in results['baseline']):
            # VAR (or PCA) available in BOTH states
            for bn in bands.keys():
                if (bn in results['ignition']['DTF_bands']) and (bn in results['baseline']['DTF_bands']):
                    Bi = results['ignition']['DTF_bands'][bn]
                    Bb = results['baseline']['DTF_bands'][bn]
                    Bi, Bb = _align(Bi, Bb)
                    dM = Bi - Bb
                    plt.figure(figsize=(5.2,4.4))
                    im = plt.imshow(dM, cmap='bwr', vmin=-np.max(np.abs(dM)), vmax=np.max(np.abs(dM)))
                    plt.colorbar(im, label='ΔDTF (Ign − Base)')
                    plt.xticks(range(len(names)), names, rotation=90, fontsize=8)
                    plt.yticks(range(len(names)), names, fontsize=8)
                    title_mode = results['ignition'].get('mode','VAR')
                    plt.title(f'Δ Directed flow — {bn} ({title_mode})')
                    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'dtf_delta_{bn}.png'), dpi=140)
                    if show: plt.show()
                    plt.close()
                else:
                    print(f"[INFO] Band '{bn}' not present in both states; skipping Δ for that band.")
        elif ('BIV_F' in results['ignition']) and ('BIV_F' in results['baseline']):
            # both are bivariate Granger
            Fi = results['ignition']['BIV_F']; Fb = results['baseline']['BIV_F']
            Fi, Fb = _align(Fi, Fb)
            dM = Fi - Fb
            plt.figure(figsize=(5.2,4.4))
            im = plt.imshow(dM, cmap='bwr', vmin=-np.max(np.abs(dM)), vmax=np.max(np.abs(dM)))
            plt.colorbar(im, label='Δ Pairwise Granger (norm.)')
            plt.xticks(range(len(names)), names, rotation=90, fontsize=8)
            plt.yticks(range(len(names)), names, fontsize=8)
            plt.title('Δ Pairwise Granger (Ign − Base)')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'granger_biv_delta.png'), dpi=140)
            if show: plt.show()
            plt.close()
        else:
            # Mixed modes (e.g., VAR/PCA in one state, BIV in the other) — we can’t compute ΔDTF cleanly
            print("[INFO] Mixed modes across states (e.g., VAR vs BIV) — skipping Δ matrix.")

    
    # ===== Plotting for PCA or BIV fallbacks =====
    if results.get(st, {}).get('mode') == 'PCA':
        names_local = results[st]['names']
        for bn,bmat in results[st]['DTF_bands'].items():
            plt.figure(figsize=(5.2,4.4))
            im = plt.imshow(bmat, vmin=0, vmax=1, cmap='magma')
            plt.colorbar(im, label='DTF (PC space)')
            plt.xticks(range(len(names_local)), names_local, rotation=90, fontsize=8)
            plt.yticks(range(len(names_local)), names_local, fontsize=8)
            plt.title(f'DTF {bn} — {st} (PCA fallback)')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'dtf_{bn}_{st}_pca.png'), dpi=140)
            if show: plt.show()
            plt.close()

    if results.get(st, {}).get('mode') == 'BIV':
        F = results[st]['BIV_F']
        names_local = results[st]['names']
        plt.figure(figsize=(5.2,4.4))
        im = plt.imshow(F, vmin=0, vmax=1, cmap='inferno')
        plt.colorbar(im, label='Bivariate Granger (norm.)')
        plt.xticks(range(len(names_local)), names_local, rotation=90, fontsize=8)
        plt.yticks(range(len(names_local)), names_local, fontsize=8)
        plt.title(f'Pairwise Granger matrix — {st}')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'granger_biv_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

    
    
    # ---- Summary CSVs ----
    summary_rows=[]
    for st in results:
        r = results[st]
        names_local = r['names']
        mode_local  = r.get('mode','VAR')
        if 'DTF_bands' in r and 'alpha' in r['DTF_bands']:
            M = r['DTF_bands']['alpha']
            n = len(names_local)-1 if names_local[-1] == 'SR' else len(names_local)
            brain = M[:n,:n]
            net = np.sum(brain, axis=1) - np.sum(brain, axis=0)
            for i in range(n):
                summary_rows.append({'state': st, 'mode': mode_local, 'node': names_local[i],
                                     'net_flow_alpha': float(net[i])})
        elif 'BIV_F' in r:
            # summarize pairwise Granger as a fallback
            F = r['BIV_F']
            n = len(names_local)-1 if names_local[-1] == 'SR' else len(names_local)
            net = np.sum(F[:n,:n], axis=1) - np.sum(F[:n,:n], axis=0)
            for i in range(n):
                summary_rows.append({'state': st, 'mode': mode_local, 'node': names_local[i],
                                     'net_flow_biv': float(net[i])})
    if summary_rows:
        pd.DataFrame(summary_rows).to_csv(os.path.join(out_dir,'summary.csv'), index=False)


    return {'results': results, 'out_dir': out_dir}


In [ ]:
# ---------- PCA reduction (nodes -> PCs) ----------
def pca_reduce_nodes(X: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    X: (n_nodes, T) z-scored. Returns (Z, U) where
    Z: (k, T) top-k component time-series, U: (n_nodes, k) loadings (orthonormal).
    """
    Xc = X - X.mean(axis=1, keepdims=True)
    # SVD of node covariance (fast & stable)
    C = Xc @ Xc.T / Xc.shape[1]         # (n, n)
    vals, vecs = np.linalg.eigh(C)
    idx = np.argsort(vals)[::-1][:k]
    U = vecs[:, idx]                    # loadings (n, k)
    Z = U.T @ Xc                        # (k, T)
    return Z, U

# ---------- time-domain pairwise (bivariate) Granger ----------
def granger_bivariate_matrix(X: np.ndarray, maxlag: int = 6) -> np.ndarray:
    """
    X: (n_nodes, T) z-scored.
    Returns F-stat matrix F_{i<-j} at the best lag (1..maxlag) per pair.
    (Time-domain Granger strength; simple validation fallback.)
    """
    n, T = X.shape
    F = np.zeros((n, n))
    for i in range(n):
        yi = X[i]
        for j in range(n):
            if i == j: continue
            yj = X[j]
            bestF = 0.0
            for p in range(1, maxlag+1):
                # build regressors
                # restricted: yi_t ~ [yi_{t-1..t-p}]
                # unrestricted: yi_t ~ [yi_{t-1..t-p}, yj_{t-1..t-p}]
                Y = yi[p:]
                Phi_i = np.column_stack([yi[p-k:-k] for k in range(1, p+1)])
                Phi_ij= np.column_stack([Phi_i] + [yj[p-k:-k] for k in range(1, p+1)])
                # LS
                beta_i  = np.linalg.lstsq(Phi_i,  Y, rcond=None)[0]
                beta_ij = np.linalg.lstsq(Phi_ij, Y, rcond=None)[0]
                rss_i   = np.sum((Y - Phi_i  @ beta_i )**2)
                rss_ij  = np.sum((Y - Phi_ij @ beta_ij)**2)
                k_num = p          # extra params
                k_den = len(Y) - 2*p
                if k_den <= 0 or rss_ij <= 0: continue
                Fp = ((rss_i - rss_ij)/k_num) / (rss_ij / k_den)
                if np.isfinite(Fp) and Fp > bestF:
                    bestF = Fp
            F[i, j] = bestF
    # normalize to 0..1 for plotting
    F = F / (np.nanmax(F) + 1e-12)
    return F

# ---------- robust VAR selector (replaces fit_var_robust earlier) ----------
def fit_var_safeguarded(X: np.ndarray,
                        fs: float,
                        order_max: int = 16,
                        ridge: float = 1e-8) -> Tuple[object, Optional[int], Optional[np.ndarray]]:
    """
    Adaptive lag cap from data length; manual BIC; PD enforcement on Sigma_u.
    Tries p in small range. If all fail, returns (None,None,None).
    """
    if not _HAS_SM: return None, None, None
    Y = X.T
    n = Y.shape[1]; N = Y.shape[0]
    # conservative lag cap: ensure N >> n*p
    p_cap = max(1, min(order_max, N // max(10, 3*n)))
    p_grid = list(range(1, min(8, p_cap)+1))  # try small lags first
    best_res = None; best_p = None; best_bic = np.inf
    model = VAR(Y)
    for p in p_grid:
        try:
            res = model.fit(p, trend='n')
            # ridge PD
            S = 0.5*(res.sigma_u_mle + res.sigma_u_mle.T) + ridge*np.eye(n)
            if np.any(np.linalg.eigvalsh(S) <= 0): continue
            T_eff = res.nobs
            ll = -0.5*T_eff * (n*np.log(2*np.pi) + np.log(np.linalg.det(S)) + n)
            k = n*n*p
            bic = -2*ll + k*np.log(max(1, T_eff))
            if bic < best_bic:
                best_bic = bic; best_res = res; best_p = p
        except Exception:
            continue
    if best_res is None:
        return None, None, None
    A = np.array(best_res.coefs)  # (p, n, n)
    return best_res, best_p, A


In [ ]:
res = run_directed_connectivity_routing(
    RECORDS,
    eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2'],   # or let roi_map group many sensors
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    roi_map=None,                                   # or dict like {'F':['F3','F4','Fz'], 'P':['P3','P4','Pz'], ...}
    sr_channel='EEG.F4',                                 # None → auto-pick posterior (e.g., Oz)
    bands={'theta':(4,8), 'alpha':(8,13), 'beta':(13,30)},
    f0=7.83,
    time_col='Timestamp',
    out_dir='exports_directed/S01',
    show=True
)
print(res['results'])

In [ ]:
"""
EEG–Schumann Coherence Testing — Simple Graphs & Validation
===========================================================

What it does
------------
1) Per-channel Welch coherence vs SR at Schumann harmonics with 95% shift-null.
2) Time–frequency wavelet coherence (WTC) with cluster-based permutation.
3) Sliding-window coherence time series at 7.83 Hz with a global 95% null line.

Inputs
------
RECORDS: pandas.DataFrame with a time column (default 'Timestamp') and EEG.* columns.
eeg_channels: list of EEG channel names (e.g., ['EEG.O1','EEG.O2',...]).
sr_channel: the Schumann/ELF reference column (e.g., magnetometer). If you don’t
            have one, you can use a posterior EEG (Oz/O1/O2) as a proxy.

Usage
-----
res = run_eeg_schumann_coherence(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8'],
    sr_channel='EEG.O1',                         # use your magnetometer if you have it
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_eeg_sr/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
import networkx as nx

# ---------------- small utilities ----------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def infer_fs(df: pd.DataFrame, time_col='Timestamp')->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs from time column.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Series '{name}' not in DataFrame.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]]):
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def bandpass(x, fs, f1, f2, order=4):
    ny=0.5*fs; f1=max(1e-6,min(f1,0.99*ny)); f2=max(f1+1e-6,min(f2,0.999*ny))
    b,a=signal.butter(order,[f1/ny,f2/ny],btype='band'); return signal.filtfilt(b,a,x)

# ---------------- Welch MSC @ harmonics with shift-null ----------------
def msc_harmonics_table(df, eeg_channels, sr_channel, wins, time_col='Timestamp',
                        harmonics=(7.83,14.3,20.8,27.3,33.8), nperseg_sec=4.0, n_null=200)->pd.DataFrame:
    fs = infer_fs(df, time_col)
    sr = slice_concat(get_series(df, sr_channel), fs, wins)
    nperseg = int(round(nperseg_sec*fs)); noverlap=nperseg//2
    rows=[]
    rng = np.random.default_rng(7)
    for ch in eeg_channels:
        x = slice_concat(get_series(df, ch), fs, wins)
        f, C = signal.coherence(x, sr, fs=fs, nperseg=nperseg, noverlap=noverlap)
        # null by circularly shifting SR
        null_vals = {h:[] for h in harmonics}
        for _ in range(n_null):
            s = int(rng.integers(1, len(sr)-1))
            sr_sh = np.r_[sr[-s:], sr[:-s]]
            _, C0 = signal.coherence(x, sr_sh, fs=fs, nperseg=nperseg, noverlap=noverlap)
            for h in harmonics:
                idx = int(np.argmin(np.abs(f - h)))
                null_vals[h].append(float(C0[idx]))
        for h in harmonics:
            idx = int(np.argmin(np.abs(f - h)))
            coh = float(C[idx])
            thr = float(np.nanpercentile(null_vals[h], 95)) if null_vals[h] else np.nan
            rows.append({'channel':ch, 'freq':float(f[idx]), 'MSC':coh, 'null95':thr})
    return pd.DataFrame(rows)

# ---------------- Wavelet coherence (TF) with cluster permutation ----------------
def wavelet_coherence_tf(df, x_name, y_name, time_col='Timestamp',
                         fmin=4, fmax=40, n_freq=64, w0=6.0,
                         n_perm=200, alpha=0.05, wins=None, show=True, out_png=None)->Dict[str,object]:
    fs = infer_fs(df, time_col)
    x = slice_concat(get_series(df, x_name), fs, wins)
    y = slice_concat(get_series(df, y_name), fs, wins)
    N = len(x)
    if N < 64:
        raise ValueError("Window too short for wavelet coherence.")

    # log-spaced frequencies
    freqs = np.exp(np.linspace(np.log(fmin), np.log(fmax), n_freq))

    # ---- Complex CWT via linear convolution (complex FFT) ----
    def cwt_linear(sig: np.ndarray) -> np.ndarray:
        sig = np.asarray(sig, float)
        Wx = []
        for f0 in freqs:
            # build complex Morlet in time
            dur = max(2.0, 8.0/f0)            # a few cycles
            L = int(np.ceil(dur*fs))
            if L % 2 == 0: L += 1
            tt = (np.arange(-(L//2), L//2+1))/fs
            sigma_t = w0/(2*np.pi*f0)
            mw = np.exp(-0.5*(tt/sigma_t)**2) * np.exp(1j*2*np.pi*f0*tt)
            mw -= mw.mean()
            mw /= (np.sqrt(np.sum(np.abs(mw)**2)) + 1e-24)

            # linear convolution via FFT, center-trim to length N
            n_lin = N + L - 1
            n_fft = int(2**np.ceil(np.log2(n_lin)))
            S = np.fft.fft(sig, n=n_fft)
            H = np.fft.fft(mw,  n=n_fft)
            conv = np.fft.ifft(S*H)[:n_lin]      # complex
            start = (L - 1)//2
            Wx.append(conv[start:start+N])
        return np.array(Wx)                      # (n_freq, N)

    Wx = cwt_linear(x)
    Wy = cwt_linear(y)

    # spectral smoothing along time (small Hann)
    def smooth(A: np.ndarray, wlen: int = 9) -> np.ndarray:
        if wlen <= 1: return A
        w = np.hanning(wlen); w /= w.sum()
        return np.apply_along_axis(lambda m: np.convolve(m, w, mode='same'), axis=1, arr=A)

    Sxx = np.abs(Wx)**2; Syy = np.abs(Wy)**2; Sxy = Wx * np.conj(Wy)
    Sxx_s, Syy_s, Sxy_s = smooth(Sxx), smooth(Syy), smooth(Sxy)
    WTC = (np.abs(Sxy_s)**2) / (Sxx_s * Syy_s + 1e-24)

    # ---- Circular-shift null on y ----
    rng = np.random.default_rng(11)
    null_max = []
    for _ in range(n_perm):
        sh = int(rng.integers(max(1, int(0.1*fs)), N - max(1, int(0.1*fs))))
        y_sh = np.r_[y[-sh:], y[:-sh]]
        Wy_s = cwt_linear(y_sh)
        Sxy0 = Wx * np.conj(Wy_s)
        WTC0 = (np.abs(smooth(Sxy0))**2) / (Sxx_s * (np.abs(smooth(Wy_s))**2) + 1e-24)
        null_max.append(np.nanmax(WTC0))
    thresh = float(np.nanpercentile(null_max, 95))
    sig = WTC >= thresh

    # ---- Plot (optional) ----
    if show or out_png:
        t = np.arange(N)/fs
        plt.figure(figsize=(10, 4))
        extent = [t[0], t[-1], freqs[0], freqs[-1]]
        plt.imshow(WTC, aspect='auto', origin='lower', extent=extent, cmap='magma',
                   vmin=0, vmax=np.nanmax(WTC))
        cb = plt.colorbar(); cb.set_label('Wavelet coherence')
        # highlight significant pixels (cluster visual)
        G = nx.grid_2d_graph(WTC.shape[0], WTC.shape[1])
        mask_idx = set(zip(*np.where(sig)))
        visited=set()
        for node in list(mask_idx):
            if node in visited: continue
            stack=[node]; comp=[]
            while stack:
                u=stack.pop()
                if u in visited or u not in mask_idx: continue
                visited.add(u); comp.append(u)
                for v in G.neighbors(u):
                    if v in mask_idx and v not in visited: stack.append(v)
            if comp:
                yy, xx = zip(*comp)
                plt.scatter(t[np.array(xx)], freqs[np.array(yy)], s=2, c='cyan', alpha=0.6)
        plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
        plt.title('EEG–SR Wavelet Coherence (cyan: > null 95%)')
        plt.tight_layout()
        if out_png: plt.savefig(out_png, dpi=140)
        if show: plt.show()
        plt.close()

    return {'WTC': WTC, 'freqs': freqs, 'thresh': thresh, 'sig_mask': sig}


# ---------------- Sliding coherence @ 7.83 Hz ----------------
def sliding_coherence_f0(df, eeg_channel, sr_channel, ignition_windows,
    f0, half, time_col='Timestamp',
    win_sec=8.0, step_sec=1.0,
    n_null=200, show=False,
    fast_mode=False, max_sec=None, max_windows=None):

    # --- Convert time to seconds from start ---
    tcol = df[time_col]
    if np.issubdtype(tcol.dtype, np.number):
        t_sec = tcol.values.astype(float)
        t_sec = t_sec - t_sec[0]
    else:
        t_dt = pd.to_datetime(tcol)
        t_sec = (t_dt - t_dt.iloc[0]).dt.total_seconds().values
    df['t_sec'] = t_sec


    # --- REMOVE hidden caps. Only crop if explicitly requested ---
    if fast_mode and (max_sec is None and max_windows is None):
        max_sec = 40.0 # opt-in fast behavior; otherwise no crop


    if max_sec is not None:
        df = df[df['t_sec'] <= max_sec]
    if max_windows is not None:
        # subsample evenly to at most max_windows centers later
        pass # (implement only if you *really* need it)


    # --- compute sliding coherence over the WHOLE (possibly cropped) df ---
    # estimate fs robustly
    dt = np.median(np.diff(df['t_sec'].values))
    fs = 1.0/float(dt)


    # centers from win/2 to T-win/2, step = step_sec
    T = df['t_sec'].iloc[-1]
    centers = np.arange(win_sec/2, max(0, T - win_sec/2) + 1e-9, step_sec)


    # compute coherence at f0 for each center (pseudo)
    coh_vals = []
    for c in centers:
        t0, t1 = c - win_sec/2, c + win_sec/2
        w = (df['t_sec'] >= t0) & (df['t_sec'] < t1)
        xe = df.loc[w, eeg_channel].values
        xs = df.loc[w, sr_channel].values
        if len(xe) < int(0.8*win_sec*fs) or len(xs) < int(0.8*win_sec*fs):
            coh_vals.append(np.nan); continue
        # bandpass around f0 ± half (your own filter or multitaper)
        # compute magnitude-squared coherence at f0 (your method)
        coh_vals.append(compute_coherence_at_f0(xe, xs, fs, f0, half))


    coh = np.asarray(coh_vals)
    # build null via surrogates (existing code)
    null95 = build_null_threshold(coh, n_null=n_null) # your existing routine


    return {
    't': centers,
    'coh': coh,
    'null95': null95,
    'fs': fs
    }

# ---------------- Orchestrator ----------------
def run_eeg_schumann_coherence(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    sr_channel: str,
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_eeg_sr/session',
    show: bool = True,
    harmonics: Tuple[float,...] = (7.83,14.3,20.8,27.3,33.8)
) -> Dict[str, object]:
    _ensure_dir(out_dir)
    # ---------- 1) Harmonic MSC bars with 95% null ----------
    tbl_ign = msc_harmonics_table(RECORDS, eeg_channels, sr_channel, ignition_windows,
                                  time_col=time_col, harmonics=harmonics, nperseg_sec=4.0, n_null=200)
    tbl_ign.to_csv(os.path.join(out_dir,'msc_harmonics_ign.csv'), index=False)
    if baseline_windows:
        tbl_bas = msc_harmonics_table(RECORDS, eeg_channels, sr_channel, baseline_windows,
                                      time_col=time_col, harmonics=harmonics, nperseg_sec=4.0, n_null=200)
        tbl_bas.to_csv(os.path.join(out_dir,'msc_harmonics_base.csv'), index=False)
    else:
        tbl_bas = None

    # bar plot (ignition)
    if show or True:
    # ---------- 1b) Build pivot & pick nearest-to-7.83 column ONCE ----------
        pivot = tbl_ign.pivot(index='channel', columns='freq', values='MSC')
        pthr  = tbl_ign.pivot(index='channel', columns='freq', values='null95')

        # convert column labels to float array & find nearest to target
        cols_raw = list(pivot.columns)
        try:
            cols = np.array(cols_raw, dtype=float)
        except Exception:
            cols = np.array([float(c) for c in cols_raw])

        if cols.size == 0:
            raise ValueError("No frequency columns in pivot; check input data / windows.")

        target = float(harmonics[0])
        j_near = int(np.nanargmin(np.abs(cols - target)))
        col_near_val = float(cols[j_near])  # for labelling only

        # Best channel at the nearest-to-7.83 bin (by POSITION, not label)
        ch_best = pivot.iloc[:, j_near].idxmax()

        # ---------- 1c) Bar plot (ignition) ----------
        if show or True:
            x = np.arange(len(cols))
            chans = list(pivot.index)
            w = 0.8 / max(1, len(chans))

            plt.figure(figsize=(min(10, 2.0 + 0.5*len(chans)), 3.2))
            for i, ch in enumerate(chans):
                vals = pivot.loc[ch, :].to_numpy()
                plt.bar(x + (i - (len(chans)-1)/2.0)*w, vals, width=w, label=ch)

            # overlay null95 (black dots) for each frequency column (by POSITION)
            for j in range(len(cols)):
                thr = pthr.iloc[:, j].to_numpy()
                plt.scatter(np.full_like(thr, x[j]), thr, s=12, c='k', zorder=5)

            plt.xticks(x, [f"{c:.2f}" for c in cols])
            plt.ylabel('MSC')
            plt.title('EEG–SR Coherence at Schumann Harmonics (Ignition)\n(black dots: shift-null 95%)')
            plt.legend(fontsize=8, ncol=min(4, len(chans)))
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, 'msc_harmonics_bars_ign.png'), dpi=140)
            if show: plt.show()
            plt.close()

    # ---------- 2) WTC using the nearest-to-7.83 best channel ----------
    wtc = wavelet_coherence_tf(RECORDS, ch_best, sr_channel, time_col=time_col,
                               fmin=4, fmax=40, n_freq=64, w0=6.0,
                               n_perm=200, alpha=0.05,
                               wins=ignition_windows, show=show,
                               out_png=os.path.join(out_dir, f'wtc_{ch_best}_ign.png'))

    # ---------- 3) Sliding coherence at ~7.83 Hz ----------
    sl = sliding_coherence_f0(RECORDS, ch_best, sr_channel, ignition_windows,
                              f0=harmonics[0], half=0.6, time_col=time_col,
                              win_sec=8.0, step_sec=1.0, n_null=200, show=show,
                              out_png=os.path.join(out_dir, f'sliding_{harmonics[0]:.2f}_{ch_best}_ign.png'))

    # ---------- 4) Summary ----------
    summary = {
        'best_channel': ch_best,
        'nearest_bin_to_7p83_Hz': col_near_val,
        'mean_MSC_nearest_bin_ign': float(pivot.iloc[:, j_near].mean()),
        'WTC_thresh': float(wtc['thresh']),
        'sliding_null95': float(sl['null95'])
    }
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir,'summary.csv'), index=False)
    return {'summary': summary, 'msc_ign': tbl_ign, 'msc_base': tbl_bas, 'wtc': wtc, 'sliding': sl, 'out_dir': out_dir}


In [ ]:
# 1) Load your data
# RECORDS = pd.read_csv(FILENAME)   # <- change this

# 2) Pick channels (posterior are best for clean theta/alpha)
eeg_channels = ['EEG.AF3','EEG.AF4','EEG.F3','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2']


# 3) Schumann reference
sr_channel = 'EEG.F4'

# 4) Windows (example)
ignition_windows = [(290,310),(580,600)]
baseline_windows = [(0,290),(325,580)]

# 5) Run
res = run_eeg_schumann_coherence(
    RECORDS,
    eeg_channels=eeg_channels,
    sr_channel=sr_channel,
    ignition_windows=ignition_windows,
    baseline_windows=baseline_windows,
    time_col='Timestamp',
    out_dir='exports_eeg_sr/S01',
    show=True  # save figures; prevents notebook output floods
)

print(res['summary'])


In [ ]:
"""
Harmonic Resonance & Spectral Mode Analysis — Simple Graphs & Validation
=======================================================================

Tests
-----
1) Spectral harmonicity:
   • For each channel, compute Welch PSD and a local SNR z-score at Schumann
     harmonics (7.83, 14.3, 20.8, 27.3, 33.8 Hz), using a robust baseline taken
     from sidebands around each target (excluding the central peak).
   • Count channels with z >= z_thr (default 2.0) at each harmonic.
   • Split the data in M epochs → repeat the test → consistency metric
     (fraction of epochs with a “hit” per channel) and a simple binomial test.

2) Spatial mode at 7–8 Hz:
   • Band-pass 7.23–8.43 Hz across channels, compute spatial covariance,
     PC1 variance ratio (how “global” the 8 Hz mode is), and “in-phase score”
     (alignment of PC1 weights with all-ones vector).

3) Off-harmonic controls:
   • Repeat spectral SNR test in an off band (16–18 Hz) → expect fewer hits.

Outputs
-------
• PNGs: mean PSD with harmonic lines, per-channel harmonic z heatmap,
        per-harmonic bar of hit counts, PC1 weights bar, PC1 variance ratio plot.
• CSV: summary with per-channel metrics and overall harmonicity indices.

Usage
-----
res = run_harmonic_resonance_spectral_modes(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6'],
    time_col='Timestamp',
    ignition_windows=[(290,310),(580,600)],     # or None for full session
    out_dir='exports_harmonics_simple/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
from scipy.stats import binom_test

# ---------- I/O / time helpers ----------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')
)->Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    # first numeric, roughly monotonic
    for c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(df)):
            arr = s.values.astype(float)
            dt = np.diff(arr[np.isfinite(arr)])
            if dt.size and np.nanmedian(dt) > 0: return c
    # datetime?
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors='raise')
            return c
        except Exception:
            pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None,
                            default_fs: float = 128.0, out_name: str = 'Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs
        return out_name
    s = df[col]
    # datetime → seconds since first
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name] = tsec.values; return out_name
    # numeric
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(df)):
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name] = sn.values
    return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs from time column.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Series '{name}' not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

# ---------- DSP helpers ----------
def bandpass(x, fs, f1, f2, order=4):
    ny=0.5*fs; f1=max(1e-6,min(f1,0.99*ny)); f2=max(f1+1e-6,min(f2,0.999*ny))
    b,a=signal.butter(order,[f1/ny,f2/ny],btype='band'); return signal.filtfilt(b,a,x)

def welch_psd(x: np.ndarray, fs: float, nperseg_sec: float = 4.0) -> Tuple[np.ndarray,np.ndarray]:
    nperseg = int(round(nperseg_sec*fs))
    f, p = signal.welch(x, fs=fs, nperseg=nperseg, noverlap=nperseg//2, nfft=None)
    return f, p

# ---------- Spectral harmonic z-score ----------
def harmonic_zscores(f: np.ndarray, p: np.ndarray,
                     harmonics=(7.83,14.3,20.8,27.3,33.8),
                     half_bw: float = 0.6,
                     side_bw: float = 2.0) -> Dict[str, float]:
    """
    For each target harmonic h, compute z = (P(h) - median(side)) / MAD(side),
    where side = [h-side_bw, h+side_bw] \ [h-half_bw, h+half_bw] (exclude central).
    Returns dict { '7.83': z, ... }  (NaN if insufficient bins).
    """
    zmap={}
    for h in harmonics:
        mask_side = (f>=h-side_bw) & (f<=h+side_bw)
        mask_excl = (f>=h-half_bw) & (f<=h+half_bw)
        side = p[mask_side & ~mask_excl]
        if side.size < 10:
            zmap[f"{h:.2f}"] = np.nan; continue
        med = np.median(side); mad = np.median(np.abs(side - med)) + 1e-12
        # nearest-bin pick for center power
        idx = int(np.nanargmin(np.abs(f - h)))
        z = (p[idx] - med) / (1.4826*mad)  # robust z via MAD
        zmap[f"{h:.2f}"] = float(z)
    return zmap

# ---------- Spatial mode (7–8 Hz) ----------
def spatial_mode_8hz(X: np.ndarray, fs: float, f0=7.83, half=0.6) -> Dict[str, object]:
    """
    X: (n_ch, T) — band-pass 7.83±half, compute covariance → PC1 variance ratio,
    and in-phase score (|corr(PC1, all-ones)|).
    """
    Xb = np.vstack([bandpass(x, fs, f0-half, f0+half) for x in X])
    # covariance
    C = Xb @ Xb.T / Xb.shape[1]
    vals, vecs = np.linalg.eigh(C)
    idx = np.argsort(vals)[::-1]
    vals, vecs = vals[idx], vecs[:, idx]
    var_ratio = float(vals[0] / (np.sum(vals)+1e-12))
    w = vecs[:, 0]
    w = w / (np.linalg.norm(w)+1e-12)
    inphase = float(np.abs(np.dot(w, np.ones_like(w))/ (np.linalg.norm(w)*np.sqrt(len(w))+1e-12)))
    return {'pc1_var_ratio': var_ratio, 'pc1_weights': w, 'inphase_score': inphase}

# ---------- Main runner ----------
def run_harmonic_resonance_spectral_modes(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    time_col: str = 'Timestamp',
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    nperseg_sec: float = 4.0,
    harmonics: Tuple[float,...] = (7.83,14.3,20.8,27.3,33.8),
    half_bw: float = 0.6,
    side_bw: float = 2.0,
    z_thr: float = 2.0,
    offband: Tuple[float,float] = (16.0,18.0),
    n_epochs: int = 4,
    out_dir: str = 'exports_harmonics_simple/session',
    show: bool = True
) -> Dict[str, object]:
    """
    High-resolution spectral harmonic test + spatial mode at 7–8 Hz.
    """
    _ensure_dir(out_dir)
    # normalize time column
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)
    # build signals matrix
    X=[]
    kept=[]
    for ch in eeg_channels:
        nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
        if nm in RECORDS.columns:
            x = get_series(RECORDS, nm)
            if ignition_windows: x = slice_concat(x, fs, ignition_windows)
            X.append(x); kept.append(nm)
    if not X: raise ValueError("No EEG channels found from eeg_channels.")
    # truncate to common length
    L = min(len(x) for x in X)
    X = np.vstack([x[:L] for x in X])   # (n_ch, T)
    n_ch, T = X.shape

    # ---- (1) Welch PSD per channel & harmonic z-scores ----
    psd = []
    ztbl_rows=[]
    for i,ch in enumerate(kept):
        f, p = welch_psd(X[i], fs, nperseg_sec=nperseg_sec)
        psd.append((f,p))
        zmap = harmonic_zscores(f, p, harmonics=harmonics, half_bw=half_bw, side_bw=side_bw)
        rec = {'channel': ch}; rec.update(zmap)
        ztbl_rows.append(rec)
    ztbl = pd.DataFrame(ztbl_rows)
    zcols = [f"{h:.2f}" for h in harmonics]

    # hit counts per harmonic
    hits = {c: int(np.sum(ztbl[c] >= z_thr)) for c in zcols if c in ztbl.columns}

    # ---- (2) Epoch consistency ----
    M = max(1, n_epochs)
    step = T//M
    ep_hits = {c: [] for c in zcols}
    for e in range(M):
        s = e*step; eidx = (e+1)*step if e<M-1 else T
        for i,ch in enumerate(kept):
            f, p = welch_psd(X[i, s:eidx], fs, nperseg_sec=nperseg_sec)
            zmap = harmonic_zscores(f,p,harmonics=harmonics,half_bw=half_bw,side_bw=side_bw)
            for c in zcols:
                ep_hits[c].append(float(zmap.get(c, np.nan)))
    # fraction of epochs with hit (per channel pooled)
    ep_consistency = {c: float(np.nanmean(np.array(ep_hits[c]) >= z_thr)) for c in zcols}

    # simple binomial test for “>=1 hit” across channels at the fundamental (p0~0.05)
    p0 = 0.05
    k = hits.get(zcols[0], 0); n = n_ch
    p_binom = float(binom_test(k, n, p0, alternative='greater'))

    # ---- (3) Spatial mode @ ~7.83 Hz ----
    mode = spatial_mode_8hz(X, fs, f0=harmonics[0], half=half_bw)

    # ---- (4) Off-harmonic control (16–18 Hz) ----
    off_hits=[]
    for i,ch in enumerate(kept):
        f, p = welch_psd(X[i], fs, nperseg_sec=nperseg_sec)
        sel = (f>=offband[0]) & (f<=offband[1])
        off_hits.append(float(np.max(p[sel]) if np.any(sel) else np.nan))
    off_mean = float(np.nanmean(off_hits))

    # ---- Plots ----
    # Mean PSD (linear freq) with harmonic lines
    plt.figure(figsize=(8,3.2))
    # plot mean PSD across channels
    f0, p0 = psd[0]
    Pstack = np.vstack([p for (f,p) in psd if len(p)==len(p0)])
    Pmean = np.nanmean(Pstack, axis=0)
    plt.plot(f0, Pmean, lw=1.6, label='Mean PSD')
    for h in harmonics:
        plt.axvline(h, color='tab:red', lw=1.0, alpha=0.7)
    plt.xlim(2, 40); plt.xlabel('Frequency (Hz)'); plt.ylabel('Power'); plt.title('Mean PSD with Schumann lines')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir,'mean_psd.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # Heatmap: per-channel harmonic z-scores
    plt.figure(figsize=(max(6, 0.4*len(kept)), 3.0))
    Z = ztbl[zcols].to_numpy(dtype=float)
    im = plt.imshow(Z, aspect='auto', origin='lower', cmap='magma', vmin=np.nanmin(Z), vmax=np.nanmax(Z))
    plt.colorbar(label='z-score vs local sidebands')
    plt.yticks(range(len(kept)), [k.split('.',1)[-1] for k in kept], fontsize=8)
    plt.xticks(range(len(zcols)), zcols)
    plt.title('Per-channel harmonic z-scores'); plt.tight_layout()
    plt.savefig(os.path.join(out_dir,'harmonic_z_heatmap.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # Bar: hit counts per harmonic (z >= z_thr)
    plt.figure(figsize=(6,3))
    xs = np.arange(len(zcols)); vals = [hits.get(c,0) for c in zcols]
    plt.bar(xs, vals, color='tab:blue', alpha=0.9)
    plt.xticks(xs, zcols); plt.ylabel(f'Channels with z≥{z_thr}')
    plt.title('Harmonic hit counts across channels')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir,'hit_counts.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # Spatial mode @ 7–8 Hz: PC1 weights bar
    plt.figure(figsize=(max(6, 0.4*len(kept)), 3.0))
    plt.bar(np.arange(len(kept)), mode['pc1_weights'], color='tab:green', alpha=0.9)
    plt.xticks(range(len(kept)), [k.split('.',1)[-1] for k in kept], rotation=0, fontsize=8)
    plt.ylabel('PC1 weight'); plt.title(f"PC1 variance ratio={mode['pc1_var_ratio']:.2f}, in-phase={mode['inphase_score']:.2f}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir,'pc1_weights.png'), dpi=140)
    if show: plt.show()
    plt.close()

    # ---- CSV summary ----
    # per-channel: z at each harmonic, plus channel-level harmonicity index (sum of z+ over harmonics)
    ztbl['HarmonicityIndex'] = np.nansum(np.clip(ztbl[zcols].to_numpy(), 0, None), axis=1)
    summary = {
        'n_channels': n_ch,
        'fund_hits': hits.get(zcols[0], 0),
        'fund_ep_consistency': ep_consistency.get(zcols[0], np.nan),
        'fund_hits_p_binom': p_binom,
        'pc1_var_ratio_8Hz': mode['pc1_var_ratio'],
        'inphase_score_8Hz': mode['inphase_score'],
        'offband_mean_power_16_18Hz': off_mean
    }
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir,'summary.csv'), index=False)
    ztbl.to_csv(os.path.join(out_dir,'per_channel_z.csv'), index=False)

    return {'summary': summary, 'per_channel': ztbl, 'out_dir': out_dir}


In [ ]:
# Example: posterior-leaning set (clean alpha/theta)
# eeg_channels = [c for c in ['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6'] if c in RECORDS.columns]

eeg_channels=['AF3','AF4','F3','F4','F7','F8','FC5','FC6','P7','P8','O1','O2','T7','T8']
# eeg_channel=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6']
res = run_harmonic_resonance_spectral_modes(
    RECORDS,
    eeg_channels=eeg_channels,
    time_col='Timestamp',
    ignition_windows=[(290,310),(580,600)],   # or None for full record
    out_dir='exports_harmonics_simple/S01',
    show=True
)
print(res['summary'])


In [ ]:
"""
Topological Data Analysis of Attractor Geometry — Simple Graphs & Validation
============================================================================

What it does
------------
1) Build a 1D robust EEG drive (mean or PCA1 over given channels).
2) Auto-pick Takens parameters (τ from ACF 1/e; m from False Nearest Neighbors).
3) Delay-embed → point cloud X in R^m (subsampled for speed).
4) Persistent homology (Ripser if available) → H0/H1/H2 diagrams:
     • b1_count_sig (number of significant loops)
     • b2_count_sig (number of significant voids)
     • max persistence in H1/H2 and p-values vs surrogates
   (Torus heuristic: b1>=2 and b2>=1 with significant persistence)
5) Fallback (if ripser missing): Recurrence Plot + RQA (RR, DET) with surrogates.
6) Saves PNGs + CSV summary to out_dir; minimal inline output if show=False.

Usage (example)
---------------
res = run_tda_attractor_topology(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8'],
    ignition_windows=[(290,310),(580,600)],     # or None for full
    time_col='Timestamp',
    out_dir='exports_tda/S01',
    show=False                                   # save figures; avoids notebook flooding
)
print(res['summary'])
"""

from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal

# Optional: Ripser / persim for persistent homology
try:
    from ripser import ripser
    from persim import plot_diagrams
    _HAS_RIPSER = True
except Exception:
    _HAS_RIPSER = False

# ---------------- I/O & time helpers ----------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')
)->Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    for c in df.columns:  # first numeric, roughly monotonic
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(df)):
            arr = s.values.astype(float); dt = np.diff(arr[np.isfinite(arr)])
            if dt.size and np.nanmedian(dt) > 0: return c
    for c in df.columns:  # datetime?
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None,
                            default_fs: float = 128.0, out_name: str = 'Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs
        return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name] = tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(df)):
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name] = sn.values; return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs from time column.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Series '{name}' not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x = np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

# ---------------- Delay-embedding tools ----------------
try:
    from sklearn.neighbors import KDTree
    _HAS_SK = True
except Exception:
    _HAS_SK = False

def estimate_delay_tau(x: np.ndarray, fs: float, max_lag_sec: float = 2.0, method='acf-1e')->int:
    nlag = int(max(1, round(max_lag_sec*fs)))
    xx = zscore(x)
    acf = signal.correlate(xx, xx, mode='full'); acf = acf[acf.size//2:acf.size//2+nlag+1]
    acf = acf/(acf[0]+1e-12)
    if method=='zero':
        idx = np.where(np.sign(acf[1:])!=np.sign(acf[:-1]))[0]
        tau = int(idx[0]+1) if idx.size else max(1,int(0.05*fs))
    else:
        idx = np.where(acf <= 1/np.e)[0]; tau = int(idx[0]) if idx.size else max(1,int(0.05*fs))
    return max(1, tau)

def takens_embedding(x: np.ndarray, m: int, tau: int)->np.ndarray:
    N = len(x) - (m-1)*tau
    if N <= 10: raise ValueError("Time series too short for requested embedding.")
    return np.column_stack([x[i:i+N] for i in range(0, m*tau, tau)]).astype(float)

def false_nearest_neighbors(x: np.ndarray, tau: int, m_list: List[int], theiler: int = 10)->pd.DataFrame:
    rows=[]
    for m in m_list:
        try:
            X_m = takens_embedding(x, m, tau); X_m1 = takens_embedding(x, m+1, tau)
            N = min(len(X_m), len(X_m1)); X_m = X_m[:N]; X_m1 = X_m1[:N]
        except Exception:
            rows.append({'m':m,'FNN%':np.nan}); continue
        if _HAS_SK:
            tree = KDTree(X_m); d, idxs = tree.query(X_m, k=2); nn = idxs[:,1]
            for i in range(N):
                if abs(nn[i]-i) <= theiler:
                    dists, idx2 = tree.query(X_m[i:i+1], k=10)
                    for cand in idx2[0,1:]:
                        if abs(cand-i) > theiler: nn[i]=cand; break
        else:
            nn = np.zeros(N, dtype=int)
            for i in range(N):
                d = np.linalg.norm(X_m - X_m[i], axis=1); d[i]=np.inf
                order = np.argsort(d); j=order[0]
                if abs(j-i) <= theiler:
                    for cand in order[1:]:
                        if abs(cand-i) > theiler: j=cand; break
                nn[i]=j
        Rtol=15.0
        dist_m  = np.linalg.norm(X_m - X_m[nn],  axis=1)
        dist_m1 = np.linalg.norm(X_m1- X_m1[nn], axis=1)
        ratio = dist_m1/(dist_m+1e-12)
        rows.append({'m':m, 'FNN%': float(np.mean(ratio>Rtol)*100.0)})
    return pd.DataFrame(rows)

# ---------------- Surrogates ----------------
def phase_randomize(x: np.ndarray)->np.ndarray:
    X = np.fft.rfft(x); mag = np.abs(X); ph = np.angle(X)
    rnd = np.random.uniform(-np.pi, np.pi, size=mag.size)
    rnd[0] = ph[0]
    if mag.size % 2 == 0: rnd[-1] = ph[-1]
    Xs = mag * np.exp(1j*rnd)
    return np.fft.irfft(Xs, n=len(x)).astype(float)

# ---------------- RQA fallback ----------------
def recurrence_plot(X: np.ndarray, eps_quant: float = 0.1)->Dict[str,object]:
    N = len(X); idx = np.random.choice(N, size=min(N, 1200), replace=False)
    Y = X[idx]; D = np.sqrt(((Y[:,None,:]-Y[None,:,:])**2).sum(axis=2))
    eps = np.quantile(D, eps_quant); R = (D <= eps).astype(int); np.fill_diagonal(R, 0)
    RR = float(np.mean(R))
    # simple DET
    det_lines=0; total_lines=0
    for i in range(R.shape[0]-1):
        run=0
        for j in range(R.shape[1]-1):
            if R[i,j]==1 and R[i+1,j+1]==1: run+=1
            else:
                if run>=1: total_lines+=1; 
                if run+1>=2: det_lines+=1
                run=0
        if run>=1: total_lines+=1; 
        if run+1>=2: det_lines+=1
    DET = det_lines/(total_lines+1e-12)
    return {'R':R, 'RR':RR, 'DET':float(DET)}

# ---------------- Main TDA runner ----------------
def run_tda_attractor_topology(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_tda/session',
    m_list: List[int] = [2,3,4,5,6,7,8],
    max_points: int = 5000,
    n_surrogates: int = 100,
    show: bool = False
)->Dict[str, object]:
    """
    Takens embedding + persistent homology (with surrogates) to test torus-like topology.
    """
    _ensure_dir(out_dir)
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)

    # robust drive: mean across given channels (z-scored)
    Xsig=[]
    for ch in eeg_channels:
        nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
        if nm in RECORDS.columns:
            x = get_series(RECORDS, nm)
            if ignition_windows: x = slice_concat(x, fs, ignition_windows)
            Xsig.append(np.asarray(x,float))
    if not Xsig: raise ValueError("No EEG channels found in RECORDS for eeg_channels.")
    L = min(len(x) for x in Xsig)
    x = zscore(np.mean(np.vstack([xx[:L] for xx in Xsig]), axis=0))

    # Takens params
    tau = estimate_delay_tau(x, fs, max_lag_sec=2.0, method='acf-1e')
    fnn = false_nearest_neighbors(x, tau, m_list)
    fnn = fnn.dropna()
    if np.any(fnn['FNN%'] <= 5.0):
        m_star = int(fnn.loc[fnn['FNN%']<=5.0,'m'].iloc[0])
    else:
        m_star = int(fnn.iloc[np.argmin(fnn['FNN%'])]['m'])
    m_star = max(3, m_star)

    # Embed
    X = takens_embedding(x, m_star, tau)
    # Subsample for speed
    if len(X) > max_points:
        idx = np.linspace(0, len(X)-1, max_points).astype(int)
        X = X[idx]

    summary = {'tau':tau, 'm':m_star}
    outputs = {}

    # ---------------- Persistent homology path ----------------
    if _HAS_RIPSER:
        rp = ripser(X, maxdim=2)     # Vietoris–Rips on point cloud
        dgms = rp['dgms']            # [H0, H1, H2]
        # persistence = death - birth; ignore inf deaths for max
        def max_persistence(dgm):
            if dgm.size == 0: return 0.0
            pers = dgm[:,1] - dgm[:,0]
            pers = pers[np.isfinite(pers)]
            return float(np.nanmax(pers)) if pers.size else 0.0

        maxH1 = max_persistence(dgms[1]); maxH2 = max_persistence(dgms[2])

        # Surrogate nulls (phase-randomize the drive, same tau/m)
        nullH1=[]; nullH2=[]
        for _ in range(n_surrogates):
            xs = zscore(phase_randomize(x))
            Xs = takens_embedding(xs, m_star, tau)
            if len(Xs) > max_points:
                idx = np.linspace(0, len(Xs)-1, max_points).astype(int)
                Xs = Xs[idx]
            rps = ripser(Xs, maxdim=2)
            d1, d2 = rps['dgms'][1], rps['dgms'][2]
            # max persistence per homology class
            def maxP(dgm):
                if dgm.size==0: return 0.0
                per = dgm[:,1]-dgm[:,0]; per = per[np.isfinite(per)]
                return float(np.nanmax(per)) if per.size else 0.0
            nullH1.append(maxP(d1)); nullH2.append(maxP(d2))
        thrH1 = float(np.nanpercentile(nullH1, 95)) if nullH1 else np.nan
        thrH2 = float(np.nanpercentile(nullH2, 95)) if nullH2 else np.nan

        # Simple counts of “significant” features (bars above threshold)
        def count_sig(dgm, thr):
            if dgm.size==0 or not np.isfinite(thr): return 0
            per = dgm[:,1]-dgm[:,0]; per = per[np.isfinite(per)]
            return int(np.sum(per >= thr))
        b1_sig = count_sig(dgms[1], thrH1)
        b2_sig = count_sig(dgms[2], thrH2)

        # Torus heuristic (T^2): H1: >=2, H2: >=1
        summary.update({
            'max_persistence_H1': maxH1, 'null95_H1': thrH1, 'b1_count_sig': b1_sig,
            'max_persistence_H2': maxH2, 'null95_H2': thrH2, 'b2_count_sig': b2_sig,
            'torus_heuristic_pass': bool((b1_sig>=2) and (b2_sig>=1))
        })

        # Plots
        # 2D/3D projections of embedding (pairwise)
        stride = max(1, len(X)//6000)
        plt.figure(figsize=(8,3))
        plt.subplot(1,2,1); plt.plot(X[::stride,0], X[::stride,1], lw=0.5, alpha=0.8)
        plt.title(f'Embedding X1–X2 (m={m_star}, τ={tau})')
        if X.shape[1]>=3:
            plt.subplot(1,2,2); plt.plot(X[::stride,1], X[::stride,2], lw=0.5, alpha=0.8)
            plt.title('Embedding X2–X3')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir,'embedding_pairwise.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # Persistence diagrams
        plt.figure(figsize=(6,3))
        plot_diagrams(dgms, show=False)
        plt.title('Persistence diagrams (H0,H1,H2)')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir,'persistence_diagrams.png'), dpi=140)
        if show: plt.show()
        plt.close()

        outputs['dgms'] = dgms

    # ---------------- Fallback: Recurrence / RQA when Ripser unavailable ----------------
    else:
        # RP & RQA for real data
        rqa = recurrence_plot(X, eps_quant=0.1)
        # nulls
        null_DET=[]
        for _ in range(n_surrogates):
            xs = zscore(phase_randomize(x))
            Xs = takens_embedding(xs, m_star, tau)
            rq = recurrence_plot(Xs, eps_quant=0.1)
            null_DET.append(rq['DET'])
        thrDET = float(np.nanpercentile(null_DET, 95)) if null_DET else np.nan
        summary.update({'DET': float(rqa['DET']), 'DET_null95': thrDET,
                        'loopiness_pass': bool(np.isfinite(thrDET) and (rqa['DET']>thrDET))})
        # Save RP image
        plt.figure(figsize=(4,4))
        plt.imshow(rqa['R'], origin='lower', cmap='binary'); plt.title(f'RP (DET={rqa["DET"]:.2f})')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir,'recurrence_plot.png'), dpi=140)
        if show: plt.show()
        plt.close()

        outputs['rqa'] = rqa

    # Save summary
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir,'summary.csv'), index=False)
    return {'summary': summary, 'out_dir': out_dir, 'params': {'tau':tau, 'm':m_star}, 'outputs': outputs}


In [ ]:
# Pick clean posterior channels for a stable attractor (low artifacts)
eeg_channels=['AF3','AF4','F3','F4','F7','F8','FC5','FC6','P7','P8','O1','O2']#,'T7','T8']

res = run_tda_attractor_topology(
    RECORDS,
    eeg_channels=eeg_channels,
    ignition_windows=[(290,310),(580,600)],   # or None for the full session
    time_col='Timestamp',
    out_dir='exports_tda/S01',
    show=True,            # save figures; avoids notebook IOPub limits
    n_surrogates=100
)
print(res['summary'])


In [ ]:
"""
Recurrence Quantification & Chaos Metrics — Simple Graphs & Validation
=====================================================================

This module validates nonlinear EEG dynamics with:
  • Recurrence Plot (RP) & RQA: RR, DET, LAM, Lmax, Lmean, Diag-Entropy, Trapping Time
  • Largest Lyapunov exponent λ_max (Rosenstein)
  • Correlation dimension D2 (Grassberger–Procaccia)
  • Phase-randomized surrogates → 95% nulls & one-sided p-values

Outputs (per state): PNGs + summary.csv in out_dir.

Usage
-----
res = run_rqa_chaos_metrics(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8'],   # clean posterior set recommended
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_rqa/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal

# -------------------- small I/O helpers --------------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')
)->Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    for c in df.columns:  # first numeric & roughly monotonic
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(df)):
            arr = s.values.astype(float); dt = np.diff(arr[np.isfinite(arr)])
            if dt.size and np.nanmedian(dt) > 0: return c
    for c in df.columns:  # datetime
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None,
                            default_fs: float = 128.0, out_name: str = 'Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs
        return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name] = tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(df)):
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name] = sn.values
    return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs from time column.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Series '{name}' not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x=np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

# -------------------- embedding tools --------------------
try:
    from sklearn.neighbors import KDTree
    _HAS_SK = True
except Exception:
    _HAS_SK = False

def estimate_delay_tau(x: np.ndarray, fs: float, max_lag_sec: float = 2.0, method='acf-1e')->int:
    nlag = int(max(1, round(max_lag_sec*fs)))
    xx = zscore(x)
    acf = signal.correlate(xx, xx, mode='full'); acf = acf[acf.size//2:acf.size//2+nlag+1]
    acf = acf/(acf[0]+1e-12)
    if method=='zero':
        idx = np.where(np.sign(acf[1:])!=np.sign(acf[:-1]))[0]; tau = int(idx[0]+1) if idx.size else max(1,int(0.05*fs))
    else:
        idx = np.where(acf <= 1/np.e)[0]; tau = int(idx[0]) if idx.size else max(1,int(0.05*fs))
    return max(1, tau)

def takens_embedding(x: np.ndarray, m: int, tau: int)->np.ndarray:
    N = len(x) - (m-1)*tau
    if N <= 10: raise ValueError("Time series too short for requested embedding.")
    return np.column_stack([x[i:i+N] for i in range(0, m*tau, tau)]).astype(float)

def false_nearest_neighbors(x: np.ndarray, tau: int, m_list: List[int], theiler: int = 10)->pd.DataFrame:
    rows=[]
    for m in m_list:
        try:
            X_m = takens_embedding(x, m, tau); X_m1 = takens_embedding(x, m+1, tau)
            N = min(len(X_m), len(X_m1)); X_m = X_m[:N]; X_m1 = X_m1[:N]
        except Exception:
            rows.append({'m':m,'FNN%':np.nan}); continue
        if _HAS_SK:
            tree = KDTree(X_m); d, idxs = tree.query(X_m, k=2); nn = idxs[:,1]
            # Theiler correction
            for i in range(N):
                if abs(nn[i]-i) <= theiler:
                    d2, idx2 = tree.query(X_m[i:i+1], k=10)
                    for cand in idx2[0,1:]:
                        if abs(cand-i) > theiler: nn[i]=cand; break
        else:
            nn = np.zeros(N, dtype=int)
            for i in range(N):
                d = np.linalg.norm(X_m - X_m[i], axis=1); d[i]=np.inf
                order = np.argsort(d); j=order[0]
                if abs(j-i) <= theiler:
                    for cand in order[1:]:
                        if abs(cand-i) > theiler: j=cand; break
                nn[i]=j
        Rtol=15.0
        dist_m  = np.linalg.norm(X_m - X_m[nn],  axis=1)
        dist_m1 = np.linalg.norm(X_m1- X_m1[nn], axis=1)
        ratio = dist_m1/(dist_m+1e-12)
        rows.append({'m':m, 'FNN%': float(np.mean(ratio>Rtol)*100.0)})
    return pd.DataFrame(rows)

# -------------------- Lyapunov (Rosenstein, robust) --------------------
def lyapunov_rosenstein(x: np.ndarray,
                        m: int, tau: int, fs: float,
                        theiler: int = 10,
                        t_fit: Tuple[int,int] = (1, 30)) -> Dict[str, object]:
    X = takens_embedding(zscore(x), m, tau)
    N = len(X)
    if N <= theiler + 2:
        return {'lambda': np.nan, 'L': np.array([]), 'k': np.array([])}
    # nearest neighbor with Theiler exclusion
    if _HAS_SK:
        tree = KDTree(X); d, idxs = tree.query(X, k=min(20, N-1))
        nn = np.zeros(N, dtype=int)
        for i in range(N):
            chosen=None
            for cand in idxs[i,1:]:
                if abs(int(cand)-i) > theiler: chosen=int(cand); break
            nn[i] = chosen if chosen is not None else int(idxs[i,1])
    else:
        nn = np.zeros(N, dtype=int)
        for i in range(N):
            d = np.linalg.norm(X - X[i], axis=1); d[i]=np.inf
            order = np.argsort(d); chosen=None
            for cand in order:
                if abs(int(cand)-i) > theiler: chosen=int(cand); break
            nn[i] = chosen if chosen is not None else int(order[0])

    max_k = max(2, min(100, N-1))
    Lvals=[]; ks=[]
    for k in range(1, max_k):
        valid = (np.arange(N)+k < N) & (nn + k < N)
        if not np.any(valid): break
        idx = np.where(valid)[0]
        if idx.size < 5: break
        d0 = np.linalg.norm(X[idx]     - X[nn[idx]],     axis=1) + 1e-24
        dk = np.linalg.norm(X[idx + k] - X[nn[idx] + k], axis=1)
        Lvals.append(np.mean(np.log(dk/d0))); ks.append(k)
    Lvals = np.asarray(Lvals,float); ks = np.asarray(ks,int)
    if Lvals.size < 5: return {'lambda': np.nan, 'L': Lvals, 'k': ks}
    k0,k1 = t_fit; k1=min(k1,int(0.6*len(Lvals)))
    if k1 <= k0+1: return {'lambda': np.nan, 'L':Lvals, 'k':ks}
    A = np.vstack([ks[k0:k1], np.ones(k1-k0)]).T
    slope, _ = np.linalg.lstsq(A, Lvals[k0:k1], rcond=None)[0]
    lam = float(slope) * fs / float(tau)
    return {'lambda': lam, 'L': Lvals, 'k': ks}

# -------------------- Correlation dimension (GP) --------------------
def correlation_dimension_gp(X: np.ndarray,
                             r_min_quant: float = 0.05,
                             r_max_quant: float = 0.30,
                             n_r: int = 20) -> Dict[str, object]:
    N = len(X)
    idx = np.random.choice(N, size=min(N, 1200), replace=False)
    D = np.sqrt(((X[idx,None,:]-X[None,idx,:])**2).sum(axis=2)).ravel()
    D = D[D>0]; Dsorted = np.sort(D)
    rmin = Dsorted[int(r_min_quant*len(Dsorted))]
    rmax = Dsorted[int(r_max_quant*len(Dsorted))]
    r_vals = np.exp(np.linspace(np.log(rmin+1e-12), np.log(rmax+1e-12), n_r))
    C = np.array([np.mean(D < r) for r in r_vals])
    x = np.log(r_vals + 1e-24); y = np.log(C + 1e-24)
    A = np.vstack([x, np.ones_like(x)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    return {'r': r_vals, 'C': C, 'D2': float(slope), 'fit': (float(slope), float(intercept))}

# -------------------- Recurrence plot & RQA --------------------
def recurrence_matrix(X: np.ndarray, eps: float, theiler: int = 0)->np.ndarray:
    D = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(axis=2))
    R = (D <= eps).astype(int)
    # suppress main diagonal band (Theiler neighborhood)
    for i in range(-theiler, theiler+1):
        if i==0: np.fill_diagonal(R, 0)
        else:
            diag = np.diag_indices_from(R)
            ii = (diag[0][max(0,i):], diag[1][max(0,-i):]) if i>=0 else (diag[0][:i], diag[1][-i:])
            R[ii] = 0
    return R

def rqa_metrics(R: np.ndarray, lmin: int = 2, vmin: int = 2)->Dict[str, float]:
    N = R.shape[0]
    RR = float(np.sum(R)/(N*N))
    # Diagonal lines
    Ls=[]
    for k in range(-(N-1), N):
        diag = np.diag(R, k=k)
        # run-lengths of ones
        run=0
        for v in diag:
            if v==1: run+=1
            elif run>0:
                if run>=lmin: Ls.append(run)
                run=0
        if run>=lmin: Ls.append(run)
    Ls = np.array(Ls, int)
    DET = float(np.sum(Ls)/ (np.sum(R) + 1e-12))
    Lmax = float(np.max(Ls) if Ls.size else 0)
    Lmean = float(np.mean(Ls) if Ls.size else 0)
    # diagonal length entropy
    if Ls.size:
        bins = np.arange(lmin, np.max(Ls)+1)
        hist, _ = np.histogram(Ls, bins=bins)
        p = hist.astype(float)/ (np.sum(hist)+1e-12)
        p = p[p>0]; Hdiag = float(-np.sum(p*np.log(p)))
    else:
        Hdiag = 0.0
    # Vertical lines → laminarity & trapping time
    Vs=[]
    for col in range(N):
        run=0
        for row in range(N):
            if R[row,col]==1: run+=1
            elif run>0:
                if run>=vmin: Vs.append(run)
                run=0
        if run>=vmin: Vs.append(run)
    Vs = np.array(Vs,int)
    LAM = float(np.sum(Vs)/ (np.sum(R) + 1e-12))
    TT  = float(np.mean(Vs) if Vs.size else 0)
    return {'RR':RR, 'DET':DET, 'LAM':LAM, 'Lmax':Lmax, 'Lmean':Lmean, 'Hdiag':Hdiag, 'TT':TT}

# -------------------- Surrogates --------------------
def phase_randomize(x: np.ndarray)->np.ndarray:
    X = np.fft.rfft(x); mag = np.abs(X); ph = np.angle(X)
    rnd = np.random.uniform(-np.pi, np.pi, size=mag.size)
    rnd[0] = ph[0]
    if mag.size % 2 == 0: rnd[-1] = ph[-1]
    Xs = mag * np.exp(1j*rnd)
    return np.fft.irfft(Xs, n=len(x)).astype(float)

# -------------------- Orchestrator --------------------
def run_rqa_chaos_metrics(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_rqa/session',
    show: bool = False,
    m_list: List[int] = [2,3,4,5,6,7,8],
    eps_quantile: float = 0.10,
    theiler: int = 0,
    lmin: int = 2, vmin: int = 2,
    max_points: int = 5000,
    n_surrogates: int = 100
)->Dict[str, object]:
    """
    RQA + Chaos metrics with surrogate validation for ignition & baseline windows.
    """
    _ensure_dir(out_dir)
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)

    # robust 1D drive: mean across provided channels
    def build_drive(wins):
        sigs=[]
        for ch in eeg_channels:
            nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
            if nm in RECORDS.columns:
                x = get_series(RECORDS, nm)
                if wins: x = slice_concat(x, fs, wins)
                sigs.append(np.asarray(x,float))
        if not sigs: raise ValueError("No EEG channels found.")
        L = min(map(len, sigs))
        return zscore(np.mean(np.vstack([s[:L] for s in sigs]), axis=0))

    states = {'ignition': ignition_windows, 'baseline': baseline_windows}
    summaries=[]; outputs={}
    for st, wins in states.items():
        if wins is None: continue
        x = build_drive(wins)
        # Takens params
        tau = estimate_delay_tau(x, fs, max_lag_sec=2.0, method='acf-1e')
        fnn = false_nearest_neighbors(x, tau, m_list)
        fnn = fnn.dropna()
        if np.any(fnn['FNN%'] <= 5.0):
            m_star = int(fnn.loc[fnn['FNN%']<=5.0,'m'].iloc[0])
        else:
            m_star = int(fnn.iloc[np.argmin(fnn['FNN%'])]['m'])
        m_star = max(3, m_star)
        # Embed & subsample
        X = takens_embedding(x, m_star, tau)
        if len(X) > max_points:
            idx = np.linspace(0, len(X)-1, max_points).astype(int)
            X = X[idx]

        # Epsilon from distance quantile
        D = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(axis=2)).ravel()
        D = D[D>0]; eps = float(np.quantile(D, eps_quantile))

        # Recurrence matrix & RQA
        R = recurrence_matrix(X, eps=eps, theiler=theiler)
        rqa = rqa_metrics(R, lmin=lmin, vmin=vmin)

        # Lyapunov & D2
        ly = lyapunov_rosenstein(x, m_star, tau, fs, theiler=max(theiler, int(0.02*fs)), t_fit=(1,30))
        gp = correlation_dimension_gp(X)

        # Surrogates
        null = {'DET':[], 'LAM':[], 'Lmax':[], 'Hdiag':[], 'TT':[], 'lambda':[], 'D2':[]}
        for _ in range(n_surrogates):
            xs = zscore(phase_randomize(x))
            Xs = takens_embedding(xs, m_star, tau)
            if len(Xs) > max_points:
                idx = np.linspace(0, len(Xs)-1, max_points).astype(int)
                Xs = Xs[idx]
            # same eps quantile on surrogate
            Ds = np.sqrt(((Xs[:,None,:]-Xs[None,:,:])**2).sum(axis=2)).ravel()
            Ds = Ds[Ds>0]; eps_s = float(np.quantile(Ds, eps_quantile))
            Rs = recurrence_matrix(Xs, eps=eps_s, theiler=theiler)
            rq = rqa_metrics(Rs, lmin=lmin, vmin=vmin)
            for k in ['DET','LAM','Lmax','Hdiag','TT']:
                null[k].append(rq[k])
            # lyapunov & D2 on surrogate
            lys = lyapunov_rosenstein(xs, m_star, tau, fs, theiler=max(theiler, int(0.02*fs)), t_fit=(1,30))
            gps = correlation_dimension_gp(Xs)
            null['lambda'].append(lys['lambda'])
            null['D2'].append(gps['D2'])

        def pval(obs, arr, greater=True):
            arr = np.asarray(arr, float)
            if not np.isfinite(obs) or arr.size==0: return np.nan
            if greater: return float((np.sum(arr >= obs)+1)/(arr.size+1))
            else:       return float((np.sum(arr <= obs)+1)/(arr.size+1))

        # p-values (one-sided): structure/chaos higher than null
        det_p   = pval(rqa['DET'],   null['DET'],   greater=True)
        lam_p   = pval(rqa['LAM'],   null['LAM'],   greater=True)
        lmax_p  = pval(rqa['Lmax'],  null['Lmax'],  greater=True)
        hdiag_p = pval(rqa['Hdiag'], null['Hdiag'], greater=True)
        tt_p    = pval(rqa['TT'],    null['TT'],    greater=True)
        ly_p    = pval(ly['lambda'], null['lambda'],greater=True)
        d2_p    = pval(gp['D2'],     null['D2'],    greater=True)

        summaries.append({
            'state': st, 'tau': tau, 'm': m_star, 'eps': eps,
            'RR': rqa['RR'], 'DET': rqa['DET'], 'LAM': rqa['LAM'],
            'Lmax': rqa['Lmax'], 'Lmean': rqa['Lmean'], 'Hdiag': rqa['Hdiag'], 'TT': rqa['TT'],
            'lambda': ly['lambda'], 'D2': gp['D2'],
            'DET_p': det_p, 'LAM_p': lam_p, 'Lmax_p': lmax_p, 'Hdiag_p': hdiag_p, 'TT_p': tt_p,
            'lambda_p': ly_p, 'D2_p': d2_p
        })

        # --------- Plots (lightweight) ---------
        # Recurrence plot
        plt.figure(figsize=(4,4))
        plt.imshow(R, origin='lower', cmap='binary')
        plt.title(f'RP — {st} (eps@{int(eps_quantile*100)}%)')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'rp_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # Diagonal-length histogram (determinism structure)
        # Already stored via Hdiag; we can visualize with a toy histogram by recomputing Ls here quickly:
        Ls=[]
        for k in range(-(R.shape[0]-1), R.shape[0]):
            diag = np.diag(R,k=k); run=0
            for v in diag:
                if v==1: run+=1
                elif run>0:
                    if run>=lmin: Ls.append(run); run=0
            if run>=lmin: Ls.append(run)
        if len(Ls):
            plt.figure(figsize=(5,3))
            plt.hist(Ls, bins=np.arange(lmin, max(Ls)+1), color='tab:blue', alpha=0.9)
            plt.xlabel('Diagonal line length'); plt.ylabel('count')
            plt.title(f'Diagonal lengths — {st} (DET={rqa["DET"]:.2f})')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'diag_lengths_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

        # Lyapunov divergence curve
        if ly['L'].size:
            plt.figure(figsize=(5,3))
            plt.plot(ly['k']/fs*tau, ly['L'], lw=1.2)
            plt.xlabel('Time (s)'); plt.ylabel('⟨log(d_k/d_0)⟩')
            plt.title(f'Lyapunov divergence — {st} (λ≈{ly["lambda"]:.3f} s⁻¹)')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'lyapunov_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

        # Correlation dimension log–log
        plt.figure(figsize=(5,3))
        x = np.log(gp['r']+1e-24); y = np.log(gp['C']+1e-24)
        plt.plot(x, y, 'o-', lw=1)
        s, b = gp['fit']
        plt.plot(x, s*x + b, 'r--', lw=1)
        plt.xlabel('log r'); plt.ylabel('log C(r)')
        plt.title(f'Correlation sum — {st} (D2≈{gp["D2"]:.2f})')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'corr_dimension_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        outputs[st] = {'rqa': rqa, 'lyap': ly, 'gp': gp, 'tau': tau, 'm': m_star, 'eps': eps}

    # Save summary CSV
    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
    return {'summary': summary_df, 'outputs': outputs, 'out_dir': out_dir}


In [ ]:
eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F4','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.O1','EEG.O2']
res = run_rqa_chaos_metrics(
    RECORDS,
    eeg_channels=eeg_channels,
    ignition_windows=[(290,310),(580,600)],   # or None for full session
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_rqa/S01',
    show=True,            # save figs, don't stream them
    n_surrogates=100
)
print(res['summary'])


In [ ]:
"""
Multi-Scale Entropy (MSE) & Fractal Scaling (DFA) — Simple Graphs & Validation
==============================================================================

What it does
------------
• Builds a robust 1-D EEG drive (mean across chosen channels, z-scored).
• MSE: Sample Entropy across coarse-grain scales (1 .. ~seconds), with
        surrogate null (phase-randomized) → 95% band & p-values (AUC, mid-scales).
• DFA: log–log slope α of RMS fluctuations across windows (0.25–20 s by default),
       with surrogate null → 95% band & p-value.
• Optionally runs both for Ignition and Baseline and writes a concise summary.

Outputs
-------
• PNGs: MSE curve with null band; DFA log–log with slope line; Ign vs Base bars.
• CSV: summary.csv with MSE_AUC, MSE_mid, DFA_alpha and p-values.

Usage
-----
res = run_mse_dfa_multiscale(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8'],   # clean posterior set recommended
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_mse_dfa/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal

# ------------------- small I/O helpers -------------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')
)->Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    # first numeric, roughly monotonic
    for c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(df)):
            x = s.values.astype(float); dt = np.diff(x[np.isfinite(x)])
            if dt.size and np.nanmedian(dt) > 0: return c
    # datetime?
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None,
                            default_fs: float = 128.0, out_name: str = 'Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs
        return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name] = tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(df)):
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name] = sn.values
    return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs from time column.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Series '{name}' not in DataFrame.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x = np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

# ------------------- MSE (Sample Entropy) -------------------
def coarse_grain(x: np.ndarray, scale: int) -> np.ndarray:
    """Non-overlapping average; drops remainder."""
    N = len(x)//scale * scale
    if N < scale: return np.array([])
    return np.mean(x[:N].reshape(-1, scale), axis=1)

def sampen(x: np.ndarray, m: int = 2, r_ratio: float = 0.2) -> float:
    """
    Sample Entropy (m,r) with Chebyshev metric.
    Returns -ln( A / B ), where:
      B = count matches of length m; A = count matches of length m+1.
    """
    x = np.asarray(x, float)
    N = len(x)
    if N < (m+2): return np.nan
    r = r_ratio * np.std(x)
    # embed
    Xm = np.column_stack([x[i:N-m+1+i] for i in range(m)])
    Xm1= np.column_stack([x[i:N-m  +i] for i in range(m+1)])
    # pairwise Chebyshev distance counts (exclude self)
    def count_matches(X, tol):
        C = 0
        M = len(X)
        for i in range(M-1):
            d = np.max(np.abs(X[i+1:] - X[i]), axis=1)
            C += np.sum(d <= tol)
        return C
    B = count_matches(Xm,  r)
    A = count_matches(Xm1, r)
    if B == 0 or A == 0: return np.nan
    return float(-np.log(A / B))

def mse_curve(x: np.ndarray,
              fs: float,
              max_scale_sec: float = 5.0,
              m: int = 2, r_ratio: float = 0.2,
              min_scale: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute MSE over integer coarse-grain scales up to max_scale_sec.
    Returns (scales_in_sec, SampEn values).
    """
    max_scale = max(min_scale, int(round(max_scale_sec * fs)))
    scales = np.arange(min_scale, max_scale+1, dtype=int)
    S = []
    for s in scales:
        cg = coarse_grain(x, s)
        if cg.size < (m+2):
            S.append(np.nan)
        else:
            S.append(sampen(cg, m=m, r_ratio=r_ratio))
    return scales/fs, np.array(S, float)

# ------------------- DFA -------------------
def dfa_alpha(x: np.ndarray,
              fs: float,
              min_win_sec: float = 0.25,
              max_win_sec: float = 20.0,
              n_win: int = 20) -> Dict[str, object]:
    """
    Detrended Fluctuation Analysis on z-scored signal (integrated profile).
    Returns alpha (slope) and log–log arrays.
    """
    x = zscore(x)
    y = np.cumsum(x - np.mean(x))
    # window sizes in samples (log-spaced)
    n_min = max(4, int(round(min_win_sec*fs)))
    n_max = max(n_min+1, int(round(max_win_sec*fs)))
    ns = np.unique(np.logspace(np.log10(n_min), np.log10(n_max), n_win).astype(int))
    F = []
    for n in ns:
        if n >= len(y): break
        # segment into non-overlapping windows
        N = len(y)//n * n
        yN = y[:N].reshape(-1, n)
        # linear detrend each segment
        t = np.arange(n)
        rms_segments=[]
        for seg in yN:
            p = np.polyfit(t, seg, 1)
            trend = np.polyval(p, t)
            rms = np.sqrt(np.mean((seg - trend)**2))
            rms_segments.append(rms)
        F.append(np.sqrt(np.mean(np.array(rms_segments)**2)))
    ns = ns[:len(F)]
    F = np.array(F, float)
    # slope on log–log
    X = np.log(ns); Y = np.log(F + 1e-24)
    A = np.vstack([X, np.ones_like(X)]).T
    slope, intercept = np.linalg.lstsq(A, Y, rcond=None)[0]
    return {'alpha': float(slope), 'ns': ns, 'F': F, 'fit': (float(slope), float(intercept))}

# ------------------- Surrogates -------------------
def phase_randomize(x: np.ndarray)->np.ndarray:
    X = np.fft.rfft(x); mag = np.abs(X); ph = np.angle(X)
    rnd = np.random.uniform(-np.pi, np.pi, size=mag.size)
    rnd[0] = ph[0]
    if mag.size % 2 == 0: rnd[-1] = ph[-1]
    Xs = mag * np.exp(1j*rnd)
    return np.fft.irfft(Xs, n=len(x)).astype(float)

# ------------------- Orchestrator -------------------
def run_mse_dfa_multiscale(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_mse_dfa/session',
    show: bool = False,
    mse_max_scale_sec: float = 5.0,
    mse_m: int = 2, mse_r_ratio: float = 0.2,
    dfa_min_sec: float = 0.25, dfa_max_sec: float = 20.0,
    n_surrogates: int = 200
)->Dict[str, object]:
    """
    MSE + DFA with surrogate validation, for ignition and baseline windows.
    """
    _ensure_dir(out_dir)
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)

    # robust 1-D drive = mean across selected channels
    def build_drive(wins):
        sigs=[]
        for ch in eeg_channels:
            nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
            if nm in RECORDS.columns:
                x = get_series(RECORDS, nm)
                if wins: x = slice_concat(x, fs, wins)
                sigs.append(np.asarray(x,float))
        if not sigs: raise ValueError("No EEG channels found.")
        L = min(map(len, sigs))
        return zscore(np.mean(np.vstack([s[:L] for s in sigs]), axis=0))

    states = {'ignition': ignition_windows, 'baseline': baseline_windows}
    summaries=[]; outputs={}
    rng = np.random.default_rng(11)

    for st, wins in states.items():
        if wins is None: continue
        x = build_drive(wins)

        # --- MSE
        scales_sec, mse = mse_curve(x, fs, max_scale_sec=mse_max_scale_sec, m=mse_m, r_ratio=mse_r_ratio, min_scale=1)
        # MSE surrogate null (AUC & mid-scale mean)
        mse_null=[]
        for _ in range(n_surrogates):
            xs = zscore(phase_randomize(x))
            _, msen = mse_curve(xs, fs, max_scale_sec=mse_max_scale_sec, m=mse_m, r_ratio=mse_r_ratio, min_scale=1)
            mse_null.append(msen)
        mse_null = np.vstack(mse_null) if len(mse_null) else np.empty((0,len(scales_sec)))
        # summary features
        mse_auc = float(np.nansum(mse))                 # simple area under curve
        mid_mask = (scales_sec>=0.2) & (scales_sec<=1.0)
        mse_mid = float(np.nanmean(mse[mid_mask])) if np.any(mid_mask) else np.nan
        # null bands
        mse_lo = np.nanpercentile(mse_null, 2.5, axis=0) if mse_null.size else np.full_like(mse, np.nan)
        mse_hi = np.nanpercentile(mse_null,97.5, axis=0) if mse_null.size else np.full_like(mse, np.nan)
        # p-values (one-sided)
        mse_auc_p = float((np.sum(np.nansum(mse_null, axis=1) >= mse_auc)+1)/(mse_null.shape[0]+1)) if mse_null.size else np.nan
        mse_mid_p = float((np.sum(np.nanmean(mse_null[:,mid_mask], axis=1) >= mse_mid)+1)/(np.sum(mid_mask)+1)) if (mse_null.size and np.any(mid_mask)) else np.nan

        # --- DFA
        dfa = dfa_alpha(x, fs, min_win_sec=dfa_min_sec, max_win_sec=dfa_max_sec, n_win=20)
        # DFA surrogate null on alpha
        alpha_null=[]
        for _ in range(n_surrogates):
            xs = zscore(phase_randomize(x))
            alpha_null.append(dfa_alpha(xs, fs, min_win_sec=dfa_min_sec, max_win_sec=dfa_max_sec, n_win=20)['alpha'])
        alpha_null = np.asarray(alpha_null, float)
        alpha_p = float((np.sum(alpha_null >= dfa['alpha'])+1)/(alpha_null.size+1)) if alpha_null.size else np.nan
        alpha_lo = float(np.nanpercentile(alpha_null, 2.5)) if alpha_null.size else np.nan
        alpha_hi = float(np.nanpercentile(alpha_null,97.5)) if alpha_null.size else np.nan

        # --- Save plots ---
        # MSE curve
        plt.figure(figsize=(6,3.2))
        plt.plot(scales_sec, mse, lw=1.8, label='MSE')
        if mse_null.size:
            plt.fill_between(scales_sec, mse_lo, mse_hi, color='k', alpha=0.12, label='surrogate 95%')
        plt.xlabel('Scale (s)'); plt.ylabel('SampEn'); plt.title(f'MSE — {st}')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'mse_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # DFA log–log
        X = np.log(dfa['ns']); Y = np.log(dfa['F']+1e-24); s,b = dfa['fit']
        plt.figure(figsize=(6,3.2))
        plt.plot(X, Y, 'o-', lw=1.0, label='log–log')
        plt.plot(X, s*X + b, 'r--', lw=1.2, label=f'α≈{dfa["alpha"]:.2f}')
        if alpha_null.size:
            plt.hlines([alpha_lo, alpha_hi], X.min(), X.max(), colors='k', linestyles=':', lw=1, label='surrogate 95% (α)')
        plt.xlabel('log window n'); plt.ylabel('log F(n)'); plt.title(f'DFA — {st}')
        plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'dfa_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        summaries.append({'state':st,
                          'MSE_AUC':mse_auc, 'MSE_AUC_p':mse_auc_p,
                          'MSE_mid':mse_mid, 'MSE_mid_p':mse_mid_p,
                          'DFA_alpha':dfa['alpha'], 'DFA_alpha_p':alpha_p,
                          'DFA_alpha_lo95':alpha_lo, 'DFA_alpha_hi95':alpha_hi})

        outputs[st] = {'mse': {'scales':scales_sec, 'S':mse, 'lo':mse_lo, 'hi':mse_hi},
                       'dfa': dfa,
                       'mse_null': mse_null, 'alpha_null': alpha_null}

    # Ignition vs Baseline bar (optional)
    if 'ignition' in outputs and 'baseline' in outputs:
        ig = [s for s in summaries if s['state']=='ignition'][0]
        ba = [s for s in summaries if s['state']=='baseline'][0]
        plt.figure(figsize=(6,3.2))
        names=['MSE_AUC','MSE_mid','DFA_alpha']
        x = np.arange(len(names)); w=0.35
        plt.bar(x-w/2, [ba[n] for n in names], width=w, label='Baseline', color='tab:orange', alpha=0.9)
        plt.bar(x+w/2, [ig[n] for n in names], width=w, label='Ignition', color='tab:blue', alpha=0.9)
        plt.xticks(x, names); plt.ylabel('value'); plt.title('Ignition vs Baseline — MSE/DFA')
        plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir,'ign_vs_base.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # Save summary
    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(os.path.join(out_dir,'summary.csv'), index=False)
    return {'summary': summary_df, 'outputs': outputs, 'out_dir': out_dir}


In [ ]:
# Choose a clean posterior set (θ/α-rich, low artifacts)
eeg_channels=['AF3','AF4','F3','F4','F7','F8','FC5','FC6','P7','P8','O1','O2']

res = run_mse_dfa_multiscale(
    RECORDS,
    eeg_channels=eeg_channels,
    ignition_windows=[(290,310),(580,600)],    # or None for full session
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_mse_dfa/S01',
    show=True,               # save figs; avoids IOPub floods
    mse_max_scale_sec=5.0,    # up to ~5 s scales for MSE
    dfa_min_sec=0.25, dfa_max_sec=20.0,
    n_surrogates=200
)

print(res['summary'])


In [ ]:
"""
Network Graph Metrics & Hub Analysis — Simple Graphs & Validation
=================================================================

What it does
------------
• Connectivity (per band) with PLI (default; robust to zero-lag / volume conduction).
• Threshold to target density → undirected weighted graph (symmetric).
• Metrics: small-world index σ (C/C_rand)/(L/L_rand), clustering C, char path L, global efficiency Eglob,
           modularity Q (greedy), participation coefficient P_i per node,
           centralities (strength, degree, betweenness).
• Degree-preserving rewires (double-edge swaps) → null for σ.
• Per-state outputs (Ignition/Baseline): heatmaps, metric bars, hub bars, summary CSV.

Inputs
------
RECORDS : DataFrame with a numeric time column (default 'Timestamp') and EEG.* columns.
eeg_channels : list of channels to include (e.g., ['EEG.O1','EEG.O2',...]).
bands : dict of name→(f1,f2); default theta/alpha/beta.
method : 'pli' (default) or 'imagcoh' (imag coherency via Welch).
density : edge density after thresholding (0..1).

Usage
-----
res = run_graph_metrics_hubs(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_graph/S01',
    show=False  # save figures; avoids notebook flooding
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, networkx as nx
from typing import Dict, List, Tuple, Optional
from scipy import signal

# ---------------- I/O & time helpers ----------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')) -> Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    # first numeric, roughly monotonic
    for c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(df)):
            x = s.values.astype(float); dt = np.diff(x[np.isfinite(x)])
            if dt.size and np.nanmedian(dt)>0: return c
    # datetime?
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None,
                            default_fs: float = 128.0, out_name: str='Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name] = tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(df)):
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name] = sn.values
    return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"{name} not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x=np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

# ---------------- Connectivity (PLI / Imag Coh) ----------------
def bandpass(x, fs, f1, f2, order=4):
    ny=0.5*fs; f1=max(1e-6,min(f1,0.99*ny)); f2=max(f1+1e-6,min(f2,0.999*ny))
    b,a=signal.butter(order,[f1/ny,f2/ny],btype='band'); return signal.filtfilt(b,a,x)

def pli_connectivity(X: np.ndarray, fs: float, f1: float, f2: float) -> np.ndarray:
    """
    PLI from band-passed analytic phases. X: (n_ch, T)
    """
    Xb = np.vstack([bandpass(x, fs, f1, f2) for x in X])
    Z = signal.hilbert(Xb, axis=1); phi = np.angle(Z)
    n = X.shape[0]
    W = np.zeros((n,n), float)
    for i in range(n):
        for j in range(i+1,n):
            dphi = phi[i]-phi[j]
            pli = np.abs(np.mean(np.sign(np.sin(dphi))))
            W[i,j]=W[j,i]=float(pli)
    np.fill_diagonal(W, 0.0)
    return W

def imagcoh_connectivity(X: np.ndarray, fs: float, f1: float, f2: float) -> np.ndarray:
    """
    Imag coherency from analytic signals. Robust to zero-lag.
    """
    Xb = np.vstack([bandpass(x, fs, f1, f2) for x in X])
    Z = signal.hilbert(Xb, axis=1)
    n = X.shape[0]
    W = np.zeros((n,n), float)
    for i in range(n):
        for j in range(i+1,n):
            Sxy = np.mean(Z[i]*np.conj(Z[j]))
            Sxx = np.mean(Z[i]*np.conj(Z[i])); Syy = np.mean(Z[j]*np.conj(Z[j]))
            coh = Sxy/np.sqrt((Sxx*Syy)+1e-24)
            val = np.abs(np.imag(coh))
            W[i,j]=W[j,i]=float(val)
    np.fill_diagonal(W, 0.0)
    return W

# ---------------- Graph build & metrics ----------------
def threshold_by_density(W: np.ndarray, density: float = 0.2) -> np.ndarray:
    """
    Keep top fraction of weights (upper triangle) to reach target density.
    """
    n = W.shape[0]
    tri = W[np.triu_indices(n,1)]
    if np.all(tri==0): return np.zeros_like(W)
    k = int(np.round(density * (n*(n-1)/2)))
    k = max(1, min(k, tri.size))
    thr = np.partition(tri, -k)[-k]  # kth largest
    WT = np.where(W >= thr, W, 0.0)
    # ensure symmetry and zero diag
    WT = np.maximum(WT, WT.T); np.fill_diagonal(WT, 0.0)
    return WT

def graph_from_weighted(W: np.ndarray) -> nx.Graph:
    G = nx.Graph()
    n = W.shape[0]
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i+1,n):
            w = float(W[i,j])
            if w>0:
                G.add_edge(i,j,weight=w, length=1.0/max(w,1e-12))  # length for distances
    return G

def global_efficiency_weighted(G: nx.Graph) -> float:
    """
    Weighted global efficiency: mean of 1/d_ij on finite shortest paths (length attr).
    """
    if G.number_of_edges()==0: return np.nan
    effs=[]
    for i in G.nodes():
        lengths = nx.single_source_dijkstra_path_length(G, i, weight='length')
        for j,l in lengths.items():
            if i!=j and np.isfinite(l) and l>0:
                effs.append(1.0/l)
    return float(np.nanmean(effs)) if effs else np.nan

def char_path_length_weighted(G: nx.Graph) -> float:
    """
    Weighted characteristic path length using 'length' as distance.
    """
    if G.number_of_edges()==0: return np.nan
    Ls=[]
    for comp in nx.connected_components(G):
        H = G.subgraph(comp)
        if H.number_of_nodes()<2: continue
        Ls.append(nx.average_shortest_path_length(H, weight='length'))
    return float(np.nanmean(Ls)) if Ls else np.nan

def clustering_weighted(G: nx.Graph) -> float:
    c = nx.clustering(G, weight='weight')
    return float(np.nanmean(list(c.values()))) if c else np.nan

def modularity_greedy(G: nx.Graph) -> Tuple[Dict[int,int], float]:
    """
    Greedy modularity communities (weighted). Returns membership dict and Q.
    """
    if G.number_of_edges()==0:
        return ({i:i for i in G.nodes()}, np.nan)
    coms = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
    memb = {}
    for ci, C in enumerate(coms):
        for node in C:
            memb[int(node)] = int(ci)
    # compute modularity Q
    Q = nx.algorithms.community.quality.modularity(G, coms, weight='weight')
    return memb, float(Q)

def participation_coeff(W: np.ndarray, memb: Dict[int,int]) -> np.ndarray:
    """
    Weighted participation coefficient: 1 - sum_s (k_is/k_i)^2
    """
    n = W.shape[0]
    k = np.sum(W, axis=1) + 1e-12
    S = {}
    for i in range(n):
        for j in range(n):
            if W[i,j]>0:
                S.setdefault((i, memb[j]), 0.0)
                S[(i, memb[j])] += W[i,j]
    P = np.zeros(n, float)
    for i in range(n):
        parts = [S.get((i,s),0.0) for s in set(memb.values())]
        P[i] = 1.0 - np.sum((np.array(parts)/k[i])**2)
    return P

def small_world_sigma(G: nx.Graph, n_rewire: int = 20) -> Tuple[float,float,float]:
    """
    Small-world index σ = (C/C_rand)/(L/L_rand) using degree-preserving nulls.
    Returns (sigma, C, L). Nulls averaged over n_rewire rewires.
    """
    if G.number_of_edges()==0 or G.number_of_nodes()<3:
        return (np.nan, np.nan, np.nan)
    C = clustering_weighted(G)
    L = char_path_length_weighted(G)
    # make a binary copy preserving degree
    B = nx.Graph()
    B.add_nodes_from(G.nodes())
    for u,v,d in G.edges(data=True):
        B.add_edge(u,v)
    Cr=[]; Lr=[]
    for _ in range(n_rewire):
        H = B.copy()
        # double-edge swaps preserve degree sequence
        try:
            nx.double_edge_swap(H, nswap=max(1, H.number_of_edges()*2), max_tries=H.number_of_edges()*10)
        except Exception:
            pass
        # put uniform weights (1) for null C,L on binary graph; lengths=1
        C0 = nx.average_clustering(H)
        if nx.is_connected(H):
            L0 = nx.average_shortest_path_length(H)
        else:
            L0 = np.nan
        Cr.append(C0); Lr.append(L0)
    Cr = np.array(Cr, float); Lr = np.array(Lr, float)
    C_rand = float(np.nanmean(Cr)) if Cr.size else np.nan
    L_rand = float(np.nanmean(Lr)) if Lr.size else np.nan
    if not np.isfinite(C_rand) or not np.isfinite(L_rand) or C_rand==0 or L_rand==0 or not np.isfinite(C) or not np.isfinite(L):
        return (np.nan, C, L)
    sigma = (C/C_rand)/(L/L_rand)
    return (float(sigma), float(C), float(L))

# ---------------- Connectivity wrapper ----------------
def compute_connectivity(RECORDS: pd.DataFrame, channels: List[str], wins,
                         band: Tuple[float,float], method: str, time_col: str) -> Tuple[np.ndarray, List[str], float]:
    """
    Returns (W, chan_names, fs) — symmetric connectivity matrix in [0,1].
    """
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)
    X=[]; names=[]
    for ch in channels:
        nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
        if nm in RECORDS.columns:
            x = get_series(RECORDS, nm)
            x = slice_concat(x, fs, wins)
            X.append(zscore(np.asarray(x,float))); names.append(nm)
    if not X: raise ValueError("No EEG channels found.")
    # truncate common length
    L = min(len(x) for x in X)
    X = np.vstack([x[:L] for x in X])
    if method.lower() == 'imagcoh':
        W = imagcoh_connectivity(X, fs, band[0], band[1])
    else:
        W = pli_connectivity(X, fs, band[0], band[1])
    return W, names, fs

# ---------------- Orchestrator ----------------
def run_graph_metrics_hubs(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    bands: Dict[str, Tuple[float,float]] = None,
    method: str = 'pli',               # 'pli' or 'imagcoh'
    density: float = 0.2,              # target graph density after threshold
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_graph/session',
    show: bool = False
)->Dict[str, object]:
    """
    Build functional graphs per band and state; compute small-worldness, clustering,
    path length, efficiency, modularity, participation & hubs; save simple plots + CSV.
    """
    _ensure_dir(out_dir)
    bands = bands or {'theta':(4,8), 'alpha':(8,13), 'beta':(13,30)}
    states = {'ignition': ignition_windows, 'baseline': baseline_windows}

    summaries=[]
    results={}
    for st, wins in states.items():
        if wins is None: continue
        for bn, bnd in bands.items():
            # Connectivity
            W, names, fs = compute_connectivity(RECORDS, eeg_channels, wins, bnd, method, time_col)
            # Threshold to density
            WT = threshold_by_density(W, density=density)
            # Graph with weights & 'length' attribute
            G = graph_from_weighted(WT)
            # Metrics
            sigma, C, L = small_world_sigma(G, n_rewire=20)
            Eglob = global_efficiency_weighted(G)
            memb, Q = modularity_greedy(G)
            # Participation
            P = participation_coeff(WT, memb) if G.number_of_edges()>0 else np.full(WT.shape[0], np.nan)
            # Hubs
            strength = np.sum(WT, axis=1)
            degree   = (WT>0).sum(axis=1)
            # Betweenness (use 'length' attr as distance → invert of weight already set)
            BC = nx.betweenness_centrality(G, weight='length', normalized=True) if G.number_of_edges()>0 else {i:np.nan for i in range(len(names))}
            betweenness = np.array([BC[i] for i in range(len(names))], float)

            # Save heatmap
            plt.figure(figsize=(4.5,4))
            plt.imshow(W, vmin=0, vmax=1, cmap='magma')
            plt.colorbar(label=f'{method.upper()}')
            plt.xticks(range(len(names)), [n.split(".",1)[-1] for n in names], rotation=90, fontsize=8)
            plt.yticks(range(len(names)), [n.split(".",1)[-1] for n in names], fontsize=8)
            plt.title(f'{bn} connectivity — {st}')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'conn_{bn}_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

            # Metric bars
            plt.figure(figsize=(6,3.2))
            keys = ['sigma','C','L','Eglob','Q']
            vals = [sigma, C, L, Eglob, Q]
            plt.bar(range(len(keys)), vals, color='tab:blue', alpha=0.9)
            plt.xticks(range(len(keys)), keys); plt.ylabel('value')
            plt.title(f'Graph metrics — {bn} / {st} (density={density:.2f})')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'metrics_{bn}_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

            # Hub bars (strength & participation)
            plt.figure(figsize=(max(6, 0.4*len(names)), 3.0))
            idx = np.argsort(strength)[::-1]
            top = idx[:min(8,len(idx))]
            plt.bar(np.arange(len(top))-0.2, strength[top], width=0.4, label='strength')
            plt.bar(np.arange(len(top))+0.2, P[top],       width=0.4, label='participation')
            plt.xticks(range(len(top)), [names[i].split('.',1)[-1] for i in top], rotation=0, fontsize=8)
            plt.ylabel('value'); plt.legend()
            plt.title(f'Hubs — {bn} / {st}')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'hubs_{bn}_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

            # Store & summarize
            results.setdefault(st, {})[bn] = {'W':W, 'WT':WT, 'names':names,
                                              'sigma':sigma, 'C':C, 'L':L, 'Eglob':Eglob,
                                              'Q':Q, 'P':P, 'strength':strength, 'degree':degree,
                                              'betweenness':betweenness, 'memb':memb}
            summaries.append({'state':st,'band':bn,'sigma':sigma,'C':C,'L':L,'Eglob':Eglob,'Q':Q})

    # Ignition vs Baseline side-by-side (if both exist)
    if 'ignition' in results and 'baseline' in results:
        for bn in bands.keys():
            if bn in results['ignition'] and bn in results['baseline']:
                plt.figure(figsize=(6,3.2))
                keys=['sigma','C','L','Eglob','Q']
                x=np.arange(len(keys)); w=0.38
                ig = results['ignition'][bn]; ba = results['baseline'][bn]
                plt.bar(x-w/2, [ba[k] for k in keys], width=w, label='Baseline', color='tab:orange', alpha=0.9)
                plt.bar(x+w/2, [ig[k] for k in keys], width=w, label='Ignition', color='tab:blue', alpha=0.9)
                plt.xticks(x, keys); plt.ylabel('value'); plt.title(f'Ignition vs Baseline — {bn}')
                plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'ign_vs_base_{bn}.png'), dpi=140)
                if show: plt.show()
                plt.close()

    # Save CSV summary
    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
    return {'summary': summary_df, 'results': results, 'out_dir': out_dir}


In [ ]:
# Clean posterior-centric set (theta/alpha; low artifacts). 6–10 channels works well.
eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F4','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.O1','EEG.O2','EEG.T7','EEG.T8']

res = run_graph_metrics_hubs(
    RECORDS,
    eeg_channels=eeg_channels,
    ignition_windows=[(290,310),(580,600)],     # or None for full session
    baseline_windows=[(0,290),(325,580)],
    bands={'theta':(4,8),'alpha':(8,13),'beta':(13,30)},
    method='pli',                  # 'pli' (robust) or 'imagcoh'
    density=0.20,                  # 20% top edges
    time_col='Timestamp',
    out_dir='exports_graph/S01',
    show=True
)
print(res['summary'])


In [ ]:
"""
EEG Microstate Segmentation — Simple Graphs & Validation
========================================================

What it does
------------
• Builds multi-channel EEG matrix X (n_ch × T) over given windows; z-scores and (optionally) band-passes.
• Finds GFP peaks and clusters their topographies (channels) with k-means (k∈{3,4,5,6}) → candidate microstate maps.
• Selects k by Global Explained Variance (GEV); backfits full sequence (polarity-invariant, argmax|corr|).
• Temporal smoothing (minimum segment duration) to avoid spurious flips.
• Metrics: GEV, mean duration (ms), coverage (%time), occurrence rate (per s),
           transition matrix (k×k), sequence Shannon entropy (bits).
• Surrogate validation: phase-randomize each channel and recompute GEV → 95% null and p-value.
• Outputs PNGs + summary.csv in out_dir.

Usage
-----
res = run_microstate_segmentation(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6'],  # 6–12 clean channels recommended
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    band=(2,40),                         # optional prefilter
    time_col='Timestamp',
    out_dir='exports_microstates/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
from sklearn.cluster import KMeans

# ---------------- I/O helpers ----------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')
)->Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    # first numeric, roughly monotonic
    for c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > max(50, 0.5*len(df)):
            x = s.values.astype(float); dt = np.diff(x[np.isfinite(x)])
            if dt.size and np.nanmedian(dt) > 0: return c
    # datetime?
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None,
                            default_fs: float = 128.0, out_name: str = 'Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs
        return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec = (pd.to_datetime(s) - pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name] = tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum() < max(50, 0.5*len(df)):
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name] = sn.values; return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs from time column.")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        x = pd.to_numeric(df[name], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    alt = 'EEG.'+name
    if alt in df.columns:
        x = pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values
        return np.asarray(x, float)
    raise ValueError(f"Series '{name}' not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x=np.asarray(x,float); return (x - np.mean(x)) / (np.std(x)+1e-12)

def bandpass(x, fs, f1, f2, order=4):
    ny=0.5*fs; f1=max(1e-6,min(f1,0.99*ny)); f2=max(f1+1e-6,min(f2,0.999*ny))
    b,a=signal.butter(order,[f1/ny,f2/ny],btype='band'); return signal.filtfilt(b,a,x)

# ---------------- Microstate core ----------------
def gfp(X: np.ndarray) -> np.ndarray:
    """Global Field Power across channels per time (std). X: (n_ch, T)"""
    return np.std(X, axis=0)

def pick_gfp_peaks(G: np.ndarray, skip: int) -> np.ndarray:
    """Pick timepoints at local maxima of GFP with a refractory 'skip' in samples."""
    peaks = []
    i = skip
    while i < len(G)-skip:
        win = G[i-skip:i+skip+1]
        if np.argmax(win) == skip:
            peaks.append(i)
            i += skip
        else:
            i += 1
    return np.array(peaks, int)

def normalize_maps(X: np.ndarray) -> np.ndarray:
    """Zero-mean & L2-normalize topographies columnwise. X: (n_ch, Nmaps)"""
    Xm = X - X.mean(axis=0, keepdims=True)
    denom = np.linalg.norm(Xm, axis=0, keepdims=True) + 1e-12
    return Xm/denom

def kmeans_microstates(Xmaps: np.ndarray, k: int, n_init: int = 20, seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    """KMeans on normalized maps (channels×N) → centers (channels×k), labels (N,)"""
    Z = Xmaps.T  # (N, n_ch)
    km = KMeans(n_clusters=k, n_init=n_init, random_state=seed)
    labels = km.fit_predict(Z)
    centers = km.cluster_centers_.T
    # normalize centers the same way
    centers = centers - centers.mean(axis=0, keepdims=True)
    centers = centers / (np.linalg.norm(centers, axis=0, keepdims=True) + 1e-12)
    return centers, labels

def backfit_sequence(X: np.ndarray, centers: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Assign each timepoint to the map with max |corr| (polarity-invariant).
    Returns labels (T,) in [0..k-1] and per-time corr_abs (T,).
    """
    # normalize data topographies per timepoint
    Xn = X - X.mean(axis=0, keepdims=True)
    denom = np.linalg.norm(Xn, axis=0, keepdims=True)+1e-12
    Xn = Xn/denom
    # centers already normalized (n_ch×k)
    C = centers
    corr = Xn.T @ C                  # (T × k)
    corr_abs = np.abs(corr)          # polarity invariance
    lab = np.argmax(corr_abs, axis=1)
    maxcorr = corr_abs[np.arange(len(lab)), lab]
    return lab.astype(int), maxcorr

def smooth_labels(labels: np.ndarray, fs: float, min_dur_ms: float = 30.0) -> np.ndarray:
    """Enforce minimum segment duration by merging short runs into neighbors."""
    min_len = max(1, int(round(min_dur_ms/1000.0 * fs)))
    L = labels.copy()
    start = 0
    while start < len(L):
        end = start
        while end+1 < len(L) and L[end+1]==L[start]:
            end += 1
        run_len = end - start + 1
        if run_len < min_len:
            # merge toward the neighboring label with longer adjacent run
            left_lab  = L[start-1] if start>0 else None
            right_lab = L[end+1]   if end+1<len(L) else None
            if left_lab is None and right_lab is None:
                pass
            elif left_lab is None:
                L[start:end+1] = right_lab
            elif right_lab is None:
                L[start:end+1] = left_lab
            else:
                # choose side with longer contiguous run
                lstart=start-1
                while lstart-1>=0 and L[lstart-1]==left_lab: lstart-=1
                rend=end+1
                while rend+1<len(L) and L[rend+1]==right_lab: rend+=1
                if (start-lstart) >= (rend-end):
                    L[start:end+1] = left_lab
                else:
                    L[start:end+1] = right_lab
        start = end+1
    return L

def microstate_metrics(labels: np.ndarray, fs: float, k: int) -> Dict[str, object]:
    """
    Mean duration (ms), coverage, occurrence rate (/s), transition matrix, sequence entropy (bits).
    """
    T = len(labels)
    cov = np.zeros(k, float)
    occ = np.zeros(k, float)
    durations=[]
    # compute runs
    i=0; seq=[]
    while i<T:
        j=i
        while j+1<T and labels[j+1]==labels[i]:
            j+=1
        L = j-i+1
        cov[labels[i]] += L
        occ[labels[i]] += 1
        durations.append((labels[i], L/fs))
        seq.extend([labels[i]]*L)
        i=j+1
    cov = cov/T
    occ = occ/ (T/fs)  # per second
    dur_ms = np.zeros(k,float)
    for s in range(k):
        ls = [d for (lab,d) in durations if lab==s]
        dur_ms[s] = 1000.0 * (np.mean(ls) if ls else np.nan)
    # transitions
    trans = np.zeros((k,k), float)
    for t in range(T-1):
        a,b = labels[t], labels[t+1]
        if a!=b:
            trans[a,b] += 1
    row_sum = trans.sum(axis=1, keepdims=True)+1e-12
    P = trans/row_sum
    # sequence entropy
    p = cov; p = p[p>0]
    H = float(-np.sum(p*np.log2(p)))
    return {'coverage':cov, 'occurrence':occ, 'duration_ms':dur_ms, 'P':P, 'H_seq':H}

def gev_score(GFP: np.ndarray, corr_abs: np.ndarray) -> float:
    """Global Explained Variance: sum(GFP^2 * corr^2)/sum(GFP^2)."""
    num = np.sum((GFP**2) * (corr_abs**2))
    den = np.sum(GFP**2) + 1e-12
    return float(num/den)

# ---------------- Orchestrator ----------------
def run_microstate_segmentation(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    band: Optional[Tuple[float,float]] = (2,40),
    ks: List[int] = [3,4,5,6],
    peak_refrac_ms: float = 10.0,
    min_seg_ms: float = 30.0,
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_microstates/session',
    n_surrogates: int = 200,
    show: bool = False
)->Dict[str, object]:
    """
    Microstate maps & metrics with surrogate validation; Ignition/Baseline comparison.
    """
    _ensure_dir(out_dir)
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)

    def build_X(wins):
        X=[]; names=[]
        for ch in eeg_channels:
            nm = ch if ch.startswith('EEG.') else 'EEG.'+ch
            if nm in RECORDS.columns:
                x = get_series(RECORDS, nm)
                if wins: x = slice_concat(x, fs, wins)
                if band is not None:
                    x = bandpass(x, fs, band[0], band[1])
                X.append(zscore(np.asarray(x,float))); names.append(nm)
        if not X: raise ValueError("No EEG channels found.")
        L = min(len(x) for x in X)
        X = np.vstack([x[:L] for x in X])
        return X, names

    states = {'ignition': ignition_windows, 'baseline': baseline_windows}
    results={}; summaries=[]

    for st, wins in states.items():
        if wins is None: continue
        X, names = build_X(wins)             # (n_ch × T)
        n_ch, T = X.shape
        G = gfp(X)
        # pick GFP peaks (every ~peak_refrac_ms)
        skip = max(1, int(round(peak_refrac_ms/1000.0 * fs)))
        idx_peaks = pick_gfp_peaks(G, skip)
        if idx_peaks.size < max(200, 5*len(ks)):
            # fallback: take top-N GFP timepoints
            N = max(200, 5*len(ks))
            idx_peaks = np.argsort(G)[-N:]

        # topography matrix at peaks
        Maps = X[:, idx_peaks]          # (n_ch × Nmaps)
        Maps = normalize_maps(Maps)     # zero-mean, L2 norms

        # try multiple k; pick best by GEV
        candidates=[]
        for k in ks:
            centers, _ = kmeans_microstates(Maps, k=k, n_init=30, seed=0)
            # backfit all timepoints
            lab_all, corr_abs = backfit_sequence(X, centers)
            # smoothing
            lab_s = smooth_labels(lab_all, fs, min_dur_ms=min_seg_ms)
            # recompute corr after smoothing (optional): use the same corr_abs for GEV
            gev = gev_score(G, corr_abs)
            candidates.append((k, centers, lab_s, corr_abs, gev))
        # select best k
        k_best, centers, labels, corr_abs, gev_best = max(candidates, key=lambda t:t[4])

        # metrics
        metrics = microstate_metrics(labels, fs, k_best)

        # surrogate GEV null (phase-randomize per channel)
        null_gev=[]
        for _ in range(n_surrogates):
            Xs = np.vstack([zscore(np.fft.irfft(np.abs(np.fft.rfft(x))*np.exp(1j*np.random.uniform(-np.pi,np.pi, size=np.fft.rfft(x).size)), n=len(x)).astype(float))
                            for x in X])
            Gs = gfp(Xs)
            # reuse same centers? fairer to re-cluster peaks of surrogate to avoid bias:
            idx_s = pick_gfp_peaks(Gs, skip)
            if idx_s.size < len(idx_peaks):
                idx_s = np.argsort(Gs)[-len(idx_peaks):]
            Maps_s = normalize_maps(Xs[:, idx_s])
            c_s, _ = kmeans_microstates(Maps_s, k=k_best, n_init=10, seed=0)
            _, corr_s = backfit_sequence(Xs, c_s)
            null_gev.append(gev_score(Gs, corr_s))
        null_gev = np.asarray(null_gev, float)
        gev_p = float((np.sum(null_gev >= gev_best)+1)/(null_gev.size+1)) if null_gev.size else np.nan
        gev_lo = float(np.nanpercentile(null_gev, 2.5)) if null_gev.size else np.nan
        gev_hi = float(np.nanpercentile(null_gev,97.5)) if null_gev.size else np.nan

        # -------- Plots --------
        # (1) GEV vs k curve
        plt.figure(figsize=(4.5,3))
        plt.plot([c[0] for c in candidates], [c[4] for c in candidates], 'o-')
        plt.xlabel('k'); plt.ylabel('GEV'); plt.title(f'GEV vs k — {st}')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'gev_vs_k_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # (2) Microstate maps (channel×k heatmap)
        plt.figure(figsize=(max(6, 0.5*k_best+4), 3.5))
        im = plt.imshow(centers, aspect='auto', cmap='coolwarm',
                        vmin=-np.max(np.abs(centers)), vmax=np.max(np.abs(centers)))
        plt.colorbar(label='weight')
        plt.yticks(range(len(names)), [n.split('.',1)[-1] for n in names], fontsize=8)
        plt.xticks(range(k_best), [f'M{k+1}' for k in range(k_best)])
        plt.title(f'Microstate maps — {st} (k={k_best})')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'maps_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # (3) Sequence strip & coverage/duration bars
        plt.figure(figsize=(8,1.8))
        plt.imshow(labels[None,:], aspect='auto', cmap='tab20', vmin=0, vmax=k_best-1)
        plt.yticks([]); plt.xlabel('Time (samples)'); plt.title(f'Sequence — {st}')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'sequence_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        plt.figure(figsize=(max(6, 0.5*k_best+4), 3.0))
        x = np.arange(k_best)
        plt.bar(x-0.2, metrics['coverage']*100, width=0.4, label='coverage %')
        plt.bar(x+0.2, metrics['duration_ms'], width=0.4, label='mean duration (ms)')
        plt.xticks(x, [f'M{k+1}' for k in range(k_best)])
        plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'coverage_duration_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # (4) Transition matrix
        plt.figure(figsize=(4,3.2))
        plt.imshow(metrics['P'], vmin=0, vmax=np.nanmax(metrics['P']) if np.isfinite(np.nanmax(metrics['P'])) else 1.0, cmap='magma')
        plt.colorbar(label='P(i→j)')
        plt.xticks(range(k_best), [f'M{k+1}' for k in range(k_best)])
        plt.yticks(range(k_best), [f'M{k+1}' for k in range(k_best)])
        plt.title(f'Transitions — {st}')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'transitions_{st}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # Store/summary
        results[st] = {'names':names, 'k':k_best, 'centers':centers,
                       'labels':labels, 'metrics':metrics,
                       'GEV':gev_best, 'GEV_null95_lo':gev_lo, 'GEV_null95_hi':gev_hi, 'GEV_p':gev_p}
        summaries.append({'state':st, 'k':k_best, 'GEV':gev_best, 'GEV_p':gev_p,
                          'coverage_mean':float(np.nanmean(metrics['coverage'])),
                          'duration_ms_mean':float(np.nanmean(metrics['duration_ms'])),
                          'H_seq':metrics['H_seq']})

    # Ignition vs Baseline quick comparison
    if 'ignition' in results and 'baseline' in results:
        ig = results['ignition']; ba = results['baseline']
        plt.figure(figsize=(6,3.2))
        names = ['GEV','coverage_mean','duration_ms_mean','H_seq']
        vals_ig = [s for s in summaries if s['state']=='ignition'][0]
        vals_ba = [s for s in summaries if s['state']=='baseline'][0]
        x = np.arange(len(names)); w=0.38
        plt.bar(x-w/2, [vals_ba[n] for n in names], width=w, label='Baseline', color='tab:orange', alpha=0.9)
        plt.bar(x+w/2, [vals_ig[n] for n in names], width=w, label='Ignition', color='tab:blue', alpha=0.9)
        plt.xticks(x, names); plt.ylabel('value'); plt.title('Ignition vs Baseline — microstate metrics')
        plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir,'ign_vs_base.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # Summary CSV
    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
    return {'summary': summary_df, 'results': results, 'out_dir': out_dir}


In [ ]:
# Use a clean posterior-centric set for stable topographies
eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F4','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.O1','EEG.O2','EEG.T7','EEG.T8']
res = run_microstate_segmentation(
    RECORDS,
    eeg_channels=eeg_channels,
    ignition_windows=[(290,310),(580,600)],   # or None for full session
    baseline_windows=[(0,290),(325,580)],
    band=(2,40),
    time_col='Timestamp',
    out_dir='exports_microstates/S01',
    show=True,
    ks=[3,4,5,6],              # candidates
    peak_refrac_ms=10.0,       # GFP peak refractory
    min_seg_ms=30.0,           # temporal smoothing: min segment
    n_surrogates=200
)
print(res['summary'])


In [ ]:
"""
Cross-Frequency & Cross-Region Coupling — Simple Graphs & Validation
====================================================================

What this module does
---------------------
1) Cross-channel PAC (phase→amplitude): slow phase in chA modulates fast amplitude in chB.
   • Comodulogram over f_slow ∈ [4..12] Hz and f_fast ∈ [30..80] Hz.
   • Surrogate null (circular shift of the **fast** channel) → cell-wise 95% threshold.
2) n:m Phase locking (PLV_{n:m}): |<exp(i*(n*phi1 − m*phi2))>|
   • Grid over f1 ∈ [4..15] Hz, f2 ∈ [20..80] Hz, ratios m∈{2..6}, n=1 (default), cross-channel.
   • Surrogate null (shift one phase) → significance map.
3) Summaries: top PAC pairs (chA→chB, f1*, f2*, MI, p), top n:m pairs (ch1↔ch2, f1*, f2*, m, PLV, p).
4) Optional Ignition vs Baseline comparison (re-run on each window set).

Outputs
-------
• PNGs: comodulograms with significant cells (cyan), n:m PLV maps, pairwise summaries.
• CSV: cfc_summary.csv (top finds & p-values) in out_dir.

Usage
-----
res = run_cfc_cross_region(
    RECORDS,
    pairs=[('EEG.F3','EEG.P8'), ('EEG.F4','EEG.P7')],  # phase@F → amp@P examples
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_cfc/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
import networkx as nx

# ---------------- I/O & time helpers ----------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')) -> Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    # numeric, roughly monotonic
    for c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum()>max(50,0.5*len(df)):
            x=s.values.astype(float); dt=np.diff(x[np.isfinite(x)])
            if dt.size and np.nanmedian(dt)>0: return c
    # datetime
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None, default_fs: float = 128.0, out_name='Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name]=tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum()<max(50,0.5*len(df)):
        df[out_name]=np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)]); df[out_name]=sn.values; return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt=np.diff(t); dt=dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs."); return 1.0
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
    alt='EEG.'+name
    if alt in df.columns:
        return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
    raise ValueError(f"{name} not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1=int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x=np.asarray(x,float); return (x-np.mean(x))/(np.std(x)+1e-12)

# ---------------- Filtering & analytic ----------------
def bandpass(x, fs, f1, f2, order=4):
    ny=0.5*fs; f1=max(1e-6,min(f1,0.99*ny)); f2=max(f1+1e-6,min(f2,0.999*ny))
    b,a=signal.butter(order,[f1/ny,f2/ny],btype='band'); return signal.filtfilt(b,a,x)

def analytic_phase_amp(x, fs, f1, f2):
    xb = bandpass(x, fs, f1, f2)
    z  = signal.hilbert(xb)
    return np.angle(z), np.abs(z)

# ---------------- PAC (Tort MI) ----------------
def pac_mi_phase_amp(phase: np.ndarray, amp: np.ndarray, nbins: int = 18) -> float:
    edges = np.linspace(-np.pi, np.pi, nbins+1)
    digit = np.digitize(phase, edges) - 1
    digit = np.clip(digit, 0, nbins-1)
    m = np.zeros(nbins)
    for k in range(nbins):
        sel = (digit==k)
        m[k] = np.mean(amp[sel]) if np.any(sel) else 0.0
    if m.sum()<=0: return 0.0
    p = m / m.sum()
    eps=1e-12
    mi = np.sum(p*np.log((p+eps)/(1.0/nbins))) / np.log(nbins)
    return float(mi)

# ---------------- n:m phase locking ----------------
def n_m_plv(phi1: np.ndarray, phi2: np.ndarray, n: int = 1, m: int = 2) -> float:
    return float(np.abs(np.mean(np.exp(1j*(n*phi1 - m*phi2)))))

# ---------------- Comodulogram & PLV grids with nulls ----------------


"""
Patch: clamp all fast-frequency scans and plots to ≤ 60 Hz (or to Nyquist, if lower).

Why you were seeing >60 Hz: the original defaults used fast bands up to 80 Hz:
- PAC:   fast_range=(30, 80)
- n:m PLV: f2_range=(20, 80)

This patch parameterizes the high cutoff and clamps it to min(user_limit, 0.999*fs/2).
It also propagates the limit through plots so color maps never show >60 Hz.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# ---------------- PAC (Tort MI) ----------------
def comodulogram_pair(x_phase: np.ndarray, y_amp: np.ndarray, fs: float,
                      slow_range=(4,12), slow_bw=1.0, slow_step=1.0,
                      fast_range=(30,80), fast_bw=5.0, fast_step=2.0,
                      n_perm: int = 200,
                      max_fast_hz: float = 60.0):
    """Phase@x vs Amp@y PAC with an explicit upper clamp for fast frequencies.

    max_fast_hz: hard upper limit for fast band (default 60 Hz). The true upper
    limit applied is min(max_fast_hz, 0.999*fs/2, fast_range[1]).
    """
    fslow = np.arange(slow_range[0], slow_range[1] + 1e-6, slow_step)
    hi_cap = min(float(max_fast_hz), 0.999*float(fs)/2.0, float(fast_range[1]))
    if hi_cap <= fast_range[0] + 1e-9:
        raise ValueError(f"fast upper limit ({hi_cap:.2f} Hz) must exceed lower bound ({fast_range[0]} Hz)")
    ffast = np.arange(fast_range[0], hi_cap + 1e-6, fast_step)

    PAC = np.zeros((len(fslow), len(ffast)), float)

    def bandpass(x, f1, f2, order=4):
        ny = 0.5*fs
        f1 = max(1e-6, min(f1, 0.99*ny))
        f2 = max(f1 + 1e-6, min(f2, 0.999*ny))
        b, a = signal.butter(order, [f1/ny, f2/ny], btype='band')
        return signal.filtfilt(b, a, x)

    def analytic_phase_amp(x, f1, f2):
        xb = bandpass(x, f1, f2)
        z  = signal.hilbert(xb)
        return np.angle(z), np.abs(z)

    # compute PAC
    for i, f1 in enumerate(fslow):
        ph, _ = analytic_phase_amp(x_phase, f1 - slow_bw/2, f1 + slow_bw/2)
        for j, f2 in enumerate(ffast):
            _, amp = analytic_phase_amp(y_amp, f2 - fast_bw/2, f2 + fast_bw/2)
            # Tort MI (simple implementation)
            nbins = 18
            edges = np.linspace(-np.pi, np.pi, nbins+1)
            digit = np.digitize(ph, edges) - 1
            digit = np.clip(digit, 0, nbins-1)
            m = np.zeros(nbins)
            for k in range(nbins):
                sel = (digit == k)
                m[k] = np.mean(amp[sel]) if np.any(sel) else 0.0
            if m.sum() <= 0:
                PAC[i, j] = 0.0
            else:
                p = m / m.sum()
                eps = 1e-12
                PAC[i, j] = float(np.sum(p * np.log((p + eps) / (1.0/nbins))) / np.log(nbins))

    # null via circular shift of amp
    rng = np.random.default_rng(7)
    null95 = np.zeros_like(PAC)
    for i, f1 in enumerate(fslow):
        ph, _ = analytic_phase_amp(x_phase, f1 - slow_bw/2, f1 + slow_bw/2)
        for j, f2 in enumerate(ffast):
            _, amp = analytic_phase_amp(y_amp, f2 - fast_bw/2, f2 + fast_bw/2)
            maxima = []
            for _ in range(n_perm):
                s = int(rng.integers(1, len(amp)-1))
                amp_sh = np.r_[amp[-s:], amp[:-s]]
                # MI again
                nbins = 18
                edges = np.linspace(-np.pi, np.pi, nbins+1)
                digit = np.digitize(ph, edges) - 1
                digit = np.clip(digit, 0, nbins-1)
                m = np.zeros(nbins)
                for k in range(nbins):
                    sel = (digit == k)
                    m[k] = np.mean(amp_sh[sel]) if np.any(sel) else 0.0
                if m.sum() <= 0:
                    maxima.append(0.0)
                else:
                    p = m / m.sum()
                    eps = 1e-12
                    mi = float(np.sum(p * np.log((p + eps) / (1.0/nbins))) / np.log(nbins))
                    maxima.append(mi)
            null95[i, j] = float(np.nanpercentile(maxima, 95))

    return fslow, ffast, PAC, null95


# ---------------- n:m phase locking ----------------
def nm_plv_grid(x1: np.ndarray, x2: np.ndarray, fs: float,
                f1_range=(4,15), f1_bw=1.0, f1_step=1.0,
                f2_range=(20,80), f2_bw=2.0, f2_step=2.0,
                m_vals=(2,3,4,5,6), n: int = 1, n_perm: int = 200,
                max_f2_hz: float = 60.0):
    """Phase locking grid with an explicit upper clamp for f2 frequencies.

    max_f2_hz: hard upper limit for f2 (default 60 Hz). True upper limit is
    min(max_f2_hz, 0.999*fs/2, f2_range[1]).
    """
    f1s = np.arange(f1_range[0], f1_range[1] + 1e-6, f1_step)
    hi_cap = min(float(max_f2_hz), 0.999*float(fs)/2.0, float(f2_range[1]))
    if hi_cap <= f2_range[0] + 1e-9:
        raise ValueError(f"f2 upper limit ({hi_cap:.2f} Hz) must exceed lower bound ({f2_range[0]} Hz)")
    f2s = np.arange(f2_range[0], hi_cap + 1e-6, f2_step)

    def bandpass(x, f1, f2, order=4):
        ny = 0.5*fs
        f1 = max(1e-6, min(f1, 0.99*ny))
        f2 = max(f1 + 1e-6, min(f2, 0.999*ny))
        b, a = signal.butter(order, [f1/ny, f2/ny], btype='band')
        return signal.filtfilt(b, a, x)

    def analytic_phase(x, f1, f2):
        xb = bandpass(x, f1, f2)
        z  = signal.hilbert(xb)
        return np.angle(z)

    PLV = np.zeros((len(f1s), len(f2s), len(m_vals)), float)
    rng = np.random.default_rng(11)
    null95 = np.zeros_like(PLV)

    for i, f1 in enumerate(f1s):
        phi1 = analytic_phase(x1, f1 - f1_bw/2, f1 + f1_bw/2)
        for j, f2 in enumerate(f2s):
            phi2 = analytic_phase(x2, f2 - f2_bw/2, f2 + f2_bw/2)
            for k, m in enumerate(m_vals):
                plv = np.abs(np.mean(np.exp(1j * (n*phi1 - m*phi2))))
                PLV[i, j, k] = float(plv)
                # Null by circular phase shift of phi2
                maxima = []
                for _ in range(n_perm):
                    s = int(rng.integers(1, len(phi2)-1))
                    phi2_sh = np.r_[phi2[-s:], phi2[:-s]]
                    plv0 = np.abs(np.mean(np.exp(1j * (n*phi1 - m*phi2_sh))))
                    maxima.append(float(plv0))
                null95[i, j, k] = float(np.nanpercentile(maxima, 95))

    return f1s, f2s, m_vals, PLV, null95


# ---------------- Orchestrator (adds a high‑freq clamp knob) ----------------
def run_cfc_cross_region(
    RECORDS,
    pairs,
    ignition_windows=None,
    baseline_windows=None,
    time_col='Timestamp',
    out_dir='exports_cfc/session',
    show=False,
    n_perm=200,
    limit_high_hz: float = 60.0  # NEW: global clamp for fast/f2 frequencies
):
    """Same orchestrator as before, but enforces a ≤limit_high_hz scan in PAC/PLV.

    Note: plotting automatically respects the truncated frequency arrays, so the
    axes will end at ≤ limit_high_hz as well.
    """
    import os, pandas as pd
    from typing import Optional, List, Tuple

    def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

    def ensure_timestamp_column(df, time_col=None, default_fs: float = 128.0, out_name='Timestamp'):
        col = time_col if time_col is not None else 'Timestamp'
        if col not in df.columns:
            df[out_name] = np.arange(len(df), dtype=float)/default_fs; return out_name
        s = df[col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[out_name]=tsec.values; return out_name
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum() < max(50, 0.5*len(df)):
            df[out_name]=np.arange(len(df), dtype=float)/default_fs; return out_name
        sn = sn - np.nanmin(sn[np.isfinite(sn)])
        df[out_name]=sn.values; return out_name

    def infer_fs(df, time_col: str) -> float:
        t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
        dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
        if dt.size == 0:
            raise ValueError("Cannot infer fs.")
        return float(1.0/np.median(dt))

    def get_series(df, name: str) -> np.ndarray:
        if name in df.columns:
            return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
        alt = 'EEG.' + name
        if alt in df.columns:
            return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
        raise ValueError(f"{name} not found.")

    _ensure_dir(out_dir)
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)

    def zscore(x):
        x = np.asarray(x, float)
        return (x - np.nanmean(x)) / (np.nanstd(x) + 1e-12)

    results = {}
    rows = []

    def get_sig(name, wins):
        x = get_series(RECORDS, name)
        if not wins:
            return zscore(x)
        # slice & concat windows
        segs = []
        n = len(x)
        for (a,b) in wins:
            i0,i1 = int(round(a*fs)), int(round(b*fs))
            i0 = max(0, i0); i1 = min(n, i1)
            if i1 > i0:
                segs.append(x[i0:i1])
        return zscore(np.concatenate(segs) if segs else x)

    for st, wins in {'ignition': ignition_windows, 'baseline': baseline_windows}.items():
        if wins is None:
            continue
        for (ch_phase, ch_other) in pairs:
            a = ch_phase if ch_phase in RECORDS.columns else ('EEG.'+ch_phase if ('EEG.'+ch_phase) in RECORDS.columns else ch_phase)
            b = ch_other if ch_other in RECORDS.columns else ('EEG.'+ch_other if ('EEG.'+ch_other) in RECORDS.columns else ch_other)

            x = get_sig(a, wins)   # phase carrier
            y = get_sig(b, wins)   # amplitude or phase carrier

            # (1) PAC with clamp
            fslow, ffast, PAC, PAC_null95 = comodulogram_pair(
                x, y, fs,
                slow_range=(4,12), slow_bw=1.0, slow_step=1.0,
                fast_range=(30,80), fast_bw=5.0, fast_step=2.0,
                n_perm=n_perm,
                max_fast_hz=float(limit_high_hz)
            )

            imax = np.unravel_index(np.argmax(PAC), PAC.shape)
            pac_best = float(PAC[imax]); pac_thr = float(PAC_null95[imax])
            f1_best = float(fslow[imax[0]]); f2_best = float(ffast[imax[1]])
            pac_sig  = bool(pac_best > pac_thr)

            plt.figure(figsize=(7.8,3.2))
            extent=[ffast[0], ffast[-1], fslow[0], fslow[-1]]
            plt.imshow(PAC, aspect='auto', origin='lower', extent=extent, cmap='magma', vmin=0, vmax=np.nanmax(PAC))
            cb=plt.colorbar(); cb.set_label('PAC (MI)')
            sig_mask = PAC > PAC_null95
            yy, xx = np.where(sig_mask)
            if yy.size:
                plt.scatter(ffast[xx], fslow[yy], s=6, c='cyan', alpha=0.6, label='> null95')
            plt.xlabel('Fast freq (Hz)'); plt.ylabel('Slow freq (Hz)')
            plt.title(f'PAC: {a} phase → {b} amplitude — {st}')
            if yy.size: plt.legend(loc='upper right', fontsize=8)
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'pac_{a}_to_{b}_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

            # (2) n:m PLV with clamp
            f1s, f2s, m_vals, PLV, PLV_null95 = nm_plv_grid(
                x, y, fs,
                f1_range=(4,15), f1_bw=1.0, f1_step=1.0,
                f2_range=(20,80), f2_bw=2.0, f2_step=2.0,
                m_vals=(2,3,4,5,6), n=1, n_perm=n_perm,
                max_f2_hz=float(limit_high_hz)
            )

            imax_plv = np.unravel_index(np.argmax(PLV), PLV.shape)
            plv_best = float(PLV[imax_plv]); plv_thr = float(PLV_null95[imax_plv])
            f1_star = float(f1s[imax_plv[0]]); f2_star = float(f2s[imax_plv[1]]); m_star = int(m_vals[imax_plv[2]])
            plv_sig = bool(plv_best > plv_thr)

            k = imax_plv[2]
            plt.figure(figsize=(7.8,3.2))
            extent=[f2s[0], f2s[-1], f1s[0], f1s[-1]]
            plt.imshow(PLV[:,:,k], aspect='auto', origin='lower', extent=extent, cmap='viridis', vmin=0, vmax=np.nanmax(PLV[:,:,k]))
            cb=plt.colorbar(); cb.set_label(f'PLV (1:{m_vals[k]})')
            sig = PLV[:,:,k] > PLV_null95[:,:,k]
            yy, xx = np.where(sig)
            if yy.size:
                plt.scatter(f2s[xx], f1s[yy], s=6, c='cyan', alpha=0.6, label='> null95')
            plt.xlabel('f2 (Hz)'); plt.ylabel('f1 (Hz)')
            plt.title(f'n:m PLV (1:{m_vals[k]}): {a} ↔ {b} — {st}')
            if yy.size: plt.legend(loc='upper right', fontsize=8)
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'plv_{a}_to_{b}_m{m_vals[k]}_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

            rows.append({'state':st, 'pair':f'{a}->{b}',
                         'PAC_best':pac_best, 'PAC_thr95':pac_thr, 'PAC_sig':pac_sig,
                         'PAC_fslow':f1_best, 'PAC_ffast':f2_best,
                         'PLV_best':plv_best, 'PLV_thr95':plv_thr, 'PLV_sig':plv_sig,
                         'PLV_f1':f1_star, 'PLV_f2':f2_star, 'PLV_m':m_star})

            results.setdefault(st, {})[f'{a}->{b}'] = {
                'fslow':fslow, 'ffast':ffast, 'PAC':PAC, 'PAC_null95':PAC_null95,
                'f1s':f1s, 'f2s':f2s, 'm_vals':m_vals, 'PLV':PLV, 'PLV_null95':PLV_null95
            }

    summary = pd.DataFrame(rows)
    _ensure_dir(out_dir)
    summary.to_csv(os.path.join(out_dir,'cfc_summary.csv'), index=False)
    return {'summary': summary, 'results': results, 'out_dir': out_dir}


# ---------------- Usage example ----------------
# res = run_cfc_cross_region(
#     RECORDS,
#     pairs=[('EEG.F3','EEG.P8'), ('EEG.F4','EEG.P7')],
#     ignition_windows=[(290,310),(580,600)],
#     baseline_windows=[(0,290),(325,580)],
#     time_col='Timestamp',
#     out_dir='exports_cfc/S01',
#     show=False,
#     n_perm=200,
#     limit_high_hz=60.0  # ← clamp to ≤60 Hz
# )




In [ ]:
# Example pairs: frontal phase → parietal amplitude (and phase↔phase)
pairs = [
    ('EEG.F3',  'EEG.P8'),  # L frontal → R parietal
    ('EEG.F4',  'EEG.P7'),  # R frontal → L parietal
    ('EEG.F3',  'EEG.O2'),  # L frontal → R occipital
    ('EEG.F4',  'EEG.O1'),  # R frontal → L occipital

    ('EEG.FC5', 'EEG.P7'),  # L FC → L parietal
    ('EEG.FC6', 'EEG.P8'),  # R FC → R parietal
    ('EEG.FC5', 'EEG.O1'),  # L FC → L occipital
    ('EEG.FC6', 'EEG.O2'),  # R FC → R occipital
]

res = run_cfc_cross_region(
    RECORDS,
    pairs=pairs,
    ignition_windows=[(290,310),(580,600)],       # or None for full record
    baseline_windows=[(0,290),(325,580)],
    time_col='Timestamp',
    out_dir='exports_cfc/S01',
    show=True,
    n_perm=200,
    limit_high_hz=59.8
)
print(res['summary'])


In [ ]:
"""
Dynamic Connectivity & Metastability — Simple Graphs & Validation
=================================================================

What it does
------------
• Sliding-window connectivity (PLV or imag-coherency) → W(t).
• Global synchrony R(t) (Kuramoto) and mean edge weight M(t).
• Metastability = var(R(t)), with phase-randomized surrogates → p-value.
• PCA on vec(W(t)) → state-space trajectory; k-means (k=2..6) -> states.
• State metrics: dwell times, coverage, transition matrix, sequence entropy.
• Optional: correlate R(t) with 7.83 Hz SR envelope (shift-null).
• Per-state (Ignition/Baseline) outputs: PNGs + CSV summary in out_dir.

Usage
-----
res = run_dynamic_connectivity_metastability(
    RECORDS,
    eeg_channels=['EEG.O1','EEG.O2','EEG.P7','EEG.P8','EEG.FC5','EEG.FC6'],
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    band=(8,13),                   # choose band; e.g., theta (4–8), alpha (8–13)
    method='pli',                  # 'pli' (default) or 'imagcoh'
    win_sec=1.0, step_sec=0.25,    # sliding window params
    sr_channel=None,               # 'EEG.O1' if you want SR-proxy coupling analysis
    time_col='Timestamp',
    out_dir='exports_dyn/S01',
    show=False
)
print(res['summary'])
"""
from __future__ import annotations
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy import signal
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# ---------------- I/O & time helpers ----------------
def _ensure_dir(d): os.makedirs(d, exist_ok=True); return d

def detect_time_col(df,
    candidates=('Timestamp','Time','time','t','seconds','sec','ms','datetime','DateTime','Datetime')) -> Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    # numeric, roughly monotonic
    for c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum()>max(50,0.5*len(df)):
            x=s.values.astype(float); dt=np.diff(x[np.isfinite(x)])
            if dt.size and np.nanmedian(dt)>0: return c
    # datetime
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors='raise'); return c
        except Exception: pass
    return None

def ensure_timestamp_column(df: pd.DataFrame, time_col: Optional[str]=None, default_fs: float = 128.0, out_name='Timestamp')->str:
    col = time_col or detect_time_col(df)
    if col is None:
        df[out_name]=np.arange(len(df), dtype=float)/default_fs; return out_name
    s = df[col]
    if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
        tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
        df[out_name]=tsec.values; return out_name
    sn = pd.to_numeric(s, errors='coerce').astype(float)
    if sn.notna().sum()<max(50,0.5*len(df)):
        df[out_name]=np.arange(len(df), dtype=float)/default_fs; return out_name
    sn = sn - np.nanmin(sn[np.isfinite(sn)])
    df[out_name]=sn.values; return out_name

def infer_fs(df: pd.DataFrame, time_col: str)->float:
    t = np.asarray(pd.to_numeric(df[time_col], errors='coerce').values, float)
    dt=np.diff(t); dt=dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError("Cannot infer fs")
    return float(1.0/np.median(dt))

def get_series(df: pd.DataFrame, name: str)->np.ndarray:
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
    alt='EEG.'+name
    if alt in df.columns:
        return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
    raise ValueError(f"{name} not found.")

def slice_concat(x: np.ndarray, fs: float, wins: Optional[List[Tuple[float,float]]])->np.ndarray:
    if not wins: return x.copy()
    segs=[]; n=len(x)
    for (a,b) in wins:
        i0,i1=int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(n,i1)
        if i1>i0: segs.append(x[i0:i1])
    return np.concatenate(segs) if segs else x.copy()

def zscore(x): x=np.asarray(x,float); return (x-np.mean(x))/(np.std(x)+1e-12)

# ---------------- Filtering & analytic ----------------
def bandpass(x, fs, f1, f2, order=4):
    ny=0.5*fs; f1=max(1e-6,min(f1,0.99*ny)); f2=max(f1+1e-6,min(f2,0.999*ny))
    b,a=signal.butter(order,[f1/ny,f2/ny],btype='band'); return signal.filtfilt(b,a,x)

def analytic_phase(x, fs, f1, f2):
    xb = bandpass(x, fs, f1, f2)
    z  = signal.hilbert(xb)
    return np.angle(z)

# ---------------- Connectivity (PLV / imagcoh) ----------------
def pli_window(Xb: np.ndarray) -> np.ndarray:
    """PLI on analytic phases; Xb: (n_ch, W) bandpassed."""
    Z = signal.hilbert(Xb, axis=1); phi = np.angle(Z)
    n = Xb.shape[0]; W = np.zeros((n,n), float)
    for i in range(n):
        for j in range(i+1,n):
            dphi = phi[i]-phi[j]
            W[i,j]=W[j,i]=float(np.abs(np.mean(np.sign(np.sin(dphi)))))
    np.fill_diagonal(W,0.0); return W

def imagcoh_window(Xb: np.ndarray) -> np.ndarray:
    """Imag coherency; Xb: (n_ch, W) bandpassed."""
    Z = signal.hilbert(Xb, axis=1); n = Xb.shape[0]
    W = np.zeros((n,n), float)
    for i in range(n):
        for j in range(i+1,n):
            Sxy = np.mean(Z[i]*np.conj(Z[j]))
            Sxx = np.mean(Z[i]*np.conj(Z[i])); Syy = np.mean(Z[j]*np.conj(Z[j]))
            coh = Sxy/np.sqrt((Sxx*Syy)+1e-24)
            W[i,j]=W[j,i]=float(np.abs(np.imag(coh)))
    np.fill_diagonal(W,0.0); return W

# ---------------- Sliding windows ----------------
def sliding_windows(X: np.ndarray, fs: float, win_sec: float, step_sec: float):
    win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
    idx=[]
    for c in range(win//2, X.shape[1]-win//2, step):
        idx.append( (c-win//2, c+win//2, c/fs) )
    return idx

# ---------------- Kuramoto R ----------------
def kuramoto_R(Xb: np.ndarray) -> float:
    Z = signal.hilbert(Xb, axis=1); phi = np.angle(Z)
    R = np.abs(np.mean(np.exp(1j*phi), axis=0))
    return float(np.mean(R))  # mean over window samples as the window’s R

# ---------------- Surrogates ----------------
def phase_randomize(x: np.ndarray)->np.ndarray:
    X = np.fft.rfft(x); mag = np.abs(X); ph = np.angle(X)
    rnd = np.random.uniform(-np.pi, np.pi, size=mag.size)
    rnd[0] = ph[0]
    if mag.size % 2 == 0: rnd[-1] = ph[-1]
    Xs = mag * np.exp(1j*rnd)
    return np.fft.irfft(Xs, n=len(x)).astype(float)

def build_surrogate_matrix(X: np.ndarray) -> np.ndarray:
    return np.vstack([zscore(phase_randomize(x)) for x in X])

# ---------------- Main runner ----------------
def run_dynamic_connectivity_metastability(
    RECORDS: pd.DataFrame,
    eeg_channels: List[str],
    ignition_windows: Optional[List[Tuple[float,float]]] = None,
    baseline_windows: Optional[List[Tuple[float,float]]] = None,
    band: Tuple[float,float] = (8,13),
    method: str = 'pli',        # 'pli' or 'imagcoh'
    win_sec: float = 1.0, step_sec: float = 0.25,
    sr_channel: Optional[str] = None,   # if provided, compute corr(R) with SR env @7.83 (shift-null)
    time_col: str = 'Timestamp',
    out_dir: str = 'exports_dyn/session',
    show: bool = False,
    n_surrogates: int = 200
)->Dict[str, object]:
    """
    Dynamic connectivity & metastability with simple graphs and tests.
    """
    _ensure_dir(out_dir)
    time_col = ensure_timestamp_column(RECORDS, time_col=time_col, default_fs=128.0)
    fs = infer_fs(RECORDS, time_col)

    def build_state_matrix(wins):
        X=[]; names=[]
        for ch in eeg_channels:
            nm = ch if ch in RECORDS.columns else ('EEG.'+ch if ('EEG.'+ch) in RECORDS.columns else ch)
            if nm in RECORDS.columns:
                x = get_series(RECORDS, nm); x = slice_concat(x, fs, wins)
                X.append(zscore(np.asarray(x,float))); names.append(nm)
        if not X: raise ValueError("No EEG channels found.")
        L = min(len(x) for x in X)
        return np.vstack([x[:L] for x in X]), names  # (n, T)

    def dyn_conn_for_state(X: np.ndarray, names: List[str], label: str):
        idx = sliding_windows(X, fs, win_sec, step_sec)
        if not idx: 
            raise ValueError("No sliding windows — extend windows or reduce win_sec.")
        Fvec=[]; Rts=[]; Mts=[]; tcent=[]
        for (s,e,t) in idx:
            Xw = X[:, s:e]
            # band-limit
            Xb = np.vstack([bandpass(x, fs, band[0], band[1]) for x in Xw])
            W = pli_window(Xb) if method.lower()=='pli' else imagcoh_window(Xb)
            # features & metrics
            ut = W[np.triu_indices(X.shape[0],1)]
            Fvec.append(ut)
            Rts.append(kuramoto_R(Xb))
            Mts.append(np.mean(ut))
            tcent.append(t)
        Fvec = np.vstack(Fvec)            # (T', E)
        Rts  = np.array(Rts, float)
        Mts  = np.array(Mts, float)
        tcent= np.array(tcent, float)

        # metastability
        meta = float(np.var(Rts))

        # surrogates for metastability (phase-randomize channels independently)
        null_meta=[]
        for _ in range(n_surrogates):
            Xs = build_surrogate_matrix(X)
            Rs=[]
            for (s,e,_) in idx:
                Xsw = Xs[:, s:e]
                Xsb = np.vstack([bandpass(x, fs, band[0], band[1]) for x in Xsw])
                Rs.append(kuramoto_R(Xsb))
            null_meta.append(np.var(Rs))
        null_meta = np.asarray(null_meta, float)
        meta_p = float((np.sum(null_meta >= meta)+1)/(n_surrogates+1))

        # PCA & clustering of connectivity states
        pca = PCA(n_components=2, random_state=0)
        Z = pca.fit_transform(Fvec)       # (T', 2)
        # choose k by silhouette (2..6)
        best_k, best_s, best_lab = 2, -np.inf, None
        for k in range(2,7):
            km = KMeans(n_clusters=k, n_init=20, random_state=0).fit(Z)
            lab = km.labels_
            if len(np.unique(lab))<2: continue
            s = silhouette_score(Z, lab)
            if s > best_s:
                best_s, best_k, best_lab = s, k, lab
        states = best_lab if best_lab is not None else KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Z)
        k_opt  = best_k

        # dwell times & transitions
        Tprime = len(states)
        cov = np.array([np.mean(states==i) for i in range(k_opt)])
        # dwell (in windows); convert to seconds
        dwell=[]
        i=0
        while i<Tprime:
            j=i
            while j+1<Tprime and states[j+1]==states[i]:
                j+=1
            dwell.append((states[i], (j-i+1)*step_sec))
            i=j+1
        dwell_mean = np.zeros(k_opt)
        for s in range(k_opt):
            ds=[d for (lab,d) in dwell if lab==s]
            dwell_mean[s]=np.mean(ds) if ds else np.nan

        trans = np.zeros((k_opt,k_opt), float)
        for t in range(Tprime-1):
            a,b = states[t], states[t+1]
            if a!=b: trans[a,b]+=1
        trans = trans/(trans.sum(axis=1, keepdims=True)+1e-12)
        # sequence entropy
        p = cov[cov>0]; H = float(-np.sum(p*np.log2(p)))

        # plots: R(t) with null band
        plt.figure(figsize=(9,3))
        zR = (Rts - np.mean(Rts))/(np.std(Rts)+1e-12)
        null95 = np.nanpercentile((null_meta - null_meta.mean())/(null_meta.std()+1e-12), 95) if null_meta.size else np.nan
        plt.plot(tcent, zR, lw=1.4, label='z-R(t)')
        if np.isfinite(null95): plt.hlines(null95, tcent[0], tcent[-1], colors='k', linestyles='--', label='null95')
        plt.xlabel('Time (s)'); plt.ylabel('z-R'); plt.title(f'Global synchrony (metastability={meta:.3f}, p={meta_p:.3f}) — {label}')
        plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'R_timeseries_{label}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # PCA trajectory colored by state
        plt.figure(figsize=(6,5))
        sc = plt.scatter(Z[:,0], Z[:,1], c=states, s=10, cmap='tab20')
        plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(f'Connectivity state-space (k={k_opt}, sil={best_s:.2f}) — {label}')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'pca_states_{label}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        # Transition matrix heatmap
        plt.figure(figsize=(4.2,3.2))
        vmax = np.nanmax(trans) if np.isfinite(np.nanmax(trans)) else 1.0
        plt.imshow(trans, vmin=0, vmax=vmax, cmap='magma'); plt.colorbar(label='P(i→j)')
        plt.xticks(range(k_opt), [f'S{k+1}' for k in range(k_opt)]); plt.yticks(range(k_opt), [f'S{k+1}' for k in range(k_opt)])
        plt.title(f'Transitions — {label}'); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'transitions_{label}.png'), dpi=140)
        if show: plt.show()
        plt.close()

        return {
            'R': Rts, 'M': Mts, 't': tcent,
            'meta': meta, 'meta_p': meta_p, 'null_meta': null_meta,
            'Z': Z, 'states': states, 'k_opt': k_opt, 'silhouette': best_s,
            'coverage': cov, 'dwell_mean_sec': dwell_mean, 'transitions': trans
        }

    # Build per-state matrices & run dynamics
    summary_rows=[]
    results={}
    for st, wins in {'ignition': ignition_windows, 'baseline': baseline_windows}.items():
        if wins is None: continue
        X, names = build_state_matrix(wins)
        # dynamic metrics
        res = dyn_conn_for_state(X, names, st)
        results[st] = res
        summary_rows.append({'state': st, 'meta_varR': res['meta'], 'meta_p': res['meta_p'],
                             'k_opt': res['k_opt'], 'silhouette': res['silhouette'],
                             'coverage_mean': float(np.nanmean(res['coverage'])),
                             'dwell_mean_sec_mean': float(np.nanmean(res['dwell_mean_sec']))})

        # optional SR coupling: corr(R(t), SR_env@7.83)
        if sr_channel is not None:
            sr = get_series(RECORDS, sr_channel)
            sr = slice_concat(sr, fs, wins)
            # 7.83 ± 0.6 Hz envelope
            ny=0.5*fs; b,a=signal.butter(4, [max(1e-6,(7.83-0.6))/ny, min(0.999,(7.83+0.6))/ny], btype='band')
            env = np.abs(signal.hilbert(signal.filtfilt(b,a, sr)))
            # sample env per window center
            env_w = np.interp(res['t'], np.arange(len(env))/fs, env)
            r = float(np.corrcoef((res['R']-np.mean(res['R']))/(np.std(res['R'])+1e-12),
                                  (env_w -np.mean(env_w))/ (np.std(env_w)+1e-12))[0,1])
            # shift-null
            rng=np.random.default_rng(13); null=[]
            for _ in range(n_surrogates):
                s=int(rng.integers(1, len(env_w)-1))
                null.append(np.corrcoef(res['R'], np.r_[env_w[-s:], env_w[:-s]])[0,1])
            thr95=float(np.nanpercentile(null,95))
            summary_rows[-1].update({'R_env_r': r, 'R_env_null95': thr95})

            plt.figure(figsize=(9,3))
            zR = (res['R'] - np.mean(res['R']))/(np.std(res['R'])+1e-12)
            zE = (env_w - np.mean(env_w))/(np.std(env_w)+1e-12)
            plt.plot(res['t'], zR, label='z-R(t)')
            plt.plot(res['t'], zE, label='z-SR env')
            plt.xlabel('Time (s)'); plt.title(f'R(t) vs SR envelope (r={r:.2f}, null95~{thr95:.2f}) — {st}')
            plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'R_vs_SR_{st}.png'), dpi=140)
            if show: plt.show()
            plt.close()

    # Ignition vs Baseline summary bar
    if 'ignition' in results and 'baseline' in results:
        ig, ba = results['ignition'], results['baseline']
        plt.figure(figsize=(6,3.2))
        keys = ['meta_varR','dwell_mean_sec_mean']
        x=np.arange(len(keys)); w=0.38
        vals_ig=[r for r in [np.var(ig['R']), np.nanmean(ig['dwell_mean_sec'])]]
        vals_ba=[r for r in [np.var(ba['R']), np.nanmean(ba['dwell_mean_sec'])]]
        plt.bar(x-w/2, vals_ba, width=w, label='Baseline', color='tab:orange', alpha=0.9)
        plt.bar(x+w/2, vals_ig, width=w, label='Ignition', color='tab:blue', alpha=0.9)
        plt.xticks(x, ['var(R)','dwell mean (s)']); plt.ylabel('value')
        plt.title('Ignition vs Baseline — metastability & dwell')
        plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'ign_vs_base.png'), dpi=140)
        if show: plt.show()
        plt.close()

    # Save CSV summary
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir,'summary.csv'), index=False)
    return {'summary': summary_df, 'results': results, 'out_dir': out_dir}


In [ ]:
# Clean, symmetric 6–10 channels work best (posterior-heavy reduces artifacts)
eeg_channels=['EEG.AF3','EEG.AF4','EEG.F3','EEG.F7','EEG.F8','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2']
res = run_dynamic_connectivity_metastability(
    RECORDS,
    eeg_channels=eeg_channels,
    ignition_windows=[(290,310),(580,600)],
    baseline_windows=[(0,290),(325,580)],
    band=(8,13),                 # try (4,8) too
    method='pli',                # or 'imagcoh'
    win_sec=1.0, step_sec=0.25,
    sr_channel="EEG.F4",             # set to 'EEG.O1' or magnetometer to enable R vs SR plot
    time_col='Timestamp',
    out_dir='exports_dyn/S01',
    show=True,
    n_surrogates=200
)
print(res['summary'])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- helpers ----------
def _auto_savgol(y, max_window=31):
    """Light, safe smoothing for visibility (odd window; poly=2)."""
    y = np.asarray(y, float)
    n = y.size
    if n < 7:
        return y
    try:
        from scipy.signal import savgol_filter
        w = max(5, min(max_window, int(round(n/15))))
        if w % 2 == 0:
            w += 1
        if w >= n:
            w = n - 1 if (n % 2 == 0) else n
        if w < 5:
            return y
        return savgol_filter(y, w, polyorder=2, mode='interp')
    except Exception:
        # moving average fallback
        w = max(5, min(max_window, int(round(n/15))))
        if w < 2:
            return y
        return np.convolve(np.nan_to_num(y), np.ones(w)/w, mode='same')

def _clip_shading(ax, windows, tmin, tmax, **kwargs):
    if not windows:
        return
    for (t0, t1) in windows:
        if t1 < tmin or t0 > tmax:
            continue
        ax.axvspan(max(t0, tmin), min(t1, tmax), **kwargs)

# ---------- main ----------

def plot_sr_ignition_signature(records,
                               eeg_channel: str,            # e.g., 'EEG.O1'
                               sr_channel: str,             # magnetometer or posterior proxy
                               ignition_windows,            # [(t0,t1), ...] in *absolute seconds*
                               time_col='Timestamp',
                               harmonics=(7.83, 14.3, 20.8, 27.3, 33.8),
                               half_bw=0.6, win_sec=8.0, step_sec=1.0,
                               n_null=200, out_png='sr_ignition_signature.png',
                               *,
                               facet=False,                 # one subplot per harmonic
                               direct_labels=True,          # write labels at line ends
                               legend_outside=True,         # legend outside if used
                               smooth=False,                # light Savitzky–Golay smoothing
                               linewidth=1.8,
                               grid=True,
                               clip_to_data=True,
                               figsize_overlay=(10, 4.6),
                               figsize_facet=(10, 8),
                               dpi=160,
                               return_fig_ax=False):
    """
    Enhanced sliding-coherence plotter: clearer lines, non-overlapping labels/legend,
    faceting option, shaded ignition clipped to data range, and optional smoothing.

    Notes
    -----
    - Assumes `sliding_coherence_f0` returns dict with keys: 't' (sec), 'coh', 'null95'.
    - If your `sliding_coherence_f0` uses *absolute seconds* for 't', shaded windows
      will align. If your 't' is relative to a slice, align your windows accordingly.
    """
    
    
    
    # 1) compute a sliding coherence trace for each harmonic
    traces = []
    for f0 in harmonics:
        sl = sliding_coherence_f0(
            records, eeg_channel, sr_channel,
            f0=f0, half=half_bw, time_col=time_col,
            win_sec=win_sec, step_sec=step_sec, n_null=n_null, show=False
        )
        if sl['t'] is None or len(sl['t']) == 0:
            continue
        coh_raw = sl['coh']
        if smooth:
            # Use the SAME callable you use to smooth the plotted trace
            smoother_fn = _auto_savgol # <-- callable, not a bool
            coh_plot = smoother_fn(coh_raw)
            thr95_s = build_null_threshold_smooth(coh_raw, n_null=n_null, method='block',smoother=smoother_fn)
            zcoh = (coh_plot - np.nanmean(coh_plot)) / (np.nanstd(coh_plot) + 1e-12)
            zthr = zscore_with_series(thr95_s, coh_plot)
        else:
            smoother_fn = None
            coh_plot = coh_raw
            thr95 = build_null_threshold_smooth(coh_raw, n_null=n_null, method='block',smoother=None)
            zcoh = (coh_plot - np.nanmean(coh_plot)) / (np.nanstd(coh_plot) + 1e-12)
            zthr = zscore_with_series(thr95, coh_plot)
        traces.append({'f0': f0, 't': np.asarray(sl['t']), 'zcoh': zcoh, 'zthr': zthr})

    if not traces:
        raise ValueError('No coherence points to plot — check inputs/windows')

    # common time extent across harmonics (for shading and xlim)
    t_all = np.concatenate([tr['t'] for tr in traces])
    tmin, tmax = float(np.nanmin(t_all)), float(np.nanmax(t_all))

    colors = plt.get_cmap('tab10').colors
    null_alpha = 0.35
    
    
    coh_raw = sl['coh']
    if smooth:
        # Use the SAME callable you use to smooth the plotted trace
        smoother_fn = _auto_savgol # <-- callable, not a bool
        coh_plot = smoother_fn(coh_raw)
        thr95_s = build_null_threshold_smooth(coh_raw, n_null=n_null, method='block',smoother=smoother_fn)
        zcoh = (coh_plot - np.nanmean(coh_plot)) / (np.nanstd(coh_plot) + 1e-12)
        zthr = zscore_with_series(thr95_s, coh_plot)
    else:
        smoother_fn = None
        coh_plot = coh_raw
        thr95 = build_null_threshold_smooth(coh_raw, n_null=n_null, method='block',smoother=None)
        zcoh = (coh_plot - np.nanmean(coh_plot)) / (np.nanstd(coh_plot) + 1e-12)
        zthr = zscore_with_series(thr95, coh_plot)

    # ---------- Faceted mode ----------
    if facet:
        fig, axes = plt.subplots(len(traces), 1, sharex=True, sharey=True,
                                 figsize=figsize_facet)
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        for i, (ax, tr) in enumerate(zip(axes, traces)):
            col = colors[i % len(colors)]
            ax.plot(tr['t'], tr['zcoh'], lw=linewidth, color=col)
            ax.hlines(tr['zthr'], tmin, tmax, colors=col, linestyles='--', lw=1.0, alpha=null_alpha)
            _clip_shading(ax, ignition_windows, tmin, tmax, color='k', alpha=0.08, zorder=0)
            if grid:
                ax.grid(True, alpha=1, linestyle=':')
            ax.set_ylabel(f"{tr['f0']:.2f} Hz")
        axes[-1].set_xlabel('Time (s)')
        fig.suptitle('EEG–SR sliding coherence at Schumann harmonics (faceted)', y=0.995)
        fig.tight_layout()
        if out_png:
            fig.savefig(out_png, dpi=dpi, bbox_inches='tight')
        if return_fig_ax:
            return fig, axes
        plt.show()
        return

    # ---------- Overlay mode ----------
    fig, ax = plt.subplots(figsize=figsize_overlay)

    lines = []
    for i, tr in enumerate(traces):
        col = colors[i % len(colors)]
        ln, = ax.plot(tr['t'], tr['zcoh'], lw=linewidth, color=col, label=f"{tr['f0']:.2f} Hz")
        lines.append((ln, tr))
        ax.hlines(tr['zthr'], tmin, tmax, colors=col, linestyles='--', lw=1.0, alpha=null_alpha)

    _clip_shading(ax, ignition_windows, tmin, tmax, color='k', alpha=0.08, zorder=0)

    if grid:
        ax.grid(True, alpha=0.25, linestyle=':')

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('z-coherence')
    ax.set_title('EEG–SR sliding coherence at Schumann harmonics (shaded = ignition)')

    # tidy legend/labels
    if clip_to_data:
        ax.set_xlim(tmin, tmax)

    used_legend = False
    if direct_labels:
        # label at the right-most finite sample for each line
        xpad = 0.01 * (tmax - tmin)
        for ln, tr in lines:
            x = tr['t']; y = tr['zcoh']
            idx = np.where(np.isfinite(y))[0]
            if idx.size == 0:
                continue
            j = idx[-1]
            ax.text(x[j] + xpad, y[j], f"{tr['f0']:.2f}", color=ln.get_color(),
                    fontsize=9, va='center', ha='left', clip_on=False)
    else:
        if legend_outside:
            ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, frameon=False)
            fig.subplots_adjust(right=0.78)
        else:
            ax.legend(loc='lower right', frameon=False)
        used_legend = True

    fig.tight_layout()
    if out_png:
        fig.savefig(out_png, dpi=dpi, bbox_inches='tight')
    if return_fig_ax:
        return fig, ax
    plt.show()


In [ ]:
import numpy as np
from scipy import signal

def compute_coherence_at_f0(xe, xs, fs, f0, half):
    """
    Magnitude-squared coherence between signals xe and xs at target frequency f0.

    Uses Welch autospectra (Pxx, Pyy) and cross-spectrum (Pxy) on the provided
    windowed segments, then returns a scalar coherence value aggregated within
    the band [f0 - half, f0 + half]. If that band contains no frequency bin,
    returns the coherence at the nearest available bin to f0.

    Parameters
    ----------
    xe : array_like
        First signal segment (e.g., EEG), 1-D.
    xs : array_like
        Second signal segment (e.g., SR proxy / magnetometer), 1-D.
    fs : float
        Sampling rate in Hz.
    f0 : float
        Target center frequency in Hz.
    half : float
        Half-bandwidth in Hz; analyze [f0 - half, f0 + half].

    Returns
    -------
    float
        Coherence (0..1) at/around f0.
    """
    xe = np.asarray(xe, dtype=float)
    xs = np.asarray(xs, dtype=float)
    N = int(min(len(xe), len(xs)))
    if N < 32:
        raise ValueError("Window too short for coherence (N < 32 samples)")
    xe = xe[:N]
    xs = xs[:N]

    # Choose segment length for spectral estimates: use half the window (typical)
    nperseg = int(max(32, min(N, N // 2)))
    noverlap = int(nperseg // 2)

    # Autospectra and cross-spectrum (Welch / CSD)
    f, Pxx = signal.welch(
        xe, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        detrend='constant', return_onesided=True, scaling='density'
    )
    _, Pyy = signal.welch(
        xs, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        detrend='constant', return_onesided=True, scaling='density'
    )
    _, Pxy = signal.csd(
        xe, xs, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        detrend='constant', return_onesided=True, scaling='density'
    )

    # Magnitude-squared coherence
    eps = 1e-20
    Cxy = (np.abs(Pxy) ** 2) / (Pxx * Pyy + eps)

    # Aggregate within the f0 ± half band; fallback to nearest bin if empty
    f_lo = max(0.0, float(f0) - float(half))
    f_hi = float(f0) + float(half)
    band = (f >= f_lo) & (f <= f_hi) & np.isfinite(Cxy)
    if np.any(band):
        c_val = float(np.nanmean(Cxy[band]))
    else:
        idx = int(np.argmin(np.abs(f - float(f0))))
        c_val = float(Cxy[idx]) if np.isfinite(Cxy[idx]) else float('nan')

    # Clamp numeric noise into [0, 1]
    if np.isfinite(c_val):
        c_val = float(np.clip(c_val, 0.0, 1.0))
    return c_val



def build_null_threshold(coh, n_null=200, method='block', block_len=None, alpha=0.05, random_state=13):
    """
    Estimate a null threshold for a sliding coherence trace by resampling the
    coherence sequence itself.

    For each surrogate, we create a bootstrap replica of the coherence
    time-series and take its maximum. The (1-alpha) percentile of these maxima
    is returned. By default, we use a block-bootstrap that preserves local
    autocorrelation structure; set method='iid' for simple i.i.d. resampling.

    Parameters
    ----------
    coh : array_like
        Coherence values (0..1) across sliding-window centers; NaNs allowed.
    n_null : int, optional
        Number of surrogate replicates (default 200).
    method : {'block','iid'}, optional
        Resampling strategy. 'block' preserves short-range correlations.
    block_len : int or None, optional
        Block length (in samples) for block-bootstrap. If None, uses ~5% of N
        (at least 5 samples).
    alpha : float, optional
        Significance level (default 0.05 → 95th percentile).
    random_state : int, optional
        RNG seed for reproducibility.

    Returns
    -------
    float
        Estimated (1 - alpha) percentile of the null maxima; clipped to [0, 1].
    """
    import numpy as np

    c = np.asarray(coh, dtype=float)
    # Keep only finite values
    c = c[np.isfinite(c)]
    if c.size == 0:
        return float('nan')

    rng = np.random.default_rng(random_state)
    N = int(c.size)

    maxima = []
    if method == 'iid':
        for _ in range(int(n_null)):
            samp = rng.choice(c, size=N, replace=True)
            maxima.append(float(np.nanmax(samp)))
    else:
        # Block bootstrap with circular wrap to preserve local structure
        if block_len is None:
            block_len = max(5, int(round(N / 20)))  # ~5% of the series
        B = int(block_len)
        for _ in range(int(n_null)):
            idx = []
            filled = 0
            while filled < N:
                start = int(rng.integers(0, N))
                end = start + B
                if end <= N:
                    idx.extend(range(start, end))
                else:
                    # circular wrap
                    idx.extend(list(range(start, N)) + list(range(0, end - N)))
                filled += B
            idx = np.asarray(idx[:N], dtype=int)
            surrogate = c[idx]
            maxima.append(float(np.nanmax(surrogate)))

    q = 100.0 * (1.0 - float(alpha))
    thr = float(np.nanpercentile(maxima, q))
    # numeric safety
    return float(np.clip(thr, 0.0, 1.0))


In [ ]:
import numpy as np

def build_null_threshold_smooth(coh_raw, n_null=200, method='block', block_len=None,
                                alpha=0.05, random_state=13, smoother=None):
    """
    Compute a (1-alpha) null threshold compatible with *smoothed* plotting.

    We generate surrogate coherence sequences from the *raw* coherence trace
    (coh_raw) via block bootstrap (or IID), then optionally apply the same
    smoothing used for the plotted trace to each surrogate before taking the
    maximum. This keeps the null line comparable to what you actually plot.

    Parameters
    ----------
    coh_raw : array_like
        UnsMoothed coherence values (0..1) across time.
    n_null : int
        Number of surrogate replicates.
    method : {'block','iid'}
        Resampling strategy for temporal dependence.
    block_len : int or None
        Block length in samples for block bootstrap. If None, ~5% of N (>=5).
    alpha : float
        Significance level (default 0.05 → 95th percentile of maxima).
    random_state : int
        RNG seed.
    smoother : callable or None
        A function y -> y_s that applies the *same* smoothing as used on the
        plotted trace. If None, no smoothing is applied to surrogates.

    Returns
    -------
    thr95_s : float
        Null threshold after applying `smoother` to surrogates (if provided).
    """
    c = np.asarray(coh_raw, float)
    c = c[np.isfinite(c)]
    if c.size == 0:
        return float('nan')

    rng = np.random.default_rng(random_state)
    N = int(c.size)

    maxima = []
    if method == 'iid':
        for _ in range(int(n_null)):
            samp = rng.choice(c, size=N, replace=True)
            if smoother is not None:
                samp = np.asarray(smoother(samp), float)
            maxima.append(float(np.nanmax(samp)))
    else:
        if block_len is None:
            block_len = max(5, int(round(N / 20)))
        B = int(block_len)
        for _ in range(int(n_null)):
            idx = []
            filled = 0
            while filled < N:
                start = int(rng.integers(0, N))
                end = start + B
                if end <= N:
                    idx.extend(range(start, end))
                else:
                    idx.extend(list(range(start, N)) + list(range(0, end - N)))
                filled += B
            idx = np.asarray(idx[:N], int)
            surrogate = c[idx]
            if smoother is not None:
                surrogate = np.asarray(smoother(surrogate), float)
            maxima.append(float(np.nanmax(surrogate)))

    q = 100.0 * (1.0 - float(alpha))
    thr95_s = float(np.nanpercentile(maxima, q))
    return float(np.clip(thr95_s, 0.0, 1.0))


def zscore_with_series(value, series, eps=1e-12):
    """Convert a scalar threshold `value` into z-units using the mean/std of `series`."""
    series = np.asarray(series, float)
    m = float(np.nanmean(series))
    s = float(np.nanstd(series)) + eps
    return float((float(value) - m) / s)

# ---- Example wiring inside your plotting code ----
# coh_raw = sl['coh']
# if smooth:
#     coh_plot = _auto_savgol(coh_raw)
#     thr95_s = build_null_threshold_smooth(coh_raw, n_null=n_null, method='block',
#                                           smoother=_auto_savgol)
#     zcoh = (coh_plot - np.nanmean(coh_plot)) / (np.nanstd(coh_plot) + 1e-12)
#     zthr = zscore_with_series(thr95_s, coh_plot)
# else:
#     zcoh = (coh_raw - np.nanmean(coh_raw)) / (np.nanstd(coh_raw) + 1e-12)
#     thr95 = build_null_threshold_smooth(coh_raw, n_null=n_null, method='block', smoother=None)
#     zthr = zscore_with_series(thr95, coh_raw)


In [ ]:
plot_sr_ignition_signature(RECORDS,
    eeg_channel="EEG.O1",            # e.g., 'EEG.O1' or a robust posterior
    sr_channel="EEG.F4",             # magnetometer or posterior proxy
    ignition_windows=[(180,200),(280,300),(430,450),(560,580),(618,638)],            # [(t0,t1), ...]
    time_col='Timestamp',
    harmonics=(7.83, 14.3, 20.8, 27.3, 33.8),
    half_bw=0.6, win_sec=20, step_sec=1,                       
    smooth=True,facet=False,legend_outside=False,direct_labels=False,figsize_overlay=(14, 8),
    n_null=200, out_png='sr_ignition_signature.png')

In [ ]:
plot_sr_ignition_signature(RECORDS,
    eeg_channel="EEG.O1",            # e.g., 'EEG.O1' or a robust posterior
    sr_channel="EEG.F4",             # magnetometer or posterior proxy
    ignition_windows=[(180,200),(280,300),(430,450),(560,580),(618,638)],            # [(t0,t1), ...]
    time_col='Timestamp',
    harmonics=(7.8,40.3,46.8,53.3,59.8),
    half_bw=0.6, win_sec=20, step_sec=1,
    smooth=True,facet=False,legend_outside=False,direct_labels=False,figsize_overlay=(14, 8),
    n_null=200, out_png='sr_ignition_signature.png')

In [ ]:
plot_sr_ignition_signature(RECORDS,
    eeg_channel="EEG.O1",            # e.g., 'EEG.O1' or a robust posterior
    sr_channel="EEG.F4",             # magnetometer or posterior proxy
    ignition_windows=[(180,200),(280,300),(430,450),(560,580),(618,638)],            # [(t0,t1), ...]
    time_col='Timestamp',
    harmonics=(7.83,3.915,2.61,1.9575,1.566),
    half_bw=0.6, win_sec=20, step_sec=1,
    smooth=True,facet=False,legend_outside=False,direct_labels=False,figsize_overlay=(14, 8),
    n_null=200, out_png='sr_ignition_signature.png')

In [ ]:
plot_sr_ignition_signature(RECORDS,
    eeg_channel="EEG.O1",            # e.g., 'EEG.O1' or a robust posterior
    sr_channel="EEG.F4",             # magnetometer or posterior proxy
    ignition_windows=[(180,200),(280,300),(430,450),(560,580),(618,638)],            # [(t0,t1), ...]
    time_col='Timestamp',
    harmonics=(1.305,1.11857,0.9788,1.2,0.783),
    half_bw=0.6, win_sec=10, step_sec=2,
    smooth=True,facet=False,legend_outside=False,direct_labels=False,figsize_overlay=(14, 8),
    n_null=200, out_png='sr_ignition_signature.png')

In [ ]:
# SR harmonic groups supplied by user + helpers to analyze/plot each group
# Requires: numpy, matplotlib, and your existing sliding_coherence_f0(...)
# Optional (already in your notebook/canvas): build_null_threshold, build_null_threshold_smooth, _auto_savgol

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# 1) Group definitions (exactly as provided)
# ----------------------------

def sr_groups():
    return {
        # H1–H5
        'Harmonics_UpTo33': (7.83, 14.3, 20.8, 27.3, 33.8),
        # H6–H10 (mind 60 Hz mains; you may cap at 58 Hz in plots)
        'HighHarmonics_40to60': (7.83, 40.3, 46.8, 53.3, 59.8),
        # Subharmonics /2..5
        'Subharmonics_2to5': (7.83, 3.915, 2.61, 1.9575, 1.566),
        # Subharmonics /6..10 (+ 1.2 as given)
        'Subharmonics_6to10_Mixed': (7.83, 1.305, 1.11857, 0.9788, 1.2, 0.783),
    }

# ----------------------------
# 2) Adaptive parameter rules per f0
# ----------------------------

def win_for_f0(f0, cycles=8, min_win=8.0, max_win=120.0):
    """Adaptive window: ensure ≥cycles of f0, clipped to [min_win, max_win]."""
    if f0 <= 1e-9:
        return max_win
    return float(np.clip(cycles/float(f0), min_win, max_win))

def half_bw_for_win(win_sec, mult=2.5, min_bw=0.1):
    """Choose half-bandwidth from spectral resolution Δf≈2/win_sec, scaled by mult."""
    df = 2.0 / max(win_sec, 1e-6)
    return float(max(min_bw, mult*df/2.0))  # ≈ mult * (Δf/2)

# Fallbacks if the smoother/null helpers aren't in scope

def _maybe_smoother():
    try:
        return _auto_savgol  # defined in your plotting utils
    except Exception:
        return None

def _build_null_for_series(coh_raw, n_null=200, smoother=None):
    try:
        return build_null_threshold_smooth(coh_raw, n_null=n_null, smoother=smoother)
    except Exception:
        return build_null_threshold(coh_raw, n_null=n_null)

# ----------------------------
# 3) Plot a single group with adaptive windows and smooth-aware nulls
# ----------------------------

def plot_sr_group_adaptive(records,
                            eeg_channel: str,
                            sr_channel: str,
                            group_name: str,
                            ignition_windows=None,
                            time_col='Timestamp',
                            cycles=8, min_win=8.0, max_win=120.0,
                            step_sec=1.0, n_null=200, smooth=True,
                            facet=True, linewidth=1.8, grid=True,
                            limit_high_hz=59.8,
                            out_png=None, dpi=160):
    """
    Compute & plot sliding z-coherence for all f0 in the chosen group with
    frequency-adaptive window lengths and (if available) smooth-aware nulls.
    """
    GROUPS = sr_groups()
    if group_name not in GROUPS:
        raise ValueError(f"Unknown group '{group_name}'. Available: {list(GROUPS)}")

    f0s = [f for f in GROUPS[group_name] if (f <= float(limit_high_hz) + 1e-9)]
    if not f0s:
        raise ValueError("No frequencies ≤ limit_high_hz in this group.")

    traces = []
    smoother_fn = _maybe_smoother() if smooth else None

    # Try to detect whether sliding_coherence_f0 accepts 'wins' kw
    import inspect
    sig = inspect.signature(sliding_coherence_f0)
    has_wins = ('wins' in sig.parameters)

    for f0 in f0s:
        win_sec = win_for_f0(f0, cycles=cycles, min_win=min_win, max_win=max_win)
        half_bw = half_bw_for_win(win_sec)
        # call sliding_coherence_f0 with or without wins
        kwargs = dict(f0=f0, half=half_bw, time_col=time_col,
                      win_sec=win_sec, step_sec=step_sec, n_null=n_null, show=False)
        if has_wins:
            kwargs['wins'] = None
        sl = sliding_coherence_f0(records, eeg_channel, sr_channel, **kwargs)
        t = np.asarray(sl['t'])
        coh_raw = np.asarray(sl['coh'], float)
        if t.size == 0 or coh_raw.size == 0:
            continue
        # smoothing for display
        if smoother_fn is not None:
            try:
                coh_plot = np.asarray(smoother_fn(coh_raw), float)
            except Exception:
                coh_plot = coh_raw
                smoother_fn = None
        else:
            coh_plot = coh_raw
        # smooth-aware null if available
        thr95 = _build_null_for_series(coh_raw, n_null=n_null, smoother=smoother_fn)
        # z-score relative to plotted series
        m = float(np.nanmean(coh_plot)); s = float(np.nanstd(coh_plot) + 1e-12)
        zcoh = (coh_plot - m)/s
        zthr = (thr95 - m)/s
        traces.append({'f0': f0, 't': t, 'zcoh': zcoh, 'zthr': zthr, 'win_sec': win_sec, 'half_bw': half_bw})

    if not traces:
        raise ValueError("No valid traces to plot.")

    # Common time range
    t_all = np.concatenate([tr['t'] for tr in traces])
    tmin, tmax = float(np.nanmin(t_all)), float(np.nanmax(t_all))

    colors = plt.get_cmap('tab10').colors
    null_alpha = 0.35

    if facet:
        fig, axes = plt.subplots(len(traces), 1, sharex=True, sharey=True,
                                 figsize=(10, max(6, 2.0*len(traces))))
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        for i, (ax, tr) in enumerate(zip(axes, traces)):
            col = colors[i % len(colors)]
            ax.plot(tr['t'], tr['zcoh'], lw=linewidth, color=col)
            ax.hlines(tr['zthr'], tmin, tmax, colors=col, linestyles='--', lw=1.0, alpha=null_alpha)
            # Shade ignitions (clipped)
            if ignition_windows:
                for (t0, t1) in ignition_windows:
                    if t1 < tmin or t0 > tmax: continue
                    ax.axvspan(max(t0, tmin), min(t1, tmax), color='k', alpha=0.08, zorder=0)
            if grid: ax.grid(True, alpha=0.25, linestyle=':')
            ax.set_ylabel(f"{tr['f0']:.3g} Hz\n(w={tr['win_sec']:.0f}s, h={tr['half_bw']:.2f})")
        axes[-1].set_xlabel('Time (s)')
        fig.suptitle(f"SR z-coherence — {group_name} (adaptive windows)")
        fig.tight_layout()
        if out_png: fig.savefig(out_png, dpi=dpi, bbox_inches='tight')
        plt.show()
        return traces
    else:
        fig, ax = plt.subplots(figsize=(10, 4.6))
        for i, tr in enumerate(traces):
            col = colors[i % len(colors)]
            ax.plot(tr['t'], tr['zcoh'], lw=linewidth, color=col, label=f"{tr['f0']:.3g} Hz")
            ax.hlines(tr['zthr'], tmin, tmax, colors=col, linestyles='--', lw=1.0, alpha=null_alpha)
        if ignition_windows:
            for (t0, t1) in ignition_windows:
                if t1 < tmin or t0 > tmax: continue
                ax.axvspan(max(t0, tmin), min(t1, tmax), color='k', alpha=0.08, zorder=0)
        if grid: ax.grid(True, alpha=0.25, linestyle=':')
        ax.set_xlim(tmin, tmax)
        ax.set_xlabel('Time (s)'); ax.set_ylabel('z-coherence')
        ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False)
        ax.set_title(f"SR z-coherence — {group_name} (adaptive windows)")
        fig.tight_layout()
        if out_png: fig.savefig(out_png, dpi=dpi, bbox_inches='tight')
        plt.show()
        return traces

# ----------------------------
# 4) Convenience: run all groups & return a tidy summary
# ----------------------------

def summarize_sr_groups(records, eeg_channel, sr_channel, ignition_windows=None,
                        groups=None, **plot_kwargs):
    groups = sr_groups() if groups is None else groups
    all_rows = []
    for name in groups:
        traces = plot_sr_group_adaptive(records, eeg_channel, sr_channel,
                                        group_name=name, ignition_windows=ignition_windows,
                                        **plot_kwargs)
        for tr in traces:
            z = tr['zcoh']; m = float(np.nanmedian(z)); mx = float(np.nanmax(z))
            cover = float(100.0*np.nanmean((z > tr['zthr']).astype(float)))
            all_rows.append({'group': name, 'f0': tr['f0'], 'win_sec': tr['win_sec'], 'half_bw': tr['half_bw'],
                             'median_z': m, 'max_z': mx, 'coverage_pct': cover})
    import pandas as pd
    return pd.DataFrame(all_rows)

# ----------------------------
# Example usage (uncomment):
# GROUPS = sr_groups()
# traces = plot_sr_group_adaptive(RECORDS, 'EEG.O1', 'EEG.Pz',
#                                 group_name='Harmonics_UpTo33',
#                                 ignition_windows=[(290,310),(580,600)],
#                                 facet=True, smooth=True, step_sec=1.0, n_null=200,
#                                 limit_high_hz=60.0)
# summary_df = summarize_sr_groups(RECORDS, 'EEG.O1', 'EEG.Pz',
#                                  ignition_windows=[(290,310),(580,600)],
#                                  facet=False, smooth=True)
# print(summary_df)


In [ ]:
# H1–H5
#         'Harmonics_UpTo33': (7.83, 14.3, 20.8, 27.3, 33.8),
#         # H6–H10 (mind 60 Hz mains; you may cap at 58 Hz in plots)
#         'HighHarmonics_40to60': (7.83, 40.3, 46.8, 53.3, 59.8),
#         # Subharmonics /2..5
#         'Subharmonics_2to5': (7.83, 3.915, 2.61, 1.9575, 1.566),
#         # Subharmonics /6..10 (+ 1.2 as given)
#         'Subharmonics_6to10_Mixed': (7.83, 1.305, 1.11857, 0.9788, 1.2, 0.783),
#     }
    
# # e.g., posterior EEG vs SR proxy
# traces = plot_sr_group_adaptive(
#     RECORDS, 'EEG.O1', 'EEG.F4',
#     group_name='Harmonics_UpTo33',
#     ignition_windows=[(290,310),(580,600)],
#     smooth=True, facet=True, step_sec=1.0, n_null=200
# )

summary = summarize_sr_groups(
    RECORDS, 'EEG.O1', 'EEG.F4',
    ignition_windows=[(162,222),(262,322),(415,475),(520,580)],
    smooth=True, facet=True
)


In [ ]:
def plot_sr_ignition_wtc_strip(
    RECORDS,
    eeg_channel: str,                 # e.g., 'EEG.O1' (or your best posterior)
    sr_channel: str,                  # magnetometer if you have one; else posterior proxy
    ignition_windows: list,           # [(t0, t1), ...] in seconds
    time_col: str = 'Timestamp',
    fmin: float = 0.5, fmax: float = 59.8, n_freq: int = 64,
    harmonics=(7.83, 14.3, 20.8, 27.3, 33.8),   # show bands for these (you can add more)
    half_band: float = 0.6,           # ±Hz shading around each harmonic
    w0: float = 6.0,                  # Morlet parameter
    n_perm: int = 200, alpha: float = 0.05,
    out_png: str = 'sr_wtc_strip.png',
    show: bool = True
):
    import numpy as np
    import matplotlib.pyplot as plt

    # 1) Run WTC on the *full* record (so ignition spans align to absolute time)
    wtc = wavelet_coherence_tf(
        RECORDS, eeg_channel, sr_channel,
        time_col=time_col, fmin=fmin, fmax=fmax, n_freq=n_freq, w0=w0,
        n_perm=n_perm, alpha=alpha, wins=None, show=True, out_png=None
    )

    # 2) Build the time axis from the DataFrame
    t_all = np.asarray(pd.to_numeric(RECORDS[time_col], errors='coerce').values, float)
    # If wavelet_coherence_tf returned a trimmed/sliced length, map to the last N samples
    N = wtc['WTC'].shape[1]
    if len(t_all) != N:
        # use the last N timestamps to match WTC length
        t = t_all[-N:]
    else:
        t = t_all

    # 3) Plot WTC with cyan significant pixels and ignition shading
    plt.figure(figsize=(11, 4))
    extent = [t[0], t[-1], wtc['freqs'][0], wtc['freqs'][-1]]

    plt.imshow(
        wtc['WTC'], aspect='auto', origin='lower', extent=extent,
        cmap='magma', vmin=0, vmax=np.nanmax(wtc['WTC'])
    )
    cb = plt.colorbar(); cb.set_label('Wavelet coherence')

    # Cyan significant pixels (cluster-based, mask already thresholded vs shift-null)
    sig = wtc['sig_mask']
    if sig is not None and np.any(sig):
        yy, xx = np.where(sig)
        plt.scatter(t[xx], wtc['freqs'][yy], s=4, c='cyan', alpha=0.7, label='> null 95%')

    # Horizontal harmonic bands (±half_band)
    for h in harmonics:
        plt.axhspan(h - half_band, h + half_band, color='white', alpha=0.08)
        plt.axhline(h, color='white', lw=0.8, alpha=0.6)

    # Shade ignition windows
    for (t0, t1) in ignition_windows:
        plt.axvspan(t0, t1, color='k', alpha=0.08)

    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.title(f'EEG–SR Wavelet Coherence (cyan = > null 95%; shaded = ignition)')
    if sig is not None and np.any(sig):
        plt.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    if out_png:
        plt.savefig(out_png, dpi=140)
    if show:
        plt.show()
    plt.close()


In [ ]:
plot_sr_ignition_wtc_strip(
    RECORDS,
    eeg_channel='EEG.O1',               # or your cleanest posterior
    sr_channel='EEG.F4',                # magnetometer if available; else O1/O2 proxy
    ignition_windows=[(179,184),(284,286),(434,429),(565,572)],
    time_col='Timestamp',
    harmonics=(1.566,1.9575,2.61,3.915,7.83, 14.3, 20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_band=0.6,
    out_png='exports_eeg_sr/S01/sr_wtc_strip.png',
    show=True
)


In [ ]:
"""
Synchrosqueezed time–frequency analysis for SR fundamentals & harmonics (0.1–60 Hz)
===================================================================================

What you get
------------
1) ssq‑CWT heatmaps (EEG & SR) with ridge‑sharp energy (uses ssqueezepy if available).
   • Fallback: high‑res STFT spectrogram if ssqueezepy is not installed.
2) Ridge extraction per target f0 (fundamental + harmonics/subharmonics), with:
   • Ridge frequency track f̂(t) and ridge power p̂(t) inside ±bw around f0.
   • Alignment error |f̂(t) − f0| statistics; coverage above null.
3) Simple validation tests (per f0):
   • Within‑band coverage vs off‑band surrogate (circular shift) → p‑value.
   • EEG↔SR ridge‑power correlation with circular‑shift null → p‑value.
4) Ready‑made plotting: heatmaps + overlaid ridge tracks + bar charts of metrics.

Limits: All analyses clamp to ≤ 60 Hz (or ≤ Nyquist).

Dependencies: numpy, scipy, matplotlib. Optional: ssqueezepy (preferred).
Install (optional): pip install ssqueezepy
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# ---------- small helpers (reuse your own if already defined) ----------

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    import pandas as pd
    if time_col in df.columns:
        s = df[time_col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values
            return time_col
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values
            return time_col
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

def get_series(df, name):
    import pandas as pd
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
    alt='EEG.'+name
    if alt in df.columns:
        return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
    raise ValueError(f'{name} not found in dataframe columns.')

# ---------- synchrosqueezed CWT (preferred) or STFT (fallback) ----------
try:
    from ssqueezepy import ssq_cwt, cwt, Wavelet
    _HAS_SSQ = True
except Exception:
    _HAS_SSQ = False


def _ssq_cwt_TFR(x, fs, fmin=0.1, fmax=60.0):
    """Return (t, f, power) using synchrosqueezed CWT if available; else STFT fallback.
    power has shape (n_freqs, n_times). t in seconds, f in Hz.
    """
    N = len(x); t = np.arange(N)/float(fs)
    ny = 0.5*fs
    fmax = float(min(fmax, 0.999*ny))
    if _HAS_SSQ:
        wv = Wavelet('morlet')  # good narrow ridges
        Wx, scales = cwt(x, wavelet=wv, fs=fs)
        Tx, fs_ssq, *_ = ssq_cwt(Wx, scales, wavelet=wv, fs=fs)
        freqs = np.asarray(fs_ssq, float)
        mask = (freqs >= float(fmin)) & (freqs <= float(fmax))
        P = np.abs(Tx[mask, :])**2
        f = freqs[mask]
        return t, f, P
    # Fallback: high‑res STFT spectrogram
    nper = int(max(fs*4, 256))      # 4 s window for LF detail
    nover = int(0.75*nper)
    nfft = int(2**np.ceil(np.log2(nper*2)))
    f, tt, Sxx = signal.spectrogram(x, fs=fs, window='hann', nperseg=nper,
                                    noverlap=nover, nfft=nfft, scaling='spectrum', mode='magnitude')
    mask = (f >= float(fmin)) & (f <= float(fmax))
    P = (Sxx[mask, :])**2
    # Interpolate STFT time grid to sample grid for comparable t axis
    if tt.size and tt[0] > 0:
        t_coarse = tt
        # return coarse t to avoid misleading densification
        return t_coarse, f[mask], P
    return t, f[mask], P

# ---------- ridge extraction around a target frequency ----------

def ridge_in_band(P, f, t, f0, bw=0.6):
    """Within [f0-bw, f0+bw], take per‑time max → ridge freq & power.
    Returns dict with arrays: f_hat(t), p_hat(t), and simple stats.
    """
    f0 = float(f0); bw = float(bw)
    band = np.where((f >= max(0.0, f0-bw)) & (f <= f0+bw))[0]
    if band.size == 0:
        # fallback: nearest single bin
        idx = int(np.argmin(np.abs(f - f0)))
        return {
            'f_hat': np.full(t.shape, f[idx]),
            'p_hat': P[idx, :].astype(float),
            'band_idx': np.array([idx]),
            'band_freqs': np.array([f[idx]])
        }
    sub = P[band, :]
    jmax = np.argmax(sub, axis=0)               # argmax over band per time
    idxs = band[jmax]
    f_hat = f[idxs]
    p_hat = sub[jmax, np.arange(sub.shape[1])]
    return {'f_hat': f_hat, 'p_hat': p_hat, 'band_idx': band, 'band_freqs': f[band]}

# ---------- simple surrogates & validation ----------

def circular_shift(a, s):
    s = int(s) % len(a)
    if s == 0: return a
    return np.r_[a[-s:], a[:-s]]


def validate_ridge(p_hat, offband_ref=None, n_perm=200, rng=None):
    """Coverage & p‑value: how often is ridge power above off‑band reference?
    offband_ref: 1D array representing background power (same length or pooled).
    If None, uses the ridge power itself with circular‑shift surrogates.
    Returns dict with coverage_pct, p_value, thr95.
    """
    rng = np.random.default_rng() if rng is None else rng
    x = np.asarray(p_hat, float)
    if offband_ref is None:
        # build null by circular shift → max distribution
        maxima = []
        for _ in range(int(n_perm)):
            s = rng.integers(1, len(x)-1)
            xs = circular_shift(x, s)
            maxima.append(float(np.nanmax(xs)))
        thr95 = float(np.nanpercentile(maxima, 95))
    else:
        ref = np.asarray(offband_ref, float)
        thr95 = float(np.nanpercentile(ref, 95))
    coverage = float(100.0*np.nanmean((x > thr95).astype(float)))
    # simple p: percentile of observed median vs surrogate medians
    meds = []
    for _ in range(int(n_perm)):
        s = rng.integers(1, len(x)-1)
        xs = circular_shift(x, s)
        meds.append(float(np.nanmedian(xs)))
    pval = float((np.sum(np.asarray(meds) >= np.nanmedian(x)) + 1) / (n_perm + 1))
    return {'coverage_pct': coverage, 'thr95': thr95, 'p_value': pval}


def validate_eeg_sr_coupling(p_eeg, p_sr, n_perm=200, rng=None):
    """Correlation between EEG & SR ridge power with circular‑shift null."""
    rng = np.random.default_rng() if rng is None else rng
    x = np.asarray(p_eeg, float); y = np.asarray(p_sr, float)
    L = min(len(x), len(y))
    x = x[:L]; y = y[:L]
    r_obs = float(np.corrcoef(x, y)[0,1]) if np.std(x)>0 and np.std(y)>0 else 0.0
    null = []
    for _ in range(int(n_perm)):
        s = rng.integers(1, L-1)
        yn = circular_shift(y, s)
        r = float(np.corrcoef(x, yn)[0,1]) if np.std(yn)>0 else 0.0
        null.append(r)
    p = float((np.sum(np.asarray(null) >= r_obs) + 1) / (n_perm + 1))
    return {'r': r_obs, 'p_value': p}

# --- helper: always make parent dir before saving ---
def _safe_savefig(path, dpi=160, **kwargs):
    if not path:
        return
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
        plt.savefig(path, dpi=dpi, **kwargs)

# ---------- main user‑facing routine ----------

def ssq_sr_validate(RECORDS,
                    eeg_channel: str,
                    sr_channel: str,
                    time_col='Timestamp',
                    freq_groups=None,
                    fmin=0.1, fmax=60.0,
                    bw=0.6, n_perm=200,
                    show=True, out_dir=None):
    """
    Compute synchrosqueezed (or STFT fallback) T–F for EEG & SR; extract ridges at
    requested frequencies; run simple validations; and draw simple graphs.

    freq_groups: dict name -> tuple/list of target f0s. If None, a default set
                 using typical SR harmonics up to 60 Hz is used.
    """
    import os, pandas as pd
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)
    ny = 0.5*fs
    fmax = float(min(fmax, 0.999*ny))

    if freq_groups is None:
        freq_groups = {
            'SR_UpTo33': (7.83, 14.3, 20.8, 27.3, 33.8),
            'SR_40to60': (7.83, 40.3, 46.8, 53.3, 59.8),
            'SR_Sub_2to5': (7.83, 3.915, 2.61, 1.9575, 1.566),
            'SR_Sub_6to10': (7.83, 1.305, 1.11857, 0.9788, 1.2, 0.783),
        }

    xe = get_series(RECORDS, eeg_channel).astype(float)
    xs = get_series(RECORDS, sr_channel).astype(float)
    
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
    
    # TFRs
    t_eeg, f_eeg, P_eeg = _ssq_cwt_TFR(xe, fs, fmin=fmin, fmax=fmax)
    t_sr,  f_sr,  P_sr  = _ssq_cwt_TFR(xs, fs, fmin=fmin, fmax=fmax)

    # heatmaps
    # --- replace your _plot_heatmap with this version ---
    def _plot_heatmap(t, f, P, title, lines=None, out_png=None, show=True):
        plt.figure(figsize=(10, 4.2))
        extent=[t[0], t[-1], f[0], f[-1]]
        plt.imshow(10*np.log10(P + 1e-20), aspect='auto', origin='lower',
        extent=extent, cmap='magma')
        cb=plt.colorbar(); cb.set_label('Power (dB)')
        if lines:
            for (freq, col) in lines:
                if f[0] <= freq <= f[-1]:
                    plt.plot([t[0], t[-1]],[freq,freq], color=col, lw=1.0, ls='--', alpha=0.7)
                    plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
                    plt.title(title)
                    plt.tight_layout()
        if out_png:
            _safe_savefig(out_png, dpi=160)
        if show:
            plt.show()
            plt.close()

    # overlay expected harmonics with colors
    COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

    # Prepare lines for all f0s
    all_f0s = sorted(set([round(float(f),6) for vals in freq_groups.values() for f in vals]))
    line_list = [(f0, COLORS[i % len(COLORS)]) for i, f0 in enumerate(all_f0s) if fmin <= f0 <= fmax]

    if show:
        _plot_heatmap(t_eeg, f_eeg, P_eeg, f'EEG T–F (ssqCWT {"on" if _HAS_SSQ else "STFT"})',
                      lines=line_list,
                      out_png=(None if out_dir is None else os.path.join(out_dir, 'EEG_TF.png')),show=show)
        _plot_heatmap(t_sr, f_sr, P_sr, f'SR T–F (ssqCWT {"on" if _HAS_SSQ else "STFT"})',
                      lines=line_list,
                      out_png=(None if out_dir is None else os.path.join(out_dir, 'SR_TF.png')),show=show)

    # For each group/f0: ridges + validations
    rows = []
    for gname, f0s in freq_groups.items():
        for idx, f0 in enumerate(f0s):
            if not (fmin <= f0 <= fmax):
                continue
            col = COLORS[idx % len(COLORS)]
            eeg_r = ridge_in_band(P_eeg, f_eeg, t_eeg, f0, bw=bw)
            sr_r  = ridge_in_band(P_sr,  f_sr,  t_sr,  f0, bw=bw)
            # alignment error (median absolute deviation in Hz)
            align_eeg = float(np.nanmedian(np.abs(eeg_r['f_hat'] - f0)))
            align_sr  = float(np.nanmedian(np.abs(sr_r['f_hat']  - f0)))
            # validations
            val_eeg = validate_ridge(eeg_r['p_hat'], n_perm=n_perm)
            val_sr  = validate_ridge(sr_r['p_hat'],  n_perm=n_perm)
            val_xy  = validate_eeg_sr_coupling(eeg_r['p_hat'], sr_r['p_hat'], n_perm=n_perm)
            rows.append({
                'group': gname, 'f0': float(f0), 'bw': float(bw),
                'EEG_align_Hz_med': align_eeg,
                'EEG_coverage_pct': val_eeg['coverage_pct'], 'EEG_pval': val_eeg['p_value'],
                'SR_align_Hz_med': align_sr,
                'SR_coverage_pct': val_sr['coverage_pct'], 'SR_pval': val_sr['p_value'],
                'EEGxSR_r': val_xy['r'], 'EEGxSR_pval': val_xy['p_value']
            })
            # quick line plots of ridges over time
            if show:
                plt.figure(figsize=(10, 2.6))
                plt.plot(t_eeg, eeg_r['f_hat'], color=col, lw=1.5, label=f'EEG ridge @ {f0:.3g} Hz')
                plt.plot(t_sr,  sr_r['f_hat'],  color=col, lw=1.0, ls='--', alpha=0.7, label='SR ridge')
                plt.axhline(f0, color=col, lw=1.0, ls=':', alpha=0.7)
                plt.ylabel('Freq (Hz)'); plt.xlabel('Time (s)'); plt.ylim(max(fmin,0.05), fmax)
                plt.title(f'Ridge tracks around {f0:.3g} Hz (±{bw:.2f} Hz)')
                plt.legend(loc='upper right', fontsize=8); plt.tight_layout()
                if out_dir: plt.savefig(os.path.join(out_dir, f'ridge_{gname}_{f0:.3g}Hz.png'), dpi=160)
                plt.show(); plt.close()

    import pandas as pd
    summary = pd.DataFrame(rows)

    # bar plot: coverage per f0 (EEG & SR)
    if show and len(rows):
        fig, ax = plt.subplots(figsize=(10,3.2))
        f0s_plot = summary['f0'].values
        ax.bar(f0s_plot-0.15, summary['EEG_coverage_pct'].values, width=0.3, label='EEG')
        ax.bar(f0s_plot+0.15, summary['SR_coverage_pct'].values, width=0.3, label='SR')
        ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Coverage > null95 (%)')
        ax.set_title('Ridge coverage vs null (EEG vs SR)')
        ax.legend(); plt.tight_layout()
        if out_dir: plt.savefig(os.path.join(out_dir, 'coverage_bar.png'), dpi=160)
        plt.show(); plt.close()

    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        summary.to_csv(os.path.join(out_dir, 'sr_ssq_summary.csv'), index=False)

    return {'summary': summary, 'EEG_TF': (t_eeg, f_eeg, P_eeg), 'SR_TF': (t_sr, f_sr, P_sr)}


# ---------------- Example usage ----------------
# res = ssq_sr_validate(
#     RECORDS,
#     eeg_channel='EEG.O1',
#     sr_channel='EEG.Pz',   # or a magnetometer if you have it
#     time_col='Timestamp',
#     freq_groups={
#        'Harmonics_UpTo33': (7.83, 14.3, 20.8, 27.3, 33.8),
#        'HighHarmonics_40to60': (7.83, 40.3, 46.8, 53.3, 59.8),
#        'Subharmonics_2to5': (7.83, 3.915, 2.61, 1.9575, 1.566),
#        'Subharmonics_6to10_Mixed': (7.83, 1.305, 1.11857, 0.9788, 1.2, 0.783),
#     },
#     fmin=0.1, fmax=60.0, bw=0.6, n_perm=200, show=True,
#     out_dir='exports_ssq'
# )


In [ ]:
res = ssq_sr_validate(
    RECORDS,
    eeg_channel='EEG.O1',
    sr_channel='EEG.F4',          # or your magnetometer channel
    time_col='Timestamp',
    freq_groups={
       'Harmonics_UpTo33': (7.83,14.3,20.8,27.3,33.8),
       'HighHarmonics_40to60': (7.83,40.3,46.8,53.3,59.8),
       'Subharmonics_2to5': (7.83,3.915,2.61,1.9575,1.566),
       'Subharmonics_6to10_Mixed': (7.83,1.305,1.11857,0.9788,1.2,0.783),
    },
    fmin=0.1, fmax=49, bw=0.2, n_perm=200, show=True,
    out_dir='exports_ssq'
)


In [ ]:
"""
Harmonic locking metrics for SR fundamentals & harmonics (0.1–60 Hz)
====================================================================
Implements:
  • H‑PLI_k(t): |< exp(i[φ_k^EEG(t) − φ_k^SR(t)]) >_{t∈W}|
  • XH‑PLI_m(t): |< exp(i[φ_m^EEG(t) − m φ_1^EEG(t)]) >_{t∈W}|
  • SubH‑PLI_n(t): |< exp(i[n φ_{1/n}^EEG(t) − φ_1^EEG(t)]) >_{t∈W}|
  • HCS(t) = Σ_k w_k · H‑PLI_k(t), default w_k=1/k

Also provides:
  • Simple surrogate tests with smooth‑aware nulls (circular time shift of phases)
  • Block‑bootstrap CIs for time‑average metrics
  • Faceted time‑series plots (PLI vs null) and bar charts with CIs
  • Summary CSV of per‑order metrics and HCS

Dependencies: numpy, scipy, matplotlib (and pandas for summaries)
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# --------------------------- helpers ---------------------------

def ensure_dir(d):
    if d:
        os.makedirs(d, exist_ok=True)
    return d

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    import pandas as pd
    if time_col in df.columns:
        s = df[time_col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values
            return time_col
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values
            return time_col
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

def get_series(df, name):
    import pandas as pd
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
    alt='EEG.'+name
    if alt in df.columns:
        return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
    raise ValueError(f'{name} not found in dataframe columns.')

# bandpass + Hilbert phase

def bandpass(x, fs, f1, f2, order=4):
    ny = 0.5*fs
    f1 = max(1e-6, min(f1, 0.99*ny))
    f2 = max(f1+1e-6, min(f2, 0.999*ny))
    b,a = signal.butter(order, [f1/ny, f2/ny], btype='band')
    return signal.filtfilt(b,a,x)

def phase_series(x, fs, f0, half):
    xb = bandpass(x, fs, float(f0)-float(half), float(f0)+float(half))
    z  = signal.hilbert(xb)
    return np.angle(z)  # in radians, wrapped

# sliding windows

def sliding_centers(N, fs, win_sec, step_sec):
    win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
    if win < 8: raise ValueError('win_sec too small')
    return np.arange(win//2, N - win//2, step, dtype=int)

# smoothing (optional visual only)

def _auto_savgol(y, max_window=31):
    y = np.asarray(y, float)
    n = y.size
    if n < 7: return y
    try:
        from scipy.signal import savgol_filter
        w = max(5, min(max_window, int(round(n/15))))
        if w % 2 == 0: w += 1
        if w >= n: w = n-1 if (n % 2 == 0) else n
        if w < 5: return y
        return savgol_filter(y, w, polyorder=2, mode='interp')
    except Exception:
        w = max(5, min(max_window, int(round(n/15))))
        if w < 2: return y
        return np.convolve(np.nan_to_num(y), np.ones(w)/w, mode='same')

# block bootstrap for CI of means

def block_bootstrap_ci(x, n_boot=1000, alpha=0.05, block_len=None, seed=13):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    N = x.size
    if N == 0: return (np.nan, np.nan)
    if block_len is None:
        block_len = max(5, int(round(N/20)))
    B = int(block_len)
    means = []
    for _ in range(int(n_boot)):
        idx = []
        filled = 0
        while filled < N:
            start = int(rng.integers(0, N))
            end = start + B
            if end <= N:
                idx.extend(range(start, end))
            else:
                idx.extend(list(range(start, N)) + list(range(0, end - N)))
            filled += B
        idx = np.asarray(idx[:N], int)
        means.append(float(np.nanmean(x[idx])))
    lo, hi = np.nanpercentile(means, [100*alpha/2, 100*(1-alpha/2)])
    return float(lo), float(hi)

# surrogate builder for PLI curves (smooth‑aware)

def pli_surrogates(phi_a, phi_b, centers, win, step, n_perm=200, smoother=None, seed=7):
    rng = np.random.default_rng(seed)
    N = len(phi_a)
    out = []
    for _ in range(int(n_perm)):
        s = int(rng.integers(win, N-1))  # shift at least one window
        phi_b_sh = np.r_[phi_b[-s:], phi_b[:-s]]
        vals = []
        for c in centers:
            sl = slice(c - win//2, c + win//2)
            dphi = phi_a[sl] - phi_b_sh[sl]
            vals.append(np.abs(np.mean(np.exp(1j*dphi))))
        v = np.asarray(vals, float)
        if smoother is not None:
            v = np.asarray(smoother(v), float)
        out.append(v)
    return np.asarray(out)  # shape (n_perm, n_centers)

# --------------------------- core metrics ---------------------------

def compute_H_PLI(phi_eeg, phi_sr, fs, win_sec=8.0, step_sec=1.0, smoother=None, n_perm=200):
    win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
    centers = sliding_centers(len(phi_eeg), fs, win_sec, step_sec)
    pli = []
    for c in centers:
        sl = slice(c - win//2, c + win//2)
        dphi = phi_eeg[sl] - phi_sr[sl]
        pli.append(np.abs(np.mean(np.exp(1j*dphi))))
    pli = np.asarray(pli, float)
    if smoother is not None:
        pli_plot = np.asarray(smoother(pli), float)
    else:
        pli_plot = pli
    # smooth‑aware null (pointwise 95th percentile) + mean p‑value
    sur = pli_surrogates(phi_eeg, phi_sr, centers, win, step, n_perm=n_perm, smoother=smoother)
    null95_curve = np.nanpercentile(sur, 95, axis=0)
    p_mean = float((np.sum(np.nanmean(sur, axis=1) >= np.nanmean(pli_plot)) + 1) / (n_perm + 1))
    return centers/fs, pli_plot, null95_curve, p_mean, pli


def compute_XH_PLI(phi_m_eeg, phi1_eeg, m, fs, win_sec=8.0, step_sec=1.0, smoother=None, n_perm=200):
    win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
    centers = sliding_centers(len(phi_m_eeg), fs, win_sec, step_sec)
    pli = []
    for c in centers:
        sl = slice(c - win//2, c + win//2)
        dphi = phi_m_eeg[sl] - m*phi1_eeg[sl]
        pli.append(np.abs(np.mean(np.exp(1j*dphi))))
    pli = np.asarray(pli, float)
    pli_plot = np.asarray(smoother(pli), float) if smoother is not None else pli
    # surrogates: circular shift the fundamental phase
    sur = []
    rng = np.random.default_rng(11)
    N = len(phi1_eeg)
    for _ in range(int(n_perm)):
        s = int(rng.integers(win, N-1))
        phi1_sh = np.r_[phi1_eeg[-s:], phi1_eeg[:-s]]
        vals = []
        for c in centers:
            sl = slice(c - win//2, c + win//2)
            dphi = phi_m_eeg[sl] - m*phi1_sh[sl]
            vals.append(np.abs(np.mean(np.exp(1j*dphi))))
        v = np.asarray(vals, float)
        if smoother is not None: v = np.asarray(smoother(v), float)
        sur.append(v)
    sur = np.asarray(sur)
    null95_curve = np.nanpercentile(sur, 95, axis=0)
    p_mean = float((np.sum(np.nanmean(sur, axis=1) >= np.nanmean(pli_plot)) + 1) / (n_perm + 1))
    return centers/fs, pli_plot, null95_curve, p_mean, pli


def compute_SubH_PLI(phi_s_eeg, phi1_eeg, n, fs, win_sec=8.0, step_sec=1.0, smoother=None, n_perm=200):
    win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
    centers = sliding_centers(len(phi_s_eeg), fs, win_sec, step_sec)
    pli = []
    for c in centers:
        sl = slice(c - win//2, c + win//2)
        dphi = n*phi_s_eeg[sl] - phi1_eeg[sl]
        pli.append(np.abs(np.mean(np.exp(1j*dphi))))
    pli = np.asarray(pli, float)
    pli_plot = np.asarray(smoother(pli), float) if smoother is not None else pli
    # surrogates: shift the subharmonic phase
    sur = []
    rng = np.random.default_rng(17)
    N = len(phi_s_eeg)
    for _ in range(int(n_perm)):
        s = int(rng.integers(win, N-1))
        phi_s_sh = np.r_[phi_s_eeg[-s:], phi_s_eeg[:-s]]
        vals = []
        for c in centers:
            sl = slice(c - win//2, c + win//2)
            dphi = n*phi_s_sh[sl] - phi1_eeg[sl]
            vals.append(np.abs(np.mean(np.exp(1j*dphi))))
        v = np.asarray(vals, float)
        if smoother is not None: v = np.asarray(smoother(v), float)
        sur.append(v)
    sur = np.asarray(sur)
    null95_curve = np.nanpercentile(sur, 95, axis=0)
    p_mean = float((np.sum(np.nanmean(sur, axis=1) >= np.nanmean(pli_plot)) + 1) / (n_perm + 1))
    return centers/fs, pli_plot, null95_curve, p_mean, pli

# --------------------------- orchestration ---------------------------

def win_for_f0(f0, cycles=8, min_win=8.0, max_win=120.0):
    if f0 <= 1e-9: return max_win
    return float(np.clip(cycles/float(f0), min_win, max_win))

def analyze_locking(RECORDS,
                    eeg_channel: str,
                    sr_channel: str,
                    fundamental=7.83,
                    harmonics=(7.83,14.3,20.8,27.3,33.8),
                    subharmonics=(3.915, 2.61, 1.9575, 1.566, 1.305, 1.11857, 0.9788, 1.2, 0.783),
                    time_col='Timestamp',
                    half_bw=0.6,
                    cycles=8, min_win=8.0, max_win=120.0,
                    step_sec=1.0,
                    n_perm=200,
                    limit_high_hz=60.0,
                    smooth=True,
                    weights_scheme='inverse_k',    # or 'equal'
                    out_dir='exports_locking',
                    show=True):
    """
    Compute H‑PLI per harmonic (EEG vs SR), cross‑order XH‑PLI_m (EEG harmonics vs EEG fundamental),
    and SubH‑PLI_n for provided subharmonics. Plot simple graphs and save a summary CSV.
    """
    import pandas as pd
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)
    ensure_dir(out_dir)

    x_eeg = get_series(RECORDS, eeg_channel).astype(float)
    x_sr  = get_series(RECORDS, sr_channel).astype(float)

    harmonics = tuple([f for f in harmonics if f <= float(limit_high_hz)+1e-9])

    smoother = _auto_savgol if smooth else None

    rows = []
    traces_H = []

    # Fundamental phase (EEG) for cross‑order comparisons
    phi1_eeg = phase_series(x_eeg, fs, fundamental, half_bw)

    # ----- H‑PLI per harmonic -----
    for fk in harmonics:
        win_sec = win_for_f0(fk, cycles=cycles, min_win=min_win, max_win=max_win)
        phi_k_eeg = phase_series(x_eeg, fs, fk, half_bw)
        phi_k_sr  = phase_series(x_sr,  fs, fk, half_bw)
        t, pli_curve, null95_curve, p_mean, pli_raw = compute_H_PLI(phi_k_eeg, phi_k_sr, fs,
                                                                    win_sec=win_sec, step_sec=step_sec,
                                                                    smoother=smoother, n_perm=n_perm)
        # CI for mean (block bootstrap)
        ci_lo, ci_hi = block_bootstrap_ci(pli_curve, n_boot=1000, alpha=0.05)
        mean_pli = float(np.nanmean(pli_curve))
        med_pli  = float(np.nanmedian(pli_curve))
        cover = float(100.0*np.nanmean((pli_curve > null95_curve).astype(float)))
        # store
        traces_H.append((fk, t, pli_curve, null95_curve, win_sec))
        k_order = max(1, int(round(fk / fundamental)))
        rows.append({'metric':'H-PLI', 'order':k_order, 'f0':fk,
                     'mean':mean_pli, 'median':med_pli, 'ci_lo':ci_lo, 'ci_hi':ci_hi,
                     'p_mean':p_mean, 'coverage_pct':cover, 'win_sec':win_sec})

    # ----- XH‑PLI vs fundamental -----
    traces_XH = []
    for fk in harmonics:
        m = max(1, int(round(fk / fundamental)))
        if m == 1:  # skip trivial
            continue
        win_sec = win_for_f0(fk, cycles=cycles, min_win=min_win, max_win=max_win)
        phi_m_eeg = phase_series(x_eeg, fs, fk, half_bw)
        t, pli_curve, null95_curve, p_mean, pli_raw = compute_XH_PLI(phi_m_eeg, phi1_eeg, m, fs,
                                                                     win_sec=win_sec, step_sec=step_sec,
                                                                     smoother=smoother, n_perm=n_perm)
        ci_lo, ci_hi = block_bootstrap_ci(pli_curve, n_boot=1000, alpha=0.05)
        mean_pli = float(np.nanmean(pli_curve))
        med_pli  = float(np.nanmedian(pli_curve))
        cover = float(100.0*np.nanmean((pli_curve > null95_curve).astype(float)))
        traces_XH.append((m, fk, t, pli_curve, null95_curve))
        rows.append({'metric':'XH-PLI', 'order':m, 'f0':fk,
                     'mean':mean_pli, 'median':med_pli, 'ci_lo':ci_lo, 'ci_hi':ci_hi,
                     'p_mean':p_mean, 'coverage_pct':cover, 'win_sec':win_sec})

    # ----- SubH‑PLI for subharmonics s=1/n -----
    traces_SH = []
    for fsb in subharmonics:
        if fsb < 0.1:  # ignore too low
            continue
        n = int(round(fundamental / fsb)) if fsb > 0 else None
        if n is None or n < 2:  # only true subharmonics
            continue
        win_sec = win_for_f0(fsb, cycles=cycles, min_win=min_win, max_win=max_win)
        phi_s_eeg = phase_series(x_eeg, fs, fsb, half_bw)
        t, pli_curve, null95_curve, p_mean, pli_raw = compute_SubH_PLI(phi_s_eeg, phi1_eeg, n, fs,
                                                                       win_sec=win_sec, step_sec=step_sec,
                                                                       smoother=smoother, n_perm=n_perm)
        ci_lo, ci_hi = block_bootstrap_ci(pli_curve, n_boot=1000, alpha=0.05)
        mean_pli = float(np.nanmean(pli_curve))
        med_pli  = float(np.nanmedian(pli_curve))
        cover = float(100.0*np.nanmean((pli_curve > null95_curve).astype(float)))
        traces_SH.append((n, fsb, t, pli_curve, null95_curve))
        rows.append({'metric':'SubH-PLI', 'order':n, 'f0':fsb,
                     'mean':mean_pli, 'median':med_pli, 'ci_lo':ci_lo, 'ci_hi':ci_hi,
                     'p_mean':p_mean, 'coverage_pct':cover, 'win_sec':win_sec})

    # ----- HCS (weighted sum over harmonics per time) -----
    # Align to common time grid (use intersections of centers)
    if traces_H:
        t_common = traces_H[0][1]
        # ensure all H traces share same centers; if not, resample by nearest
        def _resample_to(t_src, y_src, t_ref):
            idx = np.searchsorted(t_src, t_ref)
            idx = np.clip(idx, 0, len(t_src)-1)
            return y_src[idx]
        # weights
        if weights_scheme == 'inverse_k':
            weights = []
            for fk, t_k, y_k, _, _ in traces_H:
                k = max(1, int(round(fk/fundamental)))
                weights.append(1.0/float(k))
        else:
            weights = [1.0 for _ in traces_H]
        weights = np.asarray(weights, float)
        weights /= np.sum(weights)
        Y = []
        for (fk, t_k, y_k, _, _) in traces_H:
            if len(t_k) != len(t_common) or np.any(np.abs(t_k - t_common) > 1e-6):
                y_k = _resample_to(t_k, y_k, t_common)
            Y.append(y_k)
        Y = np.asarray(Y)  # shape (K, T)
        HCS_curve = np.tensordot(weights, Y, axes=(0,0))  # (T,)
        HCS_mean = float(np.nanmean(HCS_curve))
        HCS_lo, HCS_hi = block_bootstrap_ci(HCS_curve, n_boot=1000, alpha=0.05)
        rows.append({'metric':'HCS', 'order':0, 'f0':np.nan,
                     'mean':HCS_mean, 'median':float(np.nanmedian(HCS_curve)),
                     'ci_lo':HCS_lo, 'ci_hi':HCS_hi, 'p_mean':np.nan,
                     'coverage_pct':np.nan, 'win_sec':np.nan})
    else:
        HCS_curve = None; t_common = None

    summary = pd.DataFrame(rows)

    # --------------------------- plots ---------------------------
    # (1) Faceted H‑PLI traces with null curves
    if traces_H:
        nrows = len(traces_H)
        fig, axes = plt.subplots(nrows, 1, sharex=True, figsize=(10, max(6, 2.0*nrows)))
        if not isinstance(axes, np.ndarray): axes = np.array([axes])
        for i, (fk, t, y, thr, win_sec) in enumerate(traces_H):
            ax = axes[i]
            ax.plot(t, y, lw=1.8, label=f'H-PLI @ {fk:.3g} Hz')
            ax.plot(t, thr, lw=1.0, ls='--', alpha=0.7, label='null95 (smooth-aware)')
            ax.set_ylabel('PLI')
            ax.grid(True, alpha=0.25, linestyle=':')
            ax.legend(loc='upper right', fontsize=8)
        axes[-1].set_xlabel('Time (s)')
        fig.suptitle('H‑PLI (EEG vs SR) per harmonic'); fig.tight_layout()
        plt.savefig(os.path.join(out_dir, 'H_PLI_traces.png'), dpi=160, bbox_inches='tight')
        if show: plt.show(); plt.close()

    # (2) Bar charts: H‑PLI means with 95% CIs; compare to XH‑PLI means
    if len(summary):
        import pandas as pd
        S = summary
        Hbars = S[S['metric']=='H-PLI'].copy()
        if not Hbars.empty:
            x = np.arange(len(Hbars))
            fig, ax = plt.subplots(figsize=(10,3.2))
            ax.bar(x, Hbars['mean'].values, yerr=[Hbars['mean']-Hbars['ci_lo'], Hbars['ci_hi']-Hbars['mean']],
                   width=0.6, capsize=3)
            ax.set_xticks(x); ax.set_xticklabels([f"k≈{int(round(f/ fundamental))}\n{f:.2f} Hz" for f in Hbars['f0']], rotation=0)
            ax.set_ylabel('Mean H‑PLI (±CI)'); ax.set_title('H‑PLI by order')
            ax.grid(True, axis='y', alpha=0.25, linestyle=':')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'H_PLI_bars.png'), dpi=160)
            if show: plt.show(); plt.close()
        Xbars = S[S['metric']=='XH-PLI'].copy()
        if not Xbars.empty:
            # group by order m
            orders = Xbars['order'].values; means = Xbars['mean'].values
            ci_lo = Xbars['ci_lo'].values; ci_hi = Xbars['ci_hi'].values
            x = np.arange(len(orders))
            fig, ax = plt.subplots(figsize=(10,3.2))
            ax.bar(x, means, yerr=[means-ci_lo, ci_hi-means], width=0.6, capsize=3)
            ax.set_xticks(x); ax.set_xticklabels([f"m={m}" for m in orders])
            ax.set_ylabel('Mean XH‑PLI (±CI)'); ax.set_title('Cross‑order locking (waveform shape)')
            ax.grid(True, axis='y', alpha=0.25, linestyle=':')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'XH_PLI_bars.png'), dpi=160)
            if show: plt.show(); plt.close()

    # (3) HCS curve
    if HCS_curve is not None and t_common is not None:
        fig, ax = plt.subplots(figsize=(10,3.2))
        ax.plot(t_common, HCS_curve, lw=1.8)
        ax.set_xlabel('Time (s)'); ax.set_ylabel('HCS'); ax.grid(True, alpha=0.25, linestyle=':')
        ax.set_title('Harmonic Coherence Score (weighted sum of H‑PLI)')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'HCS_curve.png'), dpi=160)
        if show: plt.show(); plt.close()

    # Save summary
    summary.to_csv(os.path.join(out_dir, 'locking_summary.csv'), index=False)

    return {'summary': summary, 'H_traces': traces_H, 'XH_traces': traces_XH, 'SH_traces': traces_SH,
            'HCS': (t_common, HCS_curve) if HCS_curve is not None else None,
            'out_dir': out_dir}

# --------------------------- example usage ---------------------------
# res = analyze_locking(
#     RECORDS,
#     eeg_channel='EEG.O1',
#     sr_channel='EEG.Pz',   # or magnetometer channel if available
#     fundamental=7.83,
#     harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
#     subharmonics=(3.915, 2.61, 1.9575, 1.566, 1.305, 1.11857, 0.9788, 1.2, 0.783),
#     half_bw=0.6,
#     cycles=8, min_win=8.0, max_win=120.0,
#     step_sec=1.0, n_perm=200, limit_high_hz=60.0,
#     smooth=True, weights_scheme='inverse_k',
#     out_dir='exports_locking', show=True
# )


In [ ]:
res = analyze_locking(
    RECORDS,
    eeg_channel='EEG.O1',
    sr_channel='EEG.F4',            # or your magnetometer
    fundamental=7.83,
    harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8),
    subharmonics=(3.915,2.61,1.9575,1.566,1.305,1.11857),
    half_bw=0.6,
    cycles=8, min_win=8.0, max_win=120.0,
    step_sec=1.0, n_perm=200, limit_high_hz=60.0,
    smooth=True, weights_scheme='inverse_k',
    out_dir='exports_locking', show=True
)


In [ ]:
"""
Lead/Lag Quantification Among SR Families (Temporal Dynamics)
=============================================================
Implements simple graphs and validation tests for:
  1) Envelope cross‑correlation lag (minute‑scale 0.003–0.03 Hz “breathing”).
  2) Phase‑lead probability between slow envelopes.
  3) Consensus ordering score across families: SubH(2–5) → 7.83 → (14–34) → (40–60).

Per window (pre‑ignition, ignition, rebound) and per band/family, outputs:
  • τ̂_env (s) with block‑bootstrap 95% CI.
  • P_lead with block‑bootstrap 95% CI.
  • Summary CSV + quick bar plots.

Usage example at bottom.
Dependencies: numpy, scipy, matplotlib, pandas.
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

# ------------------ basic helpers (reuse yours if present) ------------------

def ensure_dir(d):
    if d:
        os.makedirs(d, exist_ok=True)
    return d

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    if time_col in df.columns:
        s = df[time_col]
        # datetime → seconds
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values
            return time_col
        # numeric
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values
            return time_col
    # fallback: synth time
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

# filtering

def _butter_bandpass(x, fs, lo, hi, order=4):
    ny=0.5*fs
    lo=max(1e-6,min(lo,0.99*ny)); hi=max(lo+1e-6,min(hi,0.999*ny))
    b,a=signal.butter(order,[lo/ny, hi/ny], btype='band')
    return signal.filtfilt(b,a,x)

def _butter_lowpass(x, fs, hi, order=4):
    ny=0.5*fs; hi=max(1e-6, min(hi, 0.999*ny))
    b,a=signal.butter(order, hi/ny, btype='low')
    return signal.filtfilt(b,a,x)

# narrowband analytic + slow envelope & its phase

def band_envelope_and_slow_phase(x, fs, f0, half=0.6, slow_band=(0.003,0.03)):
    """Return: slow_env (band‑passed envelope in slow_band), slow_phase (Hilbert angle)."""
    xnb = _butter_bandpass(x, fs, f0-half, f0+half)
    env = np.abs(signal.hilbert(xnb))           # amplitude envelope
    # band‑pass the envelope in minute‑scale band
    slo = _butter_bandpass(env, fs, slow_band[0], slow_band[1])
    z = signal.hilbert(slo)
    return slo, np.angle(z)

# --------------- windowing & bootstrap ---------------

def windows_to_samples(wins, fs, N):
    segs = []
    for (a,b) in wins:
        i0,i1 = int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(N,i1)
        if i1>i0: segs.append((i0,i1))
    return segs

# block bootstrap (circular)

def block_bootstrap_series(x, block_N, out_N, rng):
    idx=[]; filled=0
    while filled<out_N:
        start=int(rng.integers(0, len(x)))
        end=start+block_N
        if end<=len(x): idx.extend(range(start,end))
        else: idx.extend(list(range(start,len(x)))+list(range(0,end-len(x))))
        filled+=block_N
    idx=idx[:out_N]
    return x[idx]

# --------------- core metrics ---------------



def xcorr_lag(env_f, env_ref, fs, max_lag_s=30.0):
    # drop any NaNs pairwise
    env_f = np.asarray(env_f, float)
    env_ref = np.asarray(env_ref, float)
    m = np.isfinite(env_f) & np.isfinite(env_ref)
    env_f = env_f[m]; env_ref = env_ref[m]
    # bail if too short or flat
    if env_f.size < 8 or env_ref.size < 8:
        return np.nan, np.nan, np.array([0.0]), np.array([np.nan])
    if np.nanstd(env_f) < 1e-12 or np.nanstd(env_ref) < 1e-12:
        return np.nan, np.nan, np.array([0.0]), np.array([np.nan])

    # z-score (robust to constant signals)
    x = (env_f - np.nanmean(env_f)) / (np.nanstd(env_f) + 1e-12)
    y = (env_ref - np.nanmean(env_ref)) / (np.nanstd(env_ref) + 1e-12)

    c = signal.correlate(x, y, mode='full', method='auto')
    lags = signal.correlation_lags(len(x), len(y), mode='full')

    # restrict to ±max_lag
    L = int(round(max_lag_s * fs))
    sel = (lags >= -L) & (lags <= L)
    c = c[sel]; lags = lags[sel]

    if not np.any(np.isfinite(c)):
        return np.nan, np.nan, lags/fs, c

    # argmax ignoring NaNs
    k = int(np.nanargmax(np.nan_to_num(c, nan=-np.inf)))
    tau_s = float(lags[k]) / fs           # +τ => env_f leads env_ref
    rmax  = float(c[k]) / float(len(x))   # simple scale-stable norm
    return tau_s, rmax, lags/fs, c




def lag_ci_bootstrap(env_f, env_ref, fs, max_lag_s=30.0, n_boot=500, block_sec=10.0, seed=23):
    rng = np.random.default_rng(seed)
    N = min(len(env_f), len(env_ref))
    bN = max(8, int(round(block_sec*fs)))
    taus = []
    for _ in range(int(n_boot)):
        xf = block_bootstrap_series(env_f, bN, N, rng)
        xr = block_bootstrap_series(env_ref, bN, N, rng)
        tau,_r, _L,_C = xcorr_lag(xf, xr, fs, max_lag_s=max_lag_s)
        taus.append(tau)
    lo, hi = np.nanpercentile(taus, [2.5, 97.5])
    return float(lo), float(hi)


def phase_lead_probability(phi_f, phi_ref, n_boot=1000, block_len=200, seed=7):
    """Return P_lead = Pr(Δϕ ∈ (0,π)) with block‑bootstrap 95% CI.
    Δϕ = wrap(φ_f − φ_ref) to (−π,π].
    """
    rng = np.random.default_rng(seed)
    dphi = np.angle(np.exp(1j*(phi_f - phi_ref)))  # wrapped
    p = float(np.nanmean((dphi>0) & (dphi<np.pi)))
    N = len(dphi)
    B = max(20, int(block_len))
    ps=[]
    for _ in range(int(n_boot)):
        idx=[]; filled=0
        while filled<N:
            start=int(rng.integers(0,N))
            end=start+B
            if end<=N: idx.extend(range(start,end))
            else: idx.extend(list(range(start,N))+list(range(0,end-N)))
            filled+=B
        idx=idx[:N]
        d = dphi[idx]
        ps.append(float(np.nanmean((d>0) & (d<np.pi))))
    lo, hi = np.nanpercentile(ps, [2.5,97.5])
    return p, float(lo), float(hi)

# --------------- main analysis ---------------

def analyze_lead_lag_temporal(
    RECORDS,
    eeg_channel: str,
    windows: dict,  # {'pre':[(t0,t1),...], 'ignition':[(..)], 'rebound':[(..)]}
    time_col='Timestamp',
    # frequency families
    fundamental=7.83,
    family_subh=(3.915, 2.61, 1.9575, 1.566),      # SubH(2–5)
    family_low=(14.3, 20.8, 27.3, 33.8),           # (14–34)
    family_high=(40.3, 46.8, 53.3, 59.8),          # (40–60)
    half_bw=0.6, slow_band=(0.003,0.03),
    max_lag_s=30.0, n_boot=500, block_sec=10.0,
    coincident_tol_s=2.0,
    out_dir='exports_leadlag', show=True
):
    """Compute envelope xcorr lags & envelope‑phase lead probabilities per window and family.
    Saves summary CSV and simple bar plots with 95% CIs. Returns summary DataFrame.
    """
    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)

    x = RECORDS[eeg_channel].astype(float).values if eeg_channel in RECORDS.columns else pd.to_numeric(RECORDS['EEG.'+eeg_channel], errors='coerce').fillna(0.0).values.astype(float)
    N = len(x)

    # Precompute slow envelope + phase for fundamental and all freqs
    def env_phase_for(f):
        slo, phi = band_envelope_and_slow_phase(x, fs, f, half=half_bw, slow_band=slow_band)
        return slo, phi

    env_phi = {}
    all_freqs = {'fund':(fundamental,), 'subh':tuple(family_subh), 'low':tuple(family_low), 'high':tuple(family_high)}
    for fam, freqs in all_freqs.items():
        for f in freqs:
            env_phi[f] = env_phase_for(f)

    env_fund, phi_fund = env_phi[fundamental]

    # helper to slice windows and compute stats
    def analyze_window(win_name, segs):
        rows=[]
        # family containers for consensus ordering (use median lag across members)
        fam_lags = {'subh':[], 'fund':[], 'low':[], 'high':[]}
        for fam, freqs in [('subh', family_subh), ('fund',(fundamental,)), ('low', family_low), ('high', family_high)]:
            for f in freqs:
                env_f, phi_f = env_phi[f]
                # concatenate window segments
                idxs = windows_to_samples(segs, fs, N)
                ef = np.concatenate([env_f[i0:i1] for (i0,i1) in idxs]) if idxs else np.array([])
                er = np.concatenate([env_fund[i0:i1] for (i0,i1) in idxs]) if idxs else np.array([])
                pf = np.concatenate([phi_f[i0:i1] for (i0,i1) in idxs]) if idxs else np.array([])
                pr = np.concatenate([phi_fund[i0:i1] for (i0,i1) in idxs]) if idxs else np.array([])
                if ef.size<10 or er.size<10:
                    continue
                tau, rmax, _lags,_c = xcorr_lag(ef, er, fs, max_lag_s=max_lag_s)
                lo, hi = lag_ci_bootstrap(ef, er, fs, max_lag_s=max_lag_s, n_boot=n_boot, block_sec=block_sec)
                P, Plo, Phi = phase_lead_probability(pf, pr, n_boot=n_boot, block_len=int(block_sec*fs))
                fam_lags[fam].append(tau)
                rows.append({'window':win_name, 'family':fam, 'f0':f, 'tau_env_s':tau, 'tau_lo':lo, 'tau_hi':hi,
                             'P_lead':P, 'P_lead_lo':Plo, 'P_lead_hi':Phi, 'rmax':rmax,
                             'n_samples': int(ef.size)})
        # consensus ordering score
        score, n_tests = consensus_order_score(fam_lags, tol=coincident_tol_s)
        rows.append({'window':win_name, 'family':'CONSENSUS', 'f0':np.nan, 'tau_env_s':np.nan,
                     'tau_lo':np.nan, 'tau_hi':np.nan, 'P_lead':np.nan,
                     'P_lead_lo':np.nan, 'P_lead_hi':np.nan, 'rmax':np.nan,
                     'n_samples':int(sum(len(v) for v in fam_lags.values())),
                     'consensus_score_pct': 100.0*score, 'n_tests': n_tests})
        return rows

    def consensus_order_score(fam_lags, tol=2.0):
        """Return fraction of pairwise constraints satisfied for ordering:
        SubH → Fund → Low → High. We use median lag per family; Fund≈Low treated as coincident
        if |median(Fund) − median(Low)| ≤ tol.
        Positive τ means family leads FUND; negative means lags (by our xcorr sign choice).
        Constraints:
          median(SubH) >= median(Fund) + tol  (SubH leads)
          |median(Low) − median(Fund)| ≤ tol  (coincident)
          median(High) ≤ median(Fund) − tol  (High lags)
          median(Low) ≥ median(High) + tol   (Low ahead of High)
          median(SubH) ≥ median(Low) + tol   (SubH ahead of Low)
          median(SubH) ≥ median(High) + tol  (SubH ahead of High)
        """
        import math
        # median per family; if empty, set nan
        med = {k: (np.nan if len(v)==0 else float(np.nanmedian(v))) for k,v in fam_lags.items()}
        tests=0; ok=0
        def inc(test):
            nonlocal tests, ok
            tests += 1
            if test: ok += 1
        if not math.isnan(med['subh']) and not math.isnan(med['fund']):
            inc(med['subh'] >= med['fund'] + tol)
        if not math.isnan(med['low']) and not math.isnan(med['fund']):
            inc(abs(med['low'] - med['fund']) <= tol)
        if not math.isnan(med['high']) and not math.isnan(med['fund']):
            inc(med['high'] <= med['fund'] - tol)
        if not math.isnan(med['low']) and not math.isnan(med['high']):
            inc(med['low'] >= med['high'] + tol)
        if not math.isnan(med['subh']) and not math.isnan(med['low']):
            inc(med['subh'] >= med['low'] + tol)
        if not math.isnan(med['subh']) and not math.isnan(med['high']):
            inc(med['subh'] >= med['high'] + tol)
        return (ok / max(1, tests)), tests

    # run per window
    all_rows=[]
    for wname, segs in windows.items():
        if not segs: continue
        all_rows.extend(analyze_window(wname, segs))

    summary = pd.DataFrame(all_rows)
    csv_path = os.path.join(out_dir, 'leadlag_summary.csv')
    summary.to_csv(csv_path, index=False)

    # ------------------ simple graphs ------------------
    # Bar: τ̂_env by family (per window) with CI whiskers
    fam_order = ['subh','fund','low','high']
    for wname in sorted(set(summary['window'])):
        S = summary[(summary['window']==wname) & (summary['family'].isin(fam_order))]
        if S.empty: continue
        # aggregate per family using median across f0s
        agg = S.groupby('family').agg({'tau_env_s':'median','tau_lo':'median','tau_hi':'median'}).reindex(fam_order)
        x = np.arange(len(agg)); y = agg['tau_env_s'].values
        err_lo = y - agg['tau_lo'].values; err_hi = agg['tau_hi'].values - y
        fig, ax = plt.subplots(figsize=(8,3.2))
        ax.bar(x, y, yerr=[err_lo, err_hi], width=0.6, capsize=3)
        ax.axhline(0, color='k', lw=0.7, alpha=0.6)
        ax.set_xticks(x); ax.set_xticklabels(['SubH(2–5)','7.83','14–34','40–60'])
        ax.set_ylabel('τ̂_env (s)'); ax.set_title(f'Envelope lag by family — {wname}')
        ax.grid(True, axis='y', alpha=0.25, linestyle=':')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'lag_bars_{wname}.png'), dpi=160)
        if show: plt.show(); plt.close()

    # Bar: P_lead by family (per window)
    for wname in sorted(set(summary['window'])):
        S = summary[(summary['window']==wname) & (summary['family'].isin(fam_order))]
        if S.empty: continue
        agg = S.groupby('family').agg({'P_lead':'median','P_lead_lo':'median','P_lead_hi':'median'}).reindex(fam_order)
        x = np.arange(len(agg)); y = agg['P_lead'].values
        err_lo = y - agg['P_lead_lo'].values; err_hi = agg['P_lead_hi'].values - y
        fig, ax = plt.subplots(figsize=(8,3.2))
        ax.bar(x, y, yerr=[err_lo, err_hi], width=0.6, capsize=3)
        ax.set_ylim(0,1)
        ax.axhline(0.5, color='k', lw=0.7, ls='--', alpha=0.5)
        ax.set_xticks(x); ax.set_xticklabels(['SubH(2–5)','7.83','14–34','40–60'])
        ax.set_ylabel('P_lead'); ax.set_title(f'Phase‑lead probability (slow envelope) — {wname}')
        ax.grid(True, axis='y', alpha=0.25, linestyle=':')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'plead_bars_{wname}.png'), dpi=160)
        if show: plt.show(); plt.close()

    # Consensus score table saved separately
    CONS = summary[summary['family']=='CONSENSUS'][['window','consensus_score_pct','n_tests']]
    if not CONS.empty:
        CONS.to_csv(os.path.join(out_dir, 'consensus_scores.csv'), index=False)

    return summary

# ------------------ Example usage ------------------
# windows = {
#   'pre': [(0, 290)],
#   'ignition': [(290, 310), (580, 600)],
#   'rebound': [(325, 580)]
# }
# summary = analyze_lead_lag_temporal(
#     RECORDS,
#     eeg_channel='EEG.O1',           # or 'EEG.VIRT' if you built a virtual posterior
#     windows=windows,
#     fundamental=7.83,
#     family_subh=(3.915, 2.61, 1.9575, 1.566),
#     family_low=(14.3, 20.8, 27.3, 33.8),
#     family_high=(40.3, 46.8, 53.3, 59.8),
#     half_bw=0.6, slow_band=(0.003,0.03),
#     max_lag_s=30.0, n_boot=500, block_sec=10.0,
#     coincident_tol_s=2.0,
#     out_dir='exports_leadlag', show=True
# )


In [ ]:
def virtual_eeg_snr_weighted(RECORDS, channels, fs, f0, half=0.6, time_col='Timestamp'):
    """Return (v_sig, weights) where v_sig is SNR‑weighted sum of channels normalized to unit gain."""
    X = []
    for ch in channels:
        if ch in RECORDS.columns:
            x = pd.to_numeric(RECORDS[ch], errors='coerce').fillna(0.0).values.astype(float)
        elif ('EEG.'+ch) in RECORDS.columns:
            x = pd.to_numeric(RECORDS['EEG.'+ch], errors='coerce').fillna(0.0).values.astype(float)
        else:
            raise ValueError(f"{ch} not in dataframe")
    X.append(x)
    X = np.vstack(X) # shape (C, N)
    # SNR per channel
    snrs = np.array([snr_at_f0(x, fs, f0, half=half) for x in X])
    w = snrs / (np.sum(snrs) + 1e-12)
    v = np.dot(w, X) # (N,)
    return v, w

def snr_at_f0(x, fs, f0, half=0.6, flank=2.0):
    """SNR = band power at f0±half / average flank power at [f0±(half+δ) .. f0±(half+δ+flank)]."""
    sig = _bandpass(x, fs, f0-half, f0+half)
    p_sig = np.mean(sig**2)
    # two flanks: below and above
    lo1, lo2 = max(0.01, f0-(half+flank+0.5)), f0-(half+0.5)
    hi1, hi2 = f0+(half+0.5), f0+(half+flank+0.5)
    if lo2>lo1:
        fl = _bandpass(x, fs, lo1, lo2); p_fl = np.mean(fl**2)
    else:
        p_fl = 0.0
    if hi2>hi1:
        fh = _bandpass(x, fs, hi1, hi2); p_fh = np.mean(fh**2)
    else:
        p_fh = 0.0
    p_noise = np.mean([p for p in [p_fl, p_fh] if np.isfinite(p)]) or 1e-12
    return float(p_sig / p_noise)



def _bandpass(x, fs, lo, hi, order=4):
    ny=0.5*fs; lo=max(1e-6, min(lo, 0.99*ny)); hi=max(lo+1e-6, min(hi, 0.999*ny))
    b,a=signal.butter(order, [lo/ny, hi/ny], btype='band'); 
    return signal.filtfilt(b,a,x)

In [ ]:
windows = {
  'pre': [(0, 290)],
  'ignition': [(290, 310), (580, 600)]
}

roi = ['EEG.O1','EEG.O2','EEG.P7','EEG.P8']
v_eeg, w = virtual_eeg_snr_weighted(RECORDS, roi, 128, f0=7.83, half=0.6)
RECORDS['EEG.VIRT'] = v_eeg

summary = analyze_lead_lag_temporal(
    RECORDS,
    eeg_channel='EEG.O1',          # or your virtual posterior 'EEG.VIRT'
    windows=windows,
    fundamental=7.83,
    family_subh=(3.915, 2.61, 1.9575, 1.566),
    family_low=(14.3, 20.8, 27.3, 33.8),
    family_high=(40.3, 46.8, 53.3, 59.8),
    half_bw=0.6, slow_band=(0.002, 0.05),
    max_lag_s=30.0, n_boot=500, block_sec=10.0,
    coincident_tol_s=2.0,
    out_dir='exports_leadlag', show=True
)


In [ ]:
RECORDS['EEG.VIRT']

In [ ]:
"""
Disentangling waveform shape vs. true multi‑mode resonance (0.1–60 Hz)
=====================================================================
Implements simple tests, graphs, and summary stats for:
  (A) Cycle‑by‑cycle morphology at ~7–9 Hz (rise/decay ratio, peak/trough sharpness,
      zero‑crossing asymmetry) to quantify non‑sinusoidal shape.
  (B) IRASA‑cleaned oscillations (remove 1/f fractal) and re‑inspect harmonic peaks.
  (C) Polyspectra (auto‑bicoherence EEG; cross‑bicoherence SR→EEG) on a discrete
      SR frequency set (fundamental + harmonics up to 60 Hz).

Rule‑of‑thumb interpretation:
  • shape‑only → high auto‑bicoherence without cross‑bicoherence; harmonics shrink after IRASA;
    morphology metrics high; harmonic power correlates with sharpness/asymmetry.
  • true resonance → both auto‑ and cross‑bicoherence significant (e.g., (7.83,7.83)→15.66),
    IRASA‑oscillatory peaks persist; morphology may be neutral.

Outputs:
  • PNGs: morphology histograms/scatter, IRASA PSD plots, auto‑/cross‑bicoherence heatmaps.
  • CSV: summary table with key metrics & simple classification.

Dependencies: numpy, scipy, matplotlib, pandas.
"""

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import signal

# ----------------------------- I/O helpers -----------------------------

def ensure_dir(d):
    if d: os.makedirs(d, exist_ok=True)
    return d

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    if time_col in df.columns:
        s = df[time_col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values; return time_col
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values; return time_col
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

def get_series(df, name):
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
    alt='EEG.'+name
    if alt in df.columns:
        return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
    raise ValueError(f'{name} not in dataframe')

# ----------------------------- Filters -----------------------------

def _bandpass(x, fs, lo, hi, order=4):
    ny=0.5*fs; lo=max(1e-6, min(lo, 0.99*ny)); hi=max(lo+1e-6, min(hi, 0.999*ny))
    b,a=signal.butter(order, [lo/ny, hi/ny], btype='band'); return signal.filtfilt(b,a,x)

def _lowpass(x, fs, hi, order=4):
    ny=0.5*fs; hi=max(1e-6, min(hi, 0.999*ny))
    b,a=signal.butter(order, hi/ny, btype='low'); return signal.filtfilt(b,a,x)

# ----------------------------- (A) Cycle‑by‑cycle morphology -----------------------------

def cycles_morphology(x, fs, f0=7.83, half=1.0, sharp_win=0.02):
    """Compute cycle features from bandpassed x around f0±half.
    Returns DataFrame with period, rise/decay times, ratio, peak/trough sharpness,
    zero‑crossing asymmetry (ZC asym), and timestamps of cycles (center time).
    """
    xb = _bandpass(x, fs, f0-half, f0+half)
    # find zero crossings for cycle boundaries
    s = np.signbit(xb)
    zc_idx = np.where(np.diff(s.astype(int)) != 0)[0]
    # peaks & troughs
    peaks,_ = signal.find_peaks(xb)
    troughs,_ = signal.find_peaks(-xb)
    # helper to nearest peak after a trough etc.
    def next_idx(arr, i):
        j = arr.searchsorted(i, side='left')
        return arr[j] if j < len(arr) else None
    peaks = np.asarray(peaks); troughs = np.asarray(troughs)
    rows=[]; w = int(round(sharp_win*fs))
    for i in range(len(troughs)-1):
        t0 = troughs[i]; t1 = troughs[i+1]
        # ensure one peak between troughs
        pk = peaks[(peaks>t0)&(peaks<t1)]
        if pk.size==0: continue
        pk = pk[0]
        # zero‑crossings inside the cycle
        z = zc_idx[(zc_idx>=t0)&(zc_idx<=t1)]
        # features
        period = (t1 - t0)/fs
        rise   = (pk - t0)/fs
        decay  = (t1 - pk)/fs
        rz     = rise/(decay+1e-12)
        # sharpness via local slope/curvature proxy around extrema
        a = max(0, pk - w); b = min(len(xb), pk + w + 1)
        peak_sharp = float(xb[pk] - 0.5*(xb[a] + xb[b-1])) if (b-a)>=3 else float('nan')
        a = max(0, t0 - w); b = min(len(xb), t0 + w + 1)
        trough_sharp = float(0.5*(xb[a] + xb[b-1]) - xb[t0]) if (b-a)>=3 else float('nan')
        # ZC asymmetry: fraction of cycle spent above zero vs below
        if z.size >= 2:
            # first crossing after t0 and next crossing
            z1, z2 = z[0], z[1]
            above = (z2 - z1)/fs
            zc_asym = above / (period + 1e-12)
        else:
            zc_asym = float('nan')
        rows.append({'t_center': (t0+t1)/(2*fs), 'period_s': period, 'rise_s': rise, 'decay_s': decay,
                     'rise_decay_ratio': rz, 'peak_sharp': peak_sharp, 'trough_sharp': trough_sharp,
                     'zc_asym': zc_asym})
    return pd.DataFrame(rows)

# ----------------------------- (B) IRASA‑like background removal -----------------------------

def irasa_psd(x, fs, hset=(1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9), nperseg=None, fmax=60.0):
    """Approximate IRASA: resample by h and 1/h, compute PSDs, map freqs back by /h and *h,
    take geometric mean and then median over h to estimate fractal; subtract (in linear power)
    to get oscillatory. Returns f, Pxx, P_frac, P_osc (clipped >=0).
    """
    x = np.asarray(x, float)
    if nperseg is None:
        nperseg = int(max(2*fs, 1024))
    # base PSD
    f, P = signal.welch(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg//2,
                        nfft=int(2**np.ceil(np.log2(nperseg*2))), scaling='density')
    mask = (f>0) & (f<=min(fmax, 0.999*0.5*fs))
    f = f[mask]; P = P[mask]
    # surrogate fractal estimate
    def res_psd(y, up, down):
        z = signal.resample_poly(y, up, down)
        fs2 = fs*(up/float(down))
        f2, P2 = signal.welch(z, fs=fs2, window='hann', nperseg=nperseg, noverlap=nperseg//2,
                              nfft=int(2**np.ceil(np.log2(nperseg*2))), scaling='density')
        return f2, P2
    P_fracs=[]
    for h in hset:
        # up/down factors via rational approximation
        from fractions import Fraction
        frac = Fraction(str(h)).limit_denominator(64)
        up, down = frac.numerator, frac.denominator
        f_h, Ph = res_psd(x, up, down)
        f_hm, Phm = res_psd(x, down, up)  # 1/h
        # map both to base freq grid
        # resampling by h compresses time → expands freq by h; to map back divide freqs by h
        fh_map = f_h/float(h);  Ph_map = np.interp(f, fh_map, Ph, left=np.nan, right=np.nan)
        fhm_map = f_hm*float(h); Phm_map = np.interp(f, fhm_map, Phm, left=np.nan, right=np.nan)
        # geometric mean
        G = np.sqrt(np.maximum(Ph_map, 1e-20) * np.maximum(Phm_map, 1e-20))
        P_fracs.append(G)
    P_frac = np.nanmedian(np.vstack(P_fracs), axis=0)
    P_osc = np.clip(P - P_frac, 0, None)
    return f, P, P_frac, P_osc

# ----------------------------- (C) Discrete (cross‑)bicoherence -----------------------------

def _fft_segments(x, fs, nperseg, step):
    w = signal.hann(nperseg, sym=False)
    hop = nperseg - step
    nseg = 1 + max(0, (len(x)-nperseg)//hop)
    Xs=[]
    for i in range(nseg):
        s = i*hop; e = s + nperseg
        seg = x[s:e]
        if len(seg) < nperseg: break
        seg = seg - np.mean(seg)
        X = np.fft.rfft(w*seg, n=nperseg)
        Xs.append(X)
    Xs = np.asarray(Xs)
    freqs = np.fft.rfftfreq(nperseg, d=1.0/fs)
    return freqs, Xs


def bicoherence_discrete_auto(x, fs, f_list, nperseg=None, step=None):
    """Auto‑bicoherence on a discrete frequency list f_list (Hz). Returns matrix B[i,j] for f1=f_list[i], f2=f_list[j] with f1+f2 in grid.
    """
    if nperseg is None:
        nperseg = int(max(4*fs, 2048))
    if step is None:
        step = nperseg//2
    freqs, Xs = _fft_segments(x, fs, nperseg, step)
    # map desired freqs to bins
    def bin_idx(f): return int(np.argmin(np.abs(freqs - f)))
    idxs = [bin_idx(f) for f in f_list]
    B = np.full((len(f_list), len(f_list)), np.nan, float)
    for i, fi in enumerate(f_list):
        for j, fj in enumerate(f_list):
            fk = fi + fj
            if fk > freqs[-1]:
                continue
            ik = bin_idx(fk); ii = idxs[i]; jj = idxs[j]
            num = np.mean(Xs[:, ii] * Xs[:, jj] * np.conj(Xs[:, ik]))
            den = np.sqrt(np.mean(np.abs(Xs[:, ii]*Xs[:, jj])**2) * np.mean(np.abs(Xs[:, ik])**2) + 1e-20)
            B[i,j] = np.abs(num) / (den + 1e-20)
    return np.asarray(f_list), B


def bicoherence_discrete_cross(sr, eeg, fs, f_list, nperseg=None, step=None):
    """Cross‑bicoherence variant: B_sse(f1,f2) = <S(f1) S(f2) E*(f1+f2)> / sqrt(<|S(f1)S(f2)|^2><|E(f1+f2)|^2>).
    """
    if nperseg is None:
        nperseg = int(max(4*fs, 2048))
    if step is None:
        step = nperseg//2
    freqs, Ss = _fft_segments(sr, fs, nperseg, step)
    _,    Es = _fft_segments(eeg, fs, nperseg, step)
    def bin_idx(f): return int(np.argmin(np.abs(freqs - f)))
    idxs = [bin_idx(f) for f in f_list]
    B = np.full((len(f_list), len(f_list)), np.nan, float)
    for i, fi in enumerate(f_list):
        for j, fj in enumerate(f_list):
            fk = fi + fj
            if fk > freqs[-1]:
                continue
            ik = bin_idx(fk); ii = idxs[i]; jj = idxs[j]
            num = np.mean(Ss[:, ii] * Ss[:, jj] * np.conj(Es[:, ik]))
            den = np.sqrt(np.mean(np.abs(Ss[:, ii]*Ss[:, jj])**2) * np.mean(np.abs(Es[:, ik])**2) + 1e-20)
            B[i,j] = np.abs(num) / (den + 1e-20)
    return np.asarray(f_list), B

# ----------------------------- Orchestrator -----------------------------

def analyze_shape_vs_resonance(
    RECORDS,
    eeg_channel: str,
    sr_channel: str,
    time_col='Timestamp',
    fundamental=7.83,
    harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_bw=0.6,
    slow_band=(0.003,0.03),
    fmax=60.0,
    nperseg_irasa=None,
    nperseg_bico=None,
    out_dir='exports_shape_vs_res', show=True,
    n_perm=200
):
    """Run morphology, IRASA, and (cross‑)bicoherence tests; save figures and a summary CSV."""
    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)

    x = get_series(RECORDS, eeg_channel)
    s = get_series(RECORDS, sr_channel)

    # ---- (A) Morphology at ~fundamental ----
    morph = cycles_morphology(x, fs, f0=fundamental, half=half_bw, sharp_win=0.02)

    # Relation to harmonic power: compute PSD and ratio (sum harmonics 14–60 / power at 7.8)
    nper = int(max(4*fs, 2048))
    f_psd, P = signal.welch(x, fs=fs, window='hann', nperseg=nper, noverlap=nper//2,
                            nfft=int(2**np.ceil(np.log2(nper*2))), scaling='density')
    mask = (f_psd>0) & (f_psd<=min(fmax, 0.999*0.5*fs))
    f_psd = f_psd[mask]; P = P[mask]
    def at(freq, bw=0.4):
        m = (f_psd>=freq-bw) & (f_psd<=freq+bw)
        return float(np.trapz(P[m], f_psd[m])) if np.any(m) else 0.0
    fund_pow = at(fundamental, bw=half_bw)
    harm_bw = 0.5
    harm_list = [f for f in harmonics if f>fundamental and f<=fmax]
    harm_pow = sum(at(f, bw=harm_bw) for f in harm_list)
    harm_ratio = float(harm_pow / (fund_pow + 1e-12))

    # ---- (B) IRASA ----
    fI, Pxx, Pfrac, Posc = irasa_psd(x, fs, nperseg=nperseg_irasa, fmax=fmax)

    # ---- (C) Bicoherence on discrete SR set ----
    f_list = [f for f in harmonics if f <= min(fmax, 0.999*0.5*fs)]
    nper_b = nperseg_bico or int(max(4*fs, 4096))
    step_b = nper_b//2
    fgrid, Bauto = bicoherence_discrete_auto(x, fs, f_list, nperseg=nper_b, step=step_b)
    _,     Bcross = bicoherence_discrete_cross(s, x, fs, f_list, nperseg=nper_b, step=step_b)

    # Simple surrogates: circularly shift SR vs EEG for cross; sign‑flip segments for auto
    rng = np.random.default_rng(7)
    def sur_cross(nr=200):
        vals=[]
        _, Es = _fft_segments(x, fs, nper_b, step_b)
        freqs, Ss = _fft_segments(s, fs, nper_b, step_b)
        for _ in range(int(nr)):
            # circular shift SR in time domain by random samples
            sh = int(rng.integers(1, len(s)-1))
            s_sh = np.r_[s[-sh:], s[:-sh]]
            _,     Bc = bicoherence_discrete_cross(s_sh, x, fs, f_list, nperseg=nper_b, step=step_b)
            vals.append(Bc)
        return np.stack(vals, axis=0)  # (nr, F, F)
    def sur_auto(nr=200):
        vals=[]
        for _ in range(int(nr)):
            # randomly invert segments to break consistent triple phase
            x_sh = x.copy()
            segN = nper_b
            for start in range(0, len(x_sh)-segN, segN):
                if rng.random()<0.5:
                    x_sh[start:start+segN] *= -1
            _, Ba = bicoherence_discrete_auto(x_sh, fs, f_list, nperseg=nper_b, step=step_b)
            vals.append(Ba)
        return np.stack(vals, axis=0)

    SC = sur_cross(n_perm)
    SA = sur_auto(n_perm)
    thr_cross = np.nanpercentile(SC, 95, axis=0)
    thr_auto  = np.nanpercentile(SA, 95, axis=0)

    # Key cells near (7.83,7.83)->15.66
    def nearest_idx(arr, val):
        return int(np.argmin(np.abs(np.asarray(arr)-val)))
    i7 = nearest_idx(fgrid, fundamental)
    cross_7_7 = float(Bcross[i7, i7]) if i7 < len(fgrid) else np.nan
    auto_7_7  = float(Bauto[i7, i7])  if i7 < len(fgrid) else np.nan
    cross_thr = float(thr_cross[i7, i7]) if i7 < len(fgrid) else np.nan
    auto_thr  = float(thr_auto[i7, i7])  if i7 < len(fgrid) else np.nan

    # ----- simple classification -----
    # thresholds can be tuned; start conservative with surrogate 95th percentile
    shape_only = (auto_7_7 > auto_thr) and not (cross_7_7 > cross_thr)
    true_res   = (auto_7_7 > auto_thr) and (cross_7_7 > cross_thr)
    label = 'shape_only' if shape_only else ('true_resonance' if true_res else 'ambiguous')

    # ----------------------------- plots -----------------------------
    # Morphology distributions & scatter vs harmonic ratio
    if not morph.empty:
        fig, axes = plt.subplots(1,3, figsize=(12,3.2))
        axes[0].hist(morph['rise_decay_ratio'].dropna(), bins=30, alpha=0.9)
        axes[0].set_title('Rise/Decay ratio'); axes[0].set_xlabel('ratio'); axes[0].set_ylabel('count')
        axes[1].hist(morph['peak_sharp'].dropna(), bins=30, alpha=0.9)
        axes[1].set_title('Peak sharpness')
        axes[2].hist(morph['zc_asym'].dropna(), bins=30, alpha=0.9)
        axes[2].set_title('ZC asymmetry'); axes[2].set_xlabel('fraction of cycle above 0')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'morph_hist.png'), dpi=160)
        if show: plt.show(); plt.close()
        # scatter sharpness vs harmonic ratio
        fig, ax = plt.subplots(figsize=(5.2,3.2))
        ax.scatter(morph['peak_sharp'], np.full(len(morph), harm_ratio), s=12, alpha=0.4)
        ax.set_xlabel('Peak sharpness (a.u.)'); ax.set_ylabel('Harmonic ratio (14–60 / 7.8)')
        ax.set_title('Harmonics vs shape (quick view)')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'morph_scatter.png'), dpi=160)
        if show: plt.show(); plt.close()

    # IRASA plots
    fig, ax = plt.subplots(figsize=(7.8,3.2))
    ax.plot(fI, 10*np.log10(Pxx+1e-20), label='PSD (Welch)')
    ax.plot(fI, 10*np.log10(Pfrac+1e-20), label='Fractal (IRASA≈)')
    ax.plot(fI, 10*np.log10(Posc+1e-20), label='Oscillatory (PSD−Fractal)')
    ax.set_xlim(0, fmax); ax.set_xlabel('Hz'); ax.set_ylabel('dB')
    ax.set_title('IRASA‑cleaned oscillations'); ax.legend(); ax.grid(True, alpha=0.25, linestyle=':')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'irasa_psd.png'), dpi=160)
    if show: plt.show(); plt.close()

    # Bicoherence heatmaps
    extent=[fgrid[0], fgrid[-1], fgrid[0], fgrid[-1]]
    def heat(M, thr, title, fname):
        plt.figure(figsize=(6.4,5.4))
        plt.imshow(M, origin='lower', aspect='equal', extent=extent, vmin=0, vmax=np.nanmax(M))
        cb=plt.colorbar(); cb.set_label('bicoherence')
        yy, xx = np.where(M > thr)
        if yy.size:
            plt.scatter(fgrid[xx], fgrid[yy], s=10, c='cyan', alpha=0.7, label='> null95')
            plt.legend(loc='upper right', fontsize=8)
        plt.xlabel('f1 (Hz)'); plt.ylabel('f2 (Hz)'); plt.title(title)
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, fname), dpi=160)
        if show: plt.show(); plt.close()
    heat(Bauto, thr_auto,  'Auto‑bicoherence EEG',  'bico_auto.png')
    heat(Bcross, thr_cross,'Cross‑bicoherence SR→EEG','bico_cross.png')

    # ----------------------------- summary table -----------------------------
    rows=[{
        'fundamental': fundamental,
        'harm_ratio_14_60_over_7_8': harm_ratio,
        'auto_bico_7.83,7.83': auto_7_7,
        'auto_null95_7.83,7.83': auto_thr,
        'cross_bico_7.83,7.83': cross_7_7,
        'cross_null95_7.83,7.83': cross_thr,
        'morph_median_rise_decay': float(morph['rise_decay_ratio'].median()) if not morph.empty else np.nan,
        'morph_median_peak_sharp': float(morph['peak_sharp'].median()) if not morph.empty else np.nan,
        'morph_median_zc_asym': float(morph['zc_asym'].median()) if not morph.empty else np.nan,
        'classification': label
    }]
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(out_dir, 'shape_vs_res_summary.csv'), index=False)

    return {
        'morphology': morph,
        'irasa': (fI, Pxx, Pfrac, Posc),
        'bicoherence': {'fgrid': fgrid, 'auto': Bauto, 'cross': Bcross,
                        'thr_auto': thr_auto, 'thr_cross': thr_cross},
        'summary': summary,
        'label': label,
        'out_dir': out_dir
    }

# ----------------------------- Example usage -----------------------------
# res = analyze_shape_vs_resonance(
#     RECORDS,
#     eeg_channel='EEG.O1',
#     sr_channel='EEG.Pz',        # or a magnetometer channel
#     time_col='Timestamp',
#     fundamental=7.83,
#     harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
#     half_bw=0.6,
#     slow_band=(0.003,0.03),
#     fmax=60.0,
#     n_perm=200,
#     out_dir='exports_shape_vs_res', show=True
# )


In [ ]:
res = analyze_shape_vs_resonance(
    RECORDS,
    eeg_channel='EEG.O1',           # or your virtual posterior 'EEG.VIRT'
    sr_channel='EEG.F4',            # or magnetometer
    fundamental=7.83,
    harmonics=(1.57,1.96,2.61,3.92,7.83,14.3,20.8,27.3,33.8,40.3,46.8),
    half_bw=0.6,
    out_dir='exports_shape_vs_res',
    show=True, n_perm=200
)

In [ ]:
"""
Cross-frequency coupling expansions tied to Schumann harmonics (0.1–60 Hz)
==========================================================================
Implements simple tests + graphs for:
  1) CF-PLV across the harmonic ladder: PLV between φ_7.83 and φ_{m·7.83} using 1:m locking
     (i.e., |<e^{i(m φ1 − φm)}>|). Baseline vs ignition with surrogate p-values.
  2) Band-limited PAC per order: MVL & Tort MI with low-phase fixed at each Schumann line and
     high-frequency amplitude at the corresponding harmonic band; ignition vs baseline; 
     **shape-controlled surrogates** via within-cycle phase shuffles.
  3) Cross-frequency directionality (CFDC-like): lagged regression ΔR² where phase@7.83 predicts
     A_m at t+τ beyond A_m(t); reversed model estimates A→phase; surrogate p-values via
     circular time-shifts. Reports peak ΔR² and lag.

Outputs
-------
• PNGs: CF-PLV bars, PAC bars (MVL/MI), directionality lag curves (+ reverse), per harmonic.
• CSV: cfc_harmonics_summary.csv with metrics & p-values per window and order.

Deps: numpy, scipy, matplotlib, pandas.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

# ---------------------------- helpers ----------------------------

def ensure_dir(d):
    if d: os.makedirs(d, exist_ok=True)
    return d

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    if time_col in df.columns:
        s = df[time_col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values; return time_col
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values; return time_col
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

def get_series(df, name):
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(0.0).values.astype(float)
    alt = 'EEG.'+name
    if alt in df.columns:
        return pd.to_numeric(df[alt], errors='coerce').fillna(0.0).values.astype(float)
    raise ValueError(f'{name} not found in dataframe columns.')

# Filters & analytic

def _bandpass(x, fs, lo, hi, order=4):
    ny=0.5*fs; lo=max(1e-6, min(lo, 0.99*ny)); hi=max(lo+1e-6, min(hi, 0.999*ny))
    b,a=signal.butter(order, [lo/ny, hi/ny], btype='band'); return signal.filtfilt(b,a,x)

def phase_at(x, fs, f0, half):
    xb = _bandpass(x, fs, f0-half, f0+half)
    z  = signal.hilbert(xb)
    return np.angle(z)  # radians

def amp_envelope(x, fs, f0, half):
    xb = _bandpass(x, fs, f0-half, f0+half)
    return np.abs(signal.hilbert(xb))

# Windows utilities

def windows_to_samples(wins, fs, N):
    segs=[]
    for (a,b) in (wins or []):
        i0,i1=int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(N,i1)
        if i1>i0: segs.append((i0,i1))
    return segs

def concat_segments(x, segs):
    return np.concatenate([x[i0:i1] for (i0,i1) in segs]) if segs else np.array([])

# CF-PLV (1:m locking)

def cf_plv(phi1, phim, m, wins_samp):
    phi1_w = concat_segments(phi1, wins_samp)
    phim_w = concat_segments(phim, wins_samp)
    L = min(len(phi1_w), len(phim_w))
    if L < 50:
        return np.nan, np.nan
    dphi = (m*phi1_w[:L] - phim_w[:L])
    plv = np.abs(np.mean(np.exp(1j*dphi)))
    # Surrogates: circular shift phim relative to phi1
    rng = np.random.default_rng(7)
    null=[]
    for _ in range(200):
        s = int(rng.integers(1, L-1))
        d = (m*phi1_w[:L] - np.r_[phim_w[-s:], phim_w[:-s]][:L])
        null.append(np.abs(np.mean(np.exp(1j*d))))
    p = float((np.sum(np.asarray(null) >= plv) + 1) / (len(null) + 1))
    return float(plv), p

# PAC metrics

def mvl(phase, amp):
    amp = np.asarray(amp, float)
    phase = np.asarray(phase, float)
    return float(np.abs(np.nanmean(amp * np.exp(1j*phase))) / (np.nanmean(amp) + 1e-12))

def tort_mi(phase, amp, nbins=18):
    edges = np.linspace(-np.pi, np.pi, nbins+1)
    bins = np.digitize(phase, edges) - 1
    bins = np.clip(bins, 0, nbins-1)
    m = np.zeros(nbins)
    for k in range(nbins):
        sel = (bins==k)
        m[k] = np.mean(amp[sel]) if np.any(sel) else 0.0
    if m.sum() <= 0:
        return 0.0
    p = m / m.sum(); eps=1e-12
    return float(np.sum(p*np.log((p+eps)/(1.0/nbins))) / np.log(nbins))

# Within-cycle phase-shuffle surrogate for PAC

def pac_within_cycle_surrogate(phase_slow, amp_fast, fs, n_perm=200):
    # detect cycles by zero-crossings of the slow signal (phase unwrap is tricky)
    slow_sig = np.sin(phase_slow)  # proxy signal at the slow band
    zc = np.where(np.diff(np.signbit(slow_sig).astype(int)) != 0)[0]
    if zc.size < 4:
        # fallback: simple circular shift surrogate
        rng = np.random.default_rng(11)
        null=[]
        for _ in range(n_perm):
            s = int(rng.integers(1, len(amp_fast)-1))
            null.append((phase_slow, np.r_[amp_fast[-s:], amp_fast[:-s]]))
        return null
    segs = [(zc[i], zc[i+1]) for i in range(len(zc)-1)]
    rng = np.random.default_rng(11)
    null=[]
    for _ in range(n_perm):
        af = amp_fast.copy()
        for (i0,i1) in segs:
            L = i1 - i0
            if L <= 3: continue
            sh = int(rng.integers(0, L))
            af[i0:i1] = np.r_[af[i0:i1][-sh:], af[i0:i1][:-sh]]
        null.append((phase_slow, af))
    return null

# Directionality (CFDC-like ΔR²)

def directionality_phase_to_amp(phi1, amp_m, fs, lags=np.arange(0.0, 0.51, 0.02)):
    s = np.sin(phi1); c = np.cos(phi1)
    A = (amp_m - np.nanmean(amp_m)) / (np.nanstd(amp_m) + 1e-12)
    out = []
    for tau in lags:
        shift = int(round(tau * fs))
        if shift <= 0: y = A
        else:
            y = A[shift:]
        xA  = A[:len(y)]
        xs  = s[:len(y)]; xc = c[:len(y)]
        # baseline: y ~ xA
        X0 = np.column_stack([np.ones(len(y)), xA])
        # full: y ~ xA + sin + cos
        X1 = np.column_stack([np.ones(len(y)), xA, xs, xc])
        # solve by least squares
        b0, *_ = np.linalg.lstsq(X0, y, rcond=None)
        b1, *_ = np.linalg.lstsq(X1, y, rcond=None)
        y0 = X0 @ b0; y1 = X1 @ b1
        SS = np.sum((y - np.mean(y))**2)
        R2_0 = 1 - np.sum((y - y0)**2)/(SS + 1e-12)
        R2_1 = 1 - np.sum((y - y1)**2)/(SS + 1e-12)
        out.append(R2_1 - R2_0)
    return np.asarray(out), lags

def directionality_amp_to_phase(amp_m, phi1, fs, lags=np.arange(0.0, 0.51, 0.02)):
    # Predict sin(phi1_{t+tau}) from [sin(phi1_t), cos(phi1_t), A_m(t)]
    s = np.sin(phi1); c = np.cos(phi1)
    A = (amp_m - np.nanmean(amp_m)) / (np.nanstd(amp_m) + 1e-12)
    out = []
    for tau in lags:
        shift = int(round(tau * fs))
        if shift <= 0:
            y = s
            s0, c0, a0 = s, c, A
        else:
            y = s[shift:]
            s0, c0, a0 = s[:len(y)], c[:len(y)], A[:len(y)]
        # baseline: y ~ s0 + c0
        X0 = np.column_stack([np.ones(len(y)), s0, c0])
        # full: y ~ s0 + c0 + a0
        X1 = np.column_stack([np.ones(len(y)), s0, c0, a0])
        b0, *_ = np.linalg.lstsq(X0, y, rcond=None)
        b1, *_ = np.linalg.lstsq(X1, y, rcond=None)
        y0 = X0 @ b0; y1 = X1 @ b1
        SS = np.sum((y - np.mean(y))**2)
        R2_0 = 1 - np.sum((y - y0)**2)/(SS + 1e-12)
        R2_1 = 1 - np.sum((y - y1)**2)/(SS + 1e-12)
        out.append(R2_1 - R2_0)
    return np.asarray(out), lags

# ---------------------------- main orchestrator ----------------------------

def analyze_cfc_harmonics(
    RECORDS,
    eeg_channel: str,
    windows: dict,    # {'baseline':[(t0,t1),...], 'ignition':[(..)], ...}
    time_col='Timestamp',
    fundamental=7.83,
    harmonics=(14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_bw=0.6,
    out_dir='exports_cfc_harm', show=True
):
    """Compute CF-PLV (1:m), PAC per order (MVL & MI with within-cycle surrogates),
    and directionality ΔR² curves per harmonic. Saves PNGs + CSV summary.
    """
    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)

    x = get_series(RECORDS, eeg_channel)
    N = len(x)

    # Precompute phases & amplitudes for all needed bands
    phi1 = phase_at(x, fs, fundamental, half_bw)
    A = {}
    PHI = {}
    for fm in harmonics:
        if fm > min(60.0, 0.999*0.5*fs):
            continue
        PHI[fm] = phase_at(x, fs, fm, half_bw)
        A[fm]   = amp_envelope(x, fs, fm, half_bw)

    # Windows in samples
    W = {k: windows_to_samples(v, fs, N) for k,v in (windows or {}).items()}

    # Summary rows
    rows=[]

    # ---------- CF-PLV ----------
    for fm in PHI:
        m = int(round(fm / fundamental))
        for wname, segs in W.items():
            plv, p = cf_plv(phi1, PHI[fm], m, segs)
            rows.append({'metric':'CF-PLV', 'window':wname, 'order':m, 'f_hz':fm, 'value':plv, 'p_value':p})
        # Bar plot across windows for this fm
        vals=[]; labels=[]; errs=[]
        for wname, segs in W.items():
            plv, p = cf_plv(phi1, PHI[fm], m, segs)
            vals.append(plv); labels.append(wname); errs.append(0)
        fig, ax = plt.subplots(figsize=(6,3.0))
        ax.bar(np.arange(len(vals)), vals, width=0.6)
        ax.set_xticks(np.arange(len(vals))); ax.set_xticklabels(labels)
        ax.set_ylim(0,1); ax.set_ylabel('CF-PLV (1:m)'); ax.set_title(f'CF-PLV  φ1↔φ{m}  ({fundamental:.2f}↔{fm:.2f} Hz)')
        ax.grid(True, axis='y', alpha=0.25, linestyle=':')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'cfplv_m{m}_{fm:.2f}Hz.png'), dpi=160)
        if show: plt.show(); plt.close()

    # ---------- PAC per order (MVL & MI) ----------
    for fm in A:
        m = int(round(fm / fundamental))
        for wname, segs in W.items():
            phi_w = concat_segments(phi1, segs)
            amp_w = concat_segments(A[fm], segs)
            if len(phi_w) < 200 or len(amp_w) < 200:
                rows.append({'metric':'PAC-MVL', 'window':wname, 'order':m, 'f_hz':fm, 'value':np.nan, 'p_value':np.nan})
                rows.append({'metric':'PAC-MI',  'window':wname, 'order':m, 'f_hz':fm, 'value':np.nan, 'p_value':np.nan})
                continue
            mv = mvl(phi_w, amp_w); mi = tort_mi(phi_w, amp_w)
            # within-cycle surrogates
            null = pac_within_cycle_surrogate(phi_w, amp_w, fs)
            mv_null = []; mi_null=[]
            for (ph_s, af_s) in null:
                mv_null.append(mvl(ph_s, af_s)); mi_null.append(tort_mi(ph_s, af_s))
            p_mv = float((np.sum(np.asarray(mv_null) >= mv) + 1) / (len(mv_null)+1))
            p_mi = float((np.sum(np.asarray(mi_null) >= mi) + 1) / (len(mi_null)+1))
            rows.append({'metric':'PAC-MVL', 'window':wname, 'order':m, 'f_hz':fm, 'value':mv, 'p_value':p_mv})
            rows.append({'metric':'PAC-MI',  'window':wname, 'order':m, 'f_hz':fm, 'value':mi, 'p_value':p_mi})
        # plot
        S = [r for r in rows if r['metric']=='PAC-MI' and r['f_hz']==fm]
        labs=[r['window'] for r in S]; vals=[r['value'] for r in S]
        fig, ax = plt.subplots(figsize=(6,3.0))
        ax.bar(np.arange(len(vals)), vals, width=0.6)
        ax.set_xticks(np.arange(len(vals))); ax.set_xticklabels(labs)
        ax.set_ylabel('PAC (MI)'); ax.set_title(f'PAC MI: phase {fundamental:.2f} Hz → amp {fm:.2f} Hz')
        ax.grid(True, axis='y', alpha=0.25, linestyle=':')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'pac_mi_{fm:.2f}Hz.png'), dpi=160)
        if show: plt.show(); plt.close()

    # ---------- Directionality (ΔR² curves) ----------
    for fm in A:
        m = int(round(fm / fundamental))
        for wname, segs in W.items():
            phi_w = concat_segments(phi1, segs)
            amp_w = concat_segments(A[fm], segs)
            if len(phi_w) < 400 or len(amp_w) < 400:
                rows.append({'metric':'DIR-P2A-peak', 'window':wname, 'order':m, 'f_hz':fm, 'value':np.nan, 'p_value':np.nan})
                rows.append({'metric':'DIR-A2P-peak', 'window':wname, 'order':m, 'f_hz':fm, 'value':np.nan, 'p_value':np.nan})
                continue
            dP2A, lags = directionality_phase_to_amp(phi_w, amp_w, fs)
            dA2P, _    = directionality_amp_to_phase(amp_w, phi_w, fs)
            # surrogate: circular shift amplitude relative to phase
            rng = np.random.default_rng(19)
            nullP=[]; nullA=[]
            for _ in range(200):
                s = int(rng.integers(1, len(amp_w)-1))
                amp_s = np.r_[amp_w[-s:], amp_w[:-s]]
                dp, _ = directionality_phase_to_amp(phi_w, amp_s, fs)
                da, _ = directionality_amp_to_phase(amp_s, phi_w, fs)
                nullP.append(np.nanmax(dp)); nullA.append(np.nanmax(da))
            peakP = float(np.nanmax(dP2A)); peakA = float(np.nanmax(dA2P))
            pP = float((np.sum(np.asarray(nullP) >= peakP) + 1) / (len(nullP)+1))
            pA = float((np.sum(np.asarray(nullA) >= peakA) + 1) / (len(nullA)+1))
            rows.append({'metric':'DIR-P2A-peak', 'window':wname, 'order':m, 'f_hz':fm, 'value':peakP, 'p_value':pP})
            rows.append({'metric':'DIR-A2P-peak', 'window':wname, 'order':m, 'f_hz':fm, 'value':peakA, 'p_value':pA})
            # plot lag curves
            fig, ax = plt.subplots(figsize=(6.8,3.0))
            ax.plot(lags, dP2A, lw=1.6, label='phase→amp ΔR²')
            ax.plot(lags, dA2P, lw=1.2, ls='--', label='amp→phase ΔR²')
            ax.set_xlabel('Lag τ (s)'); ax.set_ylabel('ΔR²'); ax.set_title(f'Directionality (φ1 ↔ A_{m})  {fundamental:.2f}↔{fm:.2f} Hz')
            ax.legend(); ax.grid(True, alpha=0.25, linestyle=':')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'dir_m{m}_{fm:.2f}Hz.png'), dpi=160)
            if show: plt.show(); plt.close()

    # Save summary
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(out_dir, 'cfc_harmonics_summary.csv'), index=False)
    return summary

# ---------------------------- Example usage ----------------------------
# windows = {
#   'baseline': [(0, 290)],
#   'ignition': [(290, 310), (580, 600)],
#   'rebound':  [(325, 580)]
# }
# summary = analyze_cfc_harmonics(
#     RECORDS,
#     eeg_channel='EEG.O1',  # or 'EEG.VIRT' if you built a virtual posterior ROI
#     windows=windows,
#     fundamental=7.83,
#     harmonics=(14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
#     half_bw=0.6,
#     out_dir='exports_cfc_harm', show=True
# )


In [ ]:
windows = {
  'baseline': [(60, 120)],
  'ignition': [(180,200),(280,300)]
}
summary = analyze_cfc_harmonics(
    RECORDS,
    eeg_channel='EEG.O1',   # or your virtual posterior 'EEG.VIRT'
    windows=windows,
    fundamental=7.83,
    harmonics=(1.305,1.57,1.96,2.61,3.915,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_bw=0.6,
    out_dir='exports_cfc_harm', show=True
)

In [ ]:
"""
Spatial & Source-level Harmonics (0.1–60 Hz)
===========================================
Simple tests + graphs to validate:
  • Topographies per order: map H‑PLI_k (EEG↔SR) and HCS across electrodes
    + Global Field Synchronization (GFS) per harmonic.
  • Source localization (LCMV) at each SR line (if MNE forward model provided).
  • Connectome‑harmonic overlap: project source maps on structural eigenmodes.
  • Network graphs per order: PLV networks at 7.83, 14.3, 20.8, … → modularity, min‑cut, path length.

Dependencies: numpy, scipy, matplotlib, pandas, networkx.
Optional: mne (for topomaps + LCMV), structural connectome (SC) for eigenmodes.

Notes
-----
• H‑PLI_k at an electrode e is |<exp(i(φ_e(fk) − φ_SR(fk)))>| over sliding windows (default 8 s, 1 s step);
  per‑electrode p‑values via circular‑shift surrogates of SR phase.
• HCS (sensor‑level) is a weighted sum across orders of H‑PLI_k per electrode.
• GFS(fk): mean resultant length across channels of phases at fk (spatial phase consensus), averaged in windows.
• All analyses clamp to ≤ 60 Hz and are NaN‑safe.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import networkx as nx

# -------------------------- I/O helpers --------------------------

def ensure_dir(d):
    if d: os.makedirs(d, exist_ok=True)
    return d

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    if time_col in df.columns:
        s = df[time_col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values; return time_col
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values; return time_col
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

# -------------------------- Filtering & analytic --------------------------

def _bandpass(x, fs, lo, hi, order=4):
    ny=0.5*fs; lo=max(1e-6,min(lo,0.99*ny)); hi=max(lo+1e-6,min(hi,0.999*ny))
    b,a=signal.butter(order,[lo/ny, hi/ny], btype='band'); return signal.filtfilt(b,a,x)

def phase_series(x, fs, f0, half):
    xb = _bandpass(x, fs, f0-half, f0+half)
    return np.angle(signal.hilbert(xb))

# -------------------------- Electrode utilities --------------------------

def detect_eeg_channels(df, prefix='EEG.'):
    return [c for c in df.columns if c.startswith(prefix)]

# minimal 2D coords for common 10‑20 labels (fallback if mne not installed)
_COORDS_2D = {
    'Fp1':(-0.5, 1.0),'Fp2':(0.5,1.0),'F7':(-0.9,0.6),'F3':(-0.4,0.6),'Fz':(0,0.7),'F4':(0.4,0.6),'F8':(0.9,0.6),
    'FC5':(-0.7,0.4),'FC6':(0.7,0.4),'T7':(-1.0,0.2),'C3':(-0.5,0.2),'Cz':(0,0.2),'C4':(0.5,0.2),'T8':(1.0,0.2),
    'TP9':(-1.1,-0.1),'CP5':(-0.7,0.0),'CP6':(0.7,0.0),'TP10':(1.1,-0.1),'P7':(-0.9,-0.2),'P3':(-0.4,-0.2),
    'Pz':(0,-0.25),'P4':(0.4,-0.2),'P8':(0.9,-0.2),'POz':(0,-0.5),'O1':(-0.4,-0.7),'Oz':(0,-0.7),'O2':(0.4,-0.7),
    'AF3':(-0.25,0.8),'AF4':(0.25,0.8)
}

# -------------------------- H‑PLI topography --------------------------

def hpli_topography(RECORDS, sr_channel, harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
                    time_col='Timestamp', half_bw=0.6, win_sec=8.0, step_sec=1.0,
                    n_perm=200, windows=None, out_dir='exports_spatial', show=True):
    """Compute per‑electrode H‑PLI_k maps + p‑values for each harmonic, and HCS map.
    windows: dict name->[(t0,t1),...] to restrict; if None uses all samples.
    Returns a dict with maps and saves topography PNGs.
    """
    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)

    EEGS = detect_eeg_channels(RECORDS)
    if sr_channel not in RECORDS.columns:
        raise ValueError(f"{sr_channel} not in dataframe columns")
    x_sr = pd.to_numeric(RECORDS[sr_channel], errors='coerce').fillna(0.0).values.astype(float)

    # windows to samples
    def wins_to_seg(wins, N):
        if not wins: return [(0,N)]
        segs=[]
        for (a,b) in wins:
            i0,i1=int(round(a*fs)), int(round(b*fs))
            i0=max(0,i0); i1=min(N,i1)
            if i1>i0: segs.append((i0,i1))
        return segs

    segs = wins_to_seg((None if windows is None else sum(windows.values(), [])), len(RECORDS))

    # phases for SR per harmonic (full length)
    PHI_SR = {f0: phase_series(x_sr, fs, f0, half_bw) for f0 in harmonics if f0 <= min(60.0, 0.999*0.5*fs)}

    HPLI = {f0:{} for f0 in PHI_SR}
    PV   = {f0:{} for f0 in PHI_SR}

    rng = np.random.default_rng(7)
    win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
    centers = np.arange(win//2, len(RECORDS)-win//2, step, dtype=int)

    for ch in EEGS:
        x = pd.to_numeric(RECORDS[ch], errors='coerce').fillna(0.0).values.astype(float)
        for f0, phi_sr in PHI_SR.items():
            phi_e = phase_series(x, fs, f0, half_bw)
            # restrict to segs
            m = np.zeros_like(phi_e, dtype=bool)
            for (i0,i1) in segs: m[i0:i1] = True
            idx_centers = [c for c in centers if m[c]]
            vals=[]
            for c in idx_centers:
                sl = slice(c - win//2, c + win//2)
                dphi = phi_e[sl] - phi_sr[sl]
                vals.append(np.abs(np.mean(np.exp(1j*dphi))))
            vals = np.asarray(vals, float)
            hval = float(np.nanmean(vals)) if vals.size else np.nan
            # surrogate p: circular shift SR phase
            null=[]
            for _ in range(int(n_perm)):
                s = int(rng.integers(win, len(phi_sr)-1))
                phi_s = np.r_[phi_sr[-s:], phi_sr[:-s]]
                v=[]
                for c in idx_centers:
                    sl=slice(c-win//2, c+win//2)
                    d=phi_e[sl]-phi_s[sl]
                    v.append(np.abs(np.mean(np.exp(1j * d))))
#                     v.append(np.abs(np.mean(np.exp(1j,d))))
                null.append(np.nanmean(v) if v else np.nan)
            null = np.asarray(null, float)
            p = float((np.sum(null >= hval) + 1) / (np.sum(np.isfinite(null)) + 1)) if np.isfinite(hval) else np.nan
            HPLI[f0][ch] = hval; PV[f0][ch] = p

    # HCS per electrode (weights 1/k)
    weights = {}
    for f0 in PHI_SR.keys():
        k = max(1, int(round(f0/7.83)))
        weights[f0] = 1.0/float(k)
    wsum = sum(weights.values())
    HCS = {}
    for ch in EEGS:
        s=0.0; cnt=0
        for f0,vmap in HPLI.items():
            val = vmap.get(ch, np.nan)
            if np.isfinite(val):
                s += weights[f0]*val; cnt+=weights[f0]
        HCS[ch] = (s/(cnt+1e-12)) if cnt>0 else np.nan

    # GFS per harmonic (spatial phase consensus)
    GFS={}  # mean R across windows
    for f0 in PHI_SR:
        PHI_all=[]
        for ch in EEGS:
            x = pd.to_numeric(RECORDS[ch], errors='coerce').fillna(0.0).values.astype(float)
            PHI_all.append(phase_series(x, fs, f0, half_bw))
        PHI_all = np.vstack(PHI_all)  # (C,N)
        # restrict to segs
        m = np.zeros(PHI_all.shape[1], dtype=bool)
        for (i0,i1) in segs: m[i0:i1] = True
        R = np.abs(np.mean(np.exp(1j*PHI_all[:, m]), axis=0)) if np.any(m) else np.array([])
        GFS[f0] = float(np.nanmean(R)) if R.size else np.nan

    # -------- topography plotting --------
    try:
        import mne
        HAS_MNE = True
    except Exception:
        HAS_MNE = False

    def _plot_topo(values_dict, title, fname):
        chs=list(values_dict.keys()); vals=np.array([values_dict[c] for c in chs], float)
        if HAS_MNE:
            fs_local = fs
            info = mne.create_info(chs, sfreq=fs_local, ch_types=['eeg']*len(chs))
            montage = mne.channels.make_standard_montage('standard_1020')
            try:
                info.set_montage(montage, match_case=False)
                pos = np.array([info.get_montage().get_positions()['ch_pos'][c] for c in chs])[:, :2]
            except Exception:
                # fallback to dict
                pos = np.array([_COORDS_2D.get(c.replace('EEG.',''), (np.nan,np.nan)) for c in chs])
        else:
            pos = np.array([_COORDS_2D.get(c.replace('EEG.',''), (np.nan,np.nan)) for c in chs])
        # drop NaN positions
        mask = np.isfinite(pos).all(axis=1)
        chs = [c for c,m in zip(chs,mask) if m]
        vals = vals[mask]; pos = pos[mask]
        plt.figure(figsize=(5.2,4.6))
        sc = plt.scatter(pos[:,0], pos[:,1], c=vals, s=220, cmap='viridis', vmin=np.nanpercentile(vals,5), vmax=np.nanpercentile(vals,95))
        plt.colorbar(sc,label='value');
        for (x,y,c) in zip(pos[:,0], pos[:,1], chs):
            plt.text(x,y,c.replace('EEG.',''), ha='center', va='center', fontsize=6, color='k')
        plt.title(title); plt.axis('off'); plt.tight_layout();
        plt.savefig(os.path.join(out_dir,fname), dpi=160)
        plt.show(); plt.close()

    for f0 in PHI_SR:
        _plot_topo({ch:HPLI[f0].get(ch,np.nan) for ch in EEGS}, title=f'H‑PLI topography @ {f0:.2f} Hz', fname=f'hpli_topo_{f0:.2f}Hz.png')
    _plot_topo(HCS, title='HCS (weighted sum of H‑PLI across orders)', fname='hcs_topo.png')

    # GFS bar
    fig, ax = plt.subplots(figsize=(7,3))
    f_list = list(PHI_SR.keys()); gvals=[GFS[f] for f in f_list]
    ax.bar(np.arange(len(f_list)), gvals, width=0.7)
    ax.set_xticks(np.arange(len(f_list))); ax.set_xticklabels([f'{f:.1f}' for f in f_list])
    ax.set_ylabel('GFS (mean resultant across channels)'); ax.set_title('Global Field Synchronization per harmonic')
    ax.grid(True, axis='y', alpha=0.25, linestyle=':')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'gfs_bar.png'), dpi=160)
    plt.show(); plt.close()

    return {'HPLI':HPLI, 'PV':PV, 'HCS':HCS, 'GFS':GFS, 'EEG_channels':EEGS, 'fs':fs, 'out_dir':out_dir}

# -------------------------- PLV networks per order --------------------------

def plv_networks(RECORDS, channels=None, harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
                  time_col='Timestamp', half_bw=0.6, windows=None, thr_pct=0.2,
                  out_dir='exports_spatial', show=True):
    """Compute PLV networks at each harmonic; save adjacency heatmaps & graph plots; return stats (modularity, min‑cut, path length).
    channels: list of EEG.* columns to include; if None uses all EEG.* channels present.
    thr_pct: keep top p fraction of edges (0..1) for graph metrics.
    """
    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)
    if channels is None:
        channels = detect_eeg_channels(RECORDS)
    Nch = len(channels)

    # windows mask
    def wins_mask(N):
        if not windows: return np.ones(N, dtype=bool)
        m = np.zeros(N, dtype=bool)
        for wins in windows.values():
            for (a,b) in wins:
                i0,i1=int(round(a*fs)), int(round(b*fs))
                i0=max(0,i0); i1=min(N,i1)
                m[i0:i1]=True
        return m

    mask = wins_mask(len(RECORDS))

    # precompute phases for all channels x harmonics
    X = {ch: pd.to_numeric(RECORDS[ch], errors='coerce').fillna(0.0).values.astype(float) for ch in channels}
    PHI = {f0: {ch: phase_series(X[ch], fs, f0, half_bw) for ch in channels if f0 <= min(60.0, 0.999*0.5*fs)} for f0 in harmonics}

    stats_rows=[]

    for f0 in PHI:
        # PLV matrix
        A = np.zeros((Nch,Nch), float)
        for i,ch_i in enumerate(channels):
            ph_i = PHI[f0][ch_i][mask]
            for j,ch_j in enumerate(channels[i+1:], start=i+1):
                ph_j = PHI[f0][ch_j][mask]
                L = min(len(ph_i), len(ph_j))
                if L<100:
                    val = np.nan
                else:
                    val = float(np.abs(np.mean(np.exp(1j*(ph_i[:L]-ph_j[:L])))))
                A[i,j]=A[j,i]=val
        # heatmap
        plt.figure(figsize=(5.8,5.0))
        v = np.nan_to_num(A, nan=0.0)
        plt.imshow(v, origin='lower', cmap='viridis', vmin=0, vmax=1)
        plt.xticks(range(Nch), [c.replace('EEG.','') for c in channels], rotation=90, fontsize=7)
        plt.yticks(range(Nch), [c.replace('EEG.','') for c in channels], fontsize=7)
        plt.colorbar(label='PLV'); plt.title(f'PLV matrix @ {f0:.2f} Hz'); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'plv_matrix_{f0:.2f}Hz.png'), dpi=160)
        if show: plt.show(); plt.close()

        # threshold top p% edges (exclude diagonal)
        triu = A[np.triu_indices(Nch,1)]
        finite = triu[np.isfinite(triu)]
        if finite.size == 0:
            stats_rows.append({'f0':f0, 'modularity':np.nan, 'min_cut':np.nan, 'avg_path_len':np.nan, 'density':np.nan})
            continue
        thresh = np.nanpercentile(finite, 100*(1.0-thr_pct))
        G = nx.Graph()
        G.add_nodes_from(range(Nch))
        for i in range(Nch):
            for j in range(i+1,Nch):
                w = A[i,j]
                if np.isfinite(w) and w >= thresh:
                    G.add_edge(i,j,weight=float(w))
        # stats
        density = nx.density(G)
        # communities & modularity (falls back to NaN if <2 communities)
        try:
            comm = nx.algorithms.community.greedy_modularity_communities(G, weight='weight')
            modules = [set(c) for c in comm]
            modularity = nx.algorithms.community.modularity(G, modules, weight='weight') if len(modules)>=2 else np.nan
        except Exception:
            modularity = np.nan
        # min‑cut
        try:
            cut_val, part = nx.algorithms.connectivity.stoer_wagner(G, weight='weight')
            min_cut = float(cut_val)
        except Exception:
            min_cut = np.nan
        # path length on largest component
        if len(G) == 0 or not nx.is_connected(G.to_undirected(as_view=True)):
            comps = list(nx.connected_components(G))
            if comps:
                H = G.subgraph(max(comps, key=len)).copy()
            else:
                H = None
        else:
            H = G
        try:
            avg_pl = nx.average_shortest_path_length(H, weight=None) if H and H.number_of_edges()>0 else np.nan
        except Exception:
            avg_pl = np.nan
        stats_rows.append({'f0':f0, 'modularity':modularity, 'min_cut':min_cut, 'avg_path_len':avg_pl, 'density':density})

        # simple graph plot
        pos = nx.circular_layout(G)
        plt.figure(figsize=(6,5))
        nx.draw_networkx_nodes(G, pos, node_size=160)
        # scale edge width by weight
        widths=[2.0*G[u][v]['weight'] for u,v in G.edges()] if G.number_of_edges()>0 else []
        nx.draw_networkx_edges(G, pos, width=widths, alpha=0.7)
        nx.draw_networkx_labels(G, pos, labels={i:channels[i].replace('EEG.','') for i in G.nodes()}, font_size=8)
        plt.title(f'PLV graph (top {int(thr_pct*100)}% edges) @ {f0:.2f} Hz')
        plt.axis('off'); plt.tight_layout();
        plt.savefig(os.path.join(out_dir, f'plv_graph_{f0:.2f}Hz.png'), dpi=160)
        if show: plt.show(); plt.close()

    stats = pd.DataFrame(stats_rows)
    stats.to_csv(os.path.join(out_dir, 'plv_network_stats.csv'), index=False)
    return stats

# -------------------------- Source loc & connectome overlap --------------------------

def lcmv_sources_at_lines(RECORDS, eeg_channels, sr=None, fwd=None, noise_cov=None,
                           time_col='Timestamp', lines=(7.83,14.3,20.8,27.3,33.8),
                           half_bw=0.6, windows=None, out_dir='exports_spatial', show=True):
    """If MNE forward model & noise covariance are provided, compute LCMV source power maps
    at each SR line; else returns None. Saves one figure per line.
    """
    try:
        import mne
    except Exception:
        print('[lcmv] MNE not available; skipping source localization.')
        return None
    if fwd is None or noise_cov is None:
        print('[lcmv] Forward model or noise covariance missing; skipping source localization.')
        return None

    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)

    # build mne.Raw from dataframe columns
    info = mne.create_info(eeg_channels, sfreq=fs, ch_types=['eeg']*len(eeg_channels))
    montage = mne.channels.make_standard_montage('standard_1020')
    try:
        info.set_montage(montage, match_case=False)
    except Exception:
        pass
    data = np.vstack([pd.to_numeric(RECORDS[ch], errors='coerce').fillna(0.0).values.astype(float) for ch in eeg_channels])
    raw = mne.io.RawArray(data, info)

    # windows mask
    if windows:
        m = np.zeros(raw.n_times, dtype=bool)
        for wins in windows.values():
            for (a,b) in wins:
                i0,i1=int(round(a*fs)), int(round(b*fs))
                m[i0:i1]=True
        raw = raw.copy().load_data()
        raw._data[:, ~m] = 0.0

    src_maps = {}
    for f0 in lines:
        if f0 > min(60.0, 0.999*0.5*fs):
            continue
        l_freq = max(0.01, f0-half_bw); h_freq = f0+half_bw
        raw_f = raw.copy().filter(l_freq, h_freq, fir_design='firwin', verbose=False)
        data_cov = mne.compute_raw_covariance(raw_f, method='oas', verbose=False)
        filters = mne.beamformer.make_lcmv(raw_f.info, fwd, data_cov, reg=0.05, noise_cov=noise_cov,
                                           pick_ori='max-power', weight_norm='unit-noise-gain', verbose=False)
        stc = mne.beamformer.apply_lcmv_raw(raw_f, filters, max_ori_out='signed', verbose=False)
        src_maps[f0] = stc
        # simple brain plot (if fsaverage/subjects_dir available)
        try:
            brain = stc.plot(hemi='split', views=['lat'], time_viewer=False, smoothing_steps=5,
                             clim='auto', colormap='magma')
            brain.save_image(os.path.join(out_dir, f'lcmv_{f0:.2f}Hz.png'))
            brain.close()
        except Exception:
            pass
    return src_maps

# Connectome harmonic overlap

def connectome_overlap(source_vec, SC, k_modes=10):
    """Project a source map (N nodes) onto Laplacian eigenmodes of SC; return variance explained per mode."""
    import numpy as np
    # Laplacian
    D = np.diag(SC.sum(1))
    L = D - SC
    w, V = np.linalg.eigh(L)
    # sort by ascending eigenvalue (mode‑1 = global/slowest)
    idx = np.argsort(w); w=w[idx]; V=V[:,idx]
    x = source_vec - np.mean(source_vec)
    coeff = V.T @ x
    var = coeff**2
    frac = var/np.sum(var+1e-12)
    return {'eigs':w, 'frac':frac, 'V':V, 'coeff':coeff, 'topk':(w[:k_modes], frac[:k_modes])}

# -------------------------- Example orchestrator --------------------------

def analyze_spatial_and_source(RECORDS,
                               sr_channel,
                               windows,
                               harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
                               half_bw=0.6,
                               out_dir='exports_spatial',
                               show=True,
                               do_networks=True,
                               network_channels=None,
                               do_sources=False,
                               mne_fwd=None, mne_noise_cov=None,
                               do_connectome=False,
                               SC=None):
    """Run spatial topographies (H‑PLI, HCS), GFS bars, (optional) PLV networks and LCMV sources.
    Optional connectome overlap if SC is provided (N×N). Returns a dict of outputs.
    """
    ensure_dir(out_dir)
    topo = hpli_topography(RECORDS, sr_channel, harmonics=harmonics, half_bw=half_bw,
                           windows=windows, out_dir=out_dir, show=show)
    out = {'topography': topo}

    if do_networks:
        net_stats = plv_networks(RECORDS, channels=network_channels, harmonics=harmonics,
                                 half_bw=half_bw, windows=windows, out_dir=out_dir, show=show)
        out['network_stats'] = net_stats

    if do_sources:
        eeg_chs = detect_eeg_channels(RECORDS)
        stcs = lcmv_sources_at_lines(RECORDS, eeg_chs, fwd=mne_fwd, noise_cov=mne_noise_cov,
                                     lines=tuple([f for f in harmonics if f<=60.0]),
                                     half_bw=half_bw, windows=windows, out_dir=out_dir, show=show)
        out['sources'] = stcs
        # connectome overlap if provided and stc is vector in SC space
        if do_connectome and (SC is not None) and (stcs is not None):
            # This requires mapping stc to SC node space; placeholder expects `source_vec` aligned to SC
            # Example: take absolute mean across time for each source vertex and provide a vertex→SC mapping externally.
            pass

    return out

# -------------------------- Quick usage --------------------------
# windows = {
#   'baseline': [(0, 290)],
#   'ignition': [(290, 310), (580, 600)],
#   'rebound':  [(325, 580)]
# }
# out = analyze_spatial_and_source(
#     RECORDS,
#     sr_channel='EEG.Pz',   # or your magnetometer channel
#     windows=windows,
#     harmonics=(7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
#     half_bw=0.6,
#     out_dir='exports_spatial',
#     show=True,
#     do_networks=True,
#     network_channels=None,   # None = all EEG.* channels
#     do_sources=False,        # set True if you provide MNE forward model & noise_cov
#     mne_fwd=None, mne_noise_cov=None,
#     do_connectome=False,
#     SC=None
# )


In [ ]:
windows = {
  'baseline': [(60, 120)],
  'ignition': [(290, 310), (580, 600)]
}

out = analyze_spatial_and_source(
    RECORDS,
    sr_channel='EEG.F4',  # or your magnetometer
    windows=windows,
    harmonics=(1.305,1.57,1.96,2.61,3.915,7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_bw=0.6,
    out_dir='exports_spatial',
    show=True,
    do_networks=True,
    network_channels=None,      # None = all EEG.* channels present
    do_sources=False,           # set True if you provide mne_fwd & mne_noise_cov
    mne_fwd=None, mne_noise_cov=None,
    do_connectome=False, SC=None
)


In [ ]:
import numpy as np
from scipy import signal

def compute_coherence_at_f0(xe, xs, fs, f0, half):
    """
    Magnitude-squared coherence between signals xe and xs at target frequency f0.

    Uses Welch autospectra (Pxx, Pyy) and cross-spectrum (Pxy) on the provided
    windowed segments, then returns a scalar coherence value aggregated within
    the band [f0 - half, f0 + half]. If that band contains no frequency bin,
    returns the coherence at the nearest available bin to f0.

    Parameters
    ----------
    xe : array_like
        First signal segment (e.g., EEG), 1-D.
    xs : array_like
        Second signal segment (e.g., SR proxy / magnetometer), 1-D.
    fs : float
        Sampling rate in Hz.
    f0 : float
        Target center frequency in Hz.
    half : float
        Half-bandwidth in Hz; analyze [f0 - half, f0 + half].

    Returns
    -------
    float
        Coherence (0..1) at/around f0.
    """
    xe = np.asarray(xe, dtype=float)
    xs = np.asarray(xs, dtype=float)
    N = int(min(len(xe), len(xs)))
    if N < 32:
        raise ValueError("Window too short for coherence (N < 32 samples)")
    xe = xe[:N]
    xs = xs[:N]

    # Choose segment length for spectral estimates: use half the window (typical)
    nperseg = int(max(32, min(N, N // 2)))
    noverlap = int(nperseg // 2)

    # Autospectra and cross-spectrum (Welch / CSD)
    f, Pxx = signal.welch(
        xe, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        detrend='constant', return_onesided=True, scaling='density'
    )
    _, Pyy = signal.welch(
        xs, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        detrend='constant', return_onesided=True, scaling='density'
    )
    _, Pxy = signal.csd(
        xe, xs, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        detrend='constant', return_onesided=True, scaling='density'
    )

    # Magnitude-squared coherence
    eps = 1e-20
    Cxy = (np.abs(Pxy) ** 2) / (Pxx * Pyy + eps)

    # Aggregate within the f0 ± half band; fallback to nearest bin if empty
    f_lo = max(0.0, float(f0) - float(half))
    f_hi = float(f0) + float(half)
    band = (f >= f_lo) & (f <= f_hi) & np.isfinite(Cxy)
    if np.any(band):
        c_val = float(np.nanmean(Cxy[band]))
    else:
        idx = int(np.argmin(np.abs(f - float(f0))))
        c_val = float(Cxy[idx]) if np.isfinite(Cxy[idx]) else float('nan')

    # Clamp numeric noise into [0, 1]
    if np.isfinite(c_val):
        c_val = float(np.clip(c_val, 0.0, 1.0))
    return c_val



def build_null_threshold(coh, n_null=200, method='block', block_len=None, alpha=0.05, random_state=13):
    """
    Estimate a null threshold for a sliding coherence trace by resampling the
    coherence sequence itself.

    For each surrogate, we create a bootstrap replica of the coherence
    time-series and take its maximum. The (1-alpha) percentile of these maxima
    is returned. By default, we use a block-bootstrap that preserves local
    autocorrelation structure; set method='iid' for simple i.i.d. resampling.

    Parameters
    ----------
    coh : array_like
        Coherence values (0..1) across sliding-window centers; NaNs allowed.
    n_null : int, optional
        Number of surrogate replicates (default 200).
    method : {'block','iid'}, optional
        Resampling strategy. 'block' preserves short-range correlations.
    block_len : int or None, optional
        Block length (in samples) for block-bootstrap. If None, uses ~5% of N
        (at least 5 samples).
    alpha : float, optional
        Significance level (default 0.05 → 95th percentile).
    random_state : int, optional
        RNG seed for reproducibility.

    Returns
    -------
    float
        Estimated (1 - alpha) percentile of the null maxima; clipped to [0, 1].
    """
    import numpy as np

    c = np.asarray(coh, dtype=float)
    # Keep only finite values
    c = c[np.isfinite(c)]
    if c.size == 0:
        return float('nan')

    rng = np.random.default_rng(random_state)
    N = int(c.size)

    maxima = []
    if method == 'iid':
        for _ in range(int(n_null)):
            samp = rng.choice(c, size=N, replace=True)
            maxima.append(float(np.nanmax(samp)))
    else:
        # Block bootstrap with circular wrap to preserve local structure
        if block_len is None:
            block_len = max(5, int(round(N / 20)))  # ~5% of the series
        B = int(block_len)
        for _ in range(int(n_null)):
            idx = []
            filled = 0
            while filled < N:
                start = int(rng.integers(0, N))
                end = start + B
                if end <= N:
                    idx.extend(range(start, end))
                else:
                    # circular wrap
                    idx.extend(list(range(start, N)) + list(range(0, end - N)))
                filled += B
            idx = np.asarray(idx[:N], dtype=int)
            surrogate = c[idx]
            maxima.append(float(np.nanmax(surrogate)))

    q = 100.0 * (1.0 - float(alpha))
    thr = float(np.nanpercentile(maxima, q))
    # numeric safety
    return float(np.clip(thr, 0.0, 1.0))


In [ ]:
# ===== Directionality across systems & harmonics — self-contained loader =====
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import signal

# --- helpers ---
def ensure_dir(d):
    if d: os.makedirs(d, exist_ok=True)
    return d

def ensure_timestamp_column(df, time_col='Timestamp', default_fs=128.0):
    if time_col in df.columns:
        s = df[time_col]
        if np.issubdtype(s.dtype, np.datetime64) or 'datetime' in str(s.dtype).lower():
            tsec=(pd.to_datetime(s)-pd.to_datetime(s).iloc[0]).dt.total_seconds().astype(float)
            df[time_col] = tsec.values; return time_col
        sn = pd.to_numeric(s, errors='coerce').astype(float)
        if sn.notna().sum()>max(50,0.5*len(df)):
            sn = sn - np.nanmin(sn[np.isfinite(sn)])
            df[time_col] = sn.values; return time_col
    df[time_col] = np.arange(len(df), dtype=float)/default_fs
    return time_col

def infer_fs(df, time_col='Timestamp'):
    t = np.asarray(df[time_col].values, float)
    dt = np.diff(t); dt = dt[(dt>0)&np.isfinite(dt)]
    if dt.size==0: raise ValueError('Cannot infer fs from time column.')
    return float(1.0/np.median(dt))

def _bandpass(x, fs, lo, hi, order=4):
    ny=0.5*fs; lo=max(1e-6,min(lo,0.99*ny)); hi=max(lo+1e-6,min(hi,0.999*ny))
    b,a=signal.butter(order,[lo/ny, hi/ny],btype='band'); return signal.filtfilt(b,a,x)

def narrowband_pair(x_eeg, x_sr, fs, f0, half):
    lo=max(0.01, f0-half); hi=f0+half
    return _bandpass(x_eeg, fs, lo, hi), _bandpass(x_sr, fs, lo, hi)

def windows_to_samples(wins, fs, N):
    segs=[]
    for (a,b) in (wins or []):
        i0,i1=int(round(a*fs)), int(round(b*fs))
        i0=max(0,i0); i1=min(N,i1)
        if i1>i0: segs.append((i0,i1))
    return segs

def concat_segments(x, segs):
    return np.concatenate([x[i0:i1] for (i0,i1) in segs]) if segs else np.array([])

# --- bivariate MVAR + PDC/GC (LS) ---
def fit_mvar_2d(x, y, p=6):
    X = np.column_stack([x, y]).astype(float)
    N, k = X.shape
    if N <= p: raise ValueError('Too few samples for MVAR')
    Y = X[p:]
    Z = np.hstack([X[p-i:-i] for i in range(1, p+1)])  # (N-p) x (2p)
    B, *_ = np.linalg.lstsq(Z, Y, rcond=None)          # (2p) x 2
    A = [B[2*(i-1):2*i,:].T for i in range(1, p+1)]    # list of 2x2
    res = Y - Z @ B
    Sigma = (res.T @ res) / (len(res) - 1)
    return A, Sigma

def pdc_from_mvar(A_list, fs, f_hz):
    dt = 1.0/fs
    z = np.exp(-1j*2*np.pi*f_hz*dt)
    A_f = np.eye(2, dtype=complex)
    for k, Ak in enumerate(A_list, start=1):
        A_f = A_f - Ak * (z**k)
    denom = np.sqrt(np.sum(np.abs(A_f)**2, axis=0)) + 1e-12
    return np.abs(A_f) / denom  # 2x2

def granger_2d_refit(x, y, p=6):
    # full
    A_full, Sigma_full = fit_mvar_2d(x, y, p=p)
    sig_x_full = float(Sigma_full[0,0]); sig_y_full = float(Sigma_full[1,1])
    # Build design matrices
    X = np.column_stack([x, y]).astype(float)
    N = len(X)
    Y = X[p:]
    Z_full = np.hstack([X[p-i:-i] for i in range(1,p+1)])  # (N-p) x (2p)
    # Restricted x_t: only x lags
    Zx = np.hstack([X[p-i:-i, 0:1] for i in range(1,p+1)])
    bx, *_ = np.linalg.lstsq(Zx, Y[:,0], rcond=None)
    res_x = Y[:,0] - Zx @ bx
    sig_x_restr = float(np.var(res_x, ddof=1))
    # Restricted y_t: only y lags
    Zy = np.hstack([X[p-i:-i, 1:2] for i in range(1,p+1)])
    by, *_ = np.linalg.lstsq(Zy, Y[:,1], rcond=None)
    res_y = Y[:,1] - Zy @ by
    sig_y_restr = float(np.var(res_y, ddof=1))
    F_y_to_x = np.log((sig_x_restr + 1e-12)/(sig_x_full + 1e-12))
    F_x_to_y = np.log((sig_y_restr + 1e-12)/(sig_y_full + 1e-12))
    return F_y_to_x, F_x_to_y, A_full, Sigma_full

# --- FFT segs for bispectrum ---
def _fft_segments(x, fs, nperseg, step):
    w = signal.hann(nperseg, sym=False)
    hop = nperseg - step
    nseg = 1 + max(0, (len(x)-nperseg)//hop)
    Xs=[]
    for i in range(nseg):
        s = i*hop; e = s + nperseg
        seg = x[s:e]
        if len(seg) < nperseg: break
        seg = seg - np.mean(seg)
        X = np.fft.rfft(w*seg, n=nperseg)
        Xs.append(X)
    Xs = np.asarray(Xs)
    freqs = np.fft.rfftfreq(nperseg, d=1.0/fs)
    return freqs, Xs

# --- main orchestrator ---
def analyze_directionality_harmonics(
    RECORDS,
    eeg_channel: str,
    sr_channel: str,
    windows: dict,                 # {'baseline':[(t0,t1),...], 'ignition':[(..)], ...}
    time_col='Timestamp',
    fundamental=7.83,
    harmonics=(14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_bw=0.6,
    mvar_order=6,
    win_sec=10.0,
    step_sec=2.0,
    out_dir='exports_directionality', show=True
):
    """
    (1) TV-Granger/PDC per harmonic, (2) MIMO ARX over harmonic envelopes, (3) bispectral directionality.
    Saves figures and returns a summary dict of DataFrames.
    """
    ensure_dir(out_dir)
    ensure_timestamp_column(RECORDS, time_col=time_col)
    fs = infer_fs(RECORDS, time_col)

    # get channels robustly
    if eeg_channel in RECORDS.columns:
        x_eeg = pd.to_numeric(RECORDS[eeg_channel], errors='coerce').fillna(0.0).values.astype(float)
    elif ('EEG.'+eeg_channel) in RECORDS.columns:
        x_eeg = pd.to_numeric(RECORDS['EEG.'+eeg_channel], errors='coerce').fillna(0.0).values.astype(float)
    else:
        raise ValueError(f"{eeg_channel} not found.")

    if sr_channel in RECORDS.columns:
        x_sr = pd.to_numeric(RECORDS[sr_channel], errors='coerce').fillna(0.0).values.astype(float)
    elif ('EEG.'+sr_channel) in RECORDS.columns:
        x_sr = pd.to_numeric(RECORDS['EEG.'+sr_channel], errors='coerce').fillna(0.0).values.astype(float)
    else:
        raise ValueError(f"{sr_channel} not found.")

    # windows→samples
    W = {k: windows_to_samples(v, fs, len(RECORDS)) for k,v in (windows or {}).items()}

    # ----- (1) TV-Granger / PDC per harmonic -----
    rows=[]
    for fm in harmonics:
        if fm > min(60.0, 0.999*0.5*fs):  # clamp to ≤60 Hz
            continue
        xe_nb, xs_nb = narrowband_pair(x_eeg, x_sr, fs, fm, half_bw)
        win = int(round(win_sec*fs)); step=int(round(step_sec*fs))
        centers = np.arange(win//2, len(xe_nb)-win//2, step, dtype=int)
        gc_sr2eeg=[]; gc_eeg2sr=[]; pdc_sr2eeg=[]; pdc_eeg2sr=[]
        for c in centers:
            sl = slice(c-win//2, c+win//2)
            xw = xe_nb[sl]; yw = xs_nb[sl]
            if np.std(xw)<1e-12 or np.std(yw)<1e-12:
                continue
            try:
                Fy2x, Fx2y, A_full, Sigma = granger_2d_refit(xw, yw, p=mvar_order)
                gc_sr2eeg.append(float(Fy2x)); gc_eeg2sr.append(float(Fx2y))
                P = pdc_from_mvar(A_full, fs, fm)
                pdc_sr2eeg.append(float(P[0,1])); pdc_eeg2sr.append(float(P[1,0]))
            except Exception:
                continue
        for wname, segs in W.items():
            # (simplify) average across all valid centers
            rows.append({'metric':'GC_SR→EEG','window':wname,'f_hz':fm,'value':float(np.nanmean(gc_sr2eeg)) if gc_sr2eeg else np.nan})
            rows.append({'metric':'GC_EEG→SR','window':wname,'f_hz':fm,'value':float(np.nanmean(gc_eeg2sr)) if gc_eeg2sr else np.nan})
            rows.append({'metric':'PDC_SR→EEG','window':wname,'f_hz':fm,'value':float(np.nanmean(pdc_sr2eeg)) if pdc_sr2eeg else np.nan})
            rows.append({'metric':'PDC_EEG→SR','window':wname,'f_hz':fm,'value':float(np.nanmean(pdc_eeg2sr)) if pdc_eeg2sr else np.nan})

        # bar plots per fm
        def _bar_for(metric_prefix):
            labs=list(W.keys())
            y=[float(np.nanmean([r['value'] for r in rows if r['metric']==metric_prefix and r['f_hz']==fm and r['window']==w])) for w in labs]
            fig, ax = plt.subplots(figsize=(6,3.0))
            ax.bar(np.arange(len(y)), y, width=0.6)
            ax.set_xticks(np.arange(len(y))); ax.set_xticklabels(labs)
            ax.set_title(f'{metric_prefix} @ {fm:.2f} Hz'); ax.grid(True, axis='y', alpha=0.25, linestyle=':')
            plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"{metric_prefix.replace('→','to').replace(':','')}_{fm:.2f}Hz.png"), dpi=160)
            if show: plt.show(); plt.close()
        _bar_for('GC_SR→EEG'); _bar_for('GC_EEG→SR'); _bar_for('PDC_SR→EEG'); _bar_for('PDC_EEG→SR')

    dir_df = pd.DataFrame(rows)
    dir_df.to_csv(os.path.join(out_dir, 'directionality_summary.csv'), index=False)

    # ----- (2) MIMO ARX over harmonic envelopes -----
    freqs = [fundamental] + [f for f in harmonics if f<=min(60.0,0.999*0.5*fs)]
    K = len(freqs)
    A_eeg=[]; A_sr=[]
    for f0 in freqs:
        xe, xs = narrowband_pair(x_eeg, x_sr, fs, f0, half_bw)
        A_eeg.append(np.abs(signal.hilbert(xe)))
        A_sr.append(np.abs(signal.hilbert(xs)))
    A_eeg = np.vstack(A_eeg); A_sr = np.vstack(A_sr)  # K x N
    L = 1
    out_rows=[]
    for wname, segs in W.items():
        idxs=[]
        for (i0,i1) in segs: idxs.extend(list(range(i0+L, i1)))
        idxs = np.array(idxs, int)
        if idxs.size < 50: continue
        # SR→EEG
        Y = A_eeg[:, idxs]; X = A_sr[:, idxs-L]
        XX = X @ X.T + 1e-9*np.eye(K); G = (Y @ X.T) @ np.linalg.inv(XX)
        # surrogates for threshold
        rng = np.random.default_rng(31); null_abs=[]
        for _ in range(200):
            Xs = np.zeros_like(X)
            for k in range(K):
                s = int(rng.integers(1, X.shape[1]-1))
                Xs[k] = np.r_[X[k,-s:], X[k,:-s]]
            Gs = (Y @ Xs.T) @ np.linalg.inv(Xs @ Xs.T + 1e-9*np.eye(K))
            null_abs.append(np.abs(Gs))
        thr = np.nanpercentile(np.stack(null_abs,0), 95, axis=0); sig = (np.abs(G) > thr)
        for i in range(K):
            for j in range(K):
                out_rows.append({'window':wname,'direction':'SR→EEG','target_idx':i,'source_idx':j,
                                 'f_target':freqs[i],'f_source':freqs[j],
                                 'G':float(G[i,j]),'absG':float(abs(G[i,j])),'sig':bool(sig[i,j])})
        # EEG→SR
        Y = A_sr[:, idxs]; X = A_eeg[:, idxs-L]
        XX = X @ X.T + 1e-9*np.eye(K); G = (Y @ X.T) @ np.linalg.inv(XX)
        null_abs=[]
        for _ in range(200):
            Xs = np.zeros_like(X)
            for k in range(K):
                s = int(rng.integers(1, X.shape[1]-1))
                Xs[k] = np.r_[X[k,-s:], X[k,:-s]]
            Gs = (Y @ Xs.T) @ np.linalg.inv(Xs @ Xs.T + 1e-9*np.eye(K))
            null_abs.append(np.abs(Gs))
        thr = np.nanpercentile(np.stack(null_abs,0), 95, axis=0); sig = (np.abs(G) > thr)
        for i in range(K):
            for j in range(K):
                out_rows.append({'window':wname,'direction':'EEG→SR','target_idx':i,'source_idx':j,
                                 'f_target':freqs[i],'f_source':freqs[j],
                                 'G':float(G[i,j]),'absG':float(abs(G[i,j])),'sig':bool(sig[i,j])})
    arx_df = pd.DataFrame(out_rows); arx_df.to_csv(os.path.join(out_dir, 'arx_couplings.csv'), index=False)

    # ----- (3) Bispectral directionality -----
    nper = int(max(4*fs, 4096)); step = nper//2
    freqs_fft, Se = _fft_segments(x_eeg, fs, nper, step); _, Ss = _fft_segments(x_sr, fs, nper, step)
    def bin_idx(f): return int(np.argmin(np.abs(freqs_fft - f)))
    rows_bi=[]
    for f0 in [fundamental] + list(harmonics):
        if f0*2 > min(60.0, 0.999*0.5*fs): continue
        i = bin_idx(f0); k = bin_idx(2*f0)
        num_sse = np.mean(Ss[:, i] * Ss[:, i] * np.conj(Se[:, k]))
        num_ess = np.mean(Se[:, i] * Se[:, i] * np.conj(Ss[:, k]))
        B_sse = np.abs(num_sse); B_ess = np.abs(num_ess)
        rng = np.random.default_rng(21); null_diff=[]
        for _ in range(200):
            sh = int(rng.integers(1, len(x_sr)-1))
            xsr = np.r_[x_sr[-sh:], x_sr[:-sh]]
            _, Ssh = _fft_segments(xsr, fs, nper, step)
            num_sse_n = np.mean(Ssh[:, i] * Ssh[:, i] * np.conj(Se[:, k]))
            num_ess_n = np.mean(Se[:, i] * Se[:, i] * np.conj(Ssh[:, k]))
            null_diff.append(np.abs(num_sse_n) - np.abs(num_ess_n))
        null_diff = np.asarray(null_diff, float)
        diff = float(B_sse - B_ess)
        p = float((np.sum(null_diff >= diff) + 1) / (len(null_diff)+1))
        rows_bi.append({'metric':'BISPEC_DIR','f_hz':f0,'B_sse':float(B_sse),'B_ess':float(B_ess),'diff':diff,'p_value':p})
    bi_df = pd.DataFrame(rows_bi); bi_df.to_csv(os.path.join(out_dir, 'bispec_directionality.csv'), index=False)

    # done
    return {'gc_pdc': dir_df, 'arx': arx_df, 'bispec': bi_df, 'out_dir': out_dir}


In [ ]:
windows = {
  'baseline': [(60, 120)],
  'ignition': [(290, 310), (580, 600)]
#   'rebound':  [(325, 580)]
}

out = analyze_directionality_harmonics(
    RECORDS,
    eeg_channel='EEG.VIRT',          # or your virtual posterior 'EEG.VIRT'
    sr_channel='EEG.F4',           # or a magnetometer channel
    windows=windows,
    fundamental=7.83,
    harmonics=(1.305,1.57,1.96,2.61,3.915,7.83,14.3,20.8,27.3,33.8,40.3,46.8,53.3,59.8),
    half_bw=0.6,
    mvar_order=6,                  # try 6–10 for 250–512 Hz fs
    win_sec=10.0, step_sec=2.0,
    out_dir='exports_directionality', show=True
)
